# FranchiseOps AI · (FastAPI + GPU)

**Run order:** Runtime → Change runtime type → **T4 GPU** (or better), then *Run all*. The last cell prints the link to open the app. First start downloads NLLB-200 (translation) and Qwen2.5 (chat) onto the GPU in the background; you can already sign in while they load.

Optional Colab secrets (key icon on the left): `EMAIL_ID` + `EMAIL_PASSWORD` (Gmail app password) turn on e-mail OTP, e-mailed reports and the daily digest; `SLACK_WEBHOOK_URL`, `TEAMS_WEBHOOK_URL`, `TWILIO_*` turn on chat and SMS alerts; `HF_TOKEN` speeds up model downloads.

Demo logins are shown on the sign-in screen (admin@infosys.com / Admin@123 is the admin).

The app is served through Colab's own proxy domain (no ngrok/cloudflared needed), which avoids the browser's "dangerous site" warning.

All dates and times (database, agents, Copilot, reports) are **India Standard Time (IST)**. The Copilot understands questions like *sales today*, *shipments arriving this week* and *what time is it*.

Optional secret `NGROK_AUTHTOKEN` adds an ngrok link as a fallback to the Colab proxy link.


In [ ]:
!nvidia-smi -L || echo 'NO GPU: choose Runtime > Change runtime type > T4 GPU'

In [ ]:
!pip -q install fastapi 'uvicorn[standard]' python-multipart pyjwt pandas numpy scikit-learn vaderSentiment pdfplumber reportlab langgraph langdetect requests httpx transformers accelerate bitsandbytes sentence-transformers sentencepiece hf_transfer matplotlib huggingface_hub
!pip -q install prophet 2>/dev/null || echo 'prophet not installed (trend forecast will be used)'
!mkdir -p /content/app/static/vendor

In [ ]:
%%writefile /content/app/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


In [ ]:
%%writefile /content/app/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    _m = re.search(r"(?<![-\d])\b(\d{1,2})\b(?!\d)", q)
    if _m and 1 <= int(_m.group(1)) <= 50: n = int(_m.group(1))
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    _pl_city = _place(q)
    if _pl_city and _pl_city[0] == "city":
        _ids = set(Q("SELECT outlet_id FROM outlets WHERE lower(location)=?", (_pl_city[1].lower(),)).outlet_id)
        s = s[s.index.isin(_ids)]
        if not len(s): return {"answer": f"No outlets found in {_pl_city[1]}.", "df": None, "sql": ""}
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "What quotes need approval?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). Approve the matching *reorder* tickets in **Approvals** to raise the orders."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None):
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql)
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


In [ ]:
%%writefile /content/app/authx.py
"""Authentication for the web app: PBKDF2 password hashing, progressive lockout (5 min, 15 min, permanent), security-question reset, HttpOnly JWT cookie sessions. Independent of any UI framework."""
import re, os, hmac, hashlib, secrets, sqlite3, datetime, threading
import jwt
from config import DB_PATH, JWT_SECRET_KEY, ADMIN_EMAIL_ID, ADMIN_PASSWORD, KIND, DEMO_MODE
QUESTIONS = ["What was the name of your first pet?", "What is your mother's maiden name?", "What was the name of your first school?", "What city were you born in?", "What is your favorite book/movie?"]
LOCK_LADDER = {3: 300, 4: 900}; PERMA = 5
ROLES = {"Franchise": ["Admin", "Franchise Owner", "Store Manager", "Staff", "Customer"], "Freight": ["Admin", "Operations Manager", "Freight Broker", "Auditor", "Customer"]}[KIND]
DEMO = {"Franchise": [("owner", "owner@infosys.com", "Franchise Owner"), ("manager", "manager@infosys.com", "Store Manager"), ("staff", "staff@infosys.com", "Staff"), ("customer", "customer@infosys.com", "Customer")],
        "Freight": [("ops", "ops@infosys.com", "Operations Manager"), ("broker", "broker@infosys.com", "Freight Broker"), ("auditor", "auditor@infosys.com", "Auditor"), ("customer", "customer@infosys.com", "Customer")]}[KIND]
_L = threading.RLock()
def _c():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.row_factory = sqlite3.Row; return c
def hash_pw(pw, salt=None):
    salt = salt or secrets.token_hex(8); return f"pbkdf2${salt}${hashlib.pbkdf2_hmac('sha256', pw.encode(), salt.encode(), 120_000).hex()}"
def check_pw(pw, h):
    try: _, salt, _ = h.split("$"); return hmac.compare_digest(hash_pw(pw, salt), h)
    except Exception: return False
def init():
    with _L, _c() as c:
        c.execute("CREATE TABLE IF NOT EXISTS app_users(email TEXT PRIMARY KEY, username TEXT UNIQUE, pw TEXT, role TEXT, sq TEXT, sa TEXT, failed INT DEFAULT 0, lock_until TEXT, status TEXT DEFAULT 'active', created TEXT, last_login TEXT)")
        def add(u, e, r, pw):
            if not c.execute("SELECT 1 FROM app_users WHERE email=?", (e,)).fetchone(): c.execute("INSERT INTO app_users(email,username,pw,role,sq,sa,created) VALUES(?,?,?,?,?,?,?)", (e, u, hash_pw(pw), r, QUESTIONS[0], hashlib.sha256(b"infosys").hexdigest(), now()))
        add("admin", ADMIN_EMAIL_ID, "Admin", ADMIN_PASSWORD)
        for u, e, r in (DEMO if DEMO_MODE else []): add(u, e, r, "Demo@1234")
def now(): return datetime.datetime.now().isoformat(timespec="seconds")
def strength(pw):
    if not pw or len(pw) < 8 or not re.search(r"[A-Za-z]", pw) or not re.search(r"\d", pw): return False, "Password too weak: at least 8 characters with a letter and a digit."
    return True, "ok"
def register(username, email, pw, role, sq, sa):
    if not (username and email and pw and sa.strip()): return False, "Fill in every field."
    ok, m = strength(pw)
    if not ok: return False, m
    if role not in ("Customer", "Staff"): role = "Customer"          # self-service sign-up never grants an elevated role; Admin promotes users
    try:
        with _L, _c() as c: c.execute("INSERT INTO app_users(email,username,pw,role,sq,sa,created) VALUES(?,?,?,?,?,?,?)", (email.lower().strip(), username.strip(), hash_pw(pw), role, sq, hashlib.sha256(sa.strip().lower().encode()).hexdigest(), now()))
        return True, "Account created. Sign in."
    except sqlite3.IntegrityError: return False, "An account with this email or username already exists."
def login(login_id, pw):
    """-> (ok, message, user_dict|None)"""
    lid = (login_id or "").strip().lower()
    with _L, _c() as c:
        r = c.execute("SELECT * FROM app_users WHERE lower(email)=? OR lower(username)=?", (lid, lid)).fetchone()
        if not r: return False, "No such user.", None
        if r["status"] == "locked": return False, "Account locked after 5 failed attempts. An Admin must unlock it.", None
        if r["lock_until"] and now() < r["lock_until"]: return False, f"Temporarily locked until {r['lock_until'][11:16]} UTC.", None
        if check_pw(pw, r["pw"]):
            c.execute("UPDATE app_users SET failed=0, lock_until=NULL, last_login=? WHERE email=?", (now(), r["email"])); return True, "ok", {"email": r["email"], "username": r["username"], "role": r["role"]}
        n = (r["failed"] or 0) + 1; lock = None; st = r["status"]
        if n >= PERMA: st = "locked"
        elif n in LOCK_LADDER: lock = (datetime.datetime.now() + datetime.timedelta(seconds=LOCK_LADDER[n])).isoformat(timespec="seconds")
        c.execute("UPDATE app_users SET failed=?, lock_until=?, status=? WHERE email=?", (n, lock, st, r["email"]))
        return False, f"Wrong password. {max(0, PERMA - n)} attempt(s) left before lockout.", None
def question_for(email):
    with _c() as c: r = c.execute("SELECT sq FROM app_users WHERE lower(email)=?", (email.lower().strip(),)).fetchone(); return r["sq"] if r else None
def reset(email, answer, new_pw):
    ok, m = strength(new_pw)
    if not ok: return False, m
    with _L, _c() as c:
        r = c.execute("SELECT sa FROM app_users WHERE lower(email)=?", (email.lower().strip(),)).fetchone()
        if not r or r["sa"] != hashlib.sha256(answer.strip().lower().encode()).hexdigest(): return False, "Security answer does not match."
        c.execute("UPDATE app_users SET pw=?, failed=0, lock_until=NULL, status='active' WHERE lower(email)=?", (hash_pw(new_pw), email.lower().strip())); return True, "Password reset. Sign in."
def users():
    with _c() as c: return [dict(r) for r in c.execute("SELECT email, username, role, status, failed, last_login, created FROM app_users ORDER BY created")]
def set_role(email, role):
    with _L, _c() as c: c.execute("UPDATE app_users SET role=? WHERE email=?", (role, email))
def unlock(email):
    with _L, _c() as c: c.execute("UPDATE app_users SET status='active', failed=0, lock_until=NULL WHERE email=?", (email,))
import time as _time
def revoke(email):
    with _L, _c() as c:
        c.execute("CREATE TABLE IF NOT EXISTS revoked_before(email TEXT PRIMARY KEY, ts REAL)"); c.execute("INSERT OR REPLACE INTO revoked_before VALUES(?,?)", (email, _time.time()))
def token(u): return jwt.encode({**u, "iatf": _time.time(), "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=10)}, JWT_SECRET_KEY, algorithm="HS256")
def decode(t):
    try:
        p = jwt.decode(t, JWT_SECRET_KEY, algorithms=["HS256"])
        with _c() as c:
            c.execute("CREATE TABLE IF NOT EXISTS revoked_before(email TEXT PRIMARY KEY, ts REAL)")
            r = c.execute("SELECT role, status FROM app_users WHERE email=?", (p["email"],)).fetchone(); rv = c.execute("SELECT ts FROM revoked_before WHERE email=?", (p["email"],)).fetchone()
        if rv and float(p.get("iatf") or 0) <= float(rv[0]): return None
        return {"email": p["email"], "username": p["username"], "role": r["role"]} if r and r["status"] != "locked" else None      # role is re-read so an Admin's change applies at once
    except Exception: return None

# ---------------------------------------------------------------- e-mail OTP, profile, preferences, invitations
import mailer
def _otp_init():
    with _L, _c() as c:
        c.execute("CREATE TABLE IF NOT EXISTS otp(email TEXT, purpose TEXT, code TEXT, expires TEXT, tries INT DEFAULT 0, PRIMARY KEY(email, purpose))")
        c.execute("CREATE TABLE IF NOT EXISTS user_prefs(email TEXT PRIMARY KEY, lang TEXT DEFAULT 'English', email_alerts INT DEFAULT 0, daily_digest INT DEFAULT 0, last_digest TEXT)")
_otp_init()
def send_otp(email, purpose):
    """Mails a 6-digit code (valid 10 min). Returns (ok, message). Without SMTP settings it says so and the caller falls back to the security question."""
    email = (email or "").lower().strip()
    if "@" not in email: return False, "Enter a valid e-mail address."
    if not mailer.configured(): return False, "E-mail is not configured on this server, so codes cannot be sent. Use the security question instead."
    if purpose == "reset":
        with _c() as c:
            if not c.execute("SELECT 1 FROM app_users WHERE lower(email)=?", (email,)).fetchone(): return True, "If the account exists, a code was sent."
    code = f"{secrets.randbelow(10**6):06d}"; exp = (datetime.datetime.now() + datetime.timedelta(minutes=10)).isoformat(timespec="seconds")
    with _L, _c() as c: c.execute("INSERT OR REPLACE INTO otp(email,purpose,code,expires,tries) VALUES(?,?,?,?,0)", (email, purpose, hashlib.sha256(code.encode()).hexdigest(), exp))
    ok, m = mailer.send(email, f"Your verification code: {code}", f"Your {KIND} platform verification code is {code}. It expires in 10 minutes. If you did not ask for it, ignore this message.")
    return ok, ("Code sent to " + email + ". It expires in 10 minutes.") if ok else m
def check_otp(email, purpose, code):
    email = (email or "").lower().strip()
    with _L, _c() as c:
        r = c.execute("SELECT * FROM otp WHERE email=? AND purpose=?", (email, purpose)).fetchone()
        if not r or now() > r["expires"] or (r["tries"] or 0) >= 5: return False
        c.execute("UPDATE otp SET tries=tries+1 WHERE email=? AND purpose=?", (email, purpose))
        if hmac.compare_digest(hashlib.sha256(str(code).strip().encode()).hexdigest(), r["code"]): c.execute("DELETE FROM otp WHERE email=? AND purpose=?", (email, purpose)); return True
    return False
def reset_with_otp(email, code, new_pw):
    ok, m = strength(new_pw)
    if not ok: return False, m
    if not check_otp(email, "reset", code): return False, "The code is wrong or has expired."
    with _L, _c() as c: c.execute("UPDATE app_users SET pw=?, failed=0, lock_until=NULL, status='active' WHERE lower(email)=?", (hash_pw(new_pw), email.lower().strip()))
    return True, "Password reset. Sign in."
def register_verified(username, email, pw, role, sq, sa, code):
    if mailer.configured() and not check_otp(email, "register", code): return False, "Enter the 6-digit code that was e-mailed to you."
    return register(username, email, pw, role, sq, sa or "-")
def profile(email):
    with _c() as c:
        r = c.execute("SELECT email, username, role, created, last_login FROM app_users WHERE lower(email)=?", (email.lower(),)).fetchone(); p = c.execute("SELECT * FROM user_prefs WHERE email=?", (email.lower(),)).fetchone()
    d = dict(r) if r else {}; d["prefs"] = dict(p) if p else {"lang": "English", "email_alerts": 0, "daily_digest": 0}; return d
def set_prefs(email, **kw):
    with _L, _c() as c:
        c.execute("INSERT OR IGNORE INTO user_prefs(email) VALUES(?)", (email.lower(),))
        for k in ("lang", "email_alerts", "daily_digest", "last_digest"):
            if k in kw: c.execute(f"UPDATE user_prefs SET {k}=? WHERE email=?", (kw[k], email.lower()))
def update_name(email, username):
    username = (username or "").strip()
    if len(username) < 3: return False, "User name needs at least 3 characters."
    try:
        with _L, _c() as c: c.execute("UPDATE app_users SET username=? WHERE lower(email)=?", (username, email.lower()))
        return True, "Saved."
    except sqlite3.IntegrityError: return False, "That user name is taken."
def change_password(email, old, new):
    ok, m = strength(new)
    if not ok: return False, m
    with _L, _c() as c:
        r = c.execute("SELECT pw FROM app_users WHERE lower(email)=?", (email.lower(),)).fetchone()
        if not r or not check_pw(old, r["pw"]): return False, "Current password is wrong."
        c.execute("UPDATE app_users SET pw=? WHERE lower(email)=?", (hash_pw(new), email.lower())); return True, "Password changed."
def invite(email, username, role):
    """Admin creates an account and e-mails a one-time password (shown to the admin when e-mail is not configured)."""
    if role not in ROLES: return False, "Unknown role."
    email = email.lower().strip(); tmp = "Tmp-" + secrets.token_urlsafe(6)
    try:
        with _L, _c() as c: c.execute("INSERT INTO app_users(email,username,pw,role,sq,sa,created) VALUES(?,?,?,?,?,?,?)", (email, username.strip(), hash_pw(tmp), role, QUESTIONS[0], hashlib.sha256(secrets.token_hex(8).encode()).hexdigest(), now()))
    except sqlite3.IntegrityError: return False, "An account with this email or user name already exists."
    ok, m = mailer.send(email, "You have been invited", f"An account was created for you as {role}.\nSign in with {email} and the temporary password {tmp}, then change it under My Profile.")
    return True, (f"Invitation e-mailed to {email}." if ok else f"Account created. E-mail is not configured, so give them this temporary password yourself: {tmp}")
def deactivate(email, on=True):
    with _L, _c() as c: c.execute("UPDATE app_users SET status=? WHERE lower(email)=?", ("locked" if on else "active", email.lower()))


In [ ]:
%%writefile /content/app/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#12a594", "#3e63dd", "#f5a524", "#e5484d", "#8e4ec6", "#30a46c", "#d6409f", "#8b8d98"
PRI = "#5b5bd6"
PAL = [PRI, TEAL, AMBER, VIOLET, RED, GREEN, PINK, BLUE, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


In [ ]:
%%writefile /content/app/config.py
"""Shared configuration. APP_KIND (Franchise | Freight) is set by the launcher. No Google Drive prompts: data lives in DATA_DIR (set PERSIST_DIR to keep it between sessions)."""
import os as _os, time as _time
_os.environ["TZ"] = "Asia/Kolkata"
try: _time.tzset()
except Exception: pass          # all times in this app are India Standard Time (IST)
import os
KIND = os.environ.get("APP_KIND", "Franchise")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_default = "/content/app_data" if os.path.isdir("/content") else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FRANCHISEOPS_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(DATA_DIR, "franchise_database.db" if KIND == "Franchise" else "freight_database.db")
def _secret(key, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(key)
        if v: return str(v).strip()
    except Exception: pass
    return str(os.environ.get(key, default) or "").strip()
def _pull_drive_secrets():
    """After a Colab reset: bring the saved JWT secret / admin password back from the Drive backup first, so existing sessions and logins keep working."""
    d = os.environ.get("DRIVE_DB_DIR", "").strip()
    if not d or os.path.exists(DB_PATH): return
    import shutil as _sh
    try:
        _b = os.path.join(d, os.path.basename(DB_PATH))
        if os.path.exists(_b): _sh.copy2(_b, DB_PATH); print("persist: database restored from", _b)    # runs at import, before anything opens the database
    except Exception as _e: print("persist restore failed:", repr(_e))
    for x in (".jwt_secret", ".admin_initial"):
        try:
            if os.path.exists(os.path.join(d, x)) and not os.path.exists(os.path.join(DATA_DIR, x)): _sh.copy2(os.path.join(d, x), os.path.join(DATA_DIR, x))
        except Exception: pass
_pull_drive_secrets()
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGINGFACE_TOKEN")
def _persisted_secret():
    """No JWT_SECRET_KEY given: generate a random one once and keep it beside the database, so sessions survive restarts and are never signed with a public default."""
    import secrets as _s
    p = os.path.join(DATA_DIR, ".jwt_secret")
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = _s.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return _s.token_urlsafe(48)
JWT_SECRET_KEY = _secret("JWT_SECRET_KEY") or _persisted_secret()
ADMIN_EMAIL_ID = _secret("ADMIN_EMAIL_ID", "admin@infosys.com"); DEMO_MODE = os.environ.get("DEMO_MODE", "1") != "0"     # 1 = seeded demo accounts shown on the login page (Colab demo). 0 = production: only the admin exists, with ADMIN_PASSWORD (or a random one printed once in the server log)
def _admin_pw():
    v = _secret("ADMIN_PASSWORD")
    if v: return v
    if DEMO_MODE: return "Admin@123"
    import secrets as _s; p = os.path.join(DATA_DIR, ".admin_initial");
    if os.path.exists(p): return open(p).read().strip()
    v = _s.token_urlsafe(9) + "1aA"; open(p, "w").write(v); print("\n*** DEMO_MODE=0: initial admin password:", v, "(stored in", p, ") ***\n"); return v
ADMIN_PASSWORD = _admin_pw()
EMAIL_ID = _secret("EMAIL_ID"); EMAIL_PASSWORD = _secret("EMAIL_PASSWORD").replace(" ", "")
MODEL_NAME = os.environ.get("LLM_MODEL", "")
NLLB_MODEL = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-600M")


In [ ]:
%%writefile /content/app/data_feeder.py
"""Data Feeder: upload / folder inbox / URL feeds -> validated, auto-mapped, upserted into the live tables, then the agents re-run."""
import os, io, json, time, shutil, difflib, subprocess, datetime
import pandas as pd
from live_engine import conn, _DB_LOCK, now, emit, get_state, set_state
from config import DB_PATH
DATA_DIR = os.path.dirname(DB_PATH); INBOX = os.path.join(DATA_DIR, "inbox"); DONE = os.path.join(DATA_DIR, "inbox_done")
ALLOWED = {"Franchise": ["outlets", "staff", "inventory", "marketing", "audits", "feedback", "sales", "customers_fr", "hr_training"],
           "Freight": ["vessels", "ports", "carriers", "shipments", "customers", "freight_quotes", "freight_rates", "quote_requests", "customs_rules"]}
ALIAS = {"hr_training": {"attrition": ["attrition", "left", "churn"], "job_satisfaction": ["jobsatisfaction", "satisfaction"], "tenure_years": ["yearsatcompany", "tenure"], "work_life_balance": ["worklifebalance"], "monthly_income": ["monthlyincome", "income", "salary"], "age": ["age"], "overtime_hrs": ["overtime", "overtimehrs"]},
         "freight_rates": {"distance_nm": ["distance", "distancenm", "distance_nm"], "cargo_weight": ["weight", "weighttons", "cargoweight"], "cargo_type": ["cargotype", "cargo"], "congestion_level": ["congestion", "congestionlevel"], "freight_cost_usd": ["cost", "freightcost", "price", "freight_cost_usd"]}}
VIRTUAL = {"hr_training": ["attrition", "job_satisfaction", "tenure_years", "work_life_balance", "monthly_income", "age", "overtime_hrs"], "freight_rates": ["distance_nm", "cargo_weight", "cargo_type", "congestion_level", "freight_cost_usd"]}
def init():
    os.makedirs(INBOX, exist_ok=True); os.makedirs(DONE, exist_ok=True)
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS feed_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, source TEXT, target TEXT, rows_in INT, rows_ok INT, rows_bad INT, note TEXT)")
        c.execute("CREATE TABLE IF NOT EXISTS feed_urls(id INTEGER PRIMARY KEY AUTOINCREMENT, url TEXT, target TEXT, every_s INT, last_run REAL DEFAULT 0)")
        for col, ty in (("auth_env", "TEXT"), ("json_path", "TEXT"), ("last_status", "TEXT")):
            try: c.execute(f"ALTER TABLE feed_urls ADD COLUMN {col} {ty}")
            except Exception: pass
        c.execute("CREATE TABLE IF NOT EXISTS feed_rejects(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, source TEXT, target TEXT, row_no INT, field TEXT, reason TEXT, value TEXT)")
        c.execute("CREATE TABLE IF NOT EXISTS feed_idem(k TEXT PRIMARY KEY, ts REAL)")
        c.execute("CREATE TABLE IF NOT EXISTS ais_map(mmsi TEXT PRIMARY KEY, shipment_id TEXT)")
        c.execute("CREATE TABLE IF NOT EXISTS hr_training(attrition INT, job_satisfaction REAL, tenure_years REAL, work_life_balance REAL, monthly_income REAL, age REAL, overtime_hrs REAL)")
        c.execute("CREATE TABLE IF NOT EXISTS freight_rates(distance_nm REAL, cargo_weight REAL, cargo_type INT, congestion_level INT, freight_cost_usd REAL)")
def seen_key(k):
    """Idempotency: True if this key was already accepted (replayed webhook), else records it."""
    init()
    with _DB_LOCK, conn() as c:
        c.execute("DELETE FROM feed_idem WHERE ts<?", (time.time() - 86400,))
        try: c.execute("INSERT INTO feed_idem VALUES(?,?)", (str(k)[:200], time.time())); return False
        except Exception: return True
def mark_real(ids):
    """Entities that receive a real feed (POS or AIS) are no longer driven by the simulator for 10 minutes after the last real record."""
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS real_feed(entity TEXT PRIMARY KEY, ts REAL)")
        c.executemany("INSERT OR REPLACE INTO real_feed VALUES(?,?)", [(str(i), time.time()) for i in ids])
def real_entities(ttl=600):
    try:
        with _DB_LOCK, conn() as c: return {r[0] for r in c.execute("SELECT entity FROM real_feed WHERE ts>?", (time.time() - ttl,)).fetchall()}
    except Exception: return set()
def _ingest_positions(d, n0, mapping, source):
    """AIS-style vessel positions: needs shipment_id, lat, lon. Updates the tracked vessel in place and stops simulating it."""
    if "shipment_id" not in d.columns and "mmsi" in mapping:
        with _DB_LOCK, conn() as c: mp = dict(c.execute("SELECT mmsi, shipment_id FROM ais_map").fetchall())
        d["shipment_id"] = d["mmsi"].astype(str).map(mp)
    miss = [k for k in ("shipment_id", "lat", "lon") if k not in d.columns]
    if miss: raise ValueError(f"Required column(s) not mapped: {miss}")
    d = d.dropna(subset=["shipment_id", "lat", "lon"]).copy(); d["lat"] = pd.to_numeric(d.lat, errors="coerce"); d["lon"] = pd.to_numeric(d.lon, errors="coerce")
    d = d[d.lat.between(-90, 90) & d.lon.between(-180, 180)]; ok = 0
    with _DB_LOCK, conn() as c:
        known = {r[0] for r in c.execute("SELECT shipment_id FROM vessels").fetchall()}
        for r in d.itertuples(index=False):
            sid = str(r.shipment_id)
            if sid in known:
                sp = getattr(r, "speed_kn", None) if "speed_kn" in d.columns else None
                c.execute("UPDATE vessels SET lat=?, lon=?, updated=?" + (", speed_kn=?" if sp is not None and sp == sp else "") + " WHERE shipment_id=?", (float(r.lat), float(r.lon), now()) + ((float(sp),) if sp is not None and sp == sp else ()) + (sid,)); ok += 1
        c.execute("INSERT INTO feed_log(ts,source,target,rows_in,rows_ok,rows_bad,note) VALUES(?,?,?,?,?,?,?)", (now(), source, "vessels", n0, ok, n0 - ok, "AIS positions; unknown shipment ids rejected"))
    mark_real(d.shipment_id.astype(str).tolist())
    try: from live_engine import audit; audit(source[:40], "ingest", "vessels", f"{ok} positions")
    except Exception: pass
    return ok, n0 - ok
def schema(table):
    c = conn(); return [(r[1], r[2], r[5]) for r in c.execute(f"PRAGMA table_info({table})")]
def _norm(s): return "".join(ch for ch in str(s).lower() if ch.isalnum())
def auto_map(df, table):
    cols = [c for c, _, _ in schema(table)]; out = {}; used = set()
    for tc in cols:
        cands = [_norm(tc)] + [_norm(a) for a in ALIAS.get(table, {}).get(tc, [])]; best = None
        for sc in df.columns:
            if sc in used: continue
            if _norm(sc) in cands: best = sc; break
        if not best:
            m = difflib.get_close_matches(_norm(tc), [_norm(c) for c in df.columns if c not in used], n=1, cutoff=.82)
            best = next((c for c in df.columns if _norm(c) == m[0]), None) if m else None
        if best: out[tc] = best; used.add(best)
    return out
def read_any(name, data):
    n = name.lower()
    if n.endswith(".csv"): return pd.read_csv(io.BytesIO(data))
    if n.endswith((".xlsx", ".xls")): return pd.read_excel(io.BytesIO(data))
    if n.endswith(".json"): return pd.read_json(io.BytesIO(data))
    raise ValueError("Use CSV, XLSX or JSON")
def ingest(df, table, mapping=None, source="upload"):
    init(); mapping = mapping or auto_map(df, table); sch = schema(table)
    d = pd.DataFrame({tc: df[sc] for tc, sc in mapping.items() if sc in df.columns})
    if table == "hr_training" and "attrition" in d: d["attrition"] = d.attrition.astype(str).str.lower().isin(["1", "yes", "true", "y"]).astype(int)
    if table == "hr_training" and "overtime_hrs" in d and d.overtime_hrs.dtype == object: d["overtime_hrs"] = d.overtime_hrs.astype(str).str.lower().isin(["yes", "true", "1"]).astype(int) * 15
    if table == "freight_rates" and "cargo_type" in d and d.cargo_type.dtype == object: d["cargo_type"] = d.cargo_type.astype("category").cat.codes
    if table == "sales" and "source" not in d: d["source"] = "pos"; sch = schema(table)
    if table == "vessels": return _ingest_positions(d, n0 := len(d), mapping, source)
    for tc, ty, _ in sch:
        if tc in d and ty.upper() in ("REAL", "INTEGER", "INT", "FLOAT"): d[tc] = pd.to_numeric(d[tc], errors="coerce")
    rej = []
    if table not in VIRTUAL:
        import validate; d, rej = validate.validate(table, d, conn())
        if rej:
            with _DB_LOCK, conn() as c: c.executemany("INSERT INTO feed_rejects(ts,source,target,row_no,field,reason,value) VALUES(?,?,?,?,?,?,?)", [(now(), source[:60], table, r["row"], r["field"], r["reason"], r["value"]) for r in rej[:300]])
    if table == "feedback" and "comment" in d:
        import privacy; d["comment"] = d["comment"].map(privacy.redact)
    pk = [tc for tc, _, p in sch if p]; need = pk or [sch[0][0]] if table not in VIRTUAL else list(VIRTUAL[table])
    need = [n for n in need if not (n.endswith("_id") and n not in d and any(t.upper() == "INTEGER" and p for _, t, p in sch))]
    if table in VIRTUAL: need = [VIRTUAL[table][-1] if table == "freight_rates" else "attrition"]
    miss = [n for n in need if n not in d.columns]
    if miss: raise ValueError(f"Required column(s) not mapped: {miss}")
    if table in VIRTUAL: d = d.fillna(d.median(numeric_only=True))
    n0 = len(d) + len(rej); d = d.dropna(subset=need).drop_duplicates(subset=(need if pk else None) or None); bad = n0 - len(d)
    cols = list(d.columns); ph = ",".join("?" * len(cols))
    with _DB_LOCK, conn() as c:
        c.executemany(f"INSERT OR REPLACE INTO {table}({','.join(cols)}) VALUES({ph})", d.astype(object).where(d.notna(), None).values.tolist())
        c.execute("INSERT INTO feed_log(ts,source,target,rows_in,rows_ok,rows_bad,note) VALUES(?,?,?,?,?,?,?)", (now(), source, table, n0, len(d), bad, json.dumps(mapping)[:300]))
    if table == "sales" and "outlet_id" in d: mark_real(d.outlet_id.astype(str).unique().tolist())
    try: from live_engine import audit; audit(source[:40], "ingest", table, f"{len(d)} rows ok, {bad} rejected")
    except Exception: pass
    emit("feed", table, {"rows": len(d), "source": source}); return len(d), bad
def watch(dom_kind):
    """Called by the runtime: ingests files dropped into the inbox (name = <table>.csv or <table>__x.csv) and polls registered URLs."""
    init(); done = []
    for f in os.listdir(INBOX):
        tbl = f.split("__")[0].rsplit(".", 1)[0]
        if tbl in ALLOWED[dom_kind]:
            try:
                with open(os.path.join(INBOX, f), "rb") as fh: ingest(read_any(f, fh.read()), tbl, source="inbox:" + f)
                shutil.move(os.path.join(INBOX, f), os.path.join(DONE, f)); done.append(f)
            except Exception as e:
                with _DB_LOCK, conn() as c: c.execute("INSERT INTO feed_log(ts,source,target,rows_in,rows_ok,rows_bad,note) VALUES(?,?,?,0,0,0,?)", (now(), "inbox:" + f, tbl, "ERROR " + str(e)[:200]))
                shutil.move(os.path.join(INBOX, f), os.path.join(DONE, "failed_" + f))
    with _DB_LOCK, conn() as c: due = c.execute("SELECT id,url,target,every_s,auth_env,json_path FROM feed_urls WHERE ?-last_run>every_s", (time.time(),)).fetchall()
    for i, url, tbl, ev, auth_env, jp in due:
        try:
            import requests; hd = {"Authorization": "Bearer " + os.environ[auth_env]} if auth_env and os.environ.get(auth_env) else {}
            r = requests.get(url, timeout=15, headers=hd); r.raise_for_status()
            if jp or url.split("?")[0].lower().endswith(".json") or "json" in r.headers.get("content-type", ""):
                js = r.json()
                for k in [p for p in (jp or "").split(".") if p]: js = js[int(k)] if isinstance(js, list) else js[k]
                df = pd.DataFrame(js if isinstance(js, list) else [js])
            else: df = read_any(url.split("?")[0] if "." in url.split("/")[-1] else "x.csv", r.content)
            n_ok, n_bad = ingest(df, tbl, source="url:" + url[:60])
            with _DB_LOCK, conn() as c: c.execute("UPDATE feed_urls SET last_status=? WHERE id=?", (f"ok {n_ok} rows, {n_bad} rejected @ {now()}", i))
        except Exception as e:
            with _DB_LOCK, conn() as c: c.execute("INSERT INTO feed_log(ts,source,target,rows_in,rows_ok,rows_bad,note) VALUES(?,?,?,0,0,0,?)", (now(), "url:" + url[:60], tbl, "ERROR " + str(e)[:200])); c.execute("UPDATE feed_urls SET last_status=? WHERE id=?", ("ERROR " + str(e)[:120], i))
        with _DB_LOCK, conn() as c: c.execute("UPDATE feed_urls SET last_run=? WHERE id=?", (time.time(), i))
    return done


In [ ]:
%%writefile /content/app/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now','localtime')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


In [ ]:
%%writefile /content/app/gpu.py
"""GPU layer: Qwen2.5 (LLM), Facebook NLLB-200 (translation, the only translator) and a sentence embedder (RAG), all loaded on the GPU at startup in a background thread.
status() powers the GPU panel in the app so you can see what is on which device. Everything degrades loudly (never silently) when CUDA or a model is missing."""
import os, re, json, time, threading, sqlite3, hashlib
from config import DB_PATH, NLLB_MODEL, MODEL_NAME, HF_TOKEN
try: from lang_pack import EXTRA_LANGS, extra_detect
except Exception: EXTRA_LANGS, extra_detect = {}, (lambda t: None)

NLLB_LANGS = {"English": "eng_Latn", "Hindi (हिंदी)": "hin_Deva", "Tamil (தமிழ்)": "tam_Taml", "Telugu (తెలుగు)": "tel_Telu", "Kannada (ಕನ್ನಡ)": "kan_Knda", "Malayalam (മലയാളം)": "mal_Mlym",
    "Marathi (मराठी)": "mar_Deva", "Bengali (বাংলা)": "ben_Beng", "Gujarati (ગુજરાતી)": "guj_Gujr", "Punjabi (ਪੰਜਾਬੀ)": "pan_Guru", "Odia (ଓଡ଼ିଆ)": "ory_Orya", "Assamese (অসমীয়া)": "asm_Beng",
    "Urdu (اردو)": "urd_Arab", "Sanskrit (संस्कृतम्)": "san_Deva", "Nepali (नेपाली)": "npi_Deva", "Sindhi (سنڌي)": "snd_Arab", "Sinhala (සිංහල)": "sin_Sinh", "French (Français)": "fra_Latn",
    "German (Deutsch)": "deu_Latn", "Spanish (Español)": "spa_Latn", "Chinese (中文)": "zho_Hans", "Japanese (日本語)": "jpn_Jpan", "Arabic (العربية)": "arb_Arab"}
NLLB_LANGS.update(EXTRA_LANGS)

S = {"cuda": False, "gpu": None, "vram_gb": 0, "torch": None, "started": None,
     "models": {"llm": {"name": None, "state": "pending", "device": None, "sec": None, "err": None}, "nllb": {"name": NLLB_MODEL, "state": "pending", "device": None, "sec": None, "err": None},
                "embed": {"name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "state": "pending", "device": None, "sec": None, "err": None},
                "rerank": {"name": "cross-encoder/ms-marco-MiniLM-L-6-v2", "state": "pending", "device": None, "sec": None, "err": None}},
     "stats": {"tok_s": None, "translate_ms": None, "translations": 0, "llm_calls": 0}}
_M = {"llm": None, "tok": None, "nllb": None, "nllb_tok": None, "embed": None, "rerank": None}
_LOCK = threading.Lock(); _GEN = threading.Lock()

def _cache_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS tr_cache(k TEXT PRIMARY KEY, v TEXT)"); return c
_TC = {}

def _probe():
    try:
        import torch
        S["torch"] = torch.__version__; S["cuda"] = bool(torch.cuda.is_available())
        if S["cuda"]: p = torch.cuda.get_device_properties(0); S["gpu"] = p.name; S["vram_gb"] = round(p.total_memory / 2**30, 1)
    except Exception as e: S["torch"] = f"unavailable: {e}"

def vram_used():
    try:
        import torch
        if S["cuda"]: return round(torch.cuda.memory_allocated() / 2**30, 2), round(torch.cuda.memory_reserved() / 2**30, 2)
    except Exception: pass
    return 0, 0

def status():
    u, r = vram_used(); return {**{k: S[k] for k in ("cuda", "gpu", "vram_gb", "torch")}, "vram_used_gb": u, "vram_reserved_gb": r, "models": S["models"], "stats": S["stats"], "uptime_s": round(time.time() - (S["started"] or time.time()))}

def _load_nllb():
    m = S["models"]["nllb"]; t = time.time()
    try:
        import torch; from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        m["state"] = "loading"; tok = AutoTokenizer.from_pretrained(NLLB_MODEL, token=HF_TOKEN or None)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL, torch_dtype=torch.float16 if S["cuda"] else torch.float32, token=HF_TOKEN or None)
        dev = "cuda" if S["cuda"] else "cpu"; mdl = mdl.to(dev).eval(); _M["nllb"], _M["nllb_tok"] = mdl, tok; m.update(state="ready", device=dev, sec=round(time.time() - t, 1), err=None)
    except Exception as e: m.update(state="failed", err=str(e)[:300])

def _load_embed():
    m = S["models"]["embed"]; t = time.time()
    try:
        from sentence_transformers import SentenceTransformer
        m["state"] = "loading"; dev = "cuda" if S["cuda"] else "cpu"; _M["embed"] = SentenceTransformer(m["name"], device=dev); m.update(state="ready", device=dev, sec=round(time.time() - t, 1), err=None)
        try:
            import rag; rag.warm()
        except Exception: pass
    except Exception as e: m.update(state="failed", err=str(e)[:300])

def _load_rerank():
    m = S["models"]["rerank"]; t = time.time()
    try:
        from sentence_transformers import CrossEncoder
        m["state"] = "loading"; dev = "cuda" if S["cuda"] else "cpu"; _M["rerank"] = CrossEncoder(m["name"], device=dev, max_length=384); m.update(state="ready", device=dev, sec=round(time.time() - t, 1), err=None)
    except Exception as e: m.update(state="failed", err=str(e)[:300])
def rerank(q, passages):
    """Cross-encoder relevance scores (higher = better) or None when the reranker is not loaded."""
    import numpy as np
    if _M["rerank"] is None or not passages: return None
    return np.asarray(_M["rerank"].predict([(q, p) for p in passages], batch_size=32, show_progress_bar=False), dtype="float32")

def _download(name, m):
    """Download the weights first (with visible progress) so 'loading' never hides a 15 GB download."""
    from huggingface_hub import snapshot_download, HfApi, constants
    try: import hf_transfer; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    except Exception: pass
    pats = ["*.json", "*.safetensors", "*.txt", "*.model", "tokenizer*"]
    try: info = HfApi().model_info(name, files_metadata=True, token=HF_TOKEN or None); m["total_gb"] = round(sum((x.size or 0) for x in info.siblings if x.rfilename.endswith((".safetensors", ".json", ".txt", ".model"))) / 2**30, 2)
    except Exception: m["total_gb"] = None
    root = os.path.join(constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")); stop = threading.Event()
    def poll():
        while not stop.is_set():
            try: m["done_gb"] = round(sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(root) for f in fs) / 2**30, 2)
            except Exception: pass
            stop.wait(1.5)
    threading.Thread(target=poll, daemon=True).start(); m.update(state="downloading", done_gb=0)
    try: return snapshot_download(name, allow_patterns=pats, token=HF_TOKEN or None, max_workers=8)
    finally: stop.set()

def _load_llm():
    m = S["models"]["llm"]; t = time.time()
    try:
        import torch; from transformers import AutoTokenizer, AutoModelForCausalLM
        if not S["cuda"]: raise RuntimeError("No CUDA GPU in this runtime. In Colab: Runtime > Change runtime type > T4 GPU, then restart.")
        vram = S["vram_gb"]
        ladder = [MODEL_NAME] if MODEL_NAME else (["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct"] if vram >= 22 else ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"] if vram >= 8 else ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"])
        last = None; dt = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16      # T4 (7.5) has no real bf16: is_bf16_supported() says True by emulation, which is slow and can produce garbage
        for name in ladder:
            try:
                m.update(state="downloading", name=name, err=None); path = _download(name, m)
                m.update(state="loading"); tok = AutoTokenizer.from_pretrained(path)
                mdl = AutoModelForCausalLM.from_pretrained(path, torch_dtype=dt, device_map={"": 0}, low_cpu_mem_usage=True).eval(); _M["llm"], _M["tok"] = mdl, tok
                m.update(state="warming")
                def _ok(mm):
                    pr = tok.apply_chat_template([{"role": "user", "content": "Reply in one short sentence: what is the capital of France?"}], tokenize=False, add_generation_prompt=True); e = tok(pr, return_tensors="pt").to("cuda:0")
                    with torch.inference_mode(): o = mm.generate(**e, max_new_tokens=14, do_sample=False)
                    return "paris" in tok.decode(o[0][e["input_ids"].shape[1]:], skip_special_tokens=True).lower()
                if not _ok(mdl) and dt != torch.float32:          # the half-precision output was not sane: reload in float32 (slower but correct)
                    m.update(state="loading", err="half precision gave bad text; reloading in float32"); del mdl; torch.cuda.empty_cache(); dt = torch.float32
                    mdl = AutoModelForCausalLM.from_pretrained(path, torch_dtype=dt, device_map={"": 0}, low_cpu_mem_usage=True).eval(); _M["llm"] = mdl; m.update(state="warming", err=None)
                m.update(state="ready", device="cuda:0", sec=round(time.time() - t, 1), err=None, quant=str(dt).replace("torch.", ""))
                threading.Thread(target=_load_draft, args=(name, path, tok, dt), daemon=True).start(); return
            except Exception as e:
                last = e; _M["llm"] = None; m.update(err=f"{name}: {str(e)[:200]}")
                try: torch.cuda.empty_cache()
                except Exception: pass
        raise last or RuntimeError("no model loaded")
    except Exception as e: m.update(state="failed", err=str(e)[:300])

def _load_draft(name, path, tok, dt):
    """Speculative decoding: a 0.5B Qwen drafts tokens and the main model checks them in one pass. Kept only when it measures faster (>= 1.15x) and gives sane text."""
    import torch
    D = S.setdefault("draft", {"state": "off", "name": None, "speedup": None, "err": None})
    try:
        if not ("Qwen2.5-3B" in name and S["vram_gb"] >= 8): D.update(state="off", err="only used with the 3B model"); return
        from transformers import AutoModelForCausalLM
        dn = "Qwen/Qwen2.5-0.5B-Instruct"; D.update(state="loading", name=dn); dp = _download(dn, {})
        dm = AutoModelForCausalLM.from_pretrained(dp, torch_dtype=dt, device_map={"": 0}, low_cpu_mem_usage=True).eval(); mdl = _M["llm"]
        pr = tok.apply_chat_template([{"role": "user", "content": "List three tips for keeping stock levels healthy in a retail outlet."}], tokenize=False, add_generation_prompt=True); e = tok(pr, return_tensors="pt").to("cuda:0")
        def run(**kw):
            torch.cuda.synchronize(); t0 = time.time()
            with _GEN, torch.inference_mode(): o = mdl.generate(**e, max_new_tokens=48, do_sample=False, pad_token_id=tok.eos_token_id, **kw)
            torch.cuda.synchronize(); return time.time() - t0, tok.decode(o[0][e["input_ids"].shape[1]:], skip_special_tokens=True)
        run(); run(assistant_model=dm)                                   # warm both paths
        t1, a1 = run(); t2, a2 = run(assistant_model=dm); sp = t1 / max(t2, 1e-3)
        if sp >= 1.15 and len(a2.strip()) > 20 and a2.strip()[:30] == a1.strip()[:30]: _M["draft"] = dm; D.update(state="ready", speedup=round(sp, 2), err=None)
        else: del dm; torch.cuda.empty_cache(); D.update(state="off", speedup=round(sp, 2), err="not faster on this GPU, so it is not used")
    except Exception as ex: _M["draft"] = None; D.update(state="off", err=str(ex)[:200])
def start():
    """Called once at server start. NLLB + embeddings load in one thread, the LLM download/load runs in parallel, so the UI is usable immediately."""
    if S["started"]: return
    S["started"] = time.time(); _probe()
    threading.Thread(target=lambda: (_load_nllb(), _load_embed(), _load_rerank()), daemon=True).start(); threading.Thread(target=_load_llm, daemon=True).start()

# ---------------------------------------------------------------- translation (NLLB-200 only)
def code_of(lang): return lang if lang in NLLB_LANGS.values() else NLLB_LANGS.get(lang, "eng_Latn")
def detect(text):
    if not text: return "eng_Latn"
    ex = extra_detect(text)
    if ex: return ex
    for ch in text:
        for lo, hi, c in (("஀", "௿", "tam_Taml"), ("ऀ", "ॿ", "hin_Deva"), ("ఀ", "౿", "tel_Telu"), ("ಀ", "೿", "kan_Knda"), ("ഀ", "ൿ", "mal_Mlym"), ("ঀ", "৿", "ben_Beng"), ("઀", "૿", "guj_Gujr"), ("਀", "੿", "pan_Guru"), ("଀", "୿", "ory_Orya"), ("؀", "ۿ", "arb_Arab")):
            if lo <= ch <= hi: return c
    return "eng_Latn"
def _split(text, limit=300):
    parts = re.split(r"(?<=[.!?।。؟])\s+", str(text).strip()); out, cur = [], ""
    for x in parts:
        if cur and len(cur) + len(x) + 1 > limit: out.append(cur); cur = x
        else: cur = (cur + " " + x).strip()
    if cur: out.append(cur)
    return out or [str(text)[:limit]]
def _tid(tok, code):
    try:
        i = tok.convert_tokens_to_ids(code)
        if i is not None and i != tok.unk_token_id: return i
    except Exception: pass
    try: return tok.lang_code_to_id[code]
    except Exception: return None
def _gen_batch(segs, src, tgt, bs=32, beams=2):
    import torch
    mdl, tok = _M["nllb"], _M["nllb_tok"]; tok.src_lang = src; fid = _tid(tok, tgt); out = [None] * len(segs); order = sorted(range(len(segs)), key=lambda i: len(segs[i]))
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]; enc = tok([segs[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=200).to(mdl.device)
        with torch.inference_mode():
            g = mdl.generate(**enc, forced_bos_token_id=fid, max_new_tokens=int(min(256, enc["input_ids"].shape[1] * 1.5 + 12)), num_beams=beams, do_sample=False,
                             no_repeat_ngram_size=4, repetition_penalty=1.12, early_stopping=True)
        for i, o in zip(idx, tok.batch_decode(g, skip_special_tokens=True)): out[i] = o.strip()
    return out
def ready(): return S["models"]["nllb"]["state"] == "ready"

# ---- protected tokens: ids, numbers, units, money, urls and code are never sent through the model (it garbles them) and are put back afterwards
_PROT = re.compile(r"`[^`]+`|https?://\S+|[\w.+-]+@[\w-]+\.[\w.]+|\b[A-Z]{2,6}-[A-Z]?\d{2,}\b|#\d+|[$€£₹]\s?\d[\d,]*(?:\.\d+)?[kKmM]?|\d[\d,]*(?:\.\d+)?\s?(?:%|nm|NM|kn|kg|km/h|km|TEU|USD|INR|mm|h|d)?(?![\w])")
_TOK = "Q{}Z"
def _protect(t):
    keep = []
    def rep(m): keep.append(m.group(0)); return _TOK.format(len(keep) - 1)
    return _PROT.sub(rep, t), keep
def _restore(t, keep):
    ok = True
    for i, v in enumerate(keep):
        tk = _TOK.format(i)
        if tk not in t:
            m = re.search(r"Q\s?" + str(i) + r"\s?Z", t)
            if m: t = t[:m.start()] + tk + t[m.end():]
            else: ok = False
        t = t.replace(tk, v.strip("`") if v.startswith("`") else v)
    return t, ok
def _loops(o, src):
    """True when the output looks like a hallucination: far longer than the source, or a repeating phrase."""
    if len(o) > 3.2 * len(src) + 30: return True
    w = o.split()
    return len(w) > 8 and len(set(w)) < len(w) * 0.35
def _script_ok(o, code):
    blocks = {"hin_Deva": ("ऀ", "ॿ"), "tam_Taml": ("஀", "௿"), "tel_Telu": ("ఀ", "౿"), "kan_Knda": ("ಀ", "೿"), "mal_Mlym": ("ഀ", "ൿ"), "ben_Beng": ("ঀ", "৿"), "guj_Gujr": ("઀", "૿"), "pan_Guru": ("਀", "੿"), "arb_Arab": ("؀", "ۿ"), "urd_Arab": ("؀", "ۿ"), "rus_Cyrl": ("Ѐ", "ӿ"), "ell_Grek": ("Ͱ", "Ͽ"), "heb_Hebr": ("֐", "׿"), "tha_Thai": ("฀", "๿"), "kor_Hang": ("가", "힣"), "jpn_Jpan": ("぀", "ヿ"), "zho_Hans": ("一", "鿿"), "zho_Hant": ("一", "鿿")}
    r = blocks.get(code)
    if not r: return True
    letters = [ch for ch in o if ch.isalpha()]
    return not letters or sum(1 for ch in letters if r[0] <= ch <= r[1]) / len(letters) > 0.3
_MD = re.compile(r"^(\s*(?:[-*•]\s+|\d+[.)]\s+|#{1,6}\s+|>\s+)?)(.*?)(\s*)$")
def _units(text):
    """Split markdown/plain text into [(prefix, body, suffix, kind)] translation units. Tables are split per cell, bold markers are kept around the unit."""
    units = []
    for line in str(text).split("\n"):
        if not line.strip(): units.append(("", "", "", "blank")); continue
        if line.strip().startswith("|") and line.count("|") >= 2:
            if re.fullmatch(r"[\s|:-]+", line): units.append((line, "", "", "raw")); continue
            cells = line.strip().strip("|").split("|"); units.append(("|", [c.strip() for c in cells], "|", "row")); continue
        m = _MD.match(line); pre, body, suf = m.group(1), m.group(2), m.group(3); wrap = ""
        if body.startswith("**") and body.endswith("**") and body.count("**") == 2: wrap = "**"; body = body[2:-2]
        body = body.replace("**", "")
        units.append((pre + wrap, body, wrap + suf, "line"))
    return units
def _translate_segments(segs, src, tgt):
    """segs: raw English segments. Returns translated, with protected tokens, quality gates and retries."""
    prot = [_protect(x) for x in segs]; out = _gen_batch([p[0] for p in prot], src, tgt); final = [None] * len(segs); retry = []
    for i, ((p, keep), o) in enumerate(zip(prot, out)):
        if not o or _loops(o, p) or not _script_ok(o, tgt): retry.append(i); continue
        r, ok = _restore(o, keep)
        if not ok: retry.append(i); continue
        final[i] = r
    if retry:    # second pass: greedy, no protection (the model sees the digits itself); still gated
        o2 = _gen_batch([segs[i] for i in retry], src, tgt, bs=16, beams=1)
        for i, o in zip(retry, o2):
            final[i] = o if o and not _loops(o, segs[i]) and _script_ok(o, tgt) else segs[i]
    return final
_CV = "v3"
def translate_many(texts, tgt="eng_Latn", src="eng_Latn"):
    """Batch translate with a persistent cache. Markdown-aware, protects ids/numbers/units, gates the model output. Returns (list, error|None)."""
    src, tgt = code_of(src), code_of(tgt)
    if src == tgt: return list(texts), None
    if not ready(): return list(texts), f"NLLB-200 is {S['models']['nllb']['state']}" + (f": {S['models']['nllb']['err']}" if S["models"]["nllb"]["err"] else "")
    res = [None] * len(texts); need = {}; t0 = time.time()
    for i, tx in enumerate(texts):
        if not tx or not str(tx).strip(): res[i] = tx; continue
        k = hashlib.md5(f"{_CV}|{src}|{tgt}|{tx}".encode()).hexdigest(); hit = _TC.get(k)
        if hit is None:
            try:
                with _LOCK: r = _cache_db().execute("SELECT v FROM tr_cache WHERE k=?", (k,)).fetchone()
                hit = r[0] if r else None
            except Exception: hit = None
        if hit is not None: res[i] = hit; _TC[k] = hit
        else: need.setdefault(k, []).append(i)
    if need:
        keys = list(need); plans = {}; segs = []
        for k in keys:
            units = _units(texts[need[k][0]]); plan = []
            for pre, body, suf, kind in units:
                if kind == "row":
                    ids = []
                    for c in body:
                        ss = [x for x in _split(c)] if re.search(r"[A-Za-z]{3}", c) else []
                        ids.append((c, [len(segs) + j for j in range(len(ss))])); segs.extend(ss)
                    plan.append((pre, ids, suf, kind))
                elif kind == "line" and re.search(r"[A-Za-z]{3}", body):
                    ss = _split(body); plan.append((pre, (body, [len(segs) + j for j in range(len(ss))]), suf, kind)); segs.extend(ss)
                else: plan.append((pre, body, suf, "keep" if kind != "row" else kind))
            plans[k] = plan
        with _GEN:
            try: outs = _translate_segments(segs, src, tgt) if segs else []
            except Exception as e: return [t if r is None else r for t, r in zip(texts, res)], f"NLLB error: {str(e)[:160]}"
        def cell(c, idx): return " ".join(outs[j] for j in idx) if idx else c
        try:
            cdb = _cache_db()
            for k in keys:
                lines = []
                for pre, body, suf, kind in plans[k]:
                    if kind == "blank": lines.append("")
                    elif kind == "raw": lines.append(pre)
                    elif kind == "row": lines.append("| " + " | ".join(cell(c, idx) for c, idx in body) + " |")
                    elif kind == "line": lines.append(pre + cell(*body) + suf)
                    else: lines.append(pre + (body if isinstance(body, str) else "") + suf)
                v = "\n".join(lines).strip() or texts[need[k][0]]; _TC[k] = v
                for i in need[k]: res[i] = v
                cdb.execute("INSERT OR REPLACE INTO tr_cache VALUES(?,?)", (k, v))
            cdb.commit(); cdb.close()
        except Exception as e:
            return [t if r is None else r for t, r in zip(texts, res)], f"cache error: {str(e)[:120]}"
        S["stats"]["translate_ms"] = round((time.time() - t0) * 1000); S["stats"]["translations"] += len(segs)
    return [t if r is None else r for t, r in zip(texts, res)], None
def translate(text, tgt="eng_Latn", src=None):
    r, e = translate_many([text], tgt, src or detect(text)); return r[0], e

# ---------------------------------------------------------------- LLM
def llm_ready(): return S["models"]["llm"]["state"] == "ready"
class _Deg:
    """Stops generation when the model starts looping (the last 24 tokens use 3 or fewer distinct ids)."""
    def __init__(self, n0): self.n0 = n0
    def __call__(self, input_ids, scores, **kw):
        import torch; g = input_ids[0, self.n0:]; bad = g.shape[0] >= 24 and len(set(g[-24:].tolist())) <= 3
        return torch.full((input_ids.shape[0],), bool(bad), dtype=torch.bool, device=input_ids.device)
def llm_stream(messages, max_new_tokens=260, temperature=0.2):
    """Yields text pieces as they are generated on the GPU. Returns immediately (empty) when the model is not loaded."""
    if not llm_ready(): return
    import torch; from transformers import TextIteratorStreamer
    tok, mdl = _M["tok"], _M["llm"]; prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True); enc = tok(prompt, return_tensors="pt", truncation=True, max_length=3500).to(mdl.device)
    st = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True, timeout=120); _dr = _M.get("draft") if temperature <= 0.01 else None; kw = dict(**enc, streamer=st, max_new_tokens=max_new_tokens, do_sample=temperature > 0.01, temperature=max(temperature, 0.01), top_p=.9, repetition_penalty=1.1, pad_token_id=tok.eos_token_id, stopping_criteria=__import__('transformers').StoppingCriteriaList([_Deg(enc['input_ids'].shape[1])]))
    if _dr is not None: kw["assistant_model"] = _dr
    def run():
        try:
            with _GEN, torch.inference_mode(): mdl.generate(**kw)
        except Exception as ex:
            if "assistant_model" in kw:                                  # speculative decoding failed: switch it off, answer normally
                _M["draft"] = None; S.setdefault("draft", {}).update(state="off", err=str(ex)[:160]); kw.pop("assistant_model")
                try:
                    with _GEN, torch.inference_mode(): mdl.generate(**kw)
                except Exception: st.on_finalized_text("", stream_end=True)
            else: st.on_finalized_text("", stream_end=True)
    th = threading.Thread(target=run, daemon=True); th.start(); t0 = time.time(); n = 0
    for piece in st:
        n += 1; yield piece
    th.join(); S["stats"]["llm_calls"] += 1; S["stats"]["tok_s"] = round(n / max(time.time() - t0, 1e-3), 1)
def llm_text(messages, **kw): return "".join(llm_stream(messages, **kw)).strip()

# ---------------------------------------------------------------- embeddings + RAG store
def embed(texts):
    import numpy as np
    if _M["embed"] is None: return None
    return np.asarray(_M["embed"].encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False), dtype="float32")


In [ ]:
%%writefile /content/app/hardening.py
"""Production hardening for the FastAPI app: request ids + JSON access log, security headers (CSP, frame, permissions), per-IP rate limits on auth routes,
a global JSON error handler that never leaks tracebacks, and Prometheus-style metrics. Install with hardening.install(app)."""
import os, time, json, uuid, threading, logging, traceback, collections
from logging.handlers import RotatingFileHandler
from fastapi import Request
from fastapi.responses import JSONResponse, PlainTextResponse
import config

_LOG = logging.getLogger("access"); _LOG.setLevel(logging.INFO); _LOG.propagate = False
try:
    _h = RotatingFileHandler(os.path.join(config.DATA_DIR, "access.log"), maxBytes=2_000_000, backupCount=3); _h.setFormatter(logging.Formatter("%(message)s")); _LOG.addHandler(_h)
except Exception: pass
_ERR = logging.getLogger("errors"); _ERR.propagate = False
try:
    _e = RotatingFileHandler(os.path.join(config.DATA_DIR, "errors.log"), maxBytes=1_000_000, backupCount=2); _e.setFormatter(logging.Formatter("%(asctime)s %(message)s")); _ERR.addHandler(_e)
except Exception: pass

M = {"requests": collections.Counter(), "status": collections.Counter(), "lat_sum": collections.Counter(), "lat_n": collections.Counter(), "errors": 0, "limited": 0, "started": time.time()}
_L = threading.Lock(); _BUCKET = {}
CSP = "default-src 'self'; img-src 'self' data: blob:; style-src 'self' 'unsafe-inline'; script-src 'self' 'unsafe-inline'; connect-src 'self'; font-src 'self' data:; frame-ancestors 'none'; base-uri 'self'; form-action 'self'"
# path prefix -> (max requests, per seconds). Everything else: 900 per minute per IP.
LIMITS = {"/api/login": (10, 60), "/api/register": (6, 600), "/api/otp": (5, 300), "/api/reset": (8, 600), "/api/profile/password": (8, 600), "/api/copilot": (40, 60), "/api/translate": (60, 60)}
def _ip(r):
    f = r.headers.get("x-forwarded-for"); return (f.split(",")[0].strip() if f else (r.client.host if r.client else "?"))
def _allow(ip, path):
    key, (n, per) = next(((p, v) for p, v in LIMITS.items() if path.startswith(p)), ("*", (900, 60))); k = (ip, key); now = time.time()
    with _L:
        q = _BUCKET.setdefault(k, collections.deque())
        while q and now - q[0] > per: q.popleft()
        if len(q) >= n: return False, int(per - (now - q[0])) + 1
        q.append(now)
        if len(_BUCKET) > 20000: _BUCKET.clear()
    return True, 0
def _route(path):
    parts = path.split("/")
    return "/".join(parts[:3]) if path.startswith("/api/") else "static"

INTERNAL = uuid.uuid4().hex      # lets the server call its own API (Teach Copilot tester) without tripping the rate limit

def install(app, kind="app"):
    @app.middleware("http")
    async def guard(request: Request, call_next):
        rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]; t0 = time.time(); path = request.url.path
        if os.environ.get("RATE_LIMIT_OFF") != "1" and request.headers.get("x-internal") != INTERNAL and (path.startswith("/api/") and request.method != "GET" or path.startswith("/api/login")):
            ok, wait = _allow(_ip(request), path)
            if not ok:
                M["limited"] += 1; return JSONResponse({"ok": False, "msg": f"Too many requests. Try again in {wait}s.", "request_id": rid}, status_code=429, headers={"Retry-After": str(wait), "X-Request-ID": rid})
        try: r = await call_next(request)
        except Exception as e:
            M["errors"] += 1; _ERR.error("%s %s %s\n%s", rid, request.method, path, traceback.format_exc())
            return JSONResponse({"ok": False, "msg": "Something went wrong on the server. Quote this id to the admin: " + rid, "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
        ms = (time.time() - t0) * 1000; rt = _route(path)
        with _L: M["requests"][rt] += 1; M["status"][r.status_code // 100] += 1; M["lat_sum"][rt] += ms; M["lat_n"][rt] += 1
        h = r.headers; h["X-Request-ID"] = rid; h["X-Frame-Options"] = "DENY"; h["Content-Security-Policy"] = CSP; h["Permissions-Policy"] = "microphone=(self), camera=(), geolocation=()"; h["Cross-Origin-Opener-Policy"] = "same-origin"
        if request.headers.get("x-forwarded-proto") == "https": h["Strict-Transport-Security"] = "max-age=31536000"
        if path.startswith("/api/") and path not in ("/api/stream", "/api/pulse"): _LOG.info(json.dumps({"t": round(t0), "id": rid, "ip": _ip(request), "m": request.method, "p": path, "s": r.status_code, "ms": round(ms)}))
        return r

    @app.get("/api/ready")
    def ready(): return {"ok": True, "kind": kind, "uptime_s": round(time.time() - M["started"])}

    @app.get("/api/metrics")
    def metrics(request: Request):
        """Prometheus text format. Set METRICS_TOKEN to require ?token=..., otherwise only logged-in admins (cookie) may read it."""
        tok = os.environ.get("METRICS_TOKEN")
        if tok and request.query_params.get("token") != tok: return PlainTextResponse("forbidden", status_code=403)
        if not tok:
            try:
                import authx; u = authx.decode(request.cookies.get("sid", ""))
                if not u or u.get("role") != "Admin": return PlainTextResponse("forbidden", status_code=403)
            except Exception: return PlainTextResponse("forbidden", status_code=403)
        L = ["# TYPE app_requests_total counter"] + [f'app_requests_total{{route="{k}"}} {v}' for k, v in M["requests"].items()] + ["# TYPE app_responses_total counter"] + [f'app_responses_total{{class="{k}xx"}} {v}' for k, v in M["status"].items()]
        L += ["# TYPE app_latency_ms_avg gauge"] + [f'app_latency_ms_avg{{route="{k}"}} {M["lat_sum"][k] / max(1, M["lat_n"][k]):.1f}' for k in M["lat_n"]] + [f"app_errors_total {M['errors']}", f"app_rate_limited_total {M['limited']}", f"app_uptime_seconds {time.time() - M['started']:.0f}"]
        try:
            import live_engine as LE; c = LE.conn(); L += [f"app_open_tickets {c.execute('SELECT COUNT(*) FROM agent_actions WHERE status=?', ('open',)).fetchone()[0]}"]
        except Exception: pass
        try:
            import gpu; g = gpu.status(); L += [f'app_model_ready{{model="{k}"}} {1 if v["state"] == "ready" else 0}' for k, v in g["models"].items()]
        except Exception: pass
        return PlainTextResponse("\n".join(L) + "\n")


In [ ]:
%%writefile /content/app/kb_pack.py
"""Sample policy packs for the knowledge base. ILLUSTRATIVE internal-policy text written for this demo: replace or extend it with your own
documents (Knowledge base tab -> upload, or build a corpus in the RAG builder notebook and import it). EVAL = question -> the document that should be retrieved."""
FRANCHISE = [
 ("Inventory & Reorder Policy", "Operations SOP", "Every outlet keeps a safety stock equal to three days of average demand. A reorder is raised when days of cover fall below the supplier lead time plus two days. Perishables follow FEFO rotation: the earliest expiry is sold first and anything within 24 hours of expiry is marked down 30 percent or written off. Stock-outs of a top-10 SKU for more than 6 hours must be escalated to the regional manager. Suppliers with lead time above 5 days need a secondary supplier on file. Purchase orders above 2 lakh rupees need owner approval; below that the Store Manager may approve."),
 ("Staffing, Overtime & Attrition Policy", "HR Manual", "Staff may work at most 48 hours a week; overtime beyond 9 hours a week is flagged and needs Store Manager approval, beyond 16 hours the Franchise Owner. An outlet must keep at least 85 percent of its rostered staff present on weekends. Three unexplained absences in a month trigger a counselling meeting. Attrition risk above 70 percent from the model prompts a retention conversation within 7 days: review pay band, schedule fairness and growth path. New hires complete food-safety induction before their first solo shift."),
 ("Food Safety & FSSAI Compliance", "Compliance", "Every outlet must display a valid FSSAI licence. Cold storage stays between 1 and 4 degrees Celsius, deep freezers below minus 18. Cooking oil is discarded when total polar compounds exceed 25 percent. A failed internal audit (score below 60) requires a corrective action plan in 48 hours and a re-audit within 14 days. Critical violations such as pest evidence or temperature abuse close the affected station immediately. Audit records are retained for 24 months."),
 ("Customer Service & CSAT Standard", "Service SOP", "Outlets maintain a CSAT of at least 4.0 out of 5. Customers are greeted within 30 seconds and complaints resolved within 24 hours. An outlet whose rolling sentiment score falls below 45 out of 100 enters service recovery: manager call-back to unhappy customers, a shift briefing on the top complaint theme, and a mystery-shopper visit within 10 days. Review replies must be polite, specific and never blame the customer. Refunds up to 500 rupees can be granted by the shift lead."),
 ("Marketing Campaign & Discount Guardrails", "Marketing Policy", "Discounts above 25 percent need Franchise Owner approval. Campaigns must show positive uplift within the first 14 days or be paused: uplift is measured against the same weekday baseline. Promotions may not be run on items with fewer than 2 days of stock cover. Local campaigns spend at most 3 percent of monthly sales. Loyalty offers are capped at one per customer per day. Every campaign records its target outlets, start date and expected lift."),
 ("Outlet Performance & Health Score", "Franchise Agreement", "Each outlet receives a health score from 0 to 100 combining sales trend, inventory cover, staffing, audit results and sentiment. Below 60 is critical: the owner reviews it weekly. Below 80 is warning. Two consecutive critical months trigger a performance improvement plan with a 60-day window. Royalty is 5 percent of net sales, paid monthly; late payment attracts 1.5 percent per month. Franchisees must submit daily sales through the POS feed."),
 ("Incident Escalation & Approval Matrix", "Operations SOP", "Agent-raised tickets follow severity: Info is handled within 72 hours, Warning within 24 hours, Critical within 4 hours. Critical tickets are assigned to the Franchise Owner when unowned for 1 hour. Actions that spend money or contact customers (purchase orders, discounts, messages) require human approval; read-only checks and internal notes are automatic. Every approval and rejection is written to the audit log with the approver and time."),
 ("Data Privacy & Employee Records", "Compliance", "Customer phone numbers and e-mail addresses are used only for service and consented marketing. Employee records are visible to the Store Manager of that outlet and the Franchise Owner only. Exports containing personal data are logged. Data is retained for at most 24 months unless law requires longer. A data-subject request must be answered within 30 days."),
]
FREIGHT = [
 ("Quotation & Margin Policy", "Pricing SOP", "Quotes are built from base ocean rate, fuel surcharge (bunker adjustment factor), insurance of 0.35 percent of cargo value, customs fees and a target margin of 15 percent. Quotes with margin below 10 percent or a final price above 50,000 US dollars need Admin approval. Quotes are valid for 7 days. Fuel surcharge is re-priced when bunker price moves more than 5 percent. Spot quotes for hazardous cargo carry a 12 percent premium."),
 ("Route Selection & Canal Policy", "Operations SOP", "The default route is the shortest safe sea lane. Where the Suez Canal is disrupted or unsafe the alternative is the Cape of Good Hope, which adds roughly 3,500 nautical miles and 8 to 12 days. Panama transits depend on draft and queue length. Routes avoid active storm cells by at least 300 nautical miles. A re-route that adds more than 10 percent distance must be approved by the Operations Manager. The Strait of Hormuz and Bab-el-Mandeb are monitored for security advisories."),
 ("Port Congestion & Dwell-Time Guide", "Port Operations", "A port congestion index of 4 or higher, or dwell time above 5 days, is treated as congested. Congested ports add a delay allowance to transit time and a congestion surcharge to quotes. Where an alternative port exists within 300 nautical miles, planners should evaluate diversion. Anchorage waiting of more than 48 hours raises a Warning ticket. Berth windows are confirmed with the terminal 72 hours before arrival."),
 ("Weather & Storm Response", "Safety Manual", "Wind above 17 m/s or wave height above 4 metres is a Warning for the affected leg; wind above 25 m/s is Critical and the master may deviate. Vessels within 500 nautical miles of a named storm are re-evaluated every 6 hours. Cargo sensitive to temperature or moisture carries extra weather insurance during monsoon and typhoon seasons. Deviation costs are recorded against the shipment and shared with the customer."),
 ("Customs Documentation & Compliance", "Customs Manual", "Each shipment needs a commercial invoice, packing list, bill of lading and certificate of origin. Pharmaceuticals additionally need an import licence and drug control certificate, machinery a certificate of origin, electronics a BIS certificate, and hazardous goods an IMDG declaration. Missing documents place the shipment on Customs Hold; the broker has 48 hours to supply them. HS codes are verified against the tariff schedule before quoting duty. Misdeclaration is escalated to compliance."),
 ("Carrier Selection & Scorecard", "Procurement Policy", "Carriers are scored on on-time performance (40 percent), damage and claims (25 percent), price competitiveness (20 percent) and responsiveness (15 percent). A carrier below 70 is placed on watch and may not be assigned to high-value cargo. At least two qualified carriers must be quoted for any lane above 100,000 dollars. Contracts are reviewed annually and rates benchmarked quarterly."),
 ("Incident Escalation & Approval Matrix", "Operations SOP", "Agent tickets follow severity: Info within 72 hours, Warning within 24 hours, Critical within 4 hours. Actions that change a vessel route, commit money or contact a customer need human approval; internal notes and read-only checks are automatic. Every approval is logged with the approver and timestamp. Critical weather and customs tickets notify the Operations Manager immediately."),
 ("Cargo Insurance & Claims", "Risk Policy", "Marine cargo insurance covers all-risk at 110 percent of invoice value. Claims must be notified within 3 days of discharge with survey report, bill of lading and photos. General average declarations are handled by the carrier's average adjuster. Deductibles are 0.5 percent of cargo value, 1 percent for reefer cargo. High-value cargo above 1 million dollars requires a pre-shipment survey."),
]
EVAL = {
 "Franchise": [("When do I need to reorder stock and how much safety stock is required?", "Inventory & Reorder Policy"), ("How many overtime hours need owner approval?", "Staffing, Overtime & Attrition Policy"), ("What temperature must the cold storage be?", "Food Safety & FSSAI Compliance"),
               ("What happens when an outlet sentiment drops below 45?", "Customer Service & CSAT Standard"), ("Who approves discounts above 25 percent?", "Marketing Campaign & Discount Guardrails"), ("What is the royalty percentage?", "Outlet Performance & Health Score"), ("how long to keep audit records", "Food Safety & FSSAI Compliance")],
 "Freight": [("What margin triggers admin approval of a quote?", "Quotation & Margin Policy"), ("How far does the Cape route add compared to Suez?", "Route Selection & Canal Policy"), ("What congestion index counts as congested?", "Port Congestion & Dwell-Time Guide"),
             ("wind speed that is critical for a vessel", "Weather & Storm Response"), ("which documents are needed for pharmaceuticals", "Customs Documentation & Compliance"), ("how are carriers scored", "Carrier Selection & Scorecard"), ("claim notification deadline after discharge", "Cargo Insurance & Claims")],
}


In [ ]:
%%writefile /content/app/lang_pack.py
"""12 extra NLLB-200 languages (35 total with the apps' 23) + better source-language detection (script ranges, then langdetect for Latin-script languages)."""
EXTRA_LANGS = {"Portuguese (Português)": "por_Latn", "Russian (Русский)": "rus_Cyrl", "Korean (한국어)": "kor_Hang", "Turkish (Türkçe)": "tur_Latn", "Indonesian (Bahasa Indonesia)": "ind_Latn",
               "Vietnamese (Tiếng Việt)": "vie_Latn", "Thai (ไทย)": "tha_Thai", "Italian (Italiano)": "ita_Latn", "Dutch (Nederlands)": "nld_Latn", "Swahili (Kiswahili)": "swh_Latn",
               "Greek (Ελληνικά)": "ell_Grek", "Hebrew (עברית)": "heb_Hebr", "Polish (Polski)": "pol_Latn", "Persian (فارسی)": "pes_Arab"}
_ISO = {"fr": "fra_Latn", "de": "deu_Latn", "es": "spa_Latn", "pt": "por_Latn", "it": "ita_Latn", "nl": "nld_Latn", "id": "ind_Latn", "tr": "tur_Latn", "vi": "vie_Latn", "pl": "pol_Latn", "sw": "swh_Latn", "en": "eng_Latn"}
def extra_detect(text):
    """Returns an NLLB code or None (caller keeps its own result)."""
    kana = any('\u3040' <= c <= '\u30ff' for c in text); han = any('\u4e00' <= c <= '\u9fff' for c in text)
    for ch in text:
        if '\u0400' <= ch <= '\u04ff': return "rus_Cyrl"
        if '\uac00' <= ch <= '\ud7af': return "kor_Hang"
        if '\u0e00' <= ch <= '\u0e7f': return "tha_Thai"
        if '\u0370' <= ch <= '\u03ff': return "ell_Grek"
        if '\u0590' <= ch <= '\u05ff': return "heb_Hebr"
        if '\u0d80' <= ch <= '\u0dff': return "sin_Sinh"
    if han and not kana: return "zho_Hans"
    if kana: return "jpn_Jpan"
    if not any(ord(c) > 127 for c in text) and len(text.split()) < 4: return None
    try:
        from langdetect import detect, DetectorFactory; DetectorFactory.seed = 0; return _ISO.get(detect(text))
    except Exception: return None

In [ ]:
%%writefile /content/app/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    p = dict(getattr(dom, "DEFAULT_POLICY", {})); p.update(get_state("policy", {}) or {}); return p
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
_DEF_CHAINS = {}
def set_chain_defaults(dom):
    """Per-type default approval chains come from the domain file (DEFAULT_CHAINS for agent actions, REQUEST_KINDS for manual requests)."""
    _DEF_CHAINS.clear(); _DEF_CHAINS.update(getattr(dom, "DEFAULT_CHAINS", {}) or {}); _DEF_CHAINS.update({"req:" + k: v for k, v in (getattr(dom, "REQUEST_KINDS", {}) or {}).items()})
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}; explicit = bool(cfg.get(typ))
    ch = list(cfg[typ]) if explicit else list(_DEF_CHAINS.get(typ) or [])
    if not ch: ch = ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]; explicit = True
    if levels and int(levels) >= 2 and ch[-1] != "Admin": ch.append("Admin")
    if severity == "Critical" and not explicit and ch[-1] != "Admin": ch.append("Admin")     # a critical item also needs an Admin sign-off unless the Admin set this chain by hand
    return ch
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,status,note,approval) VALUES(?,?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, (datetime.datetime.now() + datetime.timedelta(hours=SLA_H.get(severity, 72))).isoformat(timespec="seconds"), f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "req:" + typ, levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT

In [ ]:
%%writefile /content/app/live_rbac.py
"""Capability RBAC for the live layer. Menu-level RBAC stays in each app's rbac.py; this gates actions inside the Command Center."""
CAPS = {  # view, resolve, approve, run, feeder, admin
 "Admin": {"view", "resolve", "approve", "run", "feeder", "admin", "quote"},
 "Franchise Owner": {"view", "resolve", "approve", "run", "feeder"},
 "Operations Manager": {"view", "resolve", "approve", "run", "feeder", "quote"},
 "Store Manager": {"view", "resolve"},
 "Freight Broker": {"view", "resolve", "quote"},
 "Auditor": {"view"},
 "Staff": {"view"},
 "Customer": {"quote"},
}
SCOPED = {"Store Manager", "Staff"}                 # these roles only see actions for their assigned outlet
def norm(role):
    r = (role or "").strip().lower()
    for k in CAPS:
        if k.lower() == r: return k
    return "Admin" if "admin" in r else "Customer"
def can(role, cap): return cap in CAPS.get(norm(role), set())
def scope_for(email):
    from live_engine import conn
    try:
        c = conn(); c.execute("CREATE TABLE IF NOT EXISTS user_scope(email TEXT PRIMARY KEY, entity TEXT)")
        r = c.execute("SELECT entity FROM user_scope WHERE email=?", ((email or "").lower(),)).fetchone(); return r[0] if r else None
    except Exception: return None
def set_scope(email, entity):
    from live_engine import conn
    c = conn(); c.execute("CREATE TABLE IF NOT EXISTS user_scope(email TEXT PRIMARY KEY, entity TEXT)"); c.execute("INSERT OR REPLACE INTO user_scope VALUES(?,?)", (email.lower(), entity)); c.commit()

In [ ]:
%%writefile /content/app/llm_engine.py
"""Shim so the domain modules can ask the GPU LLM for a narrative. In the background worker process no LLM is loaded, so they use their template (the server writes the GPU executive brief on demand)."""
import gpu
def is_llm_loaded(): return gpu.llm_ready()
def generate_text(messages, max_new_tokens=160, temperature=0.1): return gpu.llm_text(messages, max_new_tokens=max_new_tokens, temperature=temperature)


In [ ]:
%%writefile /content/app/mailer.py
"""One place that sends e-mail (OTP codes, reports, alerts). Gmail SMTP with an app password: set EMAIL_ID + EMAIL_PASSWORD (Colab secrets or env)."""
import os, ssl, smtplib
from email.message import EmailMessage
def creds(): return os.environ.get("EMAIL_ID") or "", (os.environ.get("EMAIL_PASSWORD") or os.environ.get("EMAIL_APP_PASSWORD") or "").replace(" ", "")
def configured(): a, b = creds(); return bool(a and b)
def send(to, subject, body, attachments=(), html=None):
    """attachments: [(filename, bytes, mimetype)]. Returns (ok, message)."""
    em, pw = creds()
    if not (em and pw): return False, "E-mail is not configured (set EMAIL_ID and EMAIL_PASSWORD)."
    try:
        m = EmailMessage(); m["Subject"] = subject; m["From"] = em; m["To"] = to; m.set_content(body)
        if html: m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments: m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s: s.login(em, pw); s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e: return False, f"Could not send: {e}"


In [ ]:
%%writefile /content/app/privacy.py
"""Retention, PII inventory, masking and data-subject requests (export / erase)."""
import re, json, time, threading, datetime, hashlib
import pandas as pd
import live_engine as LE
from config import KIND
# table -> (timestamp column, default days kept). Only operational history is purged; master data (outlets, ports, ...) is never deleted by policy.
RETENTION = {"sales": ("ts", 730), "attendance": ("ts", 365), "feedback": ("date", 730), "feed_log": ("ts", 180), "feed_rejects": ("ts", 90), "live_events": ("ts", 30), "agent_runs": ("ts", 90),
             "decisions": ("ts", 180), "audit_log": ("ts", 1095), "alerts": ("date", 365), "weather_risks": (None, 0), "purchase_orders": ("ts", 730), "quote_requests": ("ts", 730)}
PII = {"Franchise": [("staff", "name", "Employee name", "Role-masked"), ("staff", "salary", "Salary", "Role-masked"), ("staff", "age", "Age", "Admin/Owner only"), ("feedback", "comment", "Customer free text (emails and phones auto-redacted)", "Redacted at ingest"),
                     ("app_users", "email", "Login email", "Erasable"), ("customers_fr", "customer_id", "Customer spend profile (pseudonymous id)", "Pseudonymous"), ("audit_log", "actor", "Who did what", "Anonymised on erase")],
       "Freight": [("customers", "name", "Customer company", "Business data"), ("app_users", "email", "Login email", "Erasable"), ("users", "email", "Login email", "Erasable"), ("quote_requests", "customer_id", "Requester id", "Pseudonymous"), ("audit_log", "actor", "Who did what", "Anonymised on erase")]}
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"); PHONE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{8,}\d)(?!\d)")
def redact(t):
    return PHONE.sub("[phone]", EMAIL.sub("[email]", t)) if isinstance(t, str) else t
def policy():
    o = LE.get_state("retention", {}) or {}; return {t: int(o.get(t, d)) for t, (c, d) in RETENTION.items() if c}
def _exists(c, t): return c.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (t,)).fetchone() is not None
def purge(dry=True, actor="system"):
    out = []; pol = policy()
    with LE._DB_LOCK:
        c = LE.conn()
        for t, days in pol.items():
            col = RETENTION[t][0]
            if not _exists(c, t): continue
            cut = (datetime.datetime.now() - datetime.timedelta(days=days)).isoformat(timespec="seconds"); cutd = cut[:10]
            q = f"FROM {t} WHERE {col} IS NOT NULL AND {col} < ?"
            n = c.execute(f"SELECT COUNT(*) {q}", (cut if col != 'date' else cutd,)).fetchone()[0]
            if n and not dry and t != "audit_log": c.execute(f"DELETE {q}", (cut if col != 'date' else cutd,))
            elif n and not dry: c.execute(f"DELETE {q}", (cut,))
            out.append({"table": t, "keep_days": days, "older_rows": int(n), "deleted": 0 if dry else int(n)})
        if not dry:
            c.commit()
    if not dry: LE.audit(actor, "retention_purge", "all", json.dumps({r["table"]: r["deleted"] for r in out if r["deleted"]}))
    return out
def _hash(s): return "erased-" + hashlib.sha256(str(s).encode()).hexdigest()[:10]
def export_subject(key):
    """Everything held about one person: staff id (STF-/EMP-), or a login email."""
    key = str(key).strip(); data = {}
    with LE._DB_LOCK:
        c = LE.conn()
        def grab(t, col, cols="*"):
            if not _exists(c, t): return
            df = pd.read_sql_query(f"SELECT {cols} FROM {t} WHERE {col}=?", c, params=(key,))
            if len(df): data[t] = json.loads(df.to_json(orient="records"))
        if "@" in key:
            grab("app_users", "email", "email, username, role, status, created, last_login"); grab("user_prefs", "email"); grab("user_scope", "email"); grab("audit_log", "actor")
        else:
            grab("staff", "staff_id"); grab("attendance", "staff_id"); grab("customers_fr", "customer_id"); grab("audit_log", "target")
    return data
def erase_subject(key, actor):
    key = str(key).strip(); n = {}
    with LE._DB_LOCK:
        c = LE.conn()
        if "@" in key:
            for t in ("user_prefs", "user_scope"):
                if _exists(c, t): n[t] = c.execute(f"DELETE FROM {t} WHERE email=?", (key,)).rowcount
            if _exists(c, "app_users"): n["app_users"] = c.execute("DELETE FROM app_users WHERE email=?", (key,)).rowcount
            if _exists(c, "users"): n["users"] = c.execute("DELETE FROM users WHERE email=?", (key,)).rowcount
            n["audit_log"] = c.execute("UPDATE audit_log SET actor=? WHERE actor=?", (_hash(key), key)).rowcount
        else:
            if _exists(c, "staff"): n["staff"] = c.execute("UPDATE staff SET name='Erased', salary=NULL, age=NULL WHERE staff_id=?", (key,)).rowcount
        c.commit()
    LE.audit(actor, "dsar_erase", _hash(key), json.dumps(n)); return n
def mask_role(role): return role not in ("Admin", "Franchise Owner", "Operations Manager")
_T = {"t": None}
def start():
    if _T["t"]: return
    def loop():
        while True:
            time.sleep(3600)
            try:
                last = (LE.get_state("purge_last", {}) or {}).get("day")
                if last != datetime.date.today().isoformat(): purge(False, "retention-job"); LE.set_state("purge_last", {"day": datetime.date.today().isoformat()})
            except Exception: pass
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()


In [ ]:
%%writefile /content/app/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re
from blocks import Q

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index", "cheapest", "cheap", "expensive", "costliest", "priciest", "cost"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def answer(kind, q, scope=None, mask=False):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


In [ ]:
%%writefile /content/app/rag.py
"""Hybrid RAG: BM25 (own implementation) + dense embeddings (GPU) fused with Reciprocal Rank Fusion, then a cross-encoder reranker. Persistent chunk store in SQLite,\nsample policy pack per app (kb_pack.py), live-data retrieval over tickets and feedback, import/export and a recall@k evaluator."""
import re, io, os, json, math, base64, sqlite3, threading, time, collections
import numpy as np
import gpu
BUILTIN_KB = [
    {
        "title": "FSSAI Food Safety Compliance & Licensing Guidelines 2024",
        "source": "FSSAI Guidelines 2024",
        "text": "FSSAI (Food Safety and Standards Authority of India) is the statutory body under the Ministry of Health & Family Welfare, Government of India. All food business operators (FBOs), franchise outlets, and commercial kitchens must hold a valid FSSAI license/registration. Outlets must maintain strict hygiene ratings, display FSSAI license numbers on billing receipts, conduct biannual food sample testing, adhere to temperature controls (cold storage <= 5°C, hot display >= 60°C), and maintain staff hygiene records and FOSTAC certified safety supervisors."
    },
    {
        "title": "SOP-001: Customer Service & CSAT Standards",
        "source": "Franchise SOP Manual",
        "text": "All franchise outlets must maintain a minimum CSAT score of 4.0/5.0. Staff must greet customers within 30 seconds of entry. Customer complaints must be resolved within 24 hours. Mystery shopping audits are conducted monthly."
    },
    {
        "title": "SOP-002: Store Operations & Temperature Hygiene",
        "source": "Franchise SOP Manual",
        "text": "Food items must strictly follow FEFO (First-Expired, First-Out) rotation. Cold storage units must maintain temperature between 1°C and 4°C. Deep freezers must stay below -18°C. Oil TPC (Total Polar Compounds) must not exceed 25%."
    },
    {
        "title": "Maritime Shipping Industry & Port Congestion Guide 2024",
        "source": "Global Maritime Logistics Report",
        "text": "Compounding challenges in the Indian shipping industry include port infrastructure bottlenecks, high dwell times at JNPT/Mumbai ports, monsoon storm surges, and tariff adjustments. Shipping lines must optimize vessel speeds and leverage real-time AIS telemetry for route planning."
    }
]

import config
_LOCK = threading.RLock(); _C = {"ver": -1, "rows": [], "bm25": None, "emb": None, "emb_n": -1}; _VER = [0]
_STOP = set("a an the of to in on for and or is are was were be been it this that these those with as at by from how what when which who whom do does did can could should would will may might must not no if then than so we you i they he she them their our your its into about over under per each any all more most such via".split())
def _stem(w):
    for suf in ("ations", "ation", "ingly", "ings", "ing", "edly", "ed", "ies", "es", "ly", "s"):
        if len(w) > len(suf) + 3 and w.endswith(suf): return w[: -len(suf)] + ("y" if suf == "ies" else "")
    return w
def tok(t): return [_stem(w) for w in re.findall(r"[a-z0-9]+|[^\x00-\x7f\s]+", (t or "").lower()) if w not in _STOP]

class BM25:
    """Okapi BM25 (k1=1.5, b=0.75) over pre-tokenised documents; score() returns a numpy vector over all docs."""
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b; self.n = len(docs); self.tf = [collections.Counter(d) for d in docs]; self.len = np.array([len(d) for d in docs], dtype="float32"); self.avg = float(self.len.mean()) if self.n else 1.0
        df = collections.Counter(); self.post = collections.defaultdict(list)
        for i, c in enumerate(self.tf):
            for w, f in c.items(): df[w] += 1; self.post[w].append((i, f))
        self.idf = {w: math.log(1 + (self.n - d + .5) / (d + .5)) for w, d in df.items()}
    def score(self, q):
        s = np.zeros(self.n, dtype="float32")
        for w in set(q):
            if w not in self.idf: continue
            idf = self.idf[w]
            for i, f in self.post[w]: s[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

def _db():
    c = sqlite3.connect(config.DB_PATH, timeout=30, check_same_thread=False)
    c.execute("CREATE TABLE IF NOT EXISTS kb_chunks(id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT, source TEXT, owner TEXT, text TEXT, emb BLOB, builtin INT DEFAULT 0, ts TEXT DEFAULT CURRENT_TIMESTAMP)"); return c
def _chunks(text, size=700, overlap=90):
    text = re.sub(r"\s+", " ", text or "").strip()
    if len(text) <= size: return [text] if text else []
    out, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+", text):
        if len(cur) + len(s) + 1 <= size: cur = (cur + " " + s).strip()
        else:
            if cur: out.append(cur)
            cur = (cur[-overlap:] + " " + s).strip()
    if cur: out.append(cur)
    return out
def drive_dir():
    """Where the user's earlier RAG builder wrote its output (MyDrive/<App>_AI). Override with KB_DRIVE_DIR."""
    k = os.environ.get("APP_KIND", "Franchise"); return os.environ.get("KB_DRIVE_DIR") or f"/content/drive/MyDrive/{'FranchiseOps_AI' if k == 'Franchise' else 'FreightQuote_AI'}"
def _drive_chunks():
    """Chunks produced by the earlier RAG builder notebooks (synthetic_kb/all_chunks.jsonl, else faiss_indexes/*_chunks_meta.json). Returns [(title, source, text)]."""
    d = drive_dir(); out = []
    p = os.path.join(d, "synthetic_kb", "all_chunks.jsonl")
    try:
        if os.path.exists(p):
            for line in open(p, encoding="utf-8"):
                try: c = json.loads(line); out.append((c.get("title", "doc"), "your RAG builder (" + str(c.get("source", "kb")) + ")", c["text"]))
                except Exception: pass
        else:
            for f in os.listdir(os.path.join(d, "faiss_indexes")) if os.path.isdir(os.path.join(d, "faiss_indexes")) else []:
                if f.endswith("_chunks_meta.json"):
                    for c in json.load(open(os.path.join(d, "faiss_indexes", f), encoding="utf-8")): out.append((c.get("title", "doc"), "your RAG builder (" + str(c.get("source", "kb")) + ")", c["text"]))
    except Exception: pass
    return out
def import_drive():
    """(Re)load the Drive chunks as built-in knowledge (builtin=2). Returns the number loaded."""
    _seed(); rows = _drive_chunks(); c = _db()
    try:
        c.execute("DELETE FROM kb_chunks WHERE builtin=2")
        seen = set(); ins = []
        for t, s_, x in rows:
            k = hash((t, x))
            if k in seen or len(x) < 20: continue
            seen.add(k); ins.append((t, s_, None, x, 2))
        c.executemany("INSERT INTO kb_chunks(title,source,owner,text,builtin) VALUES(?,?,?,?,?)", ins); c.commit()
    finally: c.close()
    _VER[0] += 1; return len(ins)
def _seed():
    import kb_pack
    kind = os.environ.get("APP_KIND", "Franchise"); pack = kb_pack.FRANCHISE if kind == "Franchise" else kb_pack.FREIGHT
    c = _db()
    try:
        if c.execute("SELECT COUNT(*) FROM kb_chunks WHERE builtin=1").fetchone()[0] == 0:
            rows = []
            try:
                for d in json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "kb_docs.json"), encoding="utf-8")).get(kind, []): rows += [(d["title"], "Knowledge pack (your RAG builder)", None, ch, 1) for ch in _chunks(d["content"])]
            except Exception: pass
            rows += [(t, s + " (sample)", None, ch, 1) for t, s, x in pack for ch in _chunks(x)]
            c.executemany("INSERT INTO kb_chunks(title,source,owner,text,builtin) VALUES(?,?,?,?,?)", rows); c.commit()
        have_drive = c.execute("SELECT COUNT(*) FROM kb_chunks WHERE builtin=2").fetchone()[0]
    finally: c.close()
    if not have_drive and os.path.isdir(drive_dir()):
        rows = _drive_chunks()
        if rows:
            c = _db()
            try:
                seen = set(); ins = []
                for t, s_, x in rows:
                    k = hash((t, x))
                    if k in seen or len(x) < 20: continue
                    seen.add(k); ins.append((t, s_, None, x, 2))
                c.executemany("INSERT INTO kb_chunks(title,source,owner,text,builtin) VALUES(?,?,?,?,?)", ins); c.commit()
            finally: c.close()
def _load():
    """(Re)build the in-memory view: rows, BM25 index and the embedding matrix. Embeddings are computed on the GPU once and stored in SQLite."""
    with _LOCK:
        if _C["ver"] == _VER[0] and _C["rows"]: return _C
        _seed(); c = _db()
        try:
            rows = [{"id": r[0], "title": r[1], "source": r[2], "owner": r[3], "text": r[4], "emb": r[5]} for r in c.execute("SELECT id,title,source,owner,text,emb FROM kb_chunks ORDER BY id")]
        finally: c.close()
        _C.update(ver=_VER[0], rows=rows, bm25=BM25([tok(r["title"] + " " + r["text"]) for r in rows]), emb=None, emb_n=-1); return _C
def _embeddings():
    C = _load()
    if gpu.S["models"]["embed"]["state"] != "ready": return None
    with _LOCK:
        if _C["emb"] is not None and _C["emb_n"] == len(_C["rows"]): return _C["emb"]
        need = [i for i, r in enumerate(_C["rows"]) if r["emb"] is None]
        if need:
            v = gpu.embed([_C["rows"][i]["text"] for i in need])
            if v is None: return None
            c = _db()
            try:
                for i, e in zip(need, v): _C["rows"][i]["emb"] = e.astype("float32").tobytes(); c.execute("UPDATE kb_chunks SET emb=? WHERE id=?", (_C["rows"][i]["emb"], _C["rows"][i]["id"]))
                c.commit()
            finally: c.close()
        dim = len(np.frombuffer(_C["rows"][0]["emb"], dtype="float32")); mat = np.stack([np.frombuffer(r["emb"], dtype="float32") if r["emb"] and len(r["emb"]) == dim * 4 else np.zeros(dim, "float32") for r in _C["rows"]])
        _C["emb"], _C["emb_n"] = mat, len(_C["rows"]); return mat
def warm():
    """Called once the embedder is ready: index everything so the first question is fast."""
    try: _embeddings()
    except Exception: pass

def add(title, source, text, owner, embs=None):
    _load(); parts = [c for c in _chunks(text) if len(c) >= 20]
    if not parts: return 0
    c = _db()
    try:
        c.executemany("INSERT INTO kb_chunks(title,source,owner,text,emb,builtin) VALUES(?,?,?,?,?,0)", [(title, source, owner, p, (embs[i] if embs else None)) for i, p in enumerate(parts)]); c.commit()
    finally: c.close()
    _VER[0] += 1; return len(parts)
def delete_doc(title, owner):
    c = _db()
    try: n = c.execute("DELETE FROM kb_chunks WHERE title=? AND owner=? AND builtin=0", (title, owner)).rowcount; c.commit()
    finally: c.close()
    _VER[0] += 1; return n
def read_pdf(name, data):
    if name.lower().endswith(".pdf"):
        import pdfplumber
        with pdfplumber.open(io.BytesIO(data)) as pdf: return "\n".join((p.extract_text() or "") for p in pdf.pages)
    return data.decode("utf-8", errors="ignore")
def docs(owner): C = _load(); return [{k: v for k, v in r.items() if k != "emb"} for r in C["rows"] if r["owner"] in (None, owner)]

def _rank(v, mask, n=30):
    idx = np.where(mask)[0]
    if not len(idx): return []
    o = idx[np.argsort(-v[idx])][:n]; return [int(i) for i in o if v[i] > 0]
_SC = collections.OrderedDict(); _MASK = {}
def search(q, owner, k=4, mode="hybrid", rerank=True):
    """Cached front door: identical queries are answered from memory until the corpus changes (a few ms instead of an embedding + rerank pass)."""
    key = (_VER[0], q.strip().lower(), owner, k, mode, bool(rerank) and gpu.S["models"]["rerank"]["state"] == "ready", gpu.S["models"]["embed"]["state"])
    if key in _SC: _SC.move_to_end(key); return _SC[key]
    r = _search(q, owner, k, mode, rerank); _SC[key] = r
    while len(_SC) > 300: _SC.popitem(last=False)
    return r
def _search(q, owner, k=4, mode="hybrid", rerank=True):
    """mode: bm25 | dense | hybrid (BM25 + dense, RRF) ; hybrid then cross-encoder rerank when loaded. Returns hits with the score breakdown."""
    C = _load(); rows = C["rows"]
    if not rows or not q.strip(): return []
    mk = (C["ver"], owner); mask = _MASK.get(mk)
    if mask is None: _MASK.clear(); mask = _MASK[mk] = np.array([r["owner"] in (None, owner) for r in rows])
    bm = C["bm25"].score(tok(q)); dn = None; used = []
    if mode in ("dense", "hybrid"):
        try:
            mat = _embeddings(); qv = gpu.embed([q]) if mat is not None else None
            if mat is not None and qv is not None: dn = mat @ qv[0]
        except Exception: dn = None
    if mode == "dense" and dn is None: mode = "bm25"
    lists = []
    if mode in ("bm25", "hybrid"): lists.append(("bm25", _rank(bm, mask))); used.append("BM25")
    if dn is not None and mode in ("dense", "hybrid"): lists.append(("dense", _rank(dn + 1, mask))); used.append("dense")
    rrf = collections.defaultdict(float); pos = collections.defaultdict(dict)
    for nm, lst in lists:
        for r, i in enumerate(lst): rrf[i] += 1 / (60 + r + 1); pos[i][nm] = r + 1
    order = sorted(rrf, key=lambda i: -rrf[i])[:8]; rr = None
    if rerank and mode != "bm25" and order: rr = gpu.rerank(q, [rows[i]["text"] for i in order])
    if rr is not None:
        used.append("rerank"); ix = np.argsort(-rr); z = 1 / (1 + np.exp(-rr[ix])); order = [order[j] for j in ix]; sc = {i: float(zz) for i, zz in zip(order, z)}
    else:
        top = max(rrf.values()) if rrf else 1; sc = {i: rrf[i] / top for i in order}
        if mode == "bm25": mx = float(bm[mask].max()) or 1.0; sc = {i: float(bm[i]) / mx for i in order}
        elif mode == "dense" and dn is not None: sc = {i: float(dn[i]) for i in order}
    out = []
    for i in order[: max(k, 1)]:
        if bm[i] <= 0 and (dn is None or dn[i] < 0.3): continue
        r = rows[i]; why = " · ".join([f"{n} #{pos[i][n]}" for n in ("bm25", "dense") if n in pos[i]] + ([f"rerank {sc[i]:.2f}"] if rr is not None else []))
        out.append({"title": r["title"], "source": r["source"], "text": r["text"], "score": round(sc[i], 3), "why": why, "via": "+".join(used), "owner": r["owner"], "id": r["id"]})
    return out

_LH = {"t": 0, "b": None, "meta": None}
def live_hits(q, k=4):
    """BM25 over the live operational records: open tickets and (Franchise) customer feedback. Fresh on every call."""
    import live_engine as LE
    import time as _t
    if _LH["b"] is not None and _t.time() - _LH["t"] < 20:
        s = _LH["b"].score(tok(q)); o = np.argsort(-s)[:k]; meta = _LH["meta"]
        return [{"title": meta[i][0], "source": "live data", "text": meta[i][1], "score": round(float(s[i]), 2), "why": "BM25 on live records", "via": "live"} for i in o if s[i] > 0]
    docs_, meta = [], []
    try:
        for r in LE.conn().execute("SELECT action_id, type, target, detail, severity, status FROM agent_actions ORDER BY action_id DESC LIMIT 600").fetchall():
            docs_.append(f"{r[1]} {r[2]} {r[3]} {r[4]}"); meta.append((f"Ticket #{r[0]} · {r[1]} · {r[2]} · {r[4]} · {r[5]}", f"{r[3]}"))
        if os.environ.get("APP_KIND", "Franchise") == "Franchise":
            for r in LE.conn().execute("SELECT outlet_id, rating, comment FROM feedback ORDER BY rowid DESC LIMIT 800").fetchall():
                docs_.append(f"{r[0]} {r[2]}"); meta.append((f"Review · {r[0]} · {r[1]}★", str(r[2])))
    except Exception: return []
    if not docs_: return []
    b = BM25([tok(d) for d in docs_]); _LH.update(t=_t.time(), b=b, meta=meta); s = b.score(tok(q)); o = np.argsort(-s)[:k]
    return [{"title": meta[i][0], "source": "live data", "text": meta[i][1], "score": round(float(s[i]), 2), "why": "BM25 on live records", "via": "live"} for i in o if s[i] > 0]

def export_json(owner):
    C = _load(); mine = [r for r in C["rows"] if r["owner"] == owner]
    return {"format": "kb-v1", "emb_model": gpu.S["models"]["embed"]["name"], "chunks": [{"title": r["title"], "source": r["source"], "text": r["text"], "emb": base64.b64encode(r["emb"]).decode() if r["emb"] else None} for r in mine]}
def import_json(data, owner):
    if not isinstance(data, dict) or data.get("format") != "kb-v1": raise ValueError("not a kb-v1 export")
    same = data.get("emb_model") == gpu.S["models"]["embed"]["name"]; n = 0; c = _db()
    try:
        for ch in data.get("chunks", [])[:20000]:
            e = base64.b64decode(ch["emb"]) if same and ch.get("emb") else None
            c.execute("INSERT INTO kb_chunks(title,source,owner,text,emb,builtin) VALUES(?,?,?,?,?,0)", (str(ch["title"])[:200], str(ch.get("source", "imported"))[:200], owner, str(ch["text"])[:4000], e)); n += 1
        c.commit()
    finally: c.close()
    _VER[0] += 1; return n, same

def evaluate(kind, owner="eval"):
    """recall@1 / recall@3 / MRR on the built-in question set, for each retrieval mode, so the benefit of hybrid + rerank is measured, not assumed."""
    import kb_pack
    pairs = kb_pack.EVAL[kind]; rows = []
    for name, mode, rr in (("BM25 only", "bm25", False), ("Dense only", "dense", False), ("Hybrid (BM25+dense, RRF)", "hybrid", False), ("Hybrid + cross-encoder rerank", "hybrid", True)):
        r1 = r3 = mrr = 0; ok = True
        for q, want in pairs:
            hits = search(q, owner, 5, mode, rr)
            if (mode in ("dense", "hybrid") and "dense" not in (hits[0]["via"] if hits else "dense")) or (rr and hits and "rerank" not in hits[0]["via"]): ok = False
            titles = []
            for h in hits:
                if h["title"] not in titles: titles.append(h["title"])
            if want in titles: p = titles.index(want) + 1; r1 += p == 1; r3 += p <= 3; mrr += 1 / p
        n = len(pairs); rows.append({"method": name, "recall@1": round(r1 / n, 2) if ok else None, "recall@3": round(r3 / n, 2) if ok else None, "MRR": round(mrr / n, 2) if ok else None, "note": "" if ok else "model not loaded yet"})
    return rows


In [ ]:
%%writefile /content/app/reports.py
"""Reports: print-ready PDFs (reportlab + matplotlib) and CSV exports built from the live database and the agents' latest output.
Every report has a cover band, KPI tiles, charts, tables, the Intelligence Engine narrative and a prioritised action list."""
import io, os, json, datetime
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether
from blocks import Q
import entities as EN
TEAL, BLUE, AMBER, RED, INK, MUT = "#0f9f8f", "#2f6fdb", "#d99a00", "#d64545", "#0f172a", "#64748b"
ss = getSampleStyleSheet()
ST = {"h1": ParagraphStyle("h1", parent=ss["Heading1"], fontSize=15, textColor=colors.HexColor(INK), spaceBefore=10, spaceAfter=6), "p": ParagraphStyle("p", parent=ss["Normal"], fontSize=9.5, leading=13.5, textColor=colors.HexColor("#1e293b")),
      "s": ParagraphStyle("s", parent=ss["Normal"], fontSize=8, textColor=colors.HexColor(MUT)), "c": ParagraphStyle("c", parent=ss["Normal"], fontSize=7.8, leading=9.5)}

def esc(t): return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return {"score": None, "risk": "-", "narrative": "", "ev": {}, "ts": ""}
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev, "ts": d.ts[0]}
def money(v): return f"{v:,.0f}"

# ---------------------------------------------------------------- block helpers (each returns reportlab flowables)
def H(t): return [Paragraph(esc(t), ST["h1"])]
def P(t, style="p"): return [Paragraph(esc(t).replace("\n", "<br/>"), ST[style])]
def KPI(items):
    cells = [[Paragraph(f'<font size="7.5" color="{MUT}">{esc(l).upper()}</font><br/><font size="15" color="{INK}"><b>{esc(v)}</b></font>' + (f'<br/><font size="7.5" color="{MUT}">{esc(s)}</font>' if s else ""), ST["p"]) for l, v, s in items]]
    t = Table(cells, colWidths=[(A4[0] - 36 * mm) / len(items)] * len(items)); t.setStyle(TableStyle([("BOX", (0, 0), (-1, -1), .6, colors.HexColor("#cbd5e1")), ("INNERGRID", (0, 0), (-1, -1), .6, colors.HexColor("#e2e8f0")), ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#f8fafc")), ("TOPPADDING", (0, 0), (-1, -1), 7), ("BOTTOMPADDING", (0, 0), (-1, -1), 7), ("LEFTPADDING", (0, 0), (-1, -1), 9)])); return [t, Spacer(1, 8)]
def TBL(df, cols=None, widths=None, maxrows=25, tone_col=None):
    if df is None or not len(df): return P("No records.", "s")
    cols = cols or list(df.columns); df = df.head(maxrows)
    data = [[Paragraph(f"<b>{esc(c).replace('_', ' ')}</b>", ST["c"]) for c in cols]] + [[Paragraph(esc(f"{v:,.2f}" if isinstance(v, float) else v), ST["c"]) for v in r] for r in df[cols].itertuples(index=False)]
    w = widths or [(A4[0] - 36 * mm) / len(cols)] * len(cols); t = Table(data, colWidths=w, repeatRows=1)
    sty = [("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e2e8f0")), ("GRID", (0, 0), (-1, -1), .4, colors.HexColor("#cbd5e1")), ("VALIGN", (0, 0), (-1, -1), "TOP"), ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3)]
    for i in range(1, len(data)):
        if i % 2 == 0: sty.append(("BACKGROUND", (0, i), (-1, i), colors.HexColor("#f8fafc")))
    if tone_col in cols:
        j = cols.index(tone_col)
        for i, v in enumerate(df[tone_col].tolist(), 1):
            col = {"Critical": RED, "High": RED, "Fail": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE}.get(str(v))
            if col: sty.append(("TEXTCOLOR", (j, i), (j, i), colors.HexColor(col)))
    t.setStyle(TableStyle(sty)); return [t, Spacer(1, 8)]
def FIG(fn, w=170, h=62):
    fig, ax = plt.subplots(figsize=(w / 25.4, h / 25.4), dpi=170); fn(ax); ax.spines[["top", "right"]].set_visible(False); ax.tick_params(labelsize=7); ax.title.set_fontsize(8.5)
    for s in ax.spines.values(): s.set_color("#94a3b8")
    fig.tight_layout(); b = io.BytesIO(); fig.savefig(b, format="png"); plt.close(fig); b.seek(0); return [Image(b, width=w * mm, height=h * mm), Spacer(1, 6)]
def barh(labels, vals, colors_=None, title="", fmt="{:.0f}", xmax=None):
    def f(ax):
        y = np.arange(len(labels)); ax.barh(y, vals, color=colors_ or TEAL, height=.62); ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis(); ax.set_title(title, loc="left", fontweight="bold")
        for i, v in enumerate(vals): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=6.5, color=INK)
        if xmax: ax.set_xlim(0, xmax)
    return f
def colbar(labels, vals, colors_=None, title=""):
    def f(ax): ax.bar(labels, vals, color=colors_ or BLUE, width=.6); ax.set_title(title, loc="left", fontweight="bold"); ax.grid(axis="y", alpha=.25)
    return f
def linef(x, ys, title="", names=None, cols=None):
    def f(ax):
        for i, y in enumerate(ys): ax.plot(x, y, color=(cols or [TEAL, BLUE, AMBER])[i], lw=1.8, label=(names or [None] * 9)[i])
        ax.set_title(title, loc="left", fontweight="bold"); ax.grid(alpha=.25)
        if names: ax.legend(fontsize=6.5, frameon=False)
        n = len(x); ax.set_xticks(list(range(0, n, max(1, n // 8))))
    return f
def hcol(h): return RED if h < 60 else AMBER if h < 80 else TEAL
def actions(target_like=None, types=None, limit=15, scope=None):
    sql = "SELECT action_id id, severity, type, target, detail, owner, assignee, due, status FROM agent_actions WHERE status IN ('open','proposed')"; p = []
    if scope: sql += " AND target LIKE ?"; p.append(scope + "%")
    if target_like: sql += " AND target LIKE ?"; p.append(target_like + "%")
    if types: sql += " AND type IN (%s)" % ",".join("?" * len(types)); p += list(types)
    sql += " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT %d" % limit; df = Q(sql, tuple(p))
    if len(df): df["due"] = df.due.fillna("").str[:10]; df["owner"] = df.owner.fillna("-")
    return df
def ACTIONS(df, title="Prioritised action list (open tickets)"):
    out = H(title)
    if df is None or not len(df): return out + P("No open actions. Everything raised by the agents is resolved.", "s")
    return out + TBL(df, ["id", "severity", "type", "target", "detail", "owner", "due"], [14 * mm, 17 * mm, 28 * mm, 20 * mm, 60 * mm, 20 * mm, 15 * mm], 30, "severity")

# ---------------------------------------------------------------- Franchise reports
def fr_exec(c):
    d = decision(); ev = d["ev"]; hl = ev.get("health", {}); out = []
    o = Q("SELECT outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction csat, staff_headcount hc FROM outlets"); o["health"] = o.outlet_id.map(hl).astype(float).round(1)
    t = Q("SELECT severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY severity").set_index("severity").n.to_dict(); sc = d["score"]
    out += KPI([("Network health", f"{sc:.1f}" if sc is not None else "-", d["risk"] + " risk"), ("Outlets", len(o), f"{int((o.health < 60).sum())} below 60"), ("Revenue", money(o.revenue.sum()), f"margin {100 * (1 - o.operating_costs.sum() / max(o.revenue.sum(), 1)):.0f}%"), ("Open actions", sum(t.values()), f"{t.get('Critical', 0)} critical")])
    out += H("Intelligence Engine summary") + P(d["narrative"] or "No decision yet.")
    ob = o.dropna(subset=["health"]); b = ob.sort_values("health").head(10)
    out += H("Outlet benchmark: ten weakest outlets") + FIG(barh(b.outlet_id + " " + b.location.str[:14], b.health.values, [hcol(x) for x in b.health], "Health score (0-100)", "{:.1f}", 100), h=66) + TBL(b, ["outlet_id", "outlet_name", "location", "health", "csat", "hc"], maxrows=10)
    top = ob.sort_values("health", ascending=False).head(5); out += H("Growth opportunities") + P("Outlets that are the network's best operators. Their routines are the playbook for the weakest outlets above.") + TBL(top, ["outlet_id", "outlet_name", "location", "health", "csat", "revenue"], maxrows=5)
    mk = Q("SELECT channel, ROUND(AVG(actual_roi),1) roi, COUNT(*) n FROM marketing GROUP BY channel ORDER BY roi DESC")
    if len(mk): out += FIG(colbar(mk.channel, mk.roi, TEAL, "Average campaign ROI by channel (%)"), h=50) + P(f"Best channel: {mk.channel[0]} ({mk.roi[0]}% ROI). Shift budget from {mk.channel.iloc[-1]} ({mk.roi.iloc[-1]}%).")
    ag = Q("SELECT agent, risk, summary FROM agent_runs WHERE run_id IN (SELECT MAX(run_id) FROM agent_runs GROUP BY agent)"); out += H("What each agent found") + TBL(ag, ["agent", "risk", "summary"], [24 * mm, 18 * mm, 124 * mm], 12, "risk")
    return out + ACTIONS(actions(limit=12, scope=c.get("scope")))
def fr_outlet(c, oid):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (oid,))
    if not len(o): return P("Unknown outlet.")
    r = o.iloc[0]; d = decision(); h = d["ev"].get("health", {}).get(oid); out = KPI([("Health", f"{h:.1f}" if h is not None else "-", None), ("Revenue", money(r.revenue), None), ("Satisfaction", f"{r.customer_satisfaction:.2f}", "of 5"), ("Headcount", int(r.staff_headcount), None)])
    out += P(f"{r.outlet_name} · {r.location} · tier {r.tier}", "s"); bd = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(oid)
        if v is not None: bd[a] = round(v, 1)
    if bd: out += H("Why this score") + FIG(barh(list(bd), list(bd.values()), [hcol(v) for v in bd.values()], "Agent sub-scores", "{:.0f}", 100), h=52)
    s = Q("SELECT substr(ts,12,5) t, amount FROM sales WHERE outlet_id=? ORDER BY sale_id DESC LIMIT 60", (oid,)).iloc[::-1]
    if len(s): out += FIG(linef(s.t.tolist(), [s.amount.tolist()], "Latest sales (per tick)"), h=45)
    inv = Q("SELECT sku_name, current_stock stock, weekly_demand wk, lead_time_days lead, ROUND(current_stock*7.0/MAX(weekly_demand,1),1) cover_days FROM inventory WHERE outlet_id=? ORDER BY cover_days LIMIT 8", (oid,)); out += H("Lowest stock cover") + TBL(inv)
    au = Q("SELECT audit_date, score, violations, category, status FROM audits WHERE outlet_id=? ORDER BY audit_date DESC LIMIT 6", (oid,)); out += H("Recent audits") + TBL(au)
    fb = Q("SELECT date, rating, comment FROM feedback WHERE outlet_id=? ORDER BY date DESC LIMIT 6", (oid,)); out += H("Recent customer feedback") + TBL(fb, widths=[22 * mm, 14 * mm, 136 * mm])
    return out + ACTIONS(actions(oid), f"Action plan for {oid}")
def fr_inventory(c):
    i = Q("SELECT outlet_id, sku_name, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead, ROUND(stockout_risk_prob*100,0) risk_pct, ROUND(current_stock*7.0/MAX(weekly_demand,1),1) cover_days FROM inventory")
    if c.get("scope"): i = i[i.outlet_id == c["scope"]]
    short = i[i.stock <= i.thr]; out = KPI([("SKUs tracked", len(i), None), ("Below reorder point", len(short), f"{100 * len(short) / max(len(i), 1):.0f}%"), ("Avg cover", f"{i.cover_days.mean():.1f} d", None), ("Stock-out risk > 50%", int((i.risk_pct > 50).sum()), None)])
    cat = i.groupby("category").apply(lambda g: (g.stock <= g.thr).mean() * 100).sort_values(); out += FIG(barh(cat.index.tolist(), cat.values, AMBER, "Share of SKUs below reorder point, by category (%)", "{:.0f}", 100), h=55)
    out += H("Most urgent SKUs") + TBL(short.sort_values("cover_days").head(25), ["outlet_id", "sku_name", "stock", "thr", "cover_days", "lead", "risk_pct"])
    po = Q("SELECT po_id, outlet_id, sku_name, qty, status, eta_ts FROM purchase_orders ORDER BY ts DESC LIMIT 15"); out += H("Replenishment orders raised by the Inventory agent") + TBL(po)
    return out + ACTIONS(actions(types=("reorder", "pause_promo"), limit=15, scope=c.get("scope")), "Replenishment actions")
def fr_workforce(c):
    s = Q("SELECT staff_id, outlet_id, name, role, overtime_hrs, job_satisfaction, tenure_years, ROUND(predicted_attrition_prob*100,0) risk_pct FROM staff")
    if c.get("scope"): s = s[s.outlet_id == c["scope"]]
    hi = s[s.risk_pct >= 60]; out = KPI([("Employees", len(s), None), ("High attrition risk", len(hi), f"{100 * len(hi) / max(len(s), 1):.0f}%"), ("Avg overtime", f"{s.overtime_hrs.mean():.1f} h", None), ("Avg satisfaction", f"{s.job_satisfaction.mean():.1f}", "of 4")])
    out += FIG(lambda ax: (ax.hist(s.risk_pct, bins=20, color=BLUE), ax.set_title("Attrition risk distribution (%)", loc="left", fontweight="bold")), h=50)
    g = s.groupby("outlet_id").risk_pct.mean().sort_values(ascending=False).head(10); out += FIG(barh(g.index.tolist(), g.values, AMBER, "Outlets with the highest average attrition risk (%)", "{:.0f}", 100), h=58)
    out += H("Highest-risk employees") + TBL(hi.sort_values("risk_pct", ascending=False), ["staff_id", "outlet_id", "name", "role", "overtime_hrs", "job_satisfaction", "risk_pct"], maxrows=25)
    return out + ACTIONS(actions(types=("staffing_shortage", "retention_plan"), limit=15, scope=c.get("scope")), "Workforce recommendations")
def fr_marketing(c):
    m = Q("SELECT outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions FROM marketing")
    if c.get("scope"): m = m[m.outlet_id == c["scope"]]
    m["cost_per_conv"] = (m.budget / m.conversions.clip(lower=1)).round(1); ch = m.groupby("channel").agg(budget=("budget", "sum"), roi=("actual_roi", "mean"), conv=("conversions", "sum")).sort_values("roi", ascending=False).reset_index()
    out = KPI([("Campaigns", len(m), None), ("Total budget", money(m.budget.sum()), None), ("Avg ROI", f"{m.actual_roi.mean():.1f}%", None), ("Conversions", money(m.conversions.sum()), None)])
    out += FIG(colbar(ch.channel, ch.roi, [TEAL if v >= 0 else RED for v in ch.roi], "ROI by channel (%)"), h=52) + TBL(ch.round(1)); w = m.sort_values("actual_roi").head(12)
    out += H("Weakest campaigns: recommend pausing or reworking") + TBL(w, ["outlet_id", "campaign_name", "channel", "budget", "actual_roi", "cost_per_conv"]); b = m.sort_values("actual_roi", ascending=False).head(8); out += H("Best campaigns: recommend scaling") + TBL(b, ["outlet_id", "campaign_name", "channel", "budget", "actual_roi", "cost_per_conv"])
    return out + ACTIONS(actions(types=("pause_promo", "reallocate_budget", "win_back_campaign"), limit=12, scope=c.get("scope")), "Marketing actions")
def fr_audit(c):
    a = Q("SELECT outlet_id, audit_date, score, violations, category, status, notes FROM audits")
    if c.get("scope"): a = a[a.outlet_id == c["scope"]]
    last = a.sort_values("audit_date").groupby("outlet_id").tail(1); out = KPI([("Audits on file", len(a), None), ("Latest avg score", f"{last.score.mean():.1f}", None), ("Failing outlets", int((last.score < 60).sum()), None), ("Open violations", int(last.violations.sum()), None)])
    cat = a.groupby("category").score.mean().sort_values(); out += FIG(barh(cat.index.tolist(), cat.values, [hcol(v) for v in cat.values], "Average score by audit category", "{:.0f}", 100), h=55)
    out += H("Lowest-scoring outlets (latest audit)") + TBL(last.sort_values("score").head(15), ["outlet_id", "audit_date", "score", "violations", "category", "status", "notes"], [18 * mm, 20 * mm, 12 * mm, 16 * mm, 24 * mm, 17 * mm, 59 * mm])
    ca = Q("SELECT action_id id, severity, target, detail, status, owner, due FROM agent_actions WHERE type='corrective_action' ORDER BY action_id DESC LIMIT 30"); out += H("Corrective actions tracker") + TBL(ca, ["id", "severity", "target", "detail", "status", "owner", "due"], [14 * mm, 17 * mm, 20 * mm, 68 * mm, 17 * mm, 20 * mm, 15 * mm], 30, "severity")
    return out
def fr_plan(c):
    df = actions(limit=400, scope=c.get("scope")); out = KPI([("Open actions", len(df), None), ("Critical", int((df.severity == "Critical").sum()) if len(df) else 0, None), ("Awaiting approval", int((df.status == "proposed").sum()) if len(df) else 0, None), ("Outlets affected", df.target.nunique() if len(df) else 0, None)])
    out += P("A franchise action plan: every open item the agents raised, grouped by outlet, most severe first, with the owner and due date. Items marked proposed need a human approval before the system executes them.")
    if not len(df): return out
    rank = df.groupby("target").apply(lambda g: (g.severity == "Critical").sum() * 10 + len(g)).sort_values(ascending=False).head(12)
    for t in rank.index:
        g = df[df.target == t]; out.append(KeepTogether([Paragraph(f"<b>{esc(t)}</b> · {len(g)} action(s)", ST["p"])] + TBL(g, ["id", "severity", "type", "detail", "owner", "due"], [12 * mm, 17 * mm, 28 * mm, 80 * mm, 20 * mm, 17 * mm], 10, "severity")))
    return out

# ---------------------------------------------------------------- Freight reports
def fq_exec(c):
    d = decision(); q = Q("SELECT quote_id, customer_id, carrier, base_cost, final_price, margin_pct, status, created_at FROM freight_quotes"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status")
    t = Q("SELECT severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY severity").set_index("severity").n.to_dict(); appr = q[q.status == "Approved"]
    out = KPI([("Fleet health", f"{d['score']:.1f}" if d["score"] is not None else "-", d["risk"] + " risk"), ("Quotes", len(q), f"{len(appr)} approved"), ("Quoted value", money(q.final_price.sum()), "USD"), ("Avg margin", f"{q.margin_pct.mean():.1f}%" if len(q) else "-", None), ("Open actions", sum(t.values()), f"{t.get('Critical', 0)} critical")])
    out += H("Intelligence Engine summary") + P(d["narrative"] or "No decision yet.")
    if len(sh): out += FIG(colbar(sh.status, sh.n, BLUE, "Shipments by status"), h=48)
    if len(q): s = q.status.value_counts(); out += FIG(colbar(s.index, s.values, [TEAL if i == "Approved" else AMBER if "Pending" in i else RED if i == "On Hold" else "#94a3b8" for i in s.index], "Quotes by status"), h=48)
    cg = Q("SELECT name, ROUND(on_time_pct,1) on_time, ROUND(avg_cost_index,2) cost_index, risk_level FROM carriers ORDER BY on_time_pct DESC LIMIT 8"); out += H("Carrier scorecard (top 8 on-time)") + TBL(cg, tone_col="risk_level")
    ag = Q("SELECT agent, risk, summary FROM agent_runs WHERE run_id IN (SELECT MAX(run_id) FROM agent_runs GROUP BY agent)"); out += H("What each agent found") + TBL(ag, ["agent", "risk", "summary"], [28 * mm, 18 * mm, 120 * mm], 12, "risk")
    return out + ACTIONS(actions(limit=12))
def fq_quotation(c, qid):
    q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (qid,))
    if not len(q): return P("Quote not found.")
    r = q.iloc[0]; s = Q("SELECT origin_port, dest_port, eta, weight_kg, hs_code, carrier FROM shipments WHERE shipment_id=?", (r.shipment_id,)); cu = Q("SELECT name, industry FROM customers WHERE customer_id=?", (r.customer_id,)); nm = cu.name[0] if len(cu) else r.customer_id
    out = KPI([("Quote", r.quote_id, r.status), ("Total price", f"USD {r.final_price:,.0f}", "all-in"), ("Carrier", r.carrier or (s.carrier[0] if len(s) and "carrier" in s else "To be assigned"), None)]) + P(f"Prepared for {nm} on {str(r.created_at)[:10]}. Valid for 7 days.", "s")
    if len(s): x = s.iloc[0]; out += H("Shipment") + TBL(pd.DataFrame([{"origin": x.origin_port, "destination": x.dest_port, "ETA": str(x.eta)[:10], "weight_kg": x.weight_kg, "HS code": x.hs_code}]))
    rows = [("Base freight", r.base_cost), ("Insurance", r.insurance), ("Customs handling", r.customs_fee), ("Fuel surcharge", r.fuel_surcharge)]; sub = sum(v for _, v in rows); rows += [("Brokerage margin", r.final_price - sub), ("TOTAL", r.final_price)]
    out += H("Price breakdown (USD)") + TBL(pd.DataFrame([{"item": a, "amount": f"{b:,.2f}"} for a, b in rows]), widths=[110 * mm, 60 * mm]) + FIG(lambda ax: (ax.pie([max(v, 0) for _, v in rows[:-1]], labels=[a for a, _ in rows[:-1]], colors=[BLUE, TEAL, AMBER, "#8b5cf6", "#94a3b8"], autopct="%1.0f%%", textprops={"fontsize": 7}), ax.set_title("Where the price goes", loc="left", fontweight="bold")), h=62)
    return out + P("Terms: rates are valid for 7 days and subject to space and equipment availability. Weather or customs events can change transit time; the platform re-plans automatically and will notify you. Customs documents listed in the shipment record must be provided before sailing.", "s")
def fq_route(c):
    p = Q("SELECT port_name, country, ROUND(congestion_index,2) congestion, ROUND(avg_dwell_days,1) dwell_days, region FROM ports ORDER BY congestion_index DESC"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status"); v = Q("SELECT COUNT(*) n, ROUND(AVG(speed_kn),1) sp FROM vessels")
    out = KPI([("Ports tracked", len(p), None), ("Congested (>=4)", int((p.congestion >= 4).sum()), None), ("Vessels", int(v.n[0]), f"avg {v.sp[0]} kn"), ("Avg dwell", f"{p.dwell_days.mean():.1f} d", None)])
    t = p.head(12); out += FIG(barh(t.port_name.tolist(), t.congestion.values, [RED if x >= 4 else AMBER for x in t.congestion], "Most congested ports (index 0-5)", "{:.2f}", 5), h=62) + TBL(t, ["port_name", "country", "region", "congestion", "dwell_days"])
    cg = Q("SELECT name, ROUND(on_time_pct,1) on_time, ROUND(avg_cost_index,2) cost_index, rating, risk_level FROM carriers ORDER BY on_time_pct DESC LIMIT 12"); out += H("Carrier comparison") + TBL(cg, tone_col="risk_level")
    return out + ACTIONS(actions(types=("reroute", "congestion_notice"), limit=15), "Route actions")
def fq_risk(c):
    w = Q("SELECT port_name, current_severity sev, wind_speed, wave_height FROM weather_risks ORDER BY wind_speed DESC"); s = Q("SELECT s.shipment_id, s.origin_port, s.dest_port, s.carrier, ROUND(s.predicted_delay_risk*100,0) delay_risk_pct, s.status FROM shipments s ORDER BY s.predicted_delay_risk DESC LIMIT 20")
    lv = w.sev.value_counts(); out = KPI([("Ports monitored", len(w), None), ("High severity", int(lv.get("High", 0)), None), ("Max wind", f"{w.wind_speed.max():.0f} km/h" if len(w) else "-", None), ("High delay-risk shipments", int((s.delay_risk_pct >= 60).sum()), None)])
    t = w.head(12); out += FIG(barh(t.port_name.tolist(), t.wind_speed.values, [RED if x == "High" else AMBER if x == "Medium" else TEAL for x in t.sev], "Strongest port winds (km/h)", "{:.0f}"), h=62)
    out += H("Shipments with the highest predicted delay risk") + TBL(s) + ACTIONS(actions(types=("reroute",), limit=15), "Risk alerts and re-routes")
    return out
def fq_customs(c):
    sh = Q("SELECT shipment_id, origin_port, dest_port, hs_code, status FROM shipments"); hold = sh[sh.status.str.contains("Customs|Hold", case=False, na=False)]; cr = pd.DataFrame()
    try: cr = Q("SELECT * FROM customs_rules LIMIT 20")
    except Exception: pass
    out = KPI([("Shipments", len(sh), None), ("On customs hold", len(hold), f"{100 * len(hold) / max(len(sh), 1):.0f}%"), ("Delivered", int((sh.status == "Delivered").sum()) if len(sh) else 0, None)]) + H("Shipments on customs hold") + TBL(hold, maxrows=25)
    if len(cr): out += H("Customs rules in force") + TBL(cr, maxrows=20)
    return out + ACTIONS(actions(types=("customs_docs",), limit=20), "Customs readiness actions")
def fq_margin(c):
    q = Q("SELECT quote_id, customer_id, carrier, base_cost, final_price, margin_pct, status FROM freight_quotes"); out = KPI([("Quotes", len(q), None), ("Avg margin", f"{q.margin_pct.mean():.1f}%" if len(q) else "-", None), ("Gross profit", money((q.final_price - q.base_cost).sum()), "USD"), ("Low margin (<10%)", int((q.margin_pct < 10).sum()), None)])
    if len(q):
        out += FIG(lambda ax: (ax.scatter(q.base_cost, q.final_price, s=14, c=[RED if m < 10 else TEAL for m in q.margin_pct]), ax.set_xlabel("base cost", fontsize=7), ax.set_ylabel("price", fontsize=7), ax.set_title("Price vs base cost (red = margin under 10%)", loc="left", fontweight="bold")), h=62)
        g = q.assign(profit=q.final_price - q.base_cost).groupby("customer_id").agg(quotes=("quote_id", "count"), profit=("profit", "sum"), margin=("margin_pct", "mean")).sort_values("margin").head(10).round(1).reset_index(); out += H("Lowest-margin customers") + TBL(g)
        g = q.groupby("carrier").agg(quotes=("quote_id", "count"), margin=("margin_pct", "mean")).round(1).reset_index().sort_values("margin"); out += H("Brokerage performance by carrier") + TBL(g)
    return out + ACTIONS(actions(types=("margin_review", "quote_approval", "quote_hold", "quote_anomaly"), limit=15), "Pricing actions")

REPORTS = {
    "Franchise": [("fr_exec", "Executive franchise report", "Network health, outlet benchmark, growth opportunities, agent findings, action list.", fr_exec, None), ("fr_outlet", "Outlet performance report", "One outlet: score breakdown, sales, stock cover, audits, feedback, action plan.", fr_outlet, "outlet"),
                  ("fr_inventory", "Inventory report", "Stock shortages, cover days, replenishment orders.", fr_inventory, None), ("fr_workforce", "Workforce report", "Attrition risk, overtime, recommendations.", fr_workforce, None), ("fr_marketing", "Marketing ROI report", "ROI by channel, best and weakest campaigns.", fr_marketing, None),
                  ("fr_audit", "Audit & compliance report", "Scores, violations, corrective-action tracker.", fr_audit, None), ("fr_plan", "Franchise action plan", "Every open item grouped by outlet with owners and due dates.", fr_plan, None)],
    "Freight": [("fq_exec", "Executive brokerage report", "Fleet health, quotes, carriers, agent findings, action list.", fq_exec, None), ("fq_quotation", "Customer quotation (PDF)", "Customer-facing quote with price breakdown and terms.", fq_quotation, "quote"),
                ("fq_route", "Route intelligence report", "Port congestion, carrier comparison, route actions.", fq_route, None), ("fq_risk", "Shipment risk report", "Weather exposure and highest delay-risk shipments.", fq_risk, None),
                ("fq_customs", "Customs readiness report", "Shipments on hold, rules in force, document actions.", fq_customs, None), ("fq_margin", "Margin & brokerage performance", "Profit, low-margin quotes and customers, carrier margin.", fq_margin, None)]}

def build_pdf(kind, rid, c, arg=None):
    spec = next((r for r in REPORTS[kind] if r[0] == rid), None)
    if not spec: raise KeyError(rid)
    name, title = spec[1], spec[1]; ctx = c or {}
    try: body = spec[3](ctx, arg) if spec[4] else spec[3](ctx)
    except Exception as e: body = P(f"This section could not be built: {e}")
    buf = io.BytesIO(); W, Hh = A4; plat = "FranchiseOps AI" if kind == "Franchise" else "FreightQuote AI"; gen = datetime.datetime.now().strftime("%d %b %Y %H:%M")
    def deco(cv, doc):
        cv.saveState(); cv.setFillColor(colors.HexColor("#0b1220")); cv.rect(0, Hh - 24 * mm, W, 24 * mm, fill=1, stroke=0); cv.setFillColor(colors.HexColor("#2dd4bf")); cv.rect(0, Hh - 24 * mm, 6 * mm, 24 * mm, fill=1, stroke=0)
        cv.setFillColor(colors.white); cv.setFont("Helvetica-Bold", 15); cv.drawString(14 * mm, Hh - 12 * mm, title + (f" · {arg}" if arg else "")); cv.setFont("Helvetica", 8.5); cv.setFillColor(colors.HexColor("#9fb0c3")); cv.drawString(14 * mm, Hh - 18 * mm, f"{plat} · generated {gen} by the agent network" + (f" · scope {ctx['scope']}" if ctx.get("scope") else ""))
        cv.setFillColor(colors.HexColor(MUT)); cv.setFont("Helvetica", 7.5); cv.drawString(18 * mm, 10 * mm, f"{plat} · confidential"); cv.drawRightString(W - 18 * mm, 10 * mm, f"Page {doc.page}"); cv.restoreState()
    doc = BaseDocTemplate(buf, pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm, topMargin=30 * mm, bottomMargin=16 * mm, title=title, author=plat); doc.addPageTemplates([PageTemplate(id="p", frames=[Frame(18 * mm, 16 * mm, W - 36 * mm, Hh - 46 * mm, id="f", leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)], onPage=deco)])
    doc.build([Spacer(1, 2)] + body); safe = (rid + ("_" + str(arg) if arg else "") + "_" + datetime.date.today().isoformat()).replace(" ", "_"); return safe + ".pdf", buf.getvalue()

EXPORTS = {"Franchise": {"outlets": "SELECT * FROM outlets", "staff": "SELECT * FROM staff", "inventory": "SELECT * FROM inventory", "marketing": "SELECT * FROM marketing", "audits": "SELECT * FROM audits", "feedback": "SELECT * FROM feedback", "sales_latest": "SELECT * FROM sales ORDER BY sale_id DESC LIMIT 20000", "tickets": "SELECT * FROM agent_actions ORDER BY action_id DESC"},
           "Freight": {"quotes": "SELECT * FROM freight_quotes", "shipments": "SELECT * FROM shipments", "ports": "SELECT * FROM ports", "carriers": "SELECT * FROM carriers", "customers": "SELECT * FROM customers", "vessels": "SELECT * FROM vessels", "weather": "SELECT * FROM weather_risks", "tickets": "SELECT * FROM agent_actions ORDER BY action_id DESC"}}
def export_csv(kind, table, scope=None):
    sql = EXPORTS[kind][table]; df = Q(sql)
    if scope and "outlet_id" in df.columns: df = df[df.outlet_id == scope]
    if scope and table == "tickets": df = df[df.target.astype(str).str.startswith(scope)]
    return df.to_csv(index=False).encode()


In [ ]:
%%writefile /content/app/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))

In [ ]:
%%writefile /content/app/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


In [ ]:
%%writefile /content/app/selftest.py
"""Smoke test for a running app:  python selftest.py http://localhost:8000 admin@infosys.com 'password'
Logs in, opens every page the role can see, asks the Copilot exact-data questions and checks the answers against the database."""
import sys, json, httpx
def main(base, login, pw):
    c = httpx.Client(base_url=base, timeout=120); bad = []
    r = c.post("/api/login", json={"login": login, "password": pw}); assert r.status_code == 200, f"login failed: {r.text}"
    assert c.get("/api/ready").status_code == 200, "not ready"
    me = c.get("/api/me").json(); pages = [m["id"] for m in me.get("menu", [])]
    for p in pages:
        rr = c.get(f"/api/page/{p}")
        if rr.status_code != 200: bad.append((p, rr.status_code))
    kind = me.get("kind", ""); qs = ["how many outlets are there", "which outlet has the highest revenue"] if kind == "Franchise" else ["how many shipments are there", "which port has the highest congestion"]
    for q in qs:
        t = c.post("/api/copilot", json={"q": q, "lang": "English"}); fin = [json.loads(l[6:]) for l in t.text.splitlines() if l.startswith("data: ")]
        fin = [e for e in fin if e["ev"] == "final"]
        if not fin or not any(ch.isdigit() for ch in fin[0]["text"]) or not fin[0].get("sql"): bad.append(("copilot", q))
    print(f"{len(pages)} pages, {len(qs)} copilot questions, problems: {bad or 'none'}"); return 1 if bad else 0
if __name__ == "__main__": sys.exit(main(*sys.argv[1:4]))


In [ ]:
%%writefile /content/app/server.py
"""Web server: FastAPI JSON API + server-sent events + the single-page front end. Models live on the GPU in this process; the simulator and agent cycles run in worker.py."""
import os as _os, time as _time
_os.environ["TZ"] = "Asia/Kolkata"
try: _time.tzset()
except Exception: pass          # all times in this app are India Standard Time (IST)
import sqlite3, os, sys, io, re, csv, json, time, asyncio, subprocess, threading, importlib, datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config
from config import KIND, DB_PATH
from fastapi import FastAPI, Request, Response, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, HTMLResponse, PlainTextResponse
from fastapi.staticfiles import StaticFiles
import pandas as pd
import numpy as np
import mailer, reports as RP
import privacy as PV, qa as QA, analyst as AN, live_engine as LE, live_rbac as RB, data_feeder as FD, authx, gpu, rag, entities as EN, blocks as B
import views_core as VC, views_misc as VM, persist as PS
dom = importlib.import_module("domain_franchise" if KIND == "Franchise" else "domain_freight")
V = importlib.import_module("views_franchise" if KIND == "Franchise" else "views_freight")
HERE = os.path.dirname(os.path.abspath(__file__)); STATIC = os.path.join(HERE, "static")
app = FastAPI(title=f"{V.NAME} Platform", docs_url=None, redoc_url=None, openapi_url=None) if os.environ.get("PUBLIC_DOCS") != "1" else FastAPI(title=f"{V.NAME} Platform"); _worker = None; _ready = {"db": False}
import hardening; hardening.install(app, KIND); PV.start()

@app.middleware("http")
async def sec(request, call_next):
    r = await call_next(request); r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "same-origin"; r.headers["Cache-Control"] = "no-store" if request.url.path.startswith("/api") else r.headers.get("Cache-Control", "public, max-age=300"); return r

def _boot():
    from db import init_db; from seed_data import seed_all
    PS.restore()                                                  # bring the database back from Google Drive after a runtime reset
    init_db(); seed_all(); LE.init_live_tables(); authx.init(); FD.init(); _ready["db"] = True; PS.start()
    try:
        _c = LE.conn(); LE._steps_ddl(_c); _c.commit()          # the approval table exists from the first start, not only after the first ticket
    except Exception as e: print("approval table:", repr(e), file=sys.stderr)
    threading.Thread(target=_warm_up, daemon=True).start()
    threading.Thread(target=lambda: rag._seed(), daemon=True).start()   # builds the knowledge base (built-in pack + your Drive chunks) right at start
def _warm_up():
    """Runs once the models are up: one dummy search, rerank, translation and generation, so the first real question is not the one that pays for CUDA kernels and caches."""
    try:
        for _ in range(150):
            m = gpu.S["models"]
            if all(m[k]["state"] in ("ready", "failed") for k in ("embed", "rerank", "nllb")) and m["llm"]["state"] in ("ready", "failed"): break
            time.sleep(4)
        try: rag.search("refund policy and approval limits", "warm", 3, "hybrid", True)
        except Exception: pass
        try: gpu.translate("How many outlets are open today?", "hin_Deva", "eng_Latn")
        except Exception: pass
        try:
            if gpu.llm_ready(): gpu.llm_text([{"role": "user", "content": "Say OK."}], max_new_tokens=4, temperature=0.0)
        except Exception: pass
        try: VC.decision()
        except Exception: pass
    except Exception as e: print("warm-up:", repr(e), file=sys.stderr)
@app.on_event("startup")
def startup():
    global _worker
    _boot(); gpu.start()
    if os.environ.get("NO_WORKER") != "1":
        env = {**os.environ, "APP_KIND": KIND}; _worker = subprocess.Popen([sys.executable, os.path.join(HERE, "worker.py")], env=env, cwd=HERE)
@app.on_event("shutdown")
def shutdown():
    try: PS.stop()
    except Exception: pass
    if _worker:
        try: _worker.terminate()
        except Exception: pass

# ------------------------------------------------------------------ auth + context
def user_of(request: Request):
    u = authx.decode(request.cookies.get("sid", ""))
    if not u: raise HTTPException(401, "not signed in")
    return u
def ctx(u, params=None):
    role = RB.norm(u["role"]) if u["role"] in RB.CAPS else u["role"]; scope = RB.scope_for(u["email"]) if role in RB.SCOPED else None
    return {"dom": dom, "kind": KIND, "role": u["role"], "email": u["email"], "user": u, "scope": scope, "params": params or {}}
def need(c, cap):
    if not RB.can(c["role"], cap): raise HTTPException(403, f"role {c['role']} cannot {cap}")
def menu_for(role): return [m for m in V.MENU if m[0] in V.ROLE_MENU.get(role, {"copilot"})]
def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def bad(msg): return JSONResponse({"ok": False, "msg": msg}, status_code=400)

@app.post("/api/login")
async def login(request: Request, response: Response):
    d = await request.json(); okk, msg, u = await asyncio.to_thread(authx.login, d.get("login"), d.get("password"))
    if not okk: return JSONResponse({"ok": False, "msg": msg}, status_code=401)
    LE.audit(u["email"], "login", "", ""); r = JSONResponse({"ok": True, "user": u}); r.set_cookie("sid", authx.token(u), httponly=True, samesite="lax", max_age=36000, secure=request.headers.get("x-forwarded-proto") == "https"); return r
@app.post("/api/register")
async def register(request: Request):
    d = await request.json(); okk, msg = await asyncio.to_thread(authx.register_verified, d.get("username"), d.get("email"), d.get("password"), d.get("role", "Customer"), d.get("sq") or authx.QUESTIONS[0], d.get("sa", ""), d.get("otp", "")); return {"ok": okk, "msg": msg}
@app.post("/api/otp")
async def otp(request: Request):
    d = await request.json(); purpose = d.get("purpose", "register")
    if purpose not in ("register", "reset"): return bad("bad purpose")
    okk, msg = await asyncio.to_thread(authx.send_otp, d.get("email"), purpose); return {"ok": okk, "msg": msg}
@app.get("/api/question")
def question(email: str): return {"q": authx.question_for(email)}
@app.post("/api/reset")
async def reset(request: Request):
    d = await request.json()
    if d.get("otp"): okk, msg = authx.reset_with_otp(d.get("email", ""), d["otp"], d.get("password", ""))
    else: okk, msg = authx.reset(d.get("email", ""), d.get("answer", ""), d.get("password", ""))
    return {"ok": okk, "msg": msg}
@app.post("/api/logout")
def logout(request: Request):
    u = authx.decode(request.cookies.get("sid", ""))
    if u:
        authx.revoke(u["email"]); LE.audit(u["email"], "logout", "", "")
    r = JSONResponse({"ok": True}); sec_ = request.headers.get("x-forwarded-proto") == "https"
    r.set_cookie("sid", "", max_age=0, expires=0, path="/", httponly=True, samesite="lax", secure=sec_); r.delete_cookie("sid", path="/"); return r
@app.get("/api/public")
def public():
    s = LE.get_state("boot", {}) if _ready["db"] else {}
    return {"kind": KIND, "name": V.NAME, "stage": (s or {}).get("stage", "starting"), "gpu": {"cuda": gpu.S["cuda"], "name": gpu.S["gpu"]}, "questions": authx.QUESTIONS, "email_enabled": mailer.configured(), "demo": ([{"email": authx.ADMIN_EMAIL_ID, "role": "Admin", "pw": "Admin@123"}] + [{"email": e, "role": r, "pw": "Demo@1234"} for _, e, r in authx.DEMO]) if config.DEMO_MODE and config.ADMIN_PASSWORD == "Admin@123" else []}
@app.get("/api/me")
def me(request: Request):
    u = user_of(request); c = ctx(u); caps = sorted(RB.CAPS.get(u["role"], set()))
    return {"user": u, "kind": KIND, "name": V.NAME, "menu": [{"id": i, "label": l, "icon": ic} for i, l, ic in menu_for(u["role"])], "caps": caps, "scope": c["scope"], "langs": list(gpu.NLLB_LANGS), "questions": authx.QUESTIONS}

# ------------------------------------------------------------------ live pulse
def pulse():
    rt = LE.get_state("runtime", {}) or {}; boot = LE.get_state("boot", {}) or {}
    d = VC.decision(); oa = B.Q("SELECT status, severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY status, severity")
    n = lambda st, sv=None: int(oa[(oa.status == st) & ((oa.severity == sv) if sv else True)].n.sum()) if len(oa) else 0
    g = gpu.status()
    return {"stage": boot.get("stage", "starting"), "ticks": rt.get("ticks", 0), "cycles": rt.get("cycles", 0), "last_cycle": rt.get("last_cycle"), "health": d["score"] if d else None, "risk": d["risk"] if d else None, "dec": d["id"] if d else 0,
            "open": n("open"), "critical": n("open", "Critical"), "proposed": n("proposed"), "gpu": {"cuda": g["cuda"], "name": g["gpu"], "used": g["vram_used_gb"], "total": g["vram_gb"], "llm": g["models"]["llm"]["state"], "llm_name": g["models"]["llm"]["name"], "llm_done": g["models"]["llm"].get("done_gb"), "llm_total": g["models"]["llm"].get("total_gb"), "llm_err": g["models"]["llm"]["err"], "nllb": g["models"]["nllb"]["state"], "embed": g["models"]["embed"]["state"], "tok_s": g["stats"]["tok_s"]}, "worker_alive": (_worker is None or _worker.poll() is None)}
@app.get("/api/pulse")
def pulse_http(request: Request): user_of(request); return pulse()
@app.get("/api/stream")
async def stream(request: Request):
    user_of(request)
    async def gen():
        last = None; t0 = time.time()
        while True:
            if await request.is_disconnected(): break
            try: p = await asyncio.to_thread(pulse)
            except Exception as e: p = {"error": str(e)}
            k = json.dumps(p, sort_keys=True, default=str)
            if k != last or time.time() - t0 > 10: last = k; t0 = time.time(); yield f"data: {k}\n\n"
            await asyncio.sleep(1.5)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ pages
_TABS = {}
def _tabs_of(c, f):
    if f not in _TABS: _TABS[f] = f(c, None).get("tabs")
    return _TABS[f]
def _simtabs(t): return (t or [{"id": "main", "label": "Overview"}]) + [{"id": "sim", "label": "Simulator"}]
def _page(c, pid, tab):
    if pid == "command": return VC.command(c, tab or "ops")
    if pid == "approvals": return VC.approvals(c, tab or "awaiting")
    if pid == "kg": return VC.kg_page(c, tab or "net")
    if pid == "copilot": return VM.copilot(c, tab or "chat")
    if pid == "notifications": return VM.notifications(c, tab or "alerts")
    if pid == "twin": return VM.twin(c, tab)
    if pid == "feeds": return VM.feeds(c, tab or "upload")
    if pid == "admin": return VM.admin(c, tab or "runtime")
    if pid == "reports": return VM.reports(c, tab or "pdf")
    if pid == "profile": return VM.profile(c, tab or "me")
    f = V.PAGES.get(pid)
    if f:
        if tab == "sim":
            sm = VM.simulator(c, pid); return {**sm, "title": sm["title"], "tabs": _simtabs(_tabs_of(c, f)), "live": 0}
        o = f(c, None if tab == "main" else tab); _TABS[f] = o.get("tabs"); o["tabs"] = _simtabs(o.get("tabs")); return o
    raise HTTPException(404, "unknown page")
@app.get("/api/page/{pid}")
def page(pid: str, request: Request, tab: str = "", p: str = ""):
    u = user_of(request); c = ctx(u, json.loads(p) if p else {})
    if pid not in V.ROLE_MENU.get(u["role"], set()): raise HTTPException(403, "no access to this page")
    t = time.time(); out = _page(c, pid, tab or None); out["id"] = pid; out["ms"] = round((time.time() - t) * 1000); return JSONResponse(B.clean(out))
@app.get("/api/detail/{eid:path}")
def detail(eid: str, request: Request):
    u = user_of(request); c = ctx(u)
    if c["scope"] and not eid.startswith(c["scope"]): raise HTTPException(403, "outside your outlet")
    return JSONResponse(B.clean(V.detail(c, eid)))

# ------------------------------------------------------------------ tickets, cycles, policy
def _tk(aid):
    r = LE.conn().execute("SELECT target, type, status FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
    if not r: raise HTTPException(404, "ticket not found")
    return r
@app.post("/api/approval/request")
async def approval_request(request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); d = await request.json(); det = str(d.get("detail", "")).strip()[:600]
    if len(det) < 5: return bad("Describe what you need approved (a sentence or two).")
    tg = str(d.get("target", "")).strip()[:60] or "general"
    if c["scope"] and tg != "general" and not tg.startswith(c["scope"]): return bad(f"You can only raise requests for your outlet ({c['scope']}).")
    sev = d.get("severity") if d.get("severity") in ("Info", "Warning", "Critical") else "Warning"
    aid = LE.request_approval(dom, str(d.get("kind", "Other"))[:60], tg, det, sev, u["email"], d.get("levels") or 1)
    return ok(f"Request #{aid} sent. Track it under Approvals > Request approval.", refresh=True)
@app.post("/api/ticket/{aid}/{action}")
async def ticket(aid: int, action: str, request: Request):
    u = user_of(request); c = ctx(u); d = await request.json() if request.headers.get("content-length", "0") != "0" else {}; tg, typ, st = _tk(aid)
    if c["scope"] and not str(tg).startswith(c["scope"]): raise HTTPException(403, "outside your outlet")
    em = u["email"]
    if action in ("approve", "reject"):
        need(c, "approve"); yes = action == "approve"
        if st == "proposed":
            s2, msg = LE.approve_action(dom, aid, em, yes, d.get("note", ""), c["role"])
            if s2 in ("denied", "missing"): return bad(msg)
            return ok(("Approved and executed" if s2 == "final" else (msg[:1].upper() + msg[1:])) if yes else "Rejected")
        elif typ == "quote_approval" and hasattr(dom, "decide"):
            s2, msg = LE.advance(aid, em, c["role"], yes, d.get("note", ""))
            if s2 in ("denied", "missing"): return bad(msg)
            if s2 == "partial": return ok((msg[:1].upper() + msg[1:]))
            dom.decide(aid, yes, em); return ok("Approved and executed" if yes else "Rejected")
        else: raise HTTPException(400, "nothing to approve on this ticket")
    if action == "resolve": need(c, "resolve"); LE.resolve_action(aid, "resolved", f"by {em}", actor=em); return ok("Resolved")
    if action == "assign": need(c, "resolve"); LE.assign_action(aid, d.get("assignee"), em); return ok("Assigned")
    if action == "comment": need(c, "view"); LE.comment_action(aid, em, str(d.get("body", ""))[:500]); return ok("Posted")
    raise HTTPException(400, "unknown action")
@app.post("/api/run_cycle")
def run_cycle(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); t = time.time(); LE.run_cycle(dom, "manual", ""); LE.audit(u["email"], "manual_cycle", "all"); return ok(f"Agents finished in {time.time() - t:.1f}s")
@app.post("/api/run_agent")
async def run_agent(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); a = (await request.json()).get("agent")
    if a not in dom.AGENTS: raise HTTPException(404, "unknown agent")
    await asyncio.to_thread(_run_one, a); LE.audit(u["email"], "manual_agent", a); return ok(f"{a} re-ran")
def _run_one(agent):
    cid = time.strftime("%m%d-%H%M%S") + "-page"; t = time.time(); s = {"cycle_id": cid, "trigger": "page", "request": "", "queue": [], "results": {}}
    if agent == "digest":
        for a in ("outlet", "inventory", "staff", "marketing", "audit"): s["results"][a] = dom.AGENTS[a](s)
    res = dom.AGENTS[agent](s); LE._log_run(cid, "page", res, int((time.time() - t) * 1000)); s["results"][agent] = res; LE.act(dom, {"actions": res.actions}, cid, {agent: res})
@app.post("/api/sla")
def sla(request: Request): u = user_of(request); c = ctx(u); need(c, "run"); return ok(f"{LE.escalate_overdue(dom)} ticket(s) escalated")
@app.post("/api/notify_test")
def notify_test(request: Request): u = user_of(request); c = ctx(u); need(c, "admin"); LE.notify(dom, "TEST", "Critical", "Test", "Test alert from the Notifications page"); return ok("Sent to every configured channel")
@app.post("/api/policy")
async def policy(request: Request):
    u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json(); cur = LE.policy(dom)
    for t, m in d.items():
        if t in dom.ACTION_AGENT and m in ("auto", "approval") and cur.get(t, "auto") != m: LE.set_policy(t, m, u["email"])
    return ok("Policy saved")
@app.get("/api/audit.csv")
def audit_csv(request: Request):
    u = user_of(request); c = ctx(u)
    if not (RB.can(c["role"], "admin") or c["role"] == "Auditor"): raise HTTPException(403)
    return PlainTextResponse(B.Q("SELECT ts, actor, event, target, detail FROM audit_log ORDER BY audit_id DESC LIMIT 20000").to_csv(index=False), headers={"Content-Disposition": "attachment; filename=audit_log.csv"}, media_type="text/csv")

# ------------------------------------------------------------------ admin users
@app.post("/api/user/role")
async def user_role(request: Request):
    u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json()
    if d["role"] not in authx.ROLES: return bad("unknown role")
    authx.set_role(d["email"], d["role"]); LE.audit(u["email"], "role_change", d["email"], d["role"]); return ok("Role updated; it applies on the user's next request")
@app.post("/api/user/unlock")
async def user_unlock(request: Request): u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json(); authx.unlock(d["email"]); LE.audit(u["email"], "unlock", d["email"]); return ok("Unlocked")
@app.post("/api/user/scope")
async def user_scope(request: Request): u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json(); RB.set_scope(d["email"], d["outlet"]); LE.audit(u["email"], "scope_assign", d["email"], d["outlet"]); return ok("Saved")

# ------------------------------------------------------------------ digital twin
@app.post("/api/twin/run")
async def twin_run(request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); d = await request.json(); d.pop("scenario", None); c["email"] = u["email"]; t = time.time()
    if d.get("agents") not in ("yes", True): d["agents"] = False
    else: d["agents"] = True
    try: bl = await asyncio.to_thread(VM.twin_run, c, d)
    except Exception as e: return bad(f"Could not simulate: {e}")
    return ok(f"Simulated in {time.time() - t:.1f}s", blocks=B.clean(bl))
@app.post("/api/twin/saved")
async def twin_saved(request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); d = await request.json(); import twin2 as T2
    s = T2.saved().set_index("name")
    if d.get("a") not in s.index: return bad("unknown scenario")
    a = json.loads(s.loc[d["a"]].args); bl, _ = await asyncio.to_thread(T2.run, c, a); return ok("Re-run on today's data", blocks=B.clean(bl))
@app.post("/api/twin/compare")
async def twin_cmp(request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); d = await request.json()
    try: return ok("Compared", blocks=B.clean(await asyncio.to_thread(VM.twin_compare, c, d["a"], d["b"])))
    except Exception as e: return bad(f"Could not compare: {e}")
@app.post("/api/twin/propose")
async def twin_propose(request: Request):
    u = user_of(request); c = ctx(u); need(c, "approve"); d = await request.json(); import twin2 as T2
    txt = await asyncio.to_thread(T2.propose, KIND, str(d.get("action", ""))[:200], str(d.get("detail", "")), u["email"]); return ok("Sent to the approval queue (Actions > awaiting approval)", refresh=False)
@app.post("/api/sim/run")
async def sim_run(request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); d = await request.json(); d.pop("scenario", None); pg = d.pop("page"); t = time.time()
    try: bl = await asyncio.to_thread(VM.sim_run, c, pg, "sim", d)
    except Exception as e: return bad(f"Could not simulate: {e}")
    return ok(f"Simulated in {time.time() - t:.1f}s", blocks=B.clean(bl))

# ------------------------------------------------------------------ data feeds
@app.post("/api/feed/upload")
async def feed_upload(request: Request, file: UploadFile = File(...), table: str = Form(...)):
    u = user_of(request); c = ctx(u); need(c, "feeder")
    if table not in FD.ALLOWED[KIND]: return bad("table not allowed")
    try: df = FD.read_any(file.filename, await file.read()); mp = FD.auto_map(df, table); n, badn = await asyncio.to_thread(FD.ingest, df, table, mp, "upload:" + file.filename)
    except Exception as e: return bad(str(e))
    return ok(f"{n} rows ingested, {badn} rejected. Columns matched: {len(mp)}.", blocks=[B.table(df.head(8), list(df.columns), title="Preview of the file", h=240, key="fprev")])
@app.post("/api/feed/url")
async def feed_url(request: Request):
    u = user_of(request); c = ctx(u); need(c, "feeder"); d = await request.json()
    if d.get("target") not in FD.ALLOWED[KIND] or not str(d.get("url", "")).startswith("http"): return bad("need an http(s) URL and an allowed table")
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO feed_urls(url,target,every_s,auth_env,json_path) VALUES(?,?,?,?,?)", (d["url"], d["target"], max(30, int(d.get("every", 300))), (d.get("auth_env") or "").strip() or None, (d.get("json_path") or "").strip() or None))
    LE.audit(u["email"], "feed_url", d["target"], d["url"]); return ok("Feed registered", refresh=True)
@app.post("/ingest/{table}")
@app.post("/api/ingest/{table}")
async def ingest_api(table: str, request: Request, x_api_key: str = Header(default=""), idempotency_key: str = Header(default="")):
    import hmac; k = os.environ.get("INGEST_KEY", "")
    if not k or not hmac.compare_digest(x_api_key.encode(), k.encode()): raise HTTPException(401, "bad or missing X-API-Key (set INGEST_KEY)")
    if table == "ais" and KIND == "Freight": table = "vessels"
    if table not in FD.ALLOWED[KIND]: raise HTTPException(400, "table not allowed")
    if idempotency_key and await asyncio.to_thread(FD.seen_key, table + ":" + idempotency_key): return {"accepted": 0, "rejected": 0, "duplicate": True}
    body = await request.body()
    try:
        rows = [json.loads(l) for l in body.decode().splitlines() if l.strip()] if body[:1] not in (b"[", b"{") or b"\n{" in body.strip() else json.loads(body)
        if isinstance(rows, dict): rows = rows.get("rows", [rows])
        if len(rows) > 50000: raise ValueError("max 50,000 rows per request")
        n, badn = await asyncio.to_thread(FD.ingest, pd.DataFrame(rows), table, None, "api:" + table)
    except Exception as e: raise HTTPException(400, str(e))
    return {"accepted": n, "rejected": badn, "note": "rejected rows are listed in Data Feed Center > Quality"}
@app.post("/api/feed/ais_map")
async def feed_ais_map(request: Request):
    u = user_of(request); need(ctx(u), "feeder"); d = await request.json(); pairs = [l.split(",") for l in str(d.get("pairs", "")).splitlines() if "," in l]; FD.init()
    with LE._DB_LOCK, LE.conn() as cn: cn.executemany("INSERT OR REPLACE INTO ais_map VALUES(?,?)", [(a.strip(), b.strip()) for a, b in pairs])
    LE.audit(u["email"], "ais_map", "vessels", f"{len(pairs)} ids"); return ok(f"{len(pairs)} MMSI to shipment links saved", refresh=True)
@app.get("/api/feed/rejects.csv")
async def feed_rejects(request: Request):
    u = user_of(request); need(ctx(u), "feeder"); FD.init(); df = B.Q("SELECT * FROM feed_rejects ORDER BY id DESC LIMIT 5000")
    return Response(df.to_csv(index=False), media_type="text/csv", headers={"Content-Disposition": 'attachment; filename="rejected_rows.csv"'})
@app.post("/api/gov/purge")
async def gov_purge(request: Request):
    u = user_of(request); need(ctx(u), "admin"); d = await request.json(); dry = str(d.get("mode", "dry")) != "run"; rows = await asyncio.to_thread(PV.purge, dry, u["email"])
    return ok(("Dry run: nothing deleted. " if dry else "Purged. ") + f"{sum(r['older_rows'] for r in rows):,} rows past retention.", blocks=[B.table(rows, ["table", B.num_col("keep_days", "keep (days)", "int"), B.num_col("older_rows", "past retention", "int"), B.num_col("deleted", "deleted", "int")], h=300, key="gv_p")])
@app.post("/api/gov/retention")
async def gov_retention(request: Request):
    u = user_of(request); need(ctx(u), "admin"); d = await request.json(); cur = PV.policy()
    try: cur[str(d["table"])] = max(7, int(d["days"]))
    except Exception: return bad("table and days (min 7) required")
    if d["table"] not in PV.RETENTION: return bad("unknown table")
    LE.set_state("retention", cur); LE.audit(u["email"], "retention_set", d["table"], str(d["days"])); return ok("Retention updated", refresh=True)
@app.post("/api/gov/dsar")
async def gov_dsar(request: Request):
    u = user_of(request); need(ctx(u), "admin"); d = await request.json(); key = str(d.get("subject", "")).strip(); act = d.get("action", "export")
    if not key: return bad("enter a staff id or a login email")
    if act == "erase":
        n = await asyncio.to_thread(PV.erase_subject, key, u["email"]); return ok("Erased / anonymised: " + json.dumps(n))
    data = await asyncio.to_thread(PV.export_subject, key); LE.audit(u["email"], "dsar_export", PV._hash(key), ",".join(data))
    return ok(f"{sum(len(v) for v in data.values())} records found" if data else "Nothing held for this subject.", blocks=[B.md("```\n" + json.dumps(data, indent=1, default=str)[:6000] + "\n```")] if data else [])

# ------------------------------------------------------------------ Franchise actions
@app.post("/api/franchise/review")
async def fr_review(request: Request):
    import uuid; u = user_of(request); c = ctx(u); d = await request.json(); tx = str(d.get("comment", "")).strip()
    if not tx: return bad("Write a comment first")
    if c["scope"] and d["outlet"] != c["scope"]: return bad("outside your outlet")
    src = gpu.detect(tx); en = PV.redact(gpu.translate(tx, "eng_Latn", src)[0] if src != "eng_Latn" else tx)
    try: from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; sc = SentimentIntensityAnalyzer().polarity_scores(en)["compound"]
    except Exception: sc = (int(d["rating"]) - 3) / 2
    rt = int(d["rating"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)", ("FB-U" + uuid.uuid4().hex[:10], d["outlet"], rt, en, datetime.date.today().isoformat(), .6 * (rt - 3) / 2 + .4 * sc))
    LE.emit("review", d["outlet"], {"rating": rt}); return ok(f"Stored for {d['outlet']} (sentiment {sc:+.2f}{', translated from ' + src if src != 'eng_Latn' else ''}). The Sentiment agent includes it in its next cycle.", refresh=True)
@app.post("/api/franchise/audit")
async def fr_audit(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "resolve"); d = await request.json(); sc = int(d["score"])
    if c["scope"] and d["outlet"] != c["scope"]: return bad("outside your outlet")
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)", ("AUD-U" + uuid.uuid4().hex[:10], d["outlet"], datetime.date.today().isoformat(), sc, int((100 - sc) / 6), d["category"], "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", d.get("notes", "")))
    LE.emit("audit", d["outlet"], {"score": sc}); LE.audit(u["email"], "audit_logged", d["outlet"], str(sc)); await asyncio.to_thread(_run_one, "audit"); return ok("Saved and evaluated by the Audit agent.", refresh=True)

# ------------------------------------------------------------------ Freight actions
@app.post("/api/freight/quote")
async def fq_quote(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote"); d = await request.json()
    if d["o"] == d["d"]: return bad("Origin and destination must differ.")
    rid = dom.submit_request(d["o"], d["d"], d["cargo"], float(d["weight"]), d["hs"], d["customer"], "English", d.get("docs") or []); await asyncio.to_thread(dom.run_quote, rid); LE.audit(u["email"], "quote_created", str(rid), f"{d['o']}->{d['d']}")
    return ok("Quote generated", blocks=B.clean(V.quote_result(rid)))
@app.post("/api/freight/storm")
async def fq_storm(request: Request):
    from twin import PRESET_STORMS; u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json() if request.headers.get("content-length", "0") not in ("0", "2") else {}
    la, lo = PRESET_STORMS.get(d.get("area") or "Red Sea"); dom.inject_storm(la, lo, wind=float(d.get("wind", 24))); LE.audit(u["email"], "inject_storm", d.get("area", "Red Sea")); return ok("Storm injected. The weather agent will propose re-routes on its next cycle.", refresh=True)
@app.post("/api/freight/reroute")
async def fq_reroute(request: Request):
    u = user_of(request); c = ctx(u); need(c, "approve"); j = await request.json(); sid = j["shipment"]; opt = j.get("opt")
    idx = int(str(opt).split(".")[0]) - 1 if opt else None; act = {"type": "reroute", "target": sid}
    if idx is not None: act["opt_label"] = str(opt).split(" · ", 1)[-1]; act["opt_idx"] = idx
    with LE._DB_LOCK, LE.conn() as cn: note = dom.apply(act, cn)
    LE.audit(u["email"], "manual_reroute", sid, note); return ok(note or "no change", refresh=True)
@app.post("/api/freight/attach")
async def fq_attach(request: Request):
    u = user_of(request); c = ctx(u)
    if not (RB.can(c["role"], "resolve") or RB.can(c["role"], "quote")): raise HTTPException(403)
    d = await request.json(); dom.attach_doc(d["ref"], d["doc"], u["email"]); LE.audit(u["email"], "doc_attached", d["ref"], d["doc"]); return ok("Attached. Quotes re-run immediately; shipments clear on the next customs cycle.", refresh=True)
@app.post("/api/freight/readdoc")
async def fq_readdoc(request: Request, file: UploadFile = File(...)):
    u = user_of(request); data = await file.read(); txt = ""
    try:
        if file.filename.lower().endswith(".pdf"):
            import pdfplumber
            with pdfplumber.open(io.BytesIO(data)) as pdf: txt = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        else: txt = data.decode("utf-8", errors="ignore")
    except Exception as e: return bad(f"Could not read the file: {e}")
    hs = re.findall(r"HS[- ]?\d{2,8}", txt); q = re.findall(r"QTE-L\d+", txt); wt = re.findall(r"(\d+(?:\.\d+)?)\s*(?:t|tons|kg)\b", txt)
    return ok("Read", blocks=[B.table([{"field": "HS code", "value": ", ".join(dict.fromkeys(hs)) or "-"}, {"field": "Quote reference", "value": ", ".join(dict.fromkeys(q)) or "-"}, {"field": "Weights", "value": ", ".join(wt[:3]) or "-"}], ["field", "value"], title="Fields found", h=170, key="ocr"), B.md("```\n" + txt[:2500] + "\n```")])
@app.get("/api/doc/{name}")
def doc(name: str, request: Request):
    user_of(request); p = os.path.join(getattr(dom, "DATA_DIR", config.DATA_DIR), "docs", os.path.basename(name))
    if not os.path.exists(p): raise HTTPException(404); return FileResponse(p, filename=os.path.basename(name))

# ------------------------------------------------------------------ language, knowledge base
@app.post("/api/translate_batch")
async def translate_batch(request: Request):
    user_of(request); d = await request.json(); texts = [str(t)[:700] for t in d.get("texts", [])[:400]]; out, err = await asyncio.to_thread(gpu.translate_many, texts, d.get("lang", "eng_Latn")); return {"out": out, "err": err}
@app.post("/api/translate_doc")
async def translate_doc(request: Request):
    user_of(request); data = ""; text = ""; lang = "Hindi (हिंदी)"
    if "json" in request.headers.get("content-type", ""):
        j = await request.json(); text = str(j.get("text", "")); lang = j.get("lang", lang)
    else:
        fm = await request.form(); text = str(fm.get("text", "") or ""); lang = fm.get("lang", lang); file = fm.get("file")
        if file is not None and getattr(file, "filename", ""):
            try: data = rag.read_pdf(file.filename, await file.read())
            except Exception as e: return bad(f"Could not read the file: {e}")
    t = (data or text).strip()
    if not t: return bad("Paste text or choose a file")
    src = gpu.detect(t); t0 = time.time(); out, err = await asyncio.to_thread(gpu.translate, t[:20000], gpu.code_of(lang), src)
    if err: return bad(err)
    return ok(f"Translated from {src} to {gpu.code_of(lang)} in {time.time() - t0:.1f}s on {gpu.S['models']['nllb']['device']}", blocks=[B.card("Translation", B.md(out))])
@app.post("/api/kb/upload")
async def kb_upload(request: Request, file: UploadFile = File(...)):
    u = user_of(request)
    try: txt = rag.read_pdf(file.filename, await file.read())
    except Exception as e: return bad(f"Could not read the file: {e}")
    n = await asyncio.to_thread(rag.add, file.filename, "uploaded by you", txt, u["email"]); LE.audit(u["email"], "kb_upload", file.filename, f"{n} chunks"); return ok(f"{n} passages indexed ({gpu.S['models']['embed']['device'] or 'tf-idf'}).", refresh=True)
@app.post("/api/kb/search")
async def kb_search(request: Request):
    u = user_of(request); d = await request.json(); q = d.get("q", ""); mode = {"BM25 only": "bm25", "Dense only": "dense"}.get(d.get("mode"), "hybrid"); t0 = time.time()
    hits = await asyncio.to_thread(rag.search, q, u["email"], 6, mode, True)
    if d.get("live", "no") in ("yes", True): hits += await asyncio.to_thread(rag.live_hits, q, 4)
    return ok(f"{len(hits)} passage(s) in {(time.time() - t0) * 1000:.0f} ms", blocks=[B.card(f"{h['title']} · {h['source']}", B.md(h["text"]), sub=f"score {h['score']} · {h.get('why', '')}") for h in hits] or [B.callout("Nothing relevant found.", "info")])
@app.post("/api/kb/drive")
async def kb_drive(request: Request):
    u = user_of(request); need(ctx(u), "admin"); n = await asyncio.to_thread(rag.import_drive); LE.audit(u["email"], "kb_drive_import", rag.drive_dir(), f"{n} chunks")
    return ok(f"{n} chunks loaded from your Drive RAG index ({rag.drive_dir()})" if n else f"Nothing found in {rag.drive_dir()}/synthetic_kb. Mount Drive and run your RAG builder first.", refresh=True)
@app.get("/api/kb/export")
async def kb_export(request: Request):
    u = user_of(request); data = await asyncio.to_thread(rag.export_json, u["email"]); return JSONResponse(data, headers={"Content-Disposition": 'attachment; filename="kb_export.json"'})
@app.post("/api/kb/import")
async def kb_import(request: Request, file: UploadFile = File(...)):
    u = user_of(request)
    try: n, same = await asyncio.to_thread(rag.import_json, json.loads(await file.read()), u["email"])
    except Exception as e: return bad(f"Could not import: {e}")
    LE.audit(u["email"], "kb_import", file.filename, f"{n} chunks"); return ok(f"{n} passages imported" + (" with their embeddings." if same else "; embeddings will be recomputed on the GPU."), refresh=True)
@app.post("/api/kb/eval")
async def kb_eval(request: Request):
    u = user_of(request); need(c := ctx(u), "view"); rows = await asyncio.to_thread(rag.evaluate, KIND)
    return ok("Evaluated", blocks=[B.table(rows, ["method", B.num_col("recall@1", "recall@1", "num"), B.num_col("recall@3", "recall@3", "num"), B.num_col("MRR", "MRR", "num"), "note"], title="Retrieval quality on the built-in question set", h=240, key="kb_ev"), B.callout("This question set is small and written alongside the sample documents, so scores are optimistic. Use the builder notebook to evaluate on your own corpus and questions.", "info")])

# ------------------------------------------------------------------ Copilot (agents first, then the GPU LLM; NLLB for every language) + executive brief
def _sse(ev, **kw): return f"data: {json.dumps({'ev': ev, **kw}, default=str)}\n\n"
INSIGHT_SYS = "You are the operations copilot. An EXACT RESULT from the database is given in the EVIDENCE: do not repeat it or its table. In at most two short sentences add the most useful insight or next action, using the agent findings and the numbered sources when they help. Use only numbers that appear in the EVIDENCE. Cite a source as [1] when you use it. If you have nothing useful to add, answer exactly: NONE"
SYSTEM = "You are the operations copilot of an agentic platform. Answer ONLY from the EVIDENCE. FORMAT RULES: reply in exactly this layout and nothing else. Line 1: '**Answer:** ' plus one sentence of at most 25 words. Then up to 3 numbered points ('1. ...'), each at most 20 words, one fact each, with entity ids when the evidence has them. Last line, only if the evidence supports an action: '**Next step:** ' plus one short action. Quote numbers exactly as given and never invent any. Cite every fact that comes from a numbered source as [1], [2] matching the SOURCES list; facts from the database or the agents need no citation. If the evidence does not answer the question, reply exactly: I do not have that in the data or documents yet."
def _nums_ok(ans, ev):
    ans = re.sub(r"\[\d{1,2}\]|(?m:^\s*\d{1,2}[.)]\s)", " ", ans)        # citation markers and list numbering are not data
    have = set(re.findall(r"\d+(?:\.\d+)?", ev)); return all(n in have for n in re.findall(r"\d+(?:\.\d+)?", ans))
import copilot_pro as CP
CP_EXAMPLES = ["Which outlets are at highest risk?", "Why is OUT-007 flagged?", "How is inventory holding up?", "Top 5 outlets by revenue"] if KIND == "Franchise" else ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which port has the highest congestion?", "How many shipments are on customs hold?", "Quotes with margin below 10%"]
@app.post("/api/copilot")
async def copilot(request: Request):
    u = user_of(request); c = ctx(u); d = await request.json(); q = str(d.get("q", ""))[:800]; lang = d.get("lang", "English"); tgt = gpu.code_of(lang); SID = u["email"] + ":" + str(d.get("sid") or "default")[:40]
    def gen():
        t0 = time.time(); src = gpu.detect(q); en = q
        if src != "eng_Latn":
            en, err = gpu.translate(q, "eng_Latn", src); yield _sse("stage", text=f"Detected {src}; translated to English with NLLB-200 on {gpu.S['models']['nllb']['device'] or 'cpu'}")
        dec = VC.decision(); exact = None
        direct, en, pres, notes, pids = CP.prepare(KIND, en, SID, dom, VC.decision, CP_EXAMPLES, c.get("scope") or None, PV.mask_role(u["role"]))
        if notes: yield _sse("note", text=" · ".join(notes))
        if direct:
            txt = direct["text"]
            if tgt != "eng_Latn": txt = gpu.translate(txt, tgt, "eng_Latn")[0]
            CP.log_turn(q, direct.get("route", "chat"), round((time.time() - t0) * 1000))
            yield _sse("final", text=txt, sql=direct.get("sql", ""), sugg=direct.get("sugg"), en=(direct["text"] if tgt != "eng_Latn" else None), lang=lang, ms=round((time.time() - t0) * 1000), tok_s=gpu.S["stats"]["tok_s"]); yield _sse("done"); return
        analytic = bool(re.search(r"\b(why|explain|reason|recommend|should|suggest|advise|what to do|how can|how do|improve|summar)", en, re.I))
        try: exact = pres or AN.answer(KIND, en, c.get("scope") or None, dec, dom) or QA.answer(KIND, en, c.get("scope") or None, PV.mask_role(u["role"]))
        except Exception as e: print("qa failed:", repr(e), file=sys.stderr)
        try:
            if exact and exact.get("answer") and CP.is_knowledge_q(en) and CP.weak_match(en, exact): exact = None; pres = None; yield _sse("stage", text="That table does not match the question, so it is not shown. Looking in the knowledge base instead…")
        except Exception: pass
        fast = bool(exact and exact.get("answer") and (not analytic or exact.get("complete")))
        trace = []
        kno = bool(not (exact and exact.get("answer")) and not pids and (CP.is_knowledge_q(en) or CP.is_term_q(en)))
        if fast:
            planned = dom.plan(en); dd = {"narrative": dec["narrative"], "actions": [], "agents": dec["ev"].get("agents", {})} if dec else {"narrative": "", "actions": [], "agents": {}}
            trace = [{"agent": k, "risk": v["risk"], "score": v["score"], "finding": v["summary"][:160]} for k, v in dd.get("agents", {}).items() if k in planned]
            if trace: yield _sse("trace", rows=trace)
        elif kno:
            planned = []; dd = {"narrative": "", "actions": [], "agents": {}}; yield _sse("stage", text="Knowledge question: searching the documents (agents skipped).")
        else:
            planned = dom.plan(en); yield _sse("stage", text="Planner selected: " + ", ".join(planned))
            stale = (not dec) or (datetime.datetime.now() - datetime.datetime.fromisoformat(dec["ts"])).total_seconds() > 300      # the worker refreshes the agents every 60 s: reuse its latest results
            if stale:
                yield _sse("stage", text="Running the selected agents and the Intelligence Engine…"); dd = LE.run_cycle(dom, "copilot", en)
            else: dd = {"narrative": dec["narrative"], "actions": [], "agents": dec["ev"].get("agents", {})}
            yield _sse("stage", text=f"Agents ready in {time.time() - t0:.1f}s")
            trace = [{"agent": k, "risk": v["risk"], "score": v["score"], "finding": v["summary"][:160]} for k, v in dd.get("agents", {}).items() if k in planned]
            if trace: yield _sse("trace", rows=trace)
        if exact and exact.get("answer"):
            yield _sse("stage", text="Exact answer from your live data (read-only query)"); yield _sse("exact", sql=exact["sql"], rows=0 if exact["df"] is None else len(exact["df"]))
        planned = locals().get("planned") or []
        facts = [] if (fast or kno) else EN.facts(KIND, dom, en); sql_fact = ""
        if not (fast or kno):
            try: from intent_router import run_grounded_query; sql_fact, _ = run_grounded_query(en)
            except Exception: pass
        kb = (CP.relevant(en, rag.search(en, u["email"], 3, "hybrid", False)) if fast else (rag.search(en, u["email"], 4, "hybrid", gpu.llm_ready()) + rag.live_hits(en, 2))); acts = dd.get("actions", [])[:8]
        if kb: yield _sse("cites", rows=[{"n": n + 1, "title": h["title"], "source": h["source"], "why": h.get("why", ""), "score": h["score"], "text": h["text"][:240]} for n, h in enumerate(kb)])
        _k0 = None
        if kno and kb and tgt == "eng_Latn":
            _k0 = CP.kb_answer(en, [h for h in kb if h.get("via") != "live"])
            if _k0: yield _sse("partial", text=_k0)                      # instant extractive answer; a model answer replaces it only for how/why/explain questions
        ev = (f"EXACT RESULT (from the database, authoritative): {exact['answer']}\n{QA.md(exact['df'])}\n" if exact and exact.get("answer") else "") + f"{dd.get('narrative', '')} Top actions: " + " | ".join(f"{a['type']} {a['target']} ({a.get('severity')}): {a['detail']}" for a in acts) + (" Entities: " + " ".join(facts) if facts else "") + (f" Data lookup: {str(sql_fact)[:900]}" if sql_fact else "") + (" Sources: " + " ".join(f"[{n + 1}] ({h['title']}) {h['text'][:320]}" for n, h in enumerate(kb)) if kb else "")
        EXAMPLES = ["Which outlets are at highest risk?", "Why is OUT-007 flagged?", "How is inventory holding up?", "Top 5 outlets by revenue"] if KIND == "Franchise" else ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which port has the highest congestion?", "How many shipments are on customs hold?", "Quotes needing approval"]
        fallback = ("I could not compute an exact answer to that from the tables. What the agents responsible for this question found:\n" + "\n".join(f"- **{t['agent']}** ({t['risk']}): {t['finding']}" for t in trace if t["agent"] in planned) + (("\n\nRelated knowledge: " + "; ".join(h["title"] for h in kb[:3])) if kb else "") + "\n\nTry asking, for example: " + "; ".join(f'"{x}"' for x in EXAMPLES) + ".")
        ans = ""
        insight = False; _deco = None
        if exact and exact.get("answer"):
            _tb0 = (QA.md(exact["df"]) if exact.get("df") is not None else "") + (("\n\n" + exact["extra"]) if exact.get("extra") else "")
            if tgt == "eng_Latn": yield _sse("partial", text=exact["answer"] + (("\n\n" + _tb0) if _tb0 else ""))
            try:
                _deco = CP.decorate(KIND, SID, en, exact, pids)
                if _deco[0]: yield _sse("chart", block=_deco[0])
            except Exception as e: print("decorate failed:", repr(e), file=sys.stderr)
        _rows = 0 if not exact or exact.get("df") is None else len(exact["df"])
        if exact and exact.get("answer") and (not analytic or exact.get("complete")):
            if gpu.llm_ready() and analytic:
                yield _sse("stage", text=f"Writing the insight on {gpu.S['models']['llm']['device']} ({gpu.S['models']['llm']['name']})…")
                for piece in gpu.llm_stream([{"role": "system", "content": INSIGHT_SYS}, {"role": "user", "content": f"EVIDENCE:\n{ev[:2200]}\n\nQUESTION: {en}"}], max_new_tokens=70, temperature=0.0): ans += piece
                ans = ans.strip()
                if ans.upper().startswith("NONE") or len(ans) < 12 or not CP.sane(ans) or not _nums_ok(ans, ev): ans = ""
                else: ans = "**Insight:** " + ans; insight = True; yield _sse("tok", text=ans)
            elif not gpu.llm_ready(): yield _sse("stage", text="LLM not loaded (" + str(gpu.S["models"]["llm"]["state"]) + "): this answer uses SQL, the latest agent findings and the knowledge base only.")
        elif gpu.llm_ready() and not (kno and _k0 and not analytic):
            _tok_ok = False
            if fallback and tgt == "eng_Latn" and not exact: yield _sse("draft", text=fallback)      # shows the agents' findings at once; the model answer replaces it
            yield _sse("stage", text=f"Generating on {gpu.S['models']['llm']['device']} ({gpu.S['models']['llm']['name']})…")
            for piece in gpu.llm_stream([{"role": "system", "content": SYSTEM + (" An EXACT RESULT is given: start from it, do not contradict it, and add only the reasoning or advice asked for." if exact else "")}, {"role": "user", "content": f"EVIDENCE:\n{ev[:2200]}\n\nQUESTION: {en}"}], max_new_tokens=150, temperature=0.0):
                ans += piece
                if not _tok_ok:
                    if len(ans) >= 16:
                        if CP.sane(ans): _tok_ok = True; yield _sse("tok", text=ans)
                        else: break                                         # garbage from the model: stop and fall back to the evidence
                else: yield _sse("tok", text=piece)
            if not CP.sane(ans): ans = ""
            if not _nums_ok(ans, ev): yield _sse("stage", text="The model used a number that is not in the evidence: showing the evidence summary instead."); ans = ""
        else: yield _sse("stage", text="LLM not loaded (" + str(gpu.S["models"]["llm"]["state"]) + (": " + gpu.S["models"]["llm"]["err"] if gpu.S["models"]["llm"]["err"] else "") + "): answering from agent evidence.")
        if re.match(r"^\W*(no evidence|\[no evidence|i do not have that in the data)", ans.strip(), re.I) or len(ans.strip()) < 12: ans = ""
        llm_used = bool(ans.strip())
        if llm_used and not insight: ans = CP.finish(ans, kb)
        kba = None; kgap = False
        if not ans.strip() and not (exact and exact.get("answer")):
            kba = CP.kb_answer(en, [h for h in kb if h.get("via") != "live"])
            if kba: ans = kba
            elif (CP.is_knowledge_q(en) or CP.is_term_q(en)) and not pids:
                ans = "I do not have a document on that topic in the knowledge base yet, and the database has no numbers for it. Add documents with the RAG notebook or in AI Copilot → Knowledge base, then ask again."; kgap = True; CP.unanswered_note(q)
        ans = ans.strip() or (fallback if not (exact and exact.get("answer")) else "")
        tbl = (QA.md(exact["df"]) if exact and exact.get("df") is not None else "") + (("\n\n" + exact["extra"]) if exact and exact.get("extra") else ""); lead = exact["answer"] if exact and exact.get("answer") else ""
        ans_en = (lead + ("\n\n" + ans if ans else "")); final = ans_en + (("\n\n" + tbl) if tbl else "")
        if tgt != "eng_Latn":
            tr, err = gpu.translate(ans_en, tgt, "eng_Latn"); final = tr + (("\n\n" + tbl) if tbl else ""); yield _sse("stage", text=f"Translated to {lang} with NLLB-200" + (f" (failed: {err})" if err else ""))
        _drv = False
        try: _drv = os.path.isdir(rag.drive_dir())
        except Exception: pass
        yield _sse("layers", sql=bool(exact and exact.get("answer")), agents=len(trace), rag=len([h for h in kb if h.get("via") != "live"]), llm=llm_used, llm_state=str(gpu.S["models"]["llm"]["state"]), drive=_drv, kb_hit=bool(kba))
        cb, sg, has = None, ["What can you do?", "Briefing"], False
        if exact and exact.get("answer"):
            cb, sg, has = _deco or CP.decorate(KIND, SID, en, exact, pids); CP.log_turn(q, "exact" if fast or not analytic else "exact+llm", round((time.time() - t0) * 1000))
            if cb and not _deco: yield _sse("chart", block=cb)
        elif ans_en.strip().startswith("I could not compute"): CP.unanswered_note(q); CP.log_turn(q, "unanswered", round((time.time() - t0) * 1000))
        else: CP.log_turn(q, "knowledge" if kba else ("unanswered" if kgap else "agents+llm"), round((time.time() - t0) * 1000))
        yield _sse("final", text=final, sugg=sg, csv=has, rows=(0 if not exact or exact.get("df") is None else len(exact["df"])), sql=(exact or {}).get("sql", ""), en=((ans_en + (("\n\n" + tbl) if tbl else "")) if tgt != "eng_Latn" else None), lang=lang, ms=round((time.time() - t0) * 1000), tok_s=gpu.S["stats"]["tok_s"]); yield _sse("done")
    _ck = (u["role"], str(c.get("scope") or ""), tgt, re.sub(r"\s+", " ", q.lower()).strip(" ?!."))
    _fu = bool(re.search(r"^(and|also|what about|how about|then|those|same)\b|\b(it|its|them|those|that one|this one)\b", q.lower()))
    def gen_cached():
        hit = _RC.get(_ck)
        if hit and not _fu and time.time() - hit[0] < hit[2]:
            t1 = time.time()
            try: CP.prepare(KIND, q, SID, dom, VC.decision, CP_EXAMPLES, c.get("scope") or None, PV.mask_role(u["role"]))      # keeps follow-up context
            except Exception: pass
            yield _sse("stage", text=f"Instant answer: the same question was answered {int(time.time() - hit[0])} s ago (refreshes automatically).")
            for e_ in hit[1]: yield _sse(e_["ev"], **{k_: v_ for k_, v_ in e_.items() if k_ != "ev"}) if e_["ev"] != "final" else _sse("final", **{**{k_: v_ for k_, v_ in e_.items() if k_ != "ev"}, "ms": round((time.time() - t1) * 1000), "cached": True})
            return
        rec = []
        for s_ in gen():
            yield s_
            try:
                e_ = json.loads(s_[6:])
                if e_.get("ev") not in ("stage", "tok"): rec.append(e_)
            except Exception: pass
        if any(e_.get("ev") == "layers" for e_ in rec) and any(e_.get("ev") == "final" for e_ in rec):
            if len(_RC) > 200: _RC.clear()
            _lay = next((e_ for e_ in rec if e_.get("ev") == "layers"), {}); _RC[_ck] = (time.time(), rec, 600 if (not _lay.get("sql") and not _lay.get("agents")) else 60)      # document answers do not depend on live data: keep 10 min
    return StreamingResponse(gen_cached(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})
_BRIEF = {}
_RC = {}
def _chat_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS copilot_chats(id TEXT PRIMARY KEY, email TEXT, title TEXT, ts TEXT, data TEXT)"); c.execute("CREATE INDEX IF NOT EXISTS ix_chat_email ON copilot_chats(email, ts)"); return c
@app.get("/api/chats")
async def chats_list(request: Request):
    u = user_of(request)
    with _chat_db() as c: rows = c.execute("SELECT id, title, ts FROM copilot_chats WHERE email=? ORDER BY ts DESC LIMIT 60", (u["email"],)).fetchall()
    return {"ok": True, "chats": [{"id": r[0], "title": r[1], "ts": r[2]} for r in rows]}
@app.get("/api/chats/{cid}")
async def chats_get(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: r = c.execute("SELECT title, data FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"])).fetchone()
    if not r: raise HTTPException(404, "chat not found")
    return {"ok": True, "title": r[0], "msgs": json.loads(r[1])}
@app.post("/api/chats/save")
async def chats_save(request: Request):
    u = user_of(request); d = await request.json(); cid = re.sub(r"[^a-zA-Z0-9_-]", "", str(d.get("id") or ""))[:40] or ("c" + str(int(time.time() * 1000))); msgs = d.get("msgs") or []
    if not isinstance(msgs, list) or not msgs: return bad("nothing to save")
    msgs = [{"role": m.get("role"), "text": str(m.get("text", ""))[:20000], "en": (str(m["en"])[:20000] if m.get("en") else None), "lang": m.get("lang"), "sql": str(m.get("sql") or "")[:1500]} for m in msgs[-80:]]
    title = next((m["text"] for m in msgs if m["role"] == "user"), "Chat")[:70]; now = datetime.datetime.now().isoformat(timespec="seconds")
    with _chat_db() as c:
        o = c.execute("SELECT email FROM copilot_chats WHERE id=?", (cid,)).fetchone()
        if o and o[0] != u["email"]: raise HTTPException(403, "not your chat")
        c.execute("INSERT OR REPLACE INTO copilot_chats VALUES(?,?,?,?,?)", (cid, u["email"], title, now, json.dumps(msgs)))
    PS.soon()
    return {"ok": True, "id": cid}
@app.delete("/api/chats/{cid}")
async def chats_del(cid: str, request: Request):
    u = user_of(request)
    with _chat_db() as c: c.execute("DELETE FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"]))
    return {"ok": True}
@app.post("/api/weather/refresh")
async def wx_refresh(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run")
    if KIND != "Freight": raise HTTPException(404)
    import weather as WX; n, msg = await asyncio.to_thread(WX.refresh); LE.audit(u["email"], "weather_refresh", "ports", msg)
    return ok(("Real weather updated: " if n else "Could not reach Open-Meteo, simulated values stay: ") + msg, refresh=True)
@app.post("/api/brief")
async def brief(request: Request):
    u = user_of(request); d = await request.json(); lang = d.get("lang", "English"); tgt = gpu.code_of(lang); dec = VC.decision()
    def gen():
        if not dec: yield _sse("final", text="No decision yet."); return
        key = (dec["id"], tgt)
        if key in _BRIEF: yield _sse("final", text=_BRIEF[key]); return
        ev = json.dumps({k: v for k, v in dec["ev"].items() if k != "agents"}, default=str)[:1500] + " Agents: " + json.dumps(dec["ev"].get("agents", {}), default=str)[:1800]; txt = ""
        if gpu.llm_ready():
            for piece in gpu.llm_stream([{"role": "system", "content": "Write a 4-line executive briefing for the owner: status, biggest risk, what to approve now, what to watch. Use only numbers present in the data."}, {"role": "user", "content": ev}], max_new_tokens=200): txt += piece; yield _sse("tok", text=piece)
            if not _nums_ok(txt, ev): txt = ""
        txt = txt.strip() or dec["narrative"]; out = gpu.translate(txt, tgt, "eng_Latn")[0] if tgt != "eng_Latn" else txt; _BRIEF[key] = out; yield _sse("final", text=out)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ reports, exports, profile, invitations
@app.get("/api/report/{rid}")
async def report(rid: str, request: Request, arg: str = ""):
    u = user_of(request); c = ctx(u)
    if "reports" not in V.ROLE_MENU.get(u["role"], set()): raise HTTPException(403, "no access to reports")
    if c["scope"] and arg and not arg.startswith(c["scope"]) and rid == "fr_outlet": raise HTTPException(403, "outside your outlet")
    try: name, data = await asyncio.to_thread(RP.build_pdf, KIND, rid, c, arg or None)
    except KeyError: raise HTTPException(404, "unknown report")
    LE.audit(u["email"], "report", rid, arg); return Response(data, media_type="application/pdf", headers={"Content-Disposition": f'attachment; filename="{name}"'})
@app.post("/api/report/email")
async def report_email(request: Request):
    u = user_of(request); c = ctx(u); d = await request.json(); rid = d.get("rid")
    if "reports" not in V.ROLE_MENU.get(u["role"], set()): raise HTTPException(403)
    try: name, data = await asyncio.to_thread(RP.build_pdf, KIND, rid, c, d.get("arg") or None)
    except KeyError: return bad("unknown report")
    okk, msg = await asyncio.to_thread(mailer.send, u["email"], f"{V.NAME}: {name}", "Your report is attached. It was generated from the live data by the agent network.", [(name, data, "application/pdf")])
    LE.audit(u["email"], "report_email", rid, msg); return {"ok": okk, "msg": msg}
@app.get("/api/export/{table}.csv")
def export(table: str, request: Request):
    u = user_of(request); c = ctx(u)
    if "reports" not in V.ROLE_MENU.get(u["role"], set()): raise HTTPException(403)
    if table not in RP.EXPORTS[KIND]: raise HTTPException(404)
    LE.audit(u["email"], "export", table); return Response(RP.export_csv(KIND, table, c["scope"]), media_type="text/csv", headers={"Content-Disposition": f'attachment; filename="{table}.csv"'})
@app.post("/api/profile")
async def profile_save(request: Request):
    u = user_of(request); d = await request.json(); okk, msg = authx.update_name(u["email"], d.get("username", ""))
    if okk: authx.set_prefs(u["email"], lang=d.get("lang", "English")); LE.audit(u["email"], "profile", u["email"])
    return {"ok": okk, "msg": msg, "refresh": okk}
@app.post("/api/profile/prefs")
async def profile_prefs(request: Request):
    u = user_of(request); d = await request.json(); authx.set_prefs(u["email"], email_alerts=int(d.get("email_alerts") == "on"), daily_digest=int(d.get("daily_digest") == "on")); return ok("Preferences saved")
@app.post("/api/profile/password")
async def profile_pw(request: Request):
    u = user_of(request); d = await request.json(); okk, msg = await asyncio.to_thread(authx.change_password, u["email"], d.get("old", ""), d.get("new", ""))
    if okk: LE.audit(u["email"], "password_change", u["email"])
    return {"ok": okk, "msg": msg}
@app.post("/api/user/invite")
async def user_invite(request: Request):
    u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json(); okk, msg = await asyncio.to_thread(authx.invite, d.get("email", ""), d.get("username", ""), d.get("role", "")); LE.audit(u["email"], "invite", d.get("email", ""), d.get("role", "")); return {"ok": okk, "msg": msg, "refresh": okk}
@app.post("/api/user/deactivate")
async def user_deactivate(request: Request):
    u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json()
    if d["email"].lower() == u["email"].lower(): return bad("You cannot deactivate yourself.")
    authx.deactivate(d["email"], d.get("on") == "Deactivate"); LE.audit(u["email"], "deactivate" if d.get("on") == "Deactivate" else "reactivate", d["email"]); return ok("Done", refresh=True)

@app.get("/api/backup")
def backup(request: Request):
    import sqlite3, tempfile; u = user_of(request); c = ctx(u); need(c, "admin")
    fd, p = tempfile.mkstemp(suffix=".db"); os.close(fd); src = sqlite3.connect(DB_PATH); dst = sqlite3.connect(p); src.backup(dst); dst.close(); src.close()
    data = open(p, "rb").read(); os.remove(p); LE.audit(u["email"], "backup", DB_PATH); return Response(data, media_type="application/octet-stream", headers={"Content-Disposition": f'attachment; filename="{os.path.basename(DB_PATH)}"'})

@app.get("/api/health")
def health(): return {"ok": True, "gpu": gpu.status()}
@app.get("/")
def index(): return FileResponse(os.path.join(STATIC, "index.html"), headers={"Cache-Control": "no-store"})
app.mount("/static", StaticFiles(directory=STATIC), name="static")

@app.get("/api/copilot/suggest")
def copilot_suggest(request: Request, q: str = ""):
    user_of(request); return {"items": CP.suggest(q[:80], CP_EXAMPLES)}
@app.get("/api/copilot/csv")
def copilot_csv(request: Request, sid: str = "default"):
    from fastapi.responses import Response
    u = user_of(request); t = CP.last_csv(u["email"] + ":" + sid[:40])
    if t is None: raise HTTPException(404, "nothing to export yet")
    return Response(t, media_type="text/csv", headers={"Content-Disposition": "attachment; filename=copilot_result.csv"})
@app.post("/api/copilot/feedback")
async def copilot_feedback(request: Request):
    user_of(request); d = await request.json(); CP.feedback(str(d.get("q", "")), bool(d.get("helpful"))); return {"ok": True}
# ------------------------------------------------------------------ teach the Copilot (admin): find unanswered questions, add the answers
@app.post("/api/copilot/teach")
async def teach_add(request: Request):
    u = user_of(request); need(ctx(u), "admin"); d = await request.json()
    i = await asyncio.to_thread(CP.teach, d.get("question", ""), d.get("answer", ""), d.get("keywords", ""), u["email"])
    if not i: return bad("Give a question (4+ characters) and an answer (10+ characters).")
    LE.audit(u["email"], "copilot_teach", str(d.get("question", ""))[:80], "taught"); st = CP.teach_stats(); return ok(f"Saved. The Copilot will now answer this, and rephrased versions of it. {st['taught']} taught so far.", refresh=True)
@app.post("/api/copilot/teach/delete")
async def teach_delete(request: Request):
    u = user_of(request); need(ctx(u), "admin"); d = await request.json(); CP.taught_del(int(d.get("id", 0))); return ok("Removed.", refresh=True)
@app.post("/api/copilot/teach/import")
async def teach_import(request: Request, file: UploadFile = File(...)):
    u = user_of(request); need(ctx(u), "admin"); import csv as _csv, io as _io
    try: rows = list(_csv.DictReader(_io.StringIO((await file.read()).decode("utf-8-sig", errors="ignore"))))
    except Exception as e: return bad(f"Could not read the CSV: {e}")
    n = 0; skipped = 0
    for r in rows[:500]:
        r = {str(a or "").strip().lower(): (b or "") for a, b in r.items()}
        if await asyncio.to_thread(CP.teach, r.get("question") or r.get("q", ""), r.get("answer") or r.get("a", ""), r.get("keywords", ""), u["email"]): n += 1
        else: skipped += 1
    LE.audit(u["email"], "copilot_teach_import", file.filename, f"{n} taught"); return ok(f"{n} question(s) taught" + (f", {skipped} row(s) skipped (need question + answer columns)" if skipped else "") + f". {CP.teach_stats()['taught']} taught in total.", refresh=True)
@app.get("/api/copilot/teach/export")
async def teach_export(request: Request):
    u = user_of(request); need(ctx(u), "admin"); import csv as _csv, io as _io
    o = _io.StringIO(); w = _csv.writer(o); w.writerow(["question", "answer", "keywords", "added_by", "when"])
    for r in CP.taught_rows(): w.writerow([r["question"], r["answer"], r["keywords"], r["by"], r["when"]])
    return Response(o.getvalue(), media_type="text/csv", headers={"Content-Disposition": 'attachment; filename="taught_questions.csv"'})
@app.post("/api/copilot/probe")
async def copilot_probe(request: Request):
    """Runs a list of questions (one per line) through the real Copilot and reports which ones it could not answer. Unanswered ones land in the backlog."""
    u = user_of(request); need(ctx(u), "admin"); d = await request.json()
    qs = [re.sub(r"^\s*(\d+[.)]|[-*])\s*", "", x).strip() for x in str(d.get("questions", "")).splitlines()]; qs = [x for x in dict.fromkeys(qs) if len(x) > 3][:25]
    if not qs: return bad("Paste at least one question, one per line.")
    import hardening as _hd
    _sp = (request.scope.get("server") or (None, None))[1]; ports = [x for x in dict.fromkeys([_sp, int(os.environ.get("PORT", 0) or 0) or None, 8000, request.url.port]) if x]
    cookie = request.headers.get("cookie", ""); auth = request.headers.get("authorization", "")
    def run():
        import requests as _rq; out = []; base = None
        for pt in ports:                                              # find the port this server really listens on
            try:
                if _rq.get(f"http://127.0.0.1:{pt}/api/time", timeout=4).status_code < 500: base = f"http://127.0.0.1:{pt}"; break
            except Exception: continue
        for q in qs:
            route = "error"; ftxt = ""; fsql = ""; why = ""
            if not base: out.append({"question": q, "result": "Error", "route": "error", "reply": "Could not reach the server on localhost. Reload the page and try again."}); continue
            try:
                r = _rq.post(base + "/api/copilot", json={"q": q, "sid": "probe-" + u["email"]}, headers={"Cookie": cookie, "Authorization": auth, "X-Internal": _hd.INTERNAL}, stream=True, timeout=120)
                if r.status_code != 200: raise RuntimeError(f"HTTP {r.status_code}")
                for ln in r.iter_lines():
                    if ln.startswith(b"data: "):
                        try: ev = json.loads(ln[6:])
                        except Exception: continue
                        if ev.get("ev") == "final": ftxt = ev.get("text", "")
                        elif ev.get("ev") == "exact": fsql = ev.get("sql", "")
                with sqlite3.connect(DB_PATH, timeout=30) as c: row = c.execute("SELECT route FROM copilot_log WHERE q=? ORDER BY id DESC LIMIT 1", (q[:300],)).fetchone()
                route = row[0] if row else "unknown"
                if route in ("exact", "agents+llm", "exact+llm") and CP.weak_match(q, {"answer": ftxt, "sql": fsql}): route = "weak"; CP.log_turn(q, "weak", 0)
            except Exception as e: route = "error"; why = repr(e)[:80]
            out.append({"question": q, "result": "Error" if route == "error" else ("UNANSWERED" if route == "unanswered" else ("Weak match: check" if route == "weak" else "Answered")), "route": route, "reply": why if route == "error" else ftxt[:110].replace("\n", " ")})
        return out
    res = await asyncio.to_thread(run); na = sum(1 for r in res if r["result"] not in ("Answered", "Error")); er = sum(1 for r in res if r["result"] == "Error")
    return ok(f"{len(res)} tested: {len(res) - na - er} answered, {na} unanswered or weak (added to the backlog below)" + (f", {er} could not be tested (see the reply column)" if er else "") + ".", blocks=[B.table(res, ["question", B.pill_col("result"), "route", "reply"], h=min(420, 60 + 34 * len(res)), key="probe_out")], refresh=True)
@app.get("/api/copilot/insights")
def copilot_insights(request: Request):
    u = user_of(request); need(ctx(u), "admin"); return CP.insights()
@app.get("/api/time")
def api_time():
    n = datetime.datetime.now(); return {"iso": n.isoformat(timespec="seconds"), "tz": "Asia/Kolkata", "label": n.strftime("%a %d %b %Y, %I:%M %p IST")}
@app.get("/api/copilot/unanswered")
def copilot_unanswered(request: Request):
    u = user_of(request); need(ctx(u), "admin"); return {"rows": CP.unanswered()}
LE.set_chain_defaults(dom)
@app.post("/api/approval/chain")
async def approval_chain(request: Request):
    u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json(); cur = LE.get_state("approval_chain", {}) or {}; n = 0
    valid = set(dom.ACTION_AGENT) | {"req:" + k for k in dom.REQUEST_KINDS}
    for k, v in d.items():
        if k not in valid or v not in dom.CHAIN_PRESETS: continue
        ch = [x.strip() for x in v.split("→")]
        if cur.get(k) != ch: cur[k] = ch; n += 1; LE.audit(u["email"], "approval_chain", k, " → ".join(ch))
    LE.set_state("approval_chain", cur); return ok(f"{n} approval chain(s) saved. New requests use them at once.", refresh=True)


In [ ]:
%%writefile /content/app/sim_fq.py
"""Freight simulators (USD). Real sea routes (seanav), live shipments, quotes, carriers and ports; assumptions are shown on every page and are editable in the form."""
import numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
GREY = "#8b8d98"; STORM_AREAS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (15.0, 65.0), "Strait of Malacca": (3.5, 100.5), "South China Sea": (14.0, 114.0), "Bay of Biscay": (45.0, -8.0), "Cape of Good Hope": (-35.0, 20.0), "North Atlantic": (48.0, -35.0)}
def g(a, k, d):
    try: return float(a.get(k, d))
    except Exception: return float(d)
def usd(x):
    x = float(x); s = "-" if x < 0 else ""; x = abs(x); return s + ("$%.2fM" % (x / 1e6) if x >= 1e6 else "$%.1fk" % (x / 1e3) if x >= 1e4 else "${:,.0f}".format(x))
def _D():
    import domain_freight as D; return D
def hav(la1, lo1, la2, lo2):
    p = np.pi / 180; a = np.sin((la2 - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(la2 * p) * np.sin((lo2 - lo1) * p / 2) ** 2; return 2 * 3440.065 * np.arcsin(np.sqrt(a))
def _wrap(lon): return (np.asarray(lon, float) + 180) % 360 - 180
ASSUME = {"route": "Real sea routes on a land-masked graph. Speed 14 kn. Freight cost $0.012 per tonne-nautical-mile; cargo waiting costs $120 per shipment-day. Each +1 congestion adds 0.8 days at each end port. Closed chokepoints are removed from the graph; storms are avoided by a 700 nm radius.",
          "quote": "Win probability = 45% × exp(−elasticity × price change). Elasticity is a setting. Costs rise with the carrier-rate and fuel moves you set (fuel applies to the fuel surcharge). Expected profit = win probability × (price − cost).",
          "carrier": "Alternatives are ranked by rating, on-time % and cost index. Each alternative can absorb up to 30% more load. Delay cost $120 per shipment-day, and 1 point of on-time ≈ 0.08 days.",
          "weather": "Delay days = 0.25 × (wind − 12 m/s), capped at 7. A vessel is affected inside the storm radius. Demurrage and re-planning cost is a setting per shipment-day.",
          "customs": "Held shipments are those arriving soonest at the affected ports; each is delayed by the extra dwell days at the storage cost you set. Duty changes scale customs fees (average duty 7.5%)."}
def _cb(k): return B.callout(ASSUME[k], "info", "Assumptions")
def _lanes(limit=40):
    s = Q("SELECT s.shipment_id, s.origin_port o, s.dest_port d, s.carrier, s.weight_kg/1000.0 t, s.status FROM shipments s WHERE s.status IN ('In Transit','Port Congestion Delay','Anchorage Pending','Customs Hold') ORDER BY s.shipment_id")
    p = Q("SELECT port_name, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); pm = {r.port_name: r for r in p.itertuples()}
    s = s[s.o.isin(pm) & s.d.isin(pm) & (s.o != s.d)]; lanes = s.groupby(["o", "d"], as_index=False).agg(n=("shipment_id", "count"), t=("t", "sum")).sort_values("n", ascending=False).head(limit); return lanes, pm
# ------------------------------------------------------------------------------------------------ routes
_BASE = {}; LAST = {}
def run_route(a):
    import seanav; D = _D(); S = seanav.sea(); closed = [c for c in (a.get("closures") or []) if c in seanav.BLOCK_GROUPS]; area = a.get("area") or "None"; wind = g(a, "wind", 25); cong = g(a, "add", 0); dc = g(a, "delay_cost", 120); rate = g(a, "rate", 12) / 1000
    st = ((STORM_AREAS[area][0], STORM_AREAS[area][1], 700.0),) if area in STORM_AREAS and area != "None" else ()
    lanes, pm = _lanes(int(g(a, "lanes", 25))); rows = []
    for r in lanes.itertuples():
        o, d = pm[r.o], pm[r.d]; oo, dd = (o.lat, o.lon), (d.lat, d.lon)
        try:
            b = _BASE.get((r.o, r.d)) or _BASE.setdefault((r.o, r.d), S.options(oo, dd, (), (), D._access(oo), D._access(dd))); sc = S.options(oo, dd, tuple(closed), st, D._access(oo), D._access(dd)) or (S.options(oo, dd, tuple(closed), (), D._access(oo), D._access(dd)) if st else [])
        except Exception: continue
        if not b: continue
        bn = b[0]["nm"]; days0 = bn / (14 * 24) + (o.dw + d.dw) * .5
        if not sc: rows.append({"lane": f"{r.o} → {r.d}", "shipments": int(r.n), "tonnes": r.t, "nm_now": round(bn), "nm_scn": None, "days_now": round(days0, 1), "days_scn": None, "extra_cost": np.nan, "route_scn": "NO ROUTE", "route_now": b[0]["name"]}); continue
        sn = sc[0]["nm"]; days1 = sn / (14 * 24) + (o.dw + d.dw) * .5 + 0.8 * cong * 2; ex = (sn - bn) * rate * r.t + (days1 - days0) * dc * r.n
        rows.append({"dist_cost": (sn - bn) * rate * r.t, "delay_cost": (days1 - days0) * dc * r.n, "lane": f"{r.o} → {r.d}", "shipments": int(r.n), "tonnes": r.t, "nm_now": round(bn), "nm_scn": round(sn), "days_now": round(days0, 1), "days_scn": round(days1, 1), "extra_cost": ex, "route_scn": sc[0]["name"], "route_now": b[0]["name"]})
    if not rows: return {"blocks": [B.callout("No active lanes found.", "warn")], "m": {}}
    d = pd.DataFrame(rows); LAST["route"] = d[d.nm_scn.notna()]; ok = d[d.nm_scn.notna()]; nor = d[d.nm_scn.isna()]; top = ok.sort_values("extra_cost", ascending=False).head(15)
    bl = [_cb("route"), B.kpis(("Lanes re-routed", int((ok.route_scn != ok.route_now).sum()), f"of {len(d)} active lanes"), ("Lanes with no route", len(nor), "blocked completely" if len(nor) else "none", "Critical" if len(nor) else None), ("Extra cost / voyage set", usd(ok.extra_cost.sum()), f"{int(ok.shipments.sum())} shipments"), ("Average extra days", f"{(ok.days_scn - ok.days_now).mean():+.1f}", None, "Critical" if (ok.days_scn - ok.days_now).mean() > 3 else None)),
          B.card("Extra days on the worst lanes", B.chart(B.bar(top.lane, {"days now": list(top.days_now), "days under scenario": list(top.days_scn)}, horizontal=True, colors=[GREY, RED]), h=360, key="rt_c")),
          B.table(d.sort_values("extra_cost", ascending=False).head(25), ["lane", B.num_col("shipments", "shipments", "int"), B.num_col("nm_now", "nm now", "int"), B.num_col("nm_scn", "nm scenario", "int"), B.num_col("days_now", "days now", "num"), B.num_col("days_scn", "days scenario", "num"), B.num_col("extra_cost", "extra cost", "money"), "route_scn"], title="Lane by lane", h=420, key="rt_t")]
    return {"blocks": bl, "m": {"extra": float(ok.extra_cost.sum()), "no_route": len(nor), "rerouted": int((ok.route_scn != ok.route_now).sum()), "extra_days": float((ok.days_scn - ok.days_now).mean()), "lanes": len(d)}}
# ------------------------------------------------------------------------------------------------ quotes and margin
def run_quote(a):
    fuel, car, disc, fx, el = g(a, "pct", 0) / 100, g(a, "carrier", 0) / 100, g(a, "discount", 0) / 100, g(a, "fx", 0) / 100, g(a, "elasticity", 2.0)
    q = Q("SELECT quote_id, carrier, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct FROM freight_quotes WHERE final_price>0 ORDER BY created_at DESC LIMIT 400")
    if not len(q): return {"blocks": [B.callout("No quotes yet.", "warn")], "m": {}}
    p0 = q.final_price; c0 = p0 * (1 - q.margin_pct / 100); c1 = c0 + q.base_cost * ((1 + car) * (1 + fx) - 1) + q.customs_fee * fx + q.fuel_surcharge * fuel; m0 = (p0 - c0) / p0
    hold = p0; pass_ = c1 / (1 - m0.clip(.02, .6)) * (1 - disc)
    ev = lambda price, cost: (.45 * np.exp(-el * (price / p0 - 1)).clip(0, 2) * (price - cost)).sum()
    grid = np.arange(-.10, .21, .02); curve = [ev(p0 * (1 + x), c1) for x in grid]; bi = int(np.argmax(curve)); bx = float(grid[bi])
    q = q.assign(margin_now=m0, margin_hold=(hold - c1) / hold, margin_pass=(pass_ - c1) / pass_, cost_up=c1 - c0)
    kp = lambda v: float(v.mean())
    bl = [_cb("quote"), B.kpis(("Margin if prices are held", f"{kp(q.margin_hold):.1%}", f"now {kp(q.margin_now):.1%}", "Critical" if kp(q.margin_hold) < kp(q.margin_now) - .03 else None), ("Margin if costs are passed through", f"{kp(q.margin_pass):.1%}", f"price {((pass_ / p0).mean() - 1):+.1%}"), ("Quotes under 5% margin (held)", int((q.margin_hold < .05).sum()), f"now {int((q.margin_now < .05).sum())}", "Critical" if (q.margin_hold < .05).sum() > (q.margin_now < .05).sum() else None), ("Best price move", f"{bx:+.0%}", f"expected profit {usd(max(curve))} vs {usd(ev(p0, c1))} if held")),
          B.row(B.card("Expected profit vs price change", B.chart(B.line([f"{x:+.0%}" for x in grid], {"expected profit": [round(v) for v in curve]}, area=False, colors=[TEAL], markers=True), h=260, key="qt_c"), sub="Win probability falls as price rises (elasticity setting)"),
                B.card("Extra cost per quote", B.chart(B.hist(list(q.cost_up), bins=18, color=AMBER), h=260, key="qt_h"))),
          B.table(q.sort_values("margin_hold").head(15)[["quote_id", "carrier", "final_price", "cost_up", "margin_now", "margin_hold", "margin_pass"]], ["quote_id", "carrier", B.num_col("final_price", "price", "money"), B.num_col("cost_up", "extra cost", "money"), B.num_col("margin_now", "margin now", "pct"), B.num_col("margin_hold", "margin if held", "pct"), B.num_col("margin_pass", "margin if passed on", "pct")], title="Quotes to reprice first", h=400, open={"kind": "quote", "col": "quote_id"}, key="qt_t")]
    return {"blocks": bl, "m": {"margin_hold": kp(q.margin_hold), "margin_now": kp(q.margin_now), "margin_loss": float(((q.margin_now - q.margin_hold) * p0).sum()), "best_move": bx, "best_gain": float(max(curve) - ev(p0, c1)), "under5": int((q.margin_hold < .05).sum())}}
# ------------------------------------------------------------------------------------------------ carriers
def run_carrier(a):
    name = a.get("carrier"); loss = g(a, "loss", 100) / 100; dc = g(a, "delay_cost", 120); c = Q("SELECT name, rating, on_time_pct ot, avg_cost_index ci, risk_level FROM carriers")
    if not name or name not in set(c.name): name = Q("SELECT carrier, COUNT(*) n FROM shipments WHERE carrier IS NOT NULL GROUP BY carrier ORDER BY n DESC LIMIT 1").carrier[0]
    act = Q("SELECT shipment_id, carrier, weight_kg/1000.0 t, status FROM shipments WHERE status IN ('In Transit','Quoted','Anchorage Pending','Port Congestion Delay')"); load = act.groupby("carrier").size(); mine = act[act.carrier == name].sample(frac=loss, random_state=3) if len(act[act.carrier == name]) else act.iloc[0:0]
    c = c.assign(score=c.rating / 5 * 40 + c.ot * .4 - c.ci * 12 - (c.risk_level.str.lower().map({"low": 0, "medium": 6, "high": 14}).fillna(6))); cur = c[c.name == name].iloc[0]
    alts = c[(c.name != name) & c.name.isin(load.index)].sort_values("score", ascending=False).head(6).copy(); alts["cap"] = (load.reindex(alts.name).values * .30).round().clip(min=1)
    rows = []; room = dict(zip(alts.name, alts.cap))
    for r in mine.itertuples():
        pick = next((n for n in alts.name if room[n] > 0), None)
        if pick: room[pick] -= 1; al = alts[alts.name == pick].iloc[0]; rows.append({"shipment": r.shipment_id, "tonnes": r.t, "from": name, "to": pick, "cost_index": al.ci - cur.ci, "on_time": al.ot - cur.ot, "delay_cost": max(0, cur.ot - al.ot) * .08 * dc})
        else: rows.append({"shipment": r.shipment_id, "tonnes": r.t, "from": name, "to": "NO CAPACITY", "cost_index": np.nan, "on_time": np.nan, "delay_cost": np.nan})
    d = pd.DataFrame(rows)
    if not len(d): return {"blocks": [B.callout(f"{name} has no active shipments.", "info")], "m": {"unplaced": 0}}
    LAST["carrier"] = d; ok = d[d.to != "NO CAPACITY"]; ex = float((ok.cost_index * ok.tonnes * 400).sum()) + float(ok.delay_cost.sum())
    bl = [_cb("carrier"), B.kpis(("Shipments displaced", len(d), f"{name}, {loss:.0%} capacity lost"), ("Re-placed", len(ok), f"{len(d) - len(ok)} with no capacity", "Critical" if len(d) > len(ok) else None), ("Average cost index change", f"{ok.cost_index.mean():+.2f}" if len(ok) else "–"), ("Extra cost", usd(ex), "cost index × tonnes × $400 + delay cost")),
          B.card("Where the load goes", B.chart(B.bar(list(ok.groupby("to").size().index), {"shipments": list(ok.groupby("to").size().values)}, colors=[BLUE]), h=240, key="cr_c")),
          B.table(d.head(40), ["shipment", B.num_col("tonnes", "tonnes", "num"), "from", "to", B.num_col("cost_index", "cost index Δ", "num"), B.num_col("on_time", "on-time Δ (pts)", "num"), B.num_col("delay_cost", "delay cost", "money")], title="Re-assignments", h=380, key="cr_t")]
    return {"blocks": bl, "m": {"displaced": len(d), "unplaced": len(d) - len(ok), "extra": ex}}
# ------------------------------------------------------------------------------------------------ weather
def run_weather(a):
    area = a.get("area") or "Red Sea"; wind = g(a, "wind", 25); rad = g(a, "radius", 900); dem = g(a, "demurrage", 900); la, lo = STORM_AREAS.get(area, STORM_AREAS["Red Sea"])
    v = Q("SELECT v.shipment_id, v.lat, v.lon, v.progress, s.origin_port o, s.dest_port d, s.weight_kg/1000.0 t FROM vessels v JOIN shipments s USING(shipment_id)"); p = Q("SELECT port_name, lat, lon, avg_dwell_days dw, congestion_index ci FROM ports")
    v["dist"] = hav(v.lat.values, _wrap(v.lon.values), la, lo); hit = v[v.dist <= rad].copy(); p["dist"] = hav(p.lat.values, p.lon.values, la, lo); ph = p[p.dist <= rad]
    delay = lambda w: min(7.0, max(0.0, .25 * (w - 12))); dd = delay(wind); hit["delay_days"] = dd * (1 - hit.dist / rad * .5); hit["cost"] = hit.delay_days * dem
    sw = [(w, round(delay(w), 1), round(delay(w) * dem * max(len(hit), 0))) for w in (12, 18, 24, 30, 36)]
    bl = [_cb("weather"), B.kpis(("Vessels inside the storm", len(hit), f"of {len(v)} tracked", "Critical" if len(hit) > 5 else None), ("Delay per vessel", f"{dd:.1f} days", f"wind {wind:g} m/s"), ("Delay + demurrage cost", usd(hit.cost.sum())), ("Ports in the storm", len(ph), ", ".join(ph.port_name.head(3)) or "none")),
          B.card("Cost vs wind speed", B.chart(B.bar([f"{w} m/s" for w, _, _ in sw], {"cost": [c for _, _, c in sw]}, colors=[AMBER]), h=240, key="wx_s"), sub="For the vessels currently inside the radius"),
          B.table(hit.sort_values("cost", ascending=False).head(25)[["shipment_id", "o", "d", "t", "dist", "delay_days", "cost"]].rename(columns={"o": "origin", "d": "destination"}), [{"k": "shipment_id", "label": "shipment", "type": "id"}, "origin", "destination", B.num_col("t", "tonnes", "num"), B.num_col("dist", "nm from centre", "int"), B.num_col("delay_days", "delay days", "num"), B.num_col("cost", "cost", "money")], title="Vessels affected", h=380, open={"kind": "shipment", "col": "shipment_id"}, key="wx_t")]
    return {"blocks": bl, "m": {"hit": len(hit), "cost": float(hit.cost.sum()), "delay": dd}}
# ------------------------------------------------------------------------------------------------ customs
def run_customs(a):
    hp, dw, sc, duty = g(a, "hold", 20) / 100, g(a, "dwell", 4), g(a, "storage", 180), g(a, "duty", 0)
    s = Q("SELECT shipment_id, dest_port d, eta, weight_kg/1000.0 t, status FROM shipments WHERE status IN ('In Transit','Port Congestion Delay','Anchorage Pending','Customs Hold')").sort_values("eta")
    n = int(round(len(s) * hp)); held = s.head(n); cost = len(held) * dw * sc; by = held.groupby("d").size().sort_values(ascending=False).head(10)
    q = Q("SELECT SUM(customs_fee) cf, SUM(final_price) fp, COUNT(*) n FROM freight_quotes"); cf, fp = float(q.cf[0] or 0), float(q.fp[0] or 1); dcost = cf * duty / 7.5
    bl = [_cb("customs"), B.kpis(("Shipments held", len(held), f"{hp:.0%} of {len(s)} inbound", "Critical" if len(held) > 20 else None), ("Storage + delay cost", usd(cost), f"{dw:g} days × ${sc:g}"), ("Duty impact on open quotes", usd(dcost), f"{dcost / fp:.1%} of quote value", "Critical" if dcost / fp > .02 else None), ("Port most affected", by.index[0] if len(by) else "–", f"{int(by.iloc[0])} held" if len(by) else "")),
          B.card("Held shipments by destination", B.chart(B.bar(list(by.index), {"held": [int(x) for x in by.values]}, horizontal=True, colors=[VIOLET]), h=300, key="cu_c")),
          B.table(held.head(25), [{"k": "shipment_id", "label": "shipment", "type": "id"}, "d", "eta", B.num_col("t", "tonnes", "num"), "status"], title="First to be held", h=360, open={"kind": "shipment", "col": "shipment_id"}, key="cu_t")]
    return {"blocks": bl, "m": {"held": len(held), "cost": float(cost), "duty": float(dcost)}}
RUN = {"route": run_route, "carriers": run_carrier, "weather": run_weather, "quotes": run_quote, "margin": run_quote, "customs": run_customs}


In [ ]:
%%writefile /content/app/sim_fr.py
"""Franchise simulators. Every model reads the live tables, states its assumptions, and returns money-denominated results.
run_X(a) -> {"blocks": [...], "m": {metrics used by the Digital Twin}}.  All randomness is seeded, so base and scenario use common random numbers."""
import numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
GREY = "#8b8d98"
def g(a, k, d):
    try: return float(a.get(k, d))
    except Exception: return float(d)
def inr(x):
    x = float(x); s = "-" if x < 0 else ""; x = abs(x)
    return s + ("₹%.2f Cr" % (x / 1e7) if x >= 1e7 else "₹%.2f L" % (x / 1e5) if x >= 1e5 else "₹{:,.0f}".format(x))
ASSUME = {"inventory": "Daily demand is random (volatility is a setting). A reorder is placed when stock reaches the threshold and arrives after the lead time. Outlet revenue and costs are treated as monthly. Unit value = annualised revenue ÷ units demanded. Margin = 1 − operating costs ÷ revenue. 100 simulated 30-day futures per SKU, each with a shared ±10% demand swing across all SKUs; the same random numbers are used for baseline and scenario.",
          "workforce": "Hourly wage = salary ÷ 160. Overtime pays 1.5×, temp cover 1.3×. Each overtime hour per person per week adds 0.4 points of attrition probability. Replacing a leaver costs 50% of annual pay. Uncovered hours cost revenue in proportion to the lost labour share (80% productivity).",
          "pnl": "Price elasticity −1.2. 65% of operating costs are variable with volume; the rest are fixed. Revenue and costs in your outlets table are treated as monthly.",
          "marketing": "Response follows conversions = k·budget^α per channel (diminishing returns, α is a setting), with k fitted from your campaigns. The optimiser equalises marginal conversions across channels at a fixed total budget.",
          "sentiment": "New reviews are a mix of 1.5★ (negative wave) and 4.3★. Recovery reviews are 4.6★. Revenue moves 7% per star (public restaurant/retail studies; your own elasticity may differ).",
          "audit": "Probability an outlet fails = logistic(score vs pass mark 65, scale 6). Fine per violation is a setting. Remediation takes days you choose."}
def _cb(t): return B.callout(ASSUME[t], "info", "Assumptions (change the sliders; edit the model in sim_fr.py)")
# ------------------------------------------------------------------------------------------------ inventory
def _inv_frame():
    d = Q("SELECT i.outlet_id, i.sku_name, i.current_stock, i.reorder_threshold, i.weekly_demand, i.lead_time_days, o.revenue, o.operating_costs FROM inventory i JOIN outlets o USING(outlet_id)")
    d = d[d.weekly_demand > 0].reset_index(drop=True)
    tot = d.groupby("outlet_id").weekly_demand.transform("sum") * 52; d["uv"] = (d.revenue * 12 / tot).clip(1, 5000); d["gm"] = (1 - d.operating_costs / d.revenue).clip(.1, .6); return d
def _inv_core(d, dp, dl, buf, cv, T=30, R=100, seed=7):
    rng = np.random.default_rng(seed); N = len(d); dd = (d.weekly_demand.values / 7 * (1 + dp))[:, None]; lead = np.maximum(1, d.lead_time_days.values + int(round(dl)))[:, None]
    thr = (d.reorder_threshold.values * (1 + buf))[:, None]; Q_ = np.maximum(2 * d.weekly_demand.values[:, None] * (1 + dp), thr * 1.5)
    stock = np.repeat(d.current_stock.values[:, None].astype(np.float32), R, 1); pend = np.zeros((N, R), bool); arr = np.zeros((N, R)); lost = np.zeros((N, R), np.float32); out = np.zeros((T, N, R), bool)
    z = rng.standard_normal((T, N, R)).astype(np.float32); shock = np.exp(0.10 * rng.standard_normal((1, R))).astype(np.float32); day_lost = np.zeros((T, R), np.float64)
    uvv = d.uv.values[:, None]
    for t in range(T):
        dem = np.maximum(0, dd * shock * (1 + cv * z[t])); stock -= dem; short = np.maximum(0, -stock); lost += short; stock = np.maximum(stock, 0); out[t] = short > 0
        day_lost[t] = (short * uvv).sum(0)
        land = pend & (t >= arr); stock = np.where(land, stock + Q_, stock); pend &= ~land
        go = (~pend) & (stock <= thr); pend |= go; arr = np.where(go, t + lead, arr)
    return {"lost": lost, "out": out, "day_lost": day_lost, "stock0": d.current_stock.values, "dd": dd[:, 0], "Q": Q_[:, 0]}
def run_inventory(a):
    d = _inv_frame(); dp, dl, buf, cv = g(a, "pct", 0) / 100, g(a, "days", 0), g(a, "buffer", 0) / 100, g(a, "cv", 30) / 100; outs = a.get("outlets") or []
    if outs: d = d[d.outlet_id.isin(outs)].reset_index(drop=True)
    if not len(d): return {"blocks": [B.callout("No inventory rows for this selection.", "warn")], "m": {}}
    base = _inv_core(d, 0, 0, 0, cv); sc = _inv_core(d, dp, dl, 0, cv); uv = d.uv.values; gm = d.gm.values
    lv = lambda r: (r["lost"] * uv[:, None]); tot = lambda r: lv(r).sum(0)
    b_t, s_t = tot(base), tot(sc); p = lambda r: r["out"].sum(0).mean(1)
    bp, sp = p(base), p(sc); dem = d.weekly_demand.values / 7 * 30; fr = lambda r, dm: float(1 - r["lost"].mean(1).sum() / dm.sum()); gmm = float(np.average(gm, weights=uv * d.weekly_demand.values))
    daily = lambda r: np.cumsum(r["day_lost"], 0)
    cb, cs = daily(base), daily(sc); dif = (cs - cb) * gmm; x = [f"D{t + 1}" for t in range(cb.shape[0])]
    # buffer sweep under the scenario
    rows = []
    for b in (0, .1, .2, .3, .4, .5):
        r = _inv_core(d, dp, dl, b, cv); lost_m = float(tot(r).mean() * gmm); hold = float((d.reorder_threshold.values * b * uv * 0.25 * (30 / 365)).sum()); rows.append((b, lost_m, hold, lost_m + hold))
    best = min(rows, key=lambda r: r[3]); cur_total = rows[0][3]
    d["p_base"], d["p_scn"] = bp, sp; d["loss"] = lv(sc).mean(1); d["cover_days"] = d.current_stock / (d.weekly_demand / 7 * (1 + dp)).replace(0, np.nan)
    d["need_order"] = np.ceil(np.maximum(0, d.weekly_demand * (1 + dp) / 7 * (d.lead_time_days + dl + 3) - d.current_stock))
    top = d.sort_values("loss", ascending=False).head(15)
    byo = d.groupby("outlet_id").agg(loss=("loss", "sum"), skus_at_risk=("p_scn", lambda s: int((s > .5).sum()))).sort_values("loss", ascending=False).head(10).reset_index()
    bl = [_cb("inventory"), B.kpis(("Fill rate (demand served)", f"{fr(sc, dem * (1 + dp)):.1%}", f"now {fr(base, dem):.1%}", "Critical" if fr(sc, dem * (1 + dp)) < fr(base, dem) - .02 else None), ("Expected lost sales, 30 days", inr(s_t.mean()), f"P10–P90 {inr(np.percentile(s_t, 10))} – {inr(np.percentile(s_t, 90))}", None),
                              ("Lost gross margin", inr(s_t.mean() * gmm), f"+{inr((s_t.mean() - b_t.mean()) * gmm)} vs baseline"), ("Best safety buffer", f"+{best[0]:.0%}", f"saves {inr(cur_total - best[3])} vs none"), ("SKUs out of stock 5+ days", int((sp >= 5).sum()), f"now {int((bp >= 5).sum())}"))]
    bl.append(B.row(B.card("Cumulative lost sales over 30 days (expected)", B.chart(B.line(x, {"baseline": [round(float(v.mean()), 0) for v in cb], "scenario": [round(float(v.mean()), 0) for v in cs]}, area=False, colors=[GREY, RED]), h=260, key="inv_c"),
                           sub="Average of 100 simulated futures; P10 to P90 band is in the Digital Twin"),
                    B.card("Safety-buffer trade-off", B.chart(B.bar([f"+{r[0]:.0%}" for r in rows], {"lost margin": [round(r[1]) for r in rows], "holding cost": [round(r[2]) for r in rows]}, stack=True, colors=[RED, AMBER]), h=260, key="inv_b"), sub="Higher buffer: fewer stock-outs, more stock held")))
    bl.append(B.table(top[["outlet_id", "sku_name", "current_stock", "cover_days", "p_base", "p_scn", "loss", "need_order"]].rename(columns={"p_base": "days_out_now", "p_scn": "days_out_scn", "loss": "expected_lost_sales"}), ["outlet_id", "sku_name", B.num_col("current_stock", "stock", "int"), B.num_col("cover_days", "days of cover", "num"), B.num_col("days_out_now", "stock-out days now", "num"), B.num_col("days_out_scn", "under scenario", "num"), B.num_col("expected_lost_sales", "expected lost sales", "money"), B.num_col("need_order", "order now (units)", "int")], title="SKUs to act on first", h=420, open={"kind": "outlet", "col": "outlet_id"}, key="inv_t"))
    bl.append(B.table(byo, ["outlet_id", B.num_col("loss", "expected lost sales", "money"), B.num_col("skus_at_risk", "SKUs at risk", "int")], title="Most exposed outlets", h=300, open={"kind": "outlet", "col": "outlet_id"}, key="inv_o"))
    return {"blocks": bl, "m": {"d_samples": ((s_t - b_t) * gmm), "loss": float(s_t.mean() * gmm), "loss_base": float(b_t.mean() * gmm), "loss_p10": float(np.percentile(s_t, 10) * gmm), "loss_p90": float(np.percentile(s_t, 90) * gmm), "best_buffer": best[0], "buf_rows": rows, "buffer_saving": float(cur_total - best[3]), "at_risk": int((sp >= 5).sum()), "fill": fr(sc, dem * (1 + dp)), "fill_base": fr(base, dem), "curve": [float(v.mean()) for v in dif], "curve_lo": [float(np.percentile(v, 10)) for v in dif], "curve_hi": [float(np.percentile(v, 90)) for v in dif], "need": top[["outlet_id", "sku_name", "need_order"]].head(5).to_dict("records")}}
# ------------------------------------------------------------------------------------------------ workforce
def run_workforce(a):
    ab, cap, temp = g(a, "pct", 0) / 100, g(a, "ot_cap", 10), g(a, "temp", 0) / 100; outs = a.get("outlets") or []
    s = Q("SELECT outlet_id, COUNT(*) n, AVG(salary) sal, AVG(overtime_hrs) ot, AVG(predicted_attrition_prob) att FROM staff GROUP BY outlet_id"); o = Q("SELECT outlet_id, revenue FROM outlets")
    d = s.merge(o, on="outlet_id")
    if outs: d = d[d.outlet_id.isin(outs)]
    if not len(d): return {"blocks": [B.callout("No staff data.", "warn")], "m": {}}
    def model(ab_, cap_, temp_):
        n = d.n.values.astype(float); avail = n * (1 - ab_); need = (n - avail) * 40; hr = d.sal.values / 160; capw = cap_ * avail; ot_h = np.minimum(need, capw); rest = need - ot_h
        tmp_h = rest * temp_; gap = rest - tmp_h; ot_cost = ot_h * hr * 1.5 * 4.3; tmp_cost = tmp_h * hr * 1.3 * 4.3
        ot_pp = (ot_h / np.maximum(avail, 1)); att_up = np.minimum(.5, ot_pp * 0.004 * 1.0 * (1 if True else 0)); leav = n * att_up; repl = leav * d.sal.values * 12 * .5
        rev = gap / np.maximum(n * 40 * 4.3, 1) * 0.8 * d.revenue.values; return {"ot_cost": ot_cost, "tmp_cost": tmp_cost, "gap_h": gap * 4.3, "leav": leav, "repl": repl, "rev": rev, "cover": 1 - gap / np.maximum(need, 1) if need.sum() else np.ones(len(n))}
    b = model(0, cap, 0); s_ = model(ab, cap, temp); tot = lambda m: m["ot_cost"] + m["tmp_cost"] + m["repl"] + m["rev"]
    d = d.assign(ot_cost=s_["ot_cost"], tmp_cost=s_["tmp_cost"], gap_h=s_["gap_h"], leavers=s_["leav"], revenue_at_risk=s_["rev"], total=tot(s_), coverage=np.where((d.n * ab) > 0, 1 - s_["gap_h"] / np.maximum(d.n * ab * 40 * 4.3, 1), 1))
    sweep = [round(float(tot(model(v / 100, cap, temp)).sum())) for v in (0, 10, 20, 30, 40)]
    top = d.sort_values("total", ascending=False).head(15)
    bl = [_cb("workforce"), B.kpis(("Overtime + temp cost / month", inr(d.ot_cost.sum() + d.tmp_cost.sum()), f"overtime {inr(d.ot_cost.sum())}"), ("Uncovered labour hours", f"{d.gap_h.sum():,.0f}", "service gap" if d.gap_h.sum() else "fully covered", "Critical" if d.gap_h.sum() > 500 else None), ("Revenue at risk / month", inr(d.revenue_at_risk.sum())), ("Extra expected leavers", f"{d.leavers.sum():.1f}", f"replacement {inr(s_['repl'].sum())}")),
          B.row(B.card("Total monthly cost by absenteeism", B.chart(B.bar(["0%", "10%", "20%", "30%", "40%"], {"cost": sweep}, colors=[VIOLET]), h=250, key="wf_sw"), sub="Overtime, temp cover, replacements, lost revenue"),
                B.card("Outlets with the largest cost", B.chart(B.bar(top.outlet_id, {"overtime": [round(x) for x in top.ot_cost], "temp": [round(x) for x in top.tmp_cost], "revenue at risk": [round(x) for x in top.revenue_at_risk]}, stack=True, colors=[AMBER, BLUE, RED]), h=250, key="wf_o"))),
          B.table(top[["outlet_id", "n", "ot_cost", "tmp_cost", "gap_h", "leavers", "revenue_at_risk", "coverage"]], ["outlet_id", B.num_col("n", "staff", "int"), B.num_col("ot_cost", "overtime cost", "money"), B.num_col("tmp_cost", "temp cost", "money"), B.num_col("gap_h", "uncovered hrs", "int"), B.num_col("leavers", "extra leavers", "num"), B.num_col("revenue_at_risk", "revenue at risk", "money"), B.num_col("coverage", "coverage", "pct")], title="Outlets to staff first", h=420, open={"kind": "outlet", "col": "outlet_id"}, key="wf_t")]
    return {"blocks": bl, "m": {"cost": float(d.total.sum()), "ot": float(d.ot_cost.sum() + d.tmp_cost.sum()), "rev_risk": float(d.revenue_at_risk.sum()), "leavers": float(d.leavers.sum()), "gap_h": float(d.gap_h.sum()), "sweep": sweep, "model": lambda ab_, cap_, temp_: float(tot(model(ab_, cap_, temp_)).sum())}}
# ------------------------------------------------------------------------------------------------ outlet P&L
def run_pnl(a):
    dp, pp, cp = g(a, "pct", 0) / 100, g(a, "price", 0) / 100, g(a, "cost", 0) / 100; outs = a.get("outlets") or []
    o = Q("SELECT outlet_id, location, revenue, operating_costs FROM outlets")
    if outs: o = o[o.outlet_id.isin(outs)]
    e = -1.2; v = .65
    def pl(dp_, pp_, cp_):
        vol = (1 + dp_) * (1 + e * pp_); rev = o.revenue * vol * (1 + pp_); cost = o.operating_costs * (v * vol * (1 + cp_) + (1 - v) * (1 + cp_)); return rev, cost, rev - cost, vol
    r0, c0, p0, _ = pl(0, 0, 0); r1, c1, p1, vol = pl(dp, pp, cp)
    # effect decomposition (sequential)
    ps = [p0.sum(), pl(dp, 0, 0)[2].sum(), pl(dp, pp, 0)[2].sum(), p1.sum()]; deltas = [ps[1] - ps[0], ps[2] - ps[1], ps[3] - ps[2]]
    o = o.assign(profit_now=p0, profit_scn=p1, delta=p1 - p0, margin_scn=np.where(r1 > 0, p1 / r1, 0))
    sweep = [round(float(pl(x / 100, pp, cp)[2].sum())) for x in (-30, -15, 0, 15, 30)]
    worst = o.sort_values("delta").head(15)
    bl = [_cb("pnl"), B.kpis(("Total profit", inr(p1.sum()), f"{inr(p1.sum() - p0.sum())} vs now", "Critical" if p1.sum() < p0.sum() * .9 else "Low" if p1.sum() > p0.sum() * 1.02 else None), ("Margin", f"{p1.sum() / max(r1.sum(), 1):.1%}", f"now {p0.sum() / max(r0.sum(), 1):.1%}"), ("Loss-making outlets", int((p1 < 0).sum()), f"now {int((p0 < 0).sum())}", "Critical" if (p1 < 0).sum() > (p0 < 0).sum() else None), ("Revenue", inr(r1.sum()), f"{inr(r1.sum() - r0.sum())}")),
          B.row(B.card("What moved profit", B.chart(B.waterfall(["Demand", "Price", "Costs"], [round(x) for x in deltas], "Net change"), h=260, key="pl_w")),
                B.card("Profit vs demand change", B.chart(B.bar(["-30%", "-15%", "0", "+15%", "+30%"], {"profit": sweep}, colors=[TEAL]), h=260, key="pl_s"), sub="With your price and cost settings")),
          B.table(worst[["outlet_id", "location", "profit_now", "profit_scn", "delta", "margin_scn"]], ["outlet_id", "location", B.num_col("profit_now", "profit now", "money"), B.num_col("profit_scn", "profit scenario", "money"), B.num_col("delta", "change", "money"), B.num_col("margin_scn", "margin", "pct")], title="Outlets hit hardest", h=420, open={"kind": "outlet", "col": "outlet_id"}, key="pl_t")]
    return {"blocks": bl, "m": {"profit_delta": float(p1.sum() - p0.sum()), "loss_makers": int((p1 < 0).sum()), "profit": float(p1.sum()), "model": lambda dp_, pp_, cp_: float(pl(dp_, pp_, cp_)[2].sum() - p0.sum())}}
# ------------------------------------------------------------------------------------------------ marketing
def _mk_frame():
    m = Q("SELECT channel, SUM(budget) B, SUM(conversions) C, COUNT(*) n FROM marketing WHERE budget>0 GROUP BY channel"); m["channel"] = m.channel.replace({"Influencer Campaign": "Influencer", "Social Media": "Social"}); return m.groupby("channel", as_index=False).agg(B=("B", "sum"), C=("C", "sum"), n=("n", "sum"))
def run_marketing(a):
    bp, al = g(a, "pct", 0) / 100, g(a, "alpha", 70) / 100; m = _mk_frame()
    if not len(m): return {"blocks": [B.callout("No campaigns.", "warn")], "m": {}}
    k = m.C / m.B ** al; B0 = m.B.values; tot = B0.sum() * (1 + bp); conv = lambda Bv: float((k.values * Bv ** al).sum())
    w = (k.values * al) ** (1 / (1 - al)); opt = tot * w / w.sum(); keep = B0 * (1 + bp)
    m = m.assign(budget_now=B0, budget_keep=keep, budget_opt=opt, conv_now=m.C, conv_keep=k.values * keep ** al, conv_opt=k.values * opt ** al, cpc_now=B0 / m.C, cpc_opt=opt / (k.values * opt ** al))
    sweep = [round(conv(B0 * (1 + x / 100))) for x in (-30, -15, 0, 25, 50)]; up = conv(opt) - conv(keep)
    bl = [_cb("marketing"), B.kpis(("Conversions now", f"{m.C.sum():,.0f}"), ("Conversions with this budget", f"{conv(keep):,.0f}", f"{conv(keep) - m.C.sum():+,.0f}"), ("With optimal allocation", f"{conv(opt):,.0f}", f"+{up:,.0f} for the same spend", "Low" if up > 0 else None), ("Cost per conversion", inr(tot / conv(opt)), f"now {inr(B0.sum() / m.C.sum())}")),
          B.row(B.card("Budget by channel", B.chart(B.bar(m.channel, {"now": [round(x) for x in B0], "this scenario": [round(x) for x in keep], "optimal": [round(x) for x in opt]}, colors=[GREY, BLUE, TEAL]), h=270, key="mk_b")),
                B.card("Diminishing returns", B.chart(B.bar(["-30%", "-15%", "0", "+25%", "+50%"], {"conversions": sweep}, colors=[VIOLET]), h=270, key="mk_s"), sub="Total conversions as the whole budget changes")),
          B.table(m[["channel", "n", "budget_now", "budget_opt", "conv_now", "conv_opt", "cpc_now", "cpc_opt"]], ["channel", B.num_col("n", "campaigns", "int"), B.num_col("budget_now", "budget now", "money"), B.num_col("budget_opt", "optimal budget", "money"), B.num_col("conv_now", "conversions now", "int"), B.num_col("conv_opt", "conversions optimal", "int"), B.num_col("cpc_now", "cost/conv now", "money"), B.num_col("cpc_opt", "cost/conv optimal", "money")], title="Where to move money", h=330, key="mk_t")]
    return {"blocks": bl, "m": {"uplift": float(up), "conv": conv(keep), "conv_opt": conv(opt), "conv_now": float(m.C.sum())}}
# ------------------------------------------------------------------------------------------------ sentiment
def run_sentiment(a):
    neg, n_new, star = g(a, "pct", 25) / 100, g(a, "n_new", 30), 0.07; outs = a.get("outlets") or []
    f = Q("SELECT outlet_id, COUNT(*) n, AVG(rating) r FROM feedback GROUP BY outlet_id"); o = Q("SELECT outlet_id, revenue FROM outlets"); d = f.merge(o, on="outlet_id")
    if outs: d = d[d.outlet_id.isin(outs)]
    mix = neg * 1.5 + (1 - neg) * 4.3; d = d.assign(r_new=(d.n * d.r + n_new * mix) / (d.n + n_new)); d["rec_n"] = np.ceil(np.maximum(0, n_new * (d.r - mix) / (4.6 - d.r).clip(lower=.1)))
    d["rev_risk"] = np.maximum(0, d.r - d.r_new) * star * d.revenue; sw = [round(float((d.n * d.r + n_new * ((x / 100) * 1.5 + (1 - x / 100) * 4.3)).sum() / (d.n + n_new).sum()), 2) for x in (0, 10, 25, 50, 75)]
    wt = d.sort_values("rev_risk", ascending=False).head(15)
    bl = [_cb("sentiment"), B.kpis(("Average rating", f"{(d.r_new * d.n).sum() / d.n.sum():.2f}★", f"now {(d.r * d.n).sum() / d.n.sum():.2f}★"), ("Outlets below 3.5★", int((d.r_new < 3.5).sum()), f"now {int((d.r < 3.5).sum())}", "Critical" if (d.r_new < 3.5).sum() > (d.r < 3.5).sum() else None), ("Revenue at risk", inr(d.rev_risk.sum())), ("Good reviews needed to recover", f"{d.rec_n.sum():,.0f}", "4.6★ replies across the chain")),
          B.card("Chain rating vs share of negative reviews", B.chart(B.bar(["0%", "10%", "25%", "50%", "75%"], {"rating": sw}, colors=[AMBER], max_=5), h=240, key="se_s")),
          B.table(wt[["outlet_id", "n", "r", "r_new", "rev_risk", "rec_n"]], ["outlet_id", B.num_col("n", "reviews", "int"), B.num_col("r", "rating now", "num"), B.num_col("r_new", "rating after", "num"), B.num_col("rev_risk", "revenue at risk", "money"), B.num_col("rec_n", "good reviews to recover", "int")], title="Outlets most exposed", h=400, open={"kind": "outlet", "col": "outlet_id"}, key="se_t")]
    return {"blocks": bl, "m": {"rev_risk": float(d.rev_risk.sum()), "below35": int((d.r_new < 3.5).sum())}}
# ------------------------------------------------------------------------------------------------ audit
def run_audit(a):
    raise_ = g(a, "raise", 0); extra = g(a, "pct", 0) / 100; fine = g(a, "fine", 5000); days = g(a, "days", 14); outs = a.get("outlets") or []
    d = Q("SELECT outlet_id, score, violations FROM audits WHERE audit_id IN (SELECT audit_id FROM audits a2 WHERE a2.outlet_id=audits.outlet_id ORDER BY audit_date DESC LIMIT 1)")
    if outs: d = d[d.outlet_id.isin(outs)]
    if not len(d): return {"blocks": [B.callout("No audit data.", "warn")], "m": {}}
    pf = lambda sc, mark: 1 / (1 + np.exp((sc - mark) / 6)); p0 = pf(d.score.values, 65); sc1 = d.score.values - extra * 20; p1 = pf(sc1, 65 + raise_); v1 = d.violations.values * (1 + extra)
    d = d.assign(p_now=p0, p_scn=p1, viol_scn=v1, fines=p1 * v1 * fine, rem_days=np.where(p1 > .5, days, 0))
    sw = [round(float(pf(d.score.values - extra * 20, 65 + x).sum()), 1) for x in (0, 5, 10, 15)]
    wt = d.sort_values("p_scn", ascending=False).head(15)
    bl = [_cb("audit"), B.kpis(("Expected failing outlets", f"{p1.sum():.1f}", f"now {p0.sum():.1f}", "Critical" if p1.sum() > p0.sum() + 2 else None), ("Expected fines", inr(d.fines.sum())), ("Outlets likely to fail (>50%)", int((p1 > .5).sum()), f"now {int((p0 > .5).sum())}"), ("Outlet-days under remediation", f"{d.rem_days.sum():,.0f}")),
          B.card("Expected failures vs stricter pass mark", B.chart(B.bar(["65", "70", "75", "80"], {"expected failures": sw}, colors=[RED]), h=240, key="au_s")),
          B.table(wt[["outlet_id", "score", "violations", "p_now", "p_scn", "fines"]], ["outlet_id", B.num_col("score", "last score", "num"), B.num_col("violations", "violations", "int"), B.num_col("p_now", "fail risk now", "pct"), B.num_col("p_scn", "fail risk scenario", "pct"), B.num_col("fines", "expected fines", "money")], title="Outlets to prepare first", h=400, open={"kind": "outlet", "col": "outlet_id"}, key="au_t")]
    return {"blocks": bl, "m": {"fines": float(d.fines.sum()), "fail": float(p1.sum())}}
RUN = {"inventory": run_inventory, "workforce": run_workforce, "outlets": run_pnl, "marketing": run_marketing, "sentiment": run_sentiment, "audit": run_audit}


In [ ]:
%%writefile /content/app/twin.py
"""Digital Twin: a what-if engine that runs the *same agents and the same Intelligence Engine* on a private copy of the live database.
No separate model, no hand-written formula: the scenario edits the copy's data, the real agents re-evaluate it, and the page shows
the difference in health, per-agent scores and the tickets that WOULD be raised. The live database is never touched."""
import os, sqlite3, tempfile, json, copy, re
import numpy as np, pandas as pd
import live_engine as LE

def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))

def run_on_copy(dom, mutate=None):
    """Snapshot the live DB (SQLite online backup), apply `mutate(conn)`, run every agent + the engine on the copy, restore."""
    fd, tmp = tempfile.mkstemp(suffix=".twin.db"); os.close(fd); dst = sqlite3.connect(tmp); src = LE.conn()
    with LE._DB_LOCK: src.backup(dst)
    old = getattr(LE._TL, "c", None); LE._TL.c = dst
    try:
        if mutate: mutate(dst); dst.commit()
        s = {"cycle_id": "twin", "trigger": "twin", "request": "", "queue": [], "results": {}, "decision": {}}
        for name in [n for n in dom.AGENTS if n != "digest"]:
            try: s["results"][name] = dom.AGENTS[name](s)
            except Exception as e: s["results"][name] = LE.AgentResult(name, 50.0, "Medium", f"agent error on twin: {e}")
        d = dom.engine(s)
        return {"decision": d, "results": s["results"]}
    finally:
        LE._TL.c = old
        try: dst.close(); os.remove(tmp)
        except Exception: pass

def run_agents(dom, names, mutate=None):
    """Fast path for the per-agent simulators: private DB copy, run ONLY the named agents (no engine), return their results."""
    fd, tmp = tempfile.mkstemp(suffix=".twin.db"); os.close(fd); dst = sqlite3.connect(tmp); src = LE.conn()
    with LE._DB_LOCK: src.backup(dst)
    old = getattr(LE._TL, "c", None); LE._TL.c = dst
    try:
        if mutate: mutate(dst); dst.commit()
        s = {"cycle_id": "twin", "trigger": "twin", "request": "", "queue": [], "results": {}, "decision": {}}; out = {}
        for n in names:
            try: out[n] = dom.AGENTS[n](s); s["results"][n] = out[n]
            except Exception as e: out[n] = LE.AgentResult(n, 50.0, "Medium", f"agent error on twin: {e}")
        return out
    finally:
        LE._TL.c = old
        try: dst.close(); os.remove(tmp)
        except Exception: pass

def compare(dom, mutate, label=""):
    base = run_on_copy(dom, None); scen = run_on_copy(dom, mutate)
    bk = {(a["type"], str(a["target"])) for a in base["decision"]["actions"]}
    new = [a for a in scen["decision"]["actions"] if (a["type"], str(a["target"])) not in bk]
    gone = [a for a in base["decision"]["actions"] if (a["type"], str(a["target"])) not in {(x["type"], str(x["target"])) for x in scen["decision"]["actions"]}]
    rows = [{"agent": k, "before": base["results"][k].score, "after": scen["results"][k].score, "delta": round(scen["results"][k].score - base["results"][k].score, 1),
             "risk_before": base["results"][k].risk, "risk_after": scen["results"][k].risk, "summary_after": scen["results"][k].summary} for k in base["results"] if k in scen["results"]]
    per = None
    bh, sh = base["decision"].get("health"), scen["decision"].get("health")
    if bh and sh:
        per = pd.DataFrame({"before": pd.Series(bh), "after": pd.Series(sh)}).dropna(); per["delta"] = (per.after - per.before).round(1); per = per.sort_values("delta")
    return {"label": label, "base": base["decision"], "scen": scen["decision"], "agents": pd.DataFrame(rows), "new_actions": new, "cleared_actions": gone, "per_entity": per}

# ---------------------------------------------------------------- scenario catalogues (each one edits the data the real agents read)
def franchise_scenarios():
    def demand(pct, outlets):
        f = 1 + pct / 100
        def m(c):
            w = "" if not outlets else f" WHERE outlet_id IN ({','.join('?' * len(outlets))})"
            c.execute(f"UPDATE inventory SET weekly_demand=weekly_demand*?{w}", [f] + list(outlets)); c.execute(f"UPDATE sales SET amount=amount*?, tx_count=MAX(1,CAST(tx_count*? AS INT)){w}", [f, f] + list(outlets))
        return m
    def supplier(days, outlets):
        def m(c):
            w = "" if not outlets else f" WHERE outlet_id IN ({','.join('?' * len(outlets))})"; c.execute(f"UPDATE inventory SET lead_time_days=lead_time_days+?{w}", [int(days)] + list(outlets))
        return m
    def absent(pct, outlets):
        def m(c):
            w = "" if not outlets else f" AND outlet_id IN ({','.join('?' * len(outlets))})"
            ids = c.execute(f"SELECT staff_id, outlet_id FROM staff WHERE 1=1{w}", list(outlets)).fetchall(); rng = np.random.default_rng(11)
            for sid, oid in ids:
                if rng.random() < pct / 100:
                    for _ in range(12): c.execute("INSERT INTO attendance(staff_id,outlet_id,ts,status) VALUES(?,?,datetime('now','localtime'),'Absent')", (sid, oid))
                    c.execute("UPDATE staff SET overtime_hrs=MIN(45,overtime_hrs+8) WHERE staff_id=?", (sid,))
        return m
    def audit(outlets):
        def m(c):
            for o in outlets: c.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,date('now','localtime'),45,9,'Food Safety','Fail','twin')", ("AUD-TW" + o, o))
        return m
    def reviews(pct, outlets):
        def m(c):
            w = "" if not outlets else f" WHERE outlet_id IN ({','.join('?' * len(outlets))})"
            c.execute(f"UPDATE feedback SET rating=MAX(1, rating-2), comment='slow service dirty rude cold food wrong order' WHERE rowid IN (SELECT rowid FROM feedback{w} ORDER BY RANDOM() LIMIT (SELECT COUNT(*)*?/100 FROM feedback{w}))", list(outlets) + [int(pct)] + list(outlets))
        return m
    return {"Negative review wave": (reviews, ["pct", "outlets"]), "Demand change": (demand, ["pct", "outlets"]), "Supplier lead-time delay": (supplier, ["days", "outlets"]), "Staff absenteeism": (absent, ["pct", "outlets"]), "Failed inspection": (audit, ["outlets"])}

PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (15.0, 65.0), "Strait of Malacca": (3.5, 100.5), "South China Sea": (14.0, 114.0), "Bay of Biscay": (45.0, -8.0), "Cape of Good Hope": (-35.0, 20.0), "North Atlantic": (48.0, -35.0)}
def freight_scenarios():
    def storm(area, wind):
        def m(c):
            la, lo = PRESET_STORMS[area]; LE.set_state("storms", (LE.get_state("storms", []) or []) + [{"lat": la, "lon": lo, "r": 900, "wind": float(wind), "ts": LE.now()}])
        return m
    def congestion(add, region):
        def m(c):
            w = "" if region == "All" else " WHERE region=?"; c.execute(f"UPDATE ports SET congestion_index=MIN(5,congestion_index+?){w}", [float(add)] + ([] if region == "All" else [region]))
        return m
    def fuel(pct):
        def m(c):
            LE.set_state("fuel_pct", .12 * (1 + pct / 100)); k = pct / 100
            c.execute("UPDATE freight_quotes SET margin_pct=ROUND(margin_pct-(fuel_surcharge*?)/MAX(final_price,1)*100,2), fuel_surcharge=fuel_surcharge*(1+?)", (k, k))
            c.execute("UPDATE freight_quotes SET status='Pending Approval' WHERE margin_pct<10 AND status NOT IN ('On Hold','Rejected')")
        return m
    def customs(extra):
        def m(c): c.execute("UPDATE shipments SET status='Customs Hold' WHERE shipment_id IN (SELECT shipment_id FROM shipments WHERE status='In Transit' ORDER BY RANDOM() LIMIT ?)", (int(extra),))
        return m
    return {"Storm on a sea area": (storm, ["area", "wind"]), "Port congestion surge": (congestion, ["add", "region"]), "Fuel price move": (fuel, ["pct"]), "Customs crackdown": (customs, ["extra"])}

In [ ]:
%%writefile /content/app/twin2.py
"""Digital Twin v2: stack several shocks at once, run every quantitative model for this app, show the money impact with uncertainty bands, rank mitigations,
re-run the real agents on a private copy, save/compare scenarios, and send a chosen mitigation to the approval queue. Nothing here changes live data."""
import json, time
import numpy as np, pandas as pd
import blocks as B, live_engine as LE, twin as TW
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
GREY = "#8b8d98"
def _num(v, d=0.0):
    try: return float(v)
    except Exception: return float(d)
def fields(kind):
    if kind == "Franchise":
        outs = Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
        return [{"name": "pct", "label": "Demand change %", "type": "range", "min": -50, "max": 100, "value": 0}, {"name": "days", "label": "Supplier lead-time delay (days)", "type": "range", "min": 0, "max": 14, "value": 0}, {"name": "absent", "label": "Staff absent %", "type": "range", "min": 0, "max": 60, "value": 0},
                {"name": "price", "label": "Price change %", "type": "range", "min": -20, "max": 20, "value": 0}, {"name": "cost", "label": "Cost inflation %", "type": "range", "min": -20, "max": 40, "value": 0}, {"name": "neg", "label": "Share of new reviews that are negative %", "type": "range", "min": 0, "max": 100, "value": 0},
                {"name": "mk", "label": "Marketing budget change %", "type": "range", "min": -50, "max": 100, "value": 0}, {"name": "raise", "label": "Stricter audit pass mark (points)", "type": "range", "min": 0, "max": 15, "value": 0}, {"name": "cv", "label": "Demand volatility %", "type": "range", "min": 10, "max": 80, "value": 30},
                {"name": "outlets", "label": "Outlets affected (empty = all)", "type": "multi", "options": outs, "value": []}]
    ports = Q("SELECT carrier, COUNT(*) n FROM shipments WHERE carrier IS NOT NULL GROUP BY carrier ORDER BY n DESC LIMIT 25").carrier.tolist()
    return [{"name": "closures", "label": "Closed chokepoints", "type": "multi", "options": ["suez", "panama", "malacca", "hormuz", "bosphorus"], "value": []}, {"name": "area", "label": "Storm area", "type": "select", "options": ["None", "Red Sea", "Arabian Sea", "Strait of Malacca", "South China Sea", "Bay of Biscay", "Cape of Good Hope", "North Atlantic"], "value": "None"},
            {"name": "wind", "label": "Storm wind m/s", "type": "range", "min": 15, "max": 40, "value": 25}, {"name": "add", "label": "Port congestion increase", "type": "range", "min": 0, "max": 3, "step": .5, "value": 0}, {"name": "pct", "label": "Fuel price change %", "type": "range", "min": -30, "max": 80, "value": 0},
            {"name": "carrier_cost", "label": "Carrier rate change %", "type": "range", "min": -10, "max": 30, "value": 0}, {"name": "fx", "label": "Currency move %", "type": "range", "min": -10, "max": 15, "value": 0}, {"name": "carrier", "label": "Carrier with an outage", "type": "select", "options": ["None"] + ports, "value": "None"},
            {"name": "loss", "label": "Outage: share of its capacity lost %", "type": "range", "min": 10, "max": 100, "value": 100}, {"name": "hold", "label": "Extra customs holds % of inbound", "type": "range", "min": 0, "max": 60, "value": 0}, {"name": "dwell", "label": "Extra dwell days at customs", "type": "range", "min": 1, "max": 14, "value": 4},
            {"name": "elasticity", "label": "Customer price sensitivity", "type": "range", "min": 0.5, "max": 4, "step": .5, "value": 2}]
def _mut_fr(a):
    cat = TW.franchise_scenarios(); fs = []; outs = a.get("outlets") or []
    if _num(a.get("pct")): fs.append(cat["Demand change"][0](_num(a["pct"]), outs))
    if _num(a.get("days")): fs.append(cat["Supplier lead-time delay"][0](int(_num(a["days"])), outs))
    if _num(a.get("absent")): fs.append(cat["Staff absenteeism"][0](_num(a["absent"]), outs))
    if _num(a.get("neg")): fs.append(cat["Negative review wave"][0](_num(a["neg"]), outs))
    return (lambda c: [f(c) for f in fs]) if fs else None
def _mut_fq(a):
    cat = TW.freight_scenarios(); fs = []
    if a.get("area") not in (None, "None"): fs.append(cat["Storm on a sea area"][0](a["area"], _num(a.get("wind"), 25)))
    if _num(a.get("add")): fs.append(cat["Port congestion surge"][0](_num(a["add"]), "All"))
    if _num(a.get("pct")): fs.append(cat["Fuel price move"][0](_num(a["pct"])))
    if _num(a.get("hold")): fs.append(cat["Customs crackdown"][0](int(_num(a["hold"]) / 100 * 200)))
    return (lambda c: [f(c) for f in fs]) if fs else None
def _p(v): return f"{v:,.0f}"
def _cur(kind): return (lambda x: __import__("sim_fr").inr(x)) if kind == "Franchise" else (lambda x: __import__("sim_fq").usd(x))
def _fr(a):
    import sim_fr as S; inr = S.inr; args = dict(a); outs = a.get("outlets") or []
    inv = S.run_inventory({"pct": a.get("pct", 0), "days": a.get("days", 0), "buffer": 0, "cv": a.get("cv", 30), "outlets": outs}); wf = S.run_workforce({"pct": a.get("absent", 0), "ot_cap": 10, "temp": 0, "outlets": outs})
    pn = S.run_pnl({"pct": a.get("pct", 0), "price": a.get("price", 0), "cost": a.get("cost", 0), "outlets": outs}); se = S.run_sentiment({"pct": a.get("neg", 0) or 25, "n_new": 30 if _num(a.get("neg")) else 0, "outlets": outs}) if _num(a.get("neg")) else {"m": {"rev_risk": 0, "below35": 0}, "blocks": []}
    au = S.run_audit({"raise": a.get("raise", 0), "pct": min(1, max(0, _num(a.get("pct")) / 100 * -1)), "fine": 5000, "days": 14, "outlets": outs}); au0 = S.run_audit({"outlets": outs}); mk = S.run_marketing({"pct": a.get("mk", 0), "alpha": 70})
    mi, mw, mp, ms, ma, mm = inv["m"], wf["m"], pn["m"], se["m"], au["m"], mk["m"]
    comp = [("Lost sales margin (stock-outs)", -(mi.get("loss", 0) - mi.get("loss_base", 0))), ("Overtime, temp cover, attrition, lost output", -mw.get("cost", 0)), ("Profit change from demand, price, costs", mp.get("profit_delta", 0)), ("Revenue lost to rating drop", -ms.get("rev_risk", 0)), ("Audit fines", -(ma.get("fines", 0) - au0["m"].get("fines", 0)))]
    total = sum(v for _, v in comp); days = len(mi.get("curve", []))
    other = (comp[1][1] + comp[2][1] + comp[3][1] + comp[4][1])
    cur = np.array(mi.get("curve", [0])); lo = np.array(mi.get("curve_lo", [0])); hi = np.array(mi.get("curve_hi", [0]))
    acc = other * np.linspace(1 / max(days, 1), 1, max(days, 1)); p50, p10, p90 = -(cur) + acc, -(hi) + acc, -(lo) + acc
    # mitigations
    cand = []; rows = mi.get("buf_rows", [])
    for b, lm, hold, tot in rows:
        if b in (.2, .4) and rows: cand.append(("Raise safety stock +%d%% on all SKUs" % (b * 100), rows[0][3] - tot, "holding cost included", "inventory"))
    for t in (.5, 1.0):
        if mw.get("model") and _num(a.get("absent")): cand.append((f"Cover {int(t * 100)}% of the staffing gap with temps", mw["model"](_num(a["absent"]) / 100, 10, 0) - mw["model"](_num(a["absent"]) / 100, 10, t), "temp wages included", "workforce"))
    if mp.get("model"): cand.append(("Raise prices 3%", mp["model"](_num(a.get("pct")) / 100, _num(a.get("price")) / 100 + .03, _num(a.get("cost")) / 100) - mp["model"](_num(a.get("pct")) / 100, _num(a.get("price")) / 100, _num(a.get("cost")) / 100), "volume loss from elasticity included", "outlets"))
    if _num(a.get("days")) >= 2:
        i2 = S.run_inventory({"pct": a.get("pct", 0), "days": max(0, _num(a["days"]) - 2), "buffer": 0, "cv": a.get("cv", 30), "outlets": outs}); cand.append(("Expedite suppliers: recover 2 days of lead time", mi["loss"] - i2["m"]["loss"], "expedite fees NOT modelled", "inventory"))
    if mm.get("uplift", 0) > 0: cand.append((f"Re-allocate marketing to the best channels (+{mm['uplift']:,.0f} conversions at the same spend)", 0.0, "conversions, not money", "marketing"))
    cand = sorted([c for c in cand if c[1] > 0 or c[3] == "marketing"], key=lambda r: -r[1])[:6]
    parts = [inv, wf, pn, mk, se, au]
    return {"inv_samples": mi.get("d_samples"), "comp": comp, "total": total, "p10": p10, "p50": p50, "p90": p90, "mit": cand, "blocks": {"inventory": inv["blocks"], "workforce": wf["blocks"], "outlets": pn["blocks"], "marketing": mk["blocks"], "sentiment": se["blocks"], "audit": au["blocks"]},
            "kpi": {"Fill rate": f"{mi.get('fill', 0):.1%}", "Lost sales margin": inr(comp[0][1]), "Profit change": inr(comp[2][1]), "Overtime/temp/attrition": inr(comp[1][1]), "Rating revenue at risk": inr(comp[3][1]), "Audit fines": inr(comp[4][1]), "TOTAL": inr(total)}, "fmt": inr, "unit": "₹"}
def _fq(a):
    import sim_fq as S; usd = S.usd; closures = a.get("closures") or []; area = a.get("area") or "None"
    rt = S.run_route({"closures": closures, "area": area, "wind": a.get("wind", 25), "add": a.get("add", 0)}); qt = S.run_quote({"pct": a.get("pct", 0), "carrier": a.get("carrier_cost", 0), "fx": a.get("fx", 0), "discount": 0, "elasticity": a.get("elasticity", 2)})
    cr = S.run_carrier({"carrier": a.get("carrier") if a.get("carrier") != "None" else "", "loss": a.get("loss", 100)}) if a.get("carrier") not in (None, "None") else {"m": {"extra": 0, "unplaced": 0, "displaced": 0}, "blocks": []}
    wx = S.run_weather({"area": area if area != "None" else "Red Sea", "wind": a.get("wind", 25)}) if area != "None" else {"m": {"cost": 0, "hit": 0}, "blocks": []}
    cu = S.run_customs({"hold": a.get("hold", 0), "dwell": a.get("dwell", 4), "storage": 180, "duty": 0}) if _num(a.get("hold")) else {"m": {"cost": 0, "held": 0}, "blocks": []}
    mr, mq, mc, mw, mu = rt["m"], qt["m"], cr["m"], wx["m"], cu["m"]
    comp = [("Longer / re-routed voyages", -mr.get("extra", 0)), ("Margin lost on quotes if prices are held", -mq.get("margin_loss", 0)), ("Carrier outage re-placement", -mc.get("extra", 0)), ("Storm delay + demurrage", -mw.get("cost", 0)), ("Customs hold storage", -mu.get("cost", 0))]
    total = sum(v for _, v in comp); cand = []
    if mq.get("best_gain", 0) > 0: cand.append((f"Reprice quotes by {mq['best_move']:+.0%} (expected-profit optimum)", mq["best_gain"], "uses the price-sensitivity setting", "quotes"))
    if _num(a.get("hold")): cu2 = S.run_customs({"hold": _num(a["hold"]) / 2, "dwell": a.get("dwell", 4), "storage": 180, "duty": 0}); cand.append(("Pre-clear customs documents (halves holds)", mu["cost"] - cu2["m"]["cost"], "document-prep effort not modelled", "customs"))
    if area != "None" and mw.get("cost", 0) > mr.get("extra", 0): cand.append(("Reroute vessels around the storm instead of waiting", mw["cost"] - mr.get("extra", 0), "assumes the route detour is cheaper than the delay", "route"))
    if mc.get("displaced", 0): cand.append(("Pre-book capacity with the top alternative carriers", mc["extra"] * 0.25, "assumes 25% saving from pre-booking", "carriers"))
    if mr.get("no_route", 0): cand.append((f"Open an alternative gateway: {mr['no_route']} lanes have no route", 0.0, "see Route AI for the lanes", "route"))
    cand = sorted(cand, key=lambda r: -r[1])[:6]
    return {"comp": comp, "total": total, "p10": None, "p50": None, "p90": None, "mit": cand, "blocks": {"route": rt["blocks"], "quotes": qt["blocks"], "carriers": cr["blocks"], "weather": wx["blocks"], "customs": cu["blocks"]},
            "kpi": {"Lanes re-routed": mr.get("rerouted", 0), "Lanes with no route": mr.get("no_route", 0), "Margin if prices held": f"{mq.get('margin_hold', 0):.1%}", "Quotes under 5% margin": mq.get("under5", 0), "Shipments held": mu.get("held", 0), "TOTAL": usd(total)}, "fmt": usd, "unit": "$"}
_SIG = {"Lost sales margin (stock-outs)": .0, "Overtime, temp cover, attrition, lost output": .25, "Profit change from demand, price, costs": .30, "Revenue lost to rating drop": .35, "Audit fines": .40,
        "Longer / re-routed voyages": .30, "Margin lost on quotes if prices are held": .25, "Carrier outage re-placement": .35, "Storm delay + demurrage": .40, "Customs hold storage": .30}
def _mc(kind, r, f, n=3000, rho=.4):
    """Joint Monte Carlo of the total impact. Inventory uses its own simulated futures; every other driver is its modelled value times a lognormal factor (spread in _SIG) that shares a common shock (rho)."""
    import mc; rng = np.random.default_rng(5); comp = r["comp"]; z0 = rng.standard_normal(n); cols = {}; d_inv = r.get("inv_samples")
    for i, (nm, v) in enumerate(comp):
        sg = _SIG.get(nm, .3)
        if d_inv is not None and i == 0 and len(d_inv): cols[nm] = -np.asarray(rng.choice(d_inv, n), float)
        elif sg > 0 and v != 0: cols[nm] = v * np.exp(sg * (np.sqrt(rho) * z0 + np.sqrt(1 - rho) * rng.standard_normal(n)) - sg * sg / 2)
        else: cols[nm] = np.full(n, float(v))
    tot = sum(cols.values()); unit = "money"; fan = None
    if r.get("p50") is not None and len(r["p50"]): x = [f"D{i + 1}" for i in range(len(r["p50"]))]; sc = tot[:, None] / max(abs(float(r["total"])), 1.0) * np.asarray(r["p50"], float)[None, :] if r["total"] else None; fan = (x, sc) if sc is not None else None
    tor = []
    for nm, v in cols.items():
        m_ = float(v.mean()); tor.append((nm[:34], float(np.percentile(v, 10)) - m_, float(np.percentile(v, 90)) - m_))
    bl, s = mc.panel("total impact (30 days)", tot, unit, "low", today=0.0, fan=fan, tornado=tor, key="mct", note="Inventory comes from its own simulated futures; every other driver is its modelled value with a stated uncertainty (±25–40%) and a shared shock (correlation 0.4), because bad things tend to arrive together.")
    return [B.md("### Monte Carlo risk analysis (all shocks together)")] + bl
def run(c, a):
    kind = c["kind"]; dom = c["dom"]; t = time.time(); r = _fr(a) if kind == "Franchise" else _fq(a); f = r["fmt"]; comp = pd.DataFrame(r["comp"], columns=["driver", "impact"]).sort_values("impact")
    bl = [B.callout("A private what-if: every model reads a copy of your live data. All the shocks you set apply together. Negative numbers are money lost versus today.", "info")]
    tot = r["total"]; bl.append(B.kpis(("Total impact", f(tot), "vs today, next 30 days" if kind == "Franchise" else "vs today", "Critical" if tot < 0 else "Low" if tot > 0 else None), *[(k, v) for k, v in list(r["kpi"].items())[:-1][:4]]))
    bl.append(B.row(B.card("What costs the most", B.chart(B.bar(comp.driver, {"impact": [round(x) for x in comp.impact]}, horizontal=True, itemcolors=[RED if x < 0 else TEAL for x in comp.impact]), h=280, key="t2_t")),
                    B.card("Cumulative impact over 30 days" if r["p50"] is not None else "Impact by driver", B.chart(B.line([f"D{i + 1}" for i in range(len(r["p50"]))], {"expected": [round(float(x)) for x in r["p50"]]}, band=([round(float(x)) for x in r["p10"]], [round(float(x)) for x in r["p90"]]), area=False, colors=[RED]), h=280, key="t2_c"), sub="Line = expected; shaded = 10th to 90th percentile across 100 simulated futures (uncertainty from stock-outs; other costs are spread evenly)") if r["p50"] is not None else B.card("Breakdown", B.table(comp, ["driver", B.num_col("impact", "impact", "money")], h=280, key="t2_b"))))
    try: bl += _mc(kind, r, f)
    except Exception as e:
        import traceback, sys; traceback.print_exc(file=sys.stderr); bl.append(B.callout(f"Monte Carlo panel skipped: {e}", "warn"))
    if r["mit"]:
        mrows = [{"action": m[0], "benefit": m[1], "note": m[2], "area": m[3]} for m in r["mit"]]
        bl.append(B.card("Best responses (ranked by money saved)", B.table(mrows, ["action", B.num_col("benefit", "benefit", "money"), "note", "area"], h=min(80 + 42 * len(mrows), 330), key="t2_m"),
                         B.form("t2_prop", [{"name": "action", "label": "Action to send for approval", "type": "select", "options": [m[0] for m in r["mit"]], "value": r["mit"][0][0]}, {"name": "detail", "type": "hidden", "value": json.dumps({"scenario": {k: v for k, v in a.items() if v not in (None, "", [], 0, 0.0, "None")}, "kind": kind})}], "Propose to the approval queue", "/api/twin/propose", tone="ghost", title=None),
                         sub="Benefits come from the same models, assumptions listed on each simulator. Proposing creates a ticket that a human approves or rejects."))
    if a.get("agents"):
        mut = _mut_fr(a) if kind == "Franchise" else _mut_fq(a)
        if mut:
            res = TW.compare(dom, mut, "stacked"); ag = res["agents"]; b, s = res["base"], res["scen"]
            bl.append(B.card("What the real agents say on the copy", B.kpis(("Health before", b["score"]), ("Health after", s["score"], f"{s['score'] - b['score']:+.1f}", "Critical" if s["score"] < b["score"] - 1 else None), ("New tickets", len(res["new_actions"])), ("Cleared", len(res["cleared_actions"]))), B.callout(s["narrative"], "info"),
                             B.table(ag, ["agent", B.num_col("before", "before", "num"), B.num_col("after", "after", "num"), B.num_col("delta", "Δ", "num"), B.pill_col("risk_before", "risk before"), B.pill_col("risk_after", "risk after")], h=260, key="t2_ag")))
            if res["new_actions"]: bl.append(B.table(pd.DataFrame(res["new_actions"])[["type", "target", "severity", "detail"]].head(30), ["type", "target", B.pill_col("severity"), "detail"], title="Tickets this scenario would raise", h=300, key="t2_na"))
        else: bl.append(B.callout("The real agents can only be re-run for shocks that change data (demand, lead time, absence, reviews, storm, congestion, fuel, customs). Closed chokepoints are handled by the route model above.", "info"))
    bl += [B.card(k.title() + " model", *v) for k, v in r["blocks"].items() if v]
    return bl, {"total": tot, "kpi": r["kpi"], "comp": r["comp"], "unit": r["unit"]}
def _db():
    with LE._DB_LOCK, LE.conn() as c: c.execute("CREATE TABLE IF NOT EXISTS twin_scenarios(name TEXT PRIMARY KEY, args TEXT, ts TEXT, by TEXT, result TEXT)")
def save(name, args, by, result):
    _db()
    with LE._DB_LOCK, LE.conn() as c: c.execute("INSERT OR REPLACE INTO twin_scenarios VALUES(?,?,?,?,?)", (name[:60], json.dumps(args), LE.now(), by, json.dumps(result, default=str)))
def saved(): _db(); return Q("SELECT name, ts, by, args, result FROM twin_scenarios ORDER BY ts DESC")
def propose(kind, action, detail, actor):
    try: sc = json.loads(detail)
    except Exception: sc = {}
    txt = f"Digital Twin recommends: {action}. Based on scenario {json.dumps(sc.get('scenario', {}))}. Approve to put it into action; nothing has been changed yet."
    a = {"type": "twin_mitigation", "target": "NETWORK", "detail": txt, "severity": "Info"}
    with LE._DB_LOCK, LE.conn() as c: c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)", (LE.now(), a["type"], a["target"], txt, "Info", "Operations", (__import__("datetime").datetime.now() + __import__("datetime").timedelta(hours=72)).isoformat(timespec="seconds"), "twin", "proposed by " + actor, "proposed", json.dumps(a)))
    LE.audit(actor, "twin_propose", action[:60], ""); return txt


In [ ]:
%%writefile /content/app/validate.py
"""Row-level validation for every feed. Bad rows are quarantined with a reason (table feed_rejects); good rows still load.
Rules: required values, numeric ranges, allowed values, ISO dates, and foreign keys (only enforced once the parent table has rows)."""
import re, json
import pandas as pd
R = lambda lo=None, hi=None: ("range", lo, hi)
RULES = {
    "outlets": {"revenue": R(0), "operating_costs": R(0), "customer_satisfaction": R(0, 5), "staff_headcount": R(0, 5000)},
    "staff": {"salary": R(0, 1e8), "overtime_hrs": R(0, 400), "age": R(14, 90), "tenure_years": R(0, 70), "predicted_attrition_prob": R(0, 1), "job_satisfaction": R(0, 10), "work_life_balance": R(0, 10), "outlet_id": ("fk", "outlets", "outlet_id")},
    "inventory": {"current_stock": R(0), "reorder_threshold": R(0), "weekly_demand": R(0), "lead_time_days": R(0, 365), "stockout_risk_prob": R(0, 1), "outlet_id": ("fk", "outlets", "outlet_id")},
    "marketing": {"budget": R(0), "reach": R(0), "conversions": R(0), "actual_roi": R(-100, 1000), "outlet_id": ("fk", "outlets", "outlet_id"), "start_date": ("date",), "end_date": ("date",)},
    "audits": {"score": R(0, 100), "violations": R(0, 1000), "audit_date": ("date",), "outlet_id": ("fk", "outlets", "outlet_id")},
    "feedback": {"rating": R(1, 5), "sentiment_score": R(-1, 1), "date": ("date",), "outlet_id": ("fk", "outlets", "outlet_id")},
    "sales": {"amount": R(0, 1e9), "tx_count": R(0, 1e6), "ts": ("date",), "outlet_id": ("fk", "outlets", "outlet_id")},
    "ports": {"lat": R(-90, 90), "lon": R(-180, 180), "congestion_index": R(0, 10), "avg_dwell_days": R(0, 120)},
    "carriers": {"rating": R(0, 5), "on_time_pct": R(0, 100), "avg_cost_index": R(0, 10), "risk_level": ("enum", {"low", "medium", "high"})},
    "shipments": {"weight_kg": R(0, 1e9), "predicted_delay_risk": R(0, 1), "eta": ("date",), "origin_port": ("fk", "ports", "port_name"), "dest_port": ("fk", "ports", "port_name")},
    "freight_quotes": {"final_price": R(0, 1e10), "base_cost": R(0, 1e10), "margin_pct": R(-100, 100), "insurance": R(0), "customs_fee": R(0), "fuel_surcharge": R(0), "shipment_id": ("fk", "shipments", "shipment_id")},
    "vessels": {"lat": R(-90, 90), "lon": R(-180, 180), "speed_kn": R(0, 60), "progress": R(0, 1)},
    "customers": {"credit_risk": R(0, 1)},
    "quote_requests": {"weight_t": R(0, 1e6)},
    "customs_rules": {"duty_pct": R(0, 500)},
}
def _fk_values(conn, table, col):
    try: return {str(r[0]) for r in conn.execute(f"SELECT {col} FROM {table}")}
    except Exception: return set()
def validate(table, df, conn):
    """-> (clean df, rejects [{row, field, reason, value}])"""
    rules = RULES.get(table, {}); bad = {}; fk_cache = {}
    def mark(i, f, why, v): bad.setdefault(i, (f, why, str(v)[:60]))
    for col, rule in rules.items():
        if col not in df.columns: continue
        s = df[col]; kind = rule[0]
        if kind == "range":
            n = pd.to_numeric(s, errors="coerce")
            for i in df.index[s.notna() & n.isna()]: mark(i, col, "not a number", s[i])
            if rule[1] is not None:
                for i in df.index[n < rule[1]]: mark(i, col, f"below {rule[1]:g}", s[i])
            if rule[2] is not None:
                for i in df.index[n > rule[2]]: mark(i, col, f"above {rule[2]:g}", s[i])
        elif kind == "date":
            d = pd.to_datetime(s, errors="coerce", utc=True)
            for i in df.index[s.notna() & d.isna()]: mark(i, col, "not a valid date", s[i])
        elif kind == "enum":
            for i in df.index[s.notna() & ~s.astype(str).str.lower().isin(rule[1])]: mark(i, col, "value not allowed (" + "/".join(sorted(rule[1])) + ")", s[i])
        elif kind == "fk":
            key = (rule[1], rule[2])
            if key not in fk_cache: fk_cache[key] = _fk_values(conn, *key)
            if fk_cache[key]:
                for i in df.index[s.notna() & ~s.astype(str).isin(fk_cache[key])]: mark(i, col, f"unknown {rule[1]}.{rule[2]}", s[i])
    rej = [{"row": int(i) + 2, "field": f, "reason": w, "value": v} for i, (f, w, v) in sorted(bad.items())]
    return df.drop(index=list(bad)), rej


In [ ]:
%%writefile /content/app/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, live_rbac as RB, entities as EN
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE

def can(c, cap): return RB.can(c["role"], cap)
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    return df[df[col].astype(str).str.startswith(c["scope"])] if c.get("scope") and len(df) else df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_items(c, df, comments=True):
    dom = c["dom"]; now = LE.now(); ids = df.id.tolist(); cm = {}
    if comments and ids:
        for r in Q("SELECT action_id a, ts, author, body FROM ticket_comments WHERE action_id IN (%s) ORDER BY comment_id" % ",".join("?" * len(ids)), ids).itertuples(): cm.setdefault(int(r.a), []).append({"ts": r.ts[5:16], "by": r.author, "body": r.body})
    out = []; ch = LE.chains(ids) if ids else {}
    for r in df.itertuples():
        _c = ch.get(int(r.id), []); _nx = LE.next_role(_c)
        out.append({"chain": _c, "turn": _nx, "id": int(r.id), "sev": r.severity, "status": r.status, "type": r.type, "target": r.target, "detail": r.detail, "owner": r.owner if isinstance(r.owner, str) else "", "assignee": r.assignee if isinstance(r.assignee, str) else "", "due": (r.due if isinstance(r.due, str) else "")[:16], "note": r.note if isinstance(r.note, str) else "", "agent": dom.ACTION_AGENT.get(r.type, "engine"), "ts": (r.ts or "")[5:16],
                    "overdue": bool(isinstance(r.due, str) and r.due and r.due < now), "comments": cm.get(int(r.id), []), "can_approve": bool(can(c, "approve") and (not _nx or LE._role_ok(c["role"], _nx))), "can_resolve": bool(can(c, "resolve")), "quote": r.type == "quote_approval", "entity": EN.kind_of(r.target)})
    return out
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    df = tk_df(c, agent, types, target, limit=limit); return B.tickets(ticket_items(c, df), key=key or f"tk_{agent or target or 'all'}", title=title)
def hero(c, agent, title, blurb): return B.hero(agent, title, blurb, latest(agent), can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center
def command(c, tab="ops"):
    dom, kind = c["dom"], c["kind"]; tabs = [{"id": "ops", "label": "Operations"}, {"id": "agents", "label": "Agents"}, {"id": "anomaly", "label": "Anomalies"}, {"id": "net", "label": "Knowledge graph"}, {"id": "explore", "label": "Entity explorer"}]
    if c.get("scope"): sub = f"{dom.NAME} · scoped to {c['scope']}"
    else: sub = f"{dom.NAME} · live agent network"
    acts = [{"id": "run", "label": "Run agent cycle", "url": "/api/run_cycle", "tone": "primary"}] if can(c, "run") else []
    if tab == "agents": return B.page("Command Center", sub, _agents_tab(c), tabs, live=8, actions=acts)
    if tab == "anomaly": return B.page("Command Center", sub, _anomaly_tab(c), tabs, live=12, actions=acts)
    if tab == "net": return B.page("Command Center", sub, kg_blocks(c), tabs, live=12, actions=acts)
    if tab == "explore": return B.page("Command Center", sub, entity_explorer(c), tabs, live=0, actions=acts)
    return B.page("Command Center", sub, kind_ops(c), tabs, live=4, actions=acts)

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}
def approvals(c, tab="awaiting"):
    dom = c["dom"]; tabs = [{"id": "awaiting", "label": "Awaiting approval"}, {"id": "tickets", "label": "All open tickets"}, {"id": "request", "label": "Request approval"}, {"id": "matrix", "label": "Who approves what"}, {"id": "history", "label": "Decisions & audit trail"}] + ([{"id": "policy", "label": "Autonomy policy"}] if can(c, "admin") else [])
    sub = "Everything that needs a human: agent proposals, quote approvals, ML second opinions and Digital Twin recommendations. Approve to execute, reject to dismiss; every decision is audited."
    if tab == "policy" and can(c, "admin"):
        return B.page("Approvals", sub, [B.callout("Choose which agent actions run on their own (auto) and which wait here for a human (approval), and who signs off each kind. Changes apply to the next agent cycle and are written to the audit log.", "info"), policy_form(c), chain_form(c)], tabs, live=0)
    if tab == "request":
        who = c["email"]; mine = Q("SELECT action_id id, ts, detail, severity, status, approval decision FROM agent_actions WHERE type='manual_request' AND note LIKE ? ORDER BY action_id DESC LIMIT 50", (f"%requested by {who};%",)) if not can(c, "admin") else Q("SELECT action_id id, ts, detail, severity, status, approval decision FROM agent_actions WHERE type='manual_request' ORDER BY action_id DESC LIMIT 100")
        ch = LE.chains(mine.id.tolist()) if len(mine) else {}
        if len(mine): mine["progress"] = mine.id.map(lambda i: " → ".join(f"{s['role']} {'✓' if s['status']=='approved' else '✗' if s['status']=='rejected' else '…'}" for s in ch.get(int(i), [])))
        kinds = list(dom.REQUEST_KINDS)
        bl = [B.callout("Ask for a decision on anything the agents did not raise themselves. Every kind of request has its own sign-off chain (see the tab Who approves what). Pick 2 levels to add an Admin sign-off on top. Nobody can approve their own request or two steps of the same one.", "info"),
              B.form("req", [{"name": "kind", "label": "What do you need approved?", "type": "select", "options": kinds}, {"name": "target", "label": "Outlet / shipment / quote ID (optional)", "type": "text", "value": c.get("scope") or ""},
                             {"name": "detail", "label": "Details and why", "type": "textarea"}, {"name": "severity", "label": "Urgency", "type": "select", "options": ["Info", "Warning", "Critical"], "value": "Warning"},
                             {"name": "levels", "label": "Approval levels (1 = the standard chain for this request, 2 = also an Admin)", "type": "select", "options": ["1", "2"], "value": "1"}], "Send for approval", "/api/approval/request", title="New approval request")]
        bl.append(B.table(mine[["id", "ts", "detail", "severity", "progress", "status"]], ["id", "ts", "detail", B.pill_col("severity"), "progress", B.pill_col("status")], title="Requests (yours; Admin sees all)", h=300, key="ap_req") if len(mine) else B.callout("No requests yet.", "info"))
        return B.page("Approvals", sub, bl, tabs, live=0)
    if tab == "matrix":
        return B.page("Approvals", sub, approval_matrix(c), tabs, live=0)
    if tab == "history":
        h = Q("SELECT action_id id, ts, type, target, severity, approval decision, status, note FROM agent_actions WHERE approval IN ('approved','rejected') OR status IN ('resolved','rejected') ORDER BY action_id DESC LIMIT 300"); h = scoped(h, c) if len(h) else h
        bl = []
        if len(h):
            h["decision"] = h.decision.fillna(h.status); h["by"] = h.note.fillna("").str.extract(r"(?:approved|rejected) by (\S+)")[0].fillna("").str.rstrip(";,")
            n_ap = int((h.decision == "approved").sum()); n_rj = int((h.decision == "rejected").sum())
            bl.append(B.kpis(("Approved", n_ap), ("Rejected", n_rj, None, "Warning" if n_rj else None), ("Resolved", int((h.status == "resolved").sum())), ("Approval rate", f"{n_ap / max(1, n_ap + n_rj):.0%}")))
            bl.append(B.table(h.assign(type=h.type.map(lambda t: SOURCE.get(t, t)))[["id", "ts", "type", "target", "severity", "decision", "by"]], ["id", "ts", "type", "target", B.pill_col("severity"), B.pill_col("decision"), "by"], title="Recent decisions", h=360, open={"col": "target"}, key="ap_h", search=True))
        else: bl.append(B.callout("No decisions recorded yet. Approve or reject something on the first tab.", "info"))
        if can(c, "admin"):
            a = Q("SELECT ts, actor, event, target, detail FROM audit_log WHERE event IN ('approve','reject','policy','twin_propose','reroute','resolve','assign') ORDER BY audit_id DESC LIMIT 200")
            bl.append(B.table(a, ["ts", "actor", B.pill_col("event"), "target", "detail"], title="Who decided what (audit trail)", h=320, key="ap_a", search=True))
        return B.page("Approvals", sub, bl, tabs, live=0)
    df = tk_df(c, limit=300)
    if tab == "tickets":
        n_over = int(df.detail.str.startswith("OVERDUE").sum()) if len(df) else 0
        bl = [B.kpis(("Open + proposed", len(df)), ("Critical", int((df.severity == "Critical").sum()) if len(df) else 0, None, "Critical"), ("Overdue / escalated", n_over, f"escalated to {dom.ESCALATE_TO}", "Critical" if n_over else None)),
              B.tickets(ticket_items(c, df), key="tk_all", title="Tickets", sub="Assign, comment and resolve. Overdue tickets escalate automatically.")]
        if can(c, "run"): bl.append(B.button("sla", "Check SLAs now", "/api/sla", "ghost"))
        return B.page("Approvals", sub, bl, tabs, live=6)
    aw = df[(df.status == "proposed") | (df.type == "quote_approval")] if len(df) else df
    src = aw.type.map(lambda t: SOURCE.get(t, "Agent proposal")).value_counts() if len(aw) else pd.Series(dtype=int)
    try: oldest = f"{max(0, (pd.Timestamp(LE.now()) - pd.to_datetime(aw.ts, errors='coerce').min()).total_seconds() / 3600):.0f} h" if len(aw) else "–"
    except Exception: oldest = "–"
    bl = [B.kpis(("Awaiting your decision", len(aw), None, "Warning" if len(aw) else None), ("Critical", int((aw.severity == "Critical").sum()) if len(aw) else 0, None, "Critical"), ("Oldest waiting", oldest), ("From agents", int(src.get("Agent proposal", 0))), ("Quotes / ML / Twin", int(len(aw) - src.get("Agent proposal", 0))))]
    if len(aw): bl.append(B.tickets(ticket_items(c, aw), key="tk_aw", title="Needs a decision", sub="Approve executes the action (re-route, quote, stock transfer…); reject dismisses it. Add a note and it is kept in the audit trail."))
    else: bl.append(B.callout("Nothing is waiting for you. Agents act on their own where the autonomy policy says auto; anything set to approval, every ML second opinion and every Digital Twin recommendation shows up here.", "success", "All clear"))
    return B.page("Approvals", sub, bl, tabs, live=6)

def kind_ops(c):
    import importlib; return importlib.import_module("views_franchise" if c["kind"] == "Franchise" else "views_freight").ops(c)

def _tickets_tab(c):
    dom = c["dom"]; blocks = []
    df = tk_df(c, limit=250); n_prop = int((df.status == "proposed").sum()); n_over = int(df.detail.str.startswith("OVERDUE").sum()) if len(df) else 0
    blocks.append(B.kpis(("Open + proposed", len(df)), ("Awaiting your approval", n_prop, "agent proposals not yet executed", "Warning" if n_prop else None), ("Overdue / escalated", n_over, None, "Critical" if n_over else None), ("Critical", int((df.severity == "Critical").sum()), None, "Critical")))
    blocks.append(B.tickets(ticket_items(c, df), key="tk_all", title="Tickets", sub="Approve or reject what the agents propose, assign, comment and resolve. Everything is recorded in the audit log."))
    if can(c, "admin"): blocks.append(policy_form(c))
    return blocks
def _chain_txt(ch): return " → ".join(ch)
def approval_matrix(c):
    """Every approval in the product on one page: agent actions (auto or approval) and every kind of manual request, with its sign-off chain and deadline."""
    dom = c["dom"]; pol = LE.policy(dom); cfg = LE.get_state("approval_chain", {}) or {}; rows = []
    for t in sorted(dom.ACTION_AGENT):
        ch = cfg.get(t) or dom.DEFAULT_CHAINS.get(t) or []
        rows.append({"what": t.replace("_", " "), "raised by": "Agent: " + dom.ACTION_AGENT[t], "mode": pol.get(t, "auto"), "sign-off chain": _chain_txt(ch) if pol.get(t, "auto") == "approval" else "none: the agent acts and logs it", "owner": dom.OWNER.get(t, "")})
    for k in dom.REQUEST_KINDS:
        rows.append({"what": k, "raised by": "A person (Request approval tab)", "mode": "approval", "sign-off chain": _chain_txt(cfg.get("req:" + k) or dom.REQUEST_KINDS[k]), "owner": "the approvers"})
    df = pd.DataFrame(rows); n_ap = int((df["mode"] == "approval").sum())
    bl = [B.kpis(("Things that need approval", n_ap), ("Run automatically", len(df) - n_ap), ("Agent action types", len(dom.ACTION_AGENT)), ("Kinds of request", len(dom.REQUEST_KINDS))),
          B.callout("Each row is one kind of decision. approval = a human signs off, in the order shown, before anything runs. Critical items get an extra Admin sign-off. Nobody can approve their own request or two steps of the same one. An Admin can change a chain or a mode on the Autonomy policy tab.", "info"),
          B.table(df, ["what", "raised by", B.pill_col("mode"), "sign-off chain", "owner"], title="Who approves what", h=520, key="ap_matrix", search=True)]
    return bl
def chain_form(c):
    dom = c["dom"]; cfg = LE.get_state("approval_chain", {}) or {}; fields = []
    def f(key, label, default):
        cur = _chain_txt(cfg.get(key) or default); opts = list(dom.CHAIN_PRESETS) + ([cur] if cur not in dom.CHAIN_PRESETS else []); fields.append({"name": key, "label": label, "type": "select", "options": opts, "value": cur})
    for t in sorted(dom.ACTION_AGENT):
        if t in dom.DEFAULT_CHAINS or t in cfg: f(t, "Agent action: " + t.replace("_", " "), dom.DEFAULT_CHAINS.get(t, ["Operations Manager"]))
    for k, v in dom.REQUEST_KINDS.items(): f("req:" + k, "Request: " + k, v)
    return B.form("chain", fields, "Save approval chains", "/api/approval/chain", title="Approval chains", sub="Who signs off, and in which order, for each kind of approval. Critical items still get an extra Admin sign-off unless you set the chain here.")
def policy_form(c):
    pol = LE.policy(c["dom"]); fields = [{"name": t, "label": t.replace("_", " "), "type": "select", "options": ["auto", "approval"], "value": pol.get(t, "auto")} for t in sorted(c["dom"].ACTION_AGENT)]
    return B.form("policy", fields, "Save autonomy policy", "/api/policy", title="Autonomy policy", sub="auto = the agent executes and logs it. approval = the action is only proposed until a human approves it (human in the loop).")

def _agents_tab(c):
    dom = c["dom"]; blocks = []
    t = Q("SELECT substr(ts,12,8) time, cycle_id cycle, agent, risk, latency_ms ms, summary FROM agent_runs ORDER BY run_id DESC LIMIT 40")
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 90").iloc[::-1]
    blocks.append(B.row(B.card("Agent timeline (live): planner, agents, re-plan loop, engine", B.table(t, ["time", "cycle", "agent", B.pill_col("risk"), B.num_col("ms", "ms", "int"), "summary"], h=380, key="timeline")),
                        B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=170, key="trend"), B.table(pd.DataFrame(dom.feed_status(), columns=["feed", "status"]), title="Data feeds", h=170, key="feeds")), ratio=[3, 2]))
    e = Q("SELECT type, COUNT(*) raised, SUM(status IN ('resolved','approved')) resolved, SUM(status='auto-closed') auto_closed, SUM(status IN ('open','proposed')) open_, SUM(status='rejected') rejected FROM agent_actions GROUP BY type ORDER BY raised DESC")
    if len(e):
        e["agent"] = e.type.map(lambda x: dom.ACTION_AGENT.get(x, "engine")); e["fix_rate"] = ((e.resolved + e.auto_closed) / e.raised * 100).round(0)
        pf = Q("SELECT agent, COUNT(*) runs, ROUND(AVG(latency_ms)) avg_ms, ROUND(AVG(score),1) avg_score FROM agent_runs GROUP BY agent ORDER BY runs DESC")
        blocks.append(B.row(B.card("Agent effectiveness: raised, fixed by people, auto-closed once the problem went away", B.table(e, ["agent", "type", B.num_col("raised", "raised", "int"), B.num_col("resolved", "resolved", "int"), B.num_col("auto_closed", "auto-closed", "int"), B.num_col("open_", "open", "int"), B.num_col("fix_rate", "fixed %", "bar", 100)], h=300, key="eff")),
                            B.card("Latency per agent (ms)", B.chart(B.bar(pf.agent, pf.avg_ms, horizontal=True, colors=[BLUE]), h=300, key="lat")), ratio=[3, 2]))
    return blocks

def _anomaly_tab(c):
    dom, kind = c["dom"], c["kind"]; agent = "anomaly" if kind == "Franchise" else "anomaly_watch"; r = latest(agent); blocks = []
    blocks.append(B.hero(agent, "Anomaly detection (IsolationForest)", "Each outlet's sales vs baseline, volatility and sentiment." if kind == "Franchise" else "Quote economics: price/base ratio, margin, fuel share.", r, can_run=can(c, "run")))
    o = (r or {}).get("out", {}); fl = set(o.get("flagged", []))
    if kind == "Franchise" and o.get("all"):
        d = pd.DataFrame(o["all"]); d["flag"] = d.outlet_id.isin(fl) | (d.ratio < .6); d = d if not c.get("scope") else d[d.outlet_id == c["scope"]]
        g = {"Normal": [{"x": a.ratio, "y": a.vol, "label": a.outlet_id} for a in d[~d.flag].itertuples()], "Flagged": [{"x": a.ratio, "y": a.vol, "label": a.outlet_id} for a in d[d.flag].itertuples()]}
        blocks.append(B.card("Sales vs baseline and volatility (click a dot)", B.chart(B.scatter(g, "sales ÷ baseline", "volatility", colors={"Normal": TEAL, "Flagged": RED}), h=340, open={"scatter": True}, key="an_sc")))
    elif kind == "Freight":
        q = Q("SELECT quote_id, base_cost, final_price price, margin_pct margin FROM freight_quotes WHERE base_cost>0 ORDER BY rowid DESC LIMIT 600"); q["ratio"] = q.price / q.base_cost; q["flag"] = q.quote_id.isin(fl)
        g = {"Normal": [{"x": a.ratio, "y": a.margin, "label": a.quote_id} for a in q[~q.flag].itertuples()], "Flagged": [{"x": a.ratio, "y": a.margin, "label": a.quote_id} for a in q[q.flag].itertuples()]}
        blocks.append(B.card("Price ÷ base cost vs margin (click a dot)", B.chart(B.scatter(g, "price ÷ base", "margin %", colors={"Normal": TEAL, "Flagged": RED}), h=340, open={"scatter": True}, key="an_sc")))
    blocks += kg_blocks(c, compact=True)
    x = cross(c, fl, agent)
    if x: blocks.append(x)
    blocks.append(ticket_block(c, agent, title="Anomaly tickets", key="tk_anom")); return blocks

# ------------------------------------------------------------------ Knowledge graph
def kg_network(c, sev=("Critical", "Warning"), limit=110):
    dom = c["dom"]; a = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 260"); a = scoped(a, c); a = a[a.severity.isin(sev)].head(limit)
    nodes, links, seen = [], [], set()
    def add(i, kind, size, color=None, tip="", label=False, open=None):
        if i not in seen: seen.add(i); nodes.append({"id": i, "kind": kind, "size": size, "color": color, "tip": tip, "label": label, "open": open})
    for t, tg, sv, d in a.itertuples(index=False):
        ag = dom.ACTION_AGENT.get(t, "engine"); add(ag, "agent", 34, TEAL, "agent", True); add(t, "ticket type", 22, VIOLET, "action type", True); add(tg, "entity", 11, TONE.get(sv, "#8b8d98"), d[:120], False, tg); links += [(ag, t), (t, tg)]
    multi = a.groupby("target").agg(issues=("type", "nunique"), agents=("type", lambda s: len({dom.ACTION_AGENT.get(x, "engine") for x in s}))).sort_values(["agents", "issues"], ascending=False); multi = multi[multi.issues > 1].reset_index().rename(columns={"target": "entity"})
    return nodes, links, multi
def kg_blocks(c, compact=False):
    nodes, links, multi = kg_network(c)
    if not nodes: return [B.callout("No open tickets to graph.", "info")]
    out = [B.card("Live knowledge graph: agent → ticket type → entity (click an entity to open it)", B.chart(B.graph(nodes, links), h=480, open={"graph": True}, key="kg_net"), sub="Built from the agents' tickets in real time. Drag nodes, scroll to zoom.")]
    if len(multi): out.append(B.table(multi, ["entity", B.num_col("issues", "issue types", "int"), B.num_col("agents", "agents", "int")], title="Entities with several different problems at once (compounding risk)", h=220, open={"col": "entity"}, key="kg_multi"))
    return out
def kg_page(c, tab="net"):
    tabs = [{"id": "net", "label": "Network"}, {"id": "explore", "label": "Entity explorer"}]
    if tab == "explore": return B.page("Knowledge Graph", "Everything the platform knows about one entity", entity_explorer(c), tabs, live=0)
    return B.page("Knowledge Graph", "Built live from the agents' tickets and the operational tables", kg_blocks(c), tabs, live=12)
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e or opts[0]
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


In [ ]:
%%writefile /content/app/views_franchise.py
"""FranchiseOps AI pages (server-driven blocks)."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, live_rbac as RB, entities as EN
import domain_franchise as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FranchiseOps AI"
MENU = [("copilot", "AI Copilot", "bot"), ("approvals", "Approvals", "check"), ("workforce", "Agent 1 · Workforce", "users"), ("outlets", "Agent 2 · Outlets", "store"), ("inventory", "Agent 3 · Inventory", "boxes"), ("marketing", "Agent 4 · Marketing", "megaphone"), ("sentiment", "Agent 5 · Sentiment", "messages"), ("audit", "Agent 6 · Audit", "clipboard"),
        ("twin", "Digital Twin", "flask"), ("reports", "Reports", "chart"), ("notifications", "Notifications", "bell"), ("feeds", "Data Feed Center", "upload"), ("admin", "Admin", "shield"), ("profile", "My Profile", "user"), ("command", "Command Center", "radar")]
_ALL = {m[0] for m in MENU}; _OPS = _ALL - {"feeds", "admin"}
ROLE_MENU = {"Admin": _ALL, "Franchise Owner": _OPS | {"feeds"}, "Store Manager": {"copilot", "approvals", "workforce", "outlets", "inventory", "sentiment", "audit", "reports", "notifications", "twin", "profile", "command"}, "Staff": {"copilot", "approvals", "notifications", "profile", "command"}, "Customer": {"copilot", "sentiment", "profile"}}
def color_h(h): return RED if h < 60 else AMBER if h < 80 else TEAL
def _sc(c, col="outlet_id"): return (f" WHERE {col}=?", (c["scope"],)) if c.get("scope") else ("", ())
def model_card(name, keys):
    m = Q("SELECT metrics, trained_at FROM ml_models WHERE model_name=? ORDER BY model_id DESC LIMIT 1", (name,))
    if not len(m): return None
    try: d = json.loads(m.metrics[0])
    except Exception: d = {}
    return B.kpis(*[(k, d.get(k, "–")) for k in keys], ("trained", (m.trained_at[0] or "")[5:16]))

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}) if d else {}
    g = Q("SELECT g.outlet_id, g.lat, g.lon, o.outlet_name, o.location FROM outlet_geo g JOIN outlets o USING(outlet_id)"); g = g[g.outlet_id == c["scope"]] if c.get("scope") else g
    tk = Q("SELECT target, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").n; oa = tk_df(c, limit=500)
    pts = [{"id": r.outlet_id, "lat": r.lat, "lon": r.lon, "color": color_h(health.get(r.outlet_id, 70)), "r": 6 + min(10, int(tk.get(r.outlet_id, 0)) * 1.2), "tip": f"<b>{r.outlet_id}</b> {r.outlet_name}<br>{r.location}<br>health <b>{health.get(r.outlet_id, '–')}</b> · {int(tk.get(r.outlet_id, 0))} open tickets", "open": r.outlet_id} for r in g.itertuples() if r.outlet_id in health]
    mp = B.map_([{"type": "points", "id": "outlets", "pts": pts}], h=500, view={"bounds": [[7, 69], [34, 90]]}, key="ops_map", legend=[("Outlet health ≥ 80", TEAL), ("Health 60–79 (watch)", AMBER), ("Health < 60 (critical)", RED), ("Bigger dot = more open tickets", "#8b8d98")])
    hs = pd.Series(health).sort_values() if health else pd.Series(dtype=float); hs = hs[hs.index == c["scope"]] if c.get("scope") else hs
    need = pd.DataFrame({"outlet": hs.index[:10], "health": hs.values[:10]}); need["tickets"] = need.outlet.map(tk).fillna(0).astype(int)
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; sales = Q("SELECT substr(ts,12,8) time, outlet_id, ROUND(amount) amount, source FROM sales ORDER BY sale_id DESC LIMIT 7")
    by_ag = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int); ex = dom.kpi_extra()
    bl = [B.kpis(("Franchise health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), (ex[0][0], ex[0][1]), (ex[1][0], ex[1][1]))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Outlet map · click an outlet", mp), B.col(B.card("Needs attention first", B.table(need, ["outlet", B.num_col("health", "health", "bar", 100), B.num_col("tickets", "tickets", "int")], h=250, open={"col": "outlet"}, key="need")),
                                                                B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=150, key="trend"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Live sales stream", B.table(sales, ["time", "outlet_id", B.num_col("amount", "amount", "money"), B.pill_col("source")], h=240, open={"col": "outlet_id"}, key="sales")), B.card("Open tickets by agent", B.chart(B.bar(by_ag.index, by_ag.values, horizontal=True, colors=[BLUE]), h=240, key="byag"))))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k != "outlet": return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}); o = Q("SELECT outlet_name, location, revenue, customer_satisfaction csat, staff_headcount hc FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return {"title": e, "blocks": [B.callout("Unknown outlet.", "warn")]}
    r = o.iloc[0]; bd = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: bd[a] = round(v, 1)
    s = Q("SELECT substr(ts,12,8) t, amount FROM sales WHERE outlet_id=? ORDER BY sale_id DESC LIMIT 50", (e,)).iloc[::-1]
    lo = Q("SELECT sku_name, current_stock stock, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 5", (e,))
    h = health.get(e); bl = [B.kpis(("Health", h if h is not None else "–", None, "Critical" if h is not None and h < 60 else "Warning" if h is not None and h < 80 else "Low"), ("Satisfaction", f"{r.csat:.2f}"), ("Headcount", int(r.hc)), ("Revenue", f"{r.revenue:,.0f}"))]
    if bd: bl.append(B.card("Why: score from each agent", B.chart(B.bar(list(bd), list(bd.values()), horizontal=True, max_=100, itemcolors=[color_h(v) for v in bd.values()]), h=200, key="d_bd"), sub="weights: outlet 30% · inventory 20% · staff 20% · marketing 15% · audit 15%"))
    if len(s): bl.append(B.card("Live sales stream", B.chart(B.line(s.t, s.amount, colors=[TEAL]), h=170, key="d_s")))
    bl.append(B.table(lo, ["sku_name", "stock", "wk", "lead"], title="Lowest-cover SKUs", h=190, key="d_lo"))
    try:
        import alts; bl += alts.outlet_alternatives(e)
    except Exception as ex: bl.append(B.callout(f"Transfer suggestions unavailable: {ex}", "info"))
    bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
    if can(c, "run"): bl.append(B.button("rerun", "Re-run all agents now", "/api/run_cycle", "ghost"))
    return {"title": f"{e} · {r.outlet_name}", "sub": r.location, "blocks": bl}

# ------------------------------------------------------------------ agent pages
def workforce(c, tab=None):
    w, p = _sc(c); s = Q("SELECT staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction sat, predicted_attrition_prob p FROM staff" + w, p)
    bl = [hero(c, "staff", "Workforce agent", "Attrition risk (LogisticRegression), attendance and overtime for every employee. Feeds 20% of the outlet health score."),
          B.kpis(("Employees", f"{len(s):,}"), ("Avg attrition risk", f"{s.p.mean():.0%}"), ("High risk (>60%)", int((s.p > .6).sum()), None, "Warning"), ("Avg overtime h", round(s.overtime_hrs.mean(), 1)))]
    g = s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(12); at = Q("SELECT status, COUNT(*) n FROM attendance" + w + " GROUP BY status", p)
    bl.append(B.row(B.card("Attrition probability distribution", B.chart(B.hist(s.p, 24, TEAL, pct=True), h=270, key="wf_h")), B.card("Outlets with highest average attrition risk (click)", B.chart(B.bar(g.index, (g.values * 100).round(1), horizontal=True, colors=[AMBER], pct=True), h=270, open={"bar": "y"}, key="wf_b"))))
    t = dom.attrition_table(c.get("scope"), 15); imp = dom.attrition_importance()
    bl.append(B.row(B.card("Live attendance (all recorded shifts)", B.chart(B.pie(at.set_index("status").n.to_dict(), colors={"Present": TEAL, "Late": AMBER, "Absent": RED}), h=270, key="wf_a")),
                    B.card("Highest-risk employees and their main driver", B.table(t[["staff_id", "outlet_id", "name", "role", "prob", "driver"]] if len(t) else pd.DataFrame(), ["staff_id", "outlet_id", "name", "role", B.num_col("prob", "attrition", "pct"), "driver"], h=270, open={"col": "outlet_id"}, key="wf_t"))))
    if imp: ip = pd.Series(imp).sort_values(); bl.append(B.card("What pushes attrition up (positive) or down (negative)", B.chart(B.bar(ip.index, ip.values.round(3), horizontal=True, colors=[VIOLET]), h=240, key="wf_i")))
    mc = model_card("logreg_attrition", ["accuracy", "auc", "source"]); bl += [mc] if mc else []
    x = cross(c, s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(15).index, "staff"); bl += [x] if x else []
    bl.append(ticket_block(c, "staff", title="Open tickets from this agent", key="tk_staff")); return B.page("Workforce", "Agent 1 · attrition, attendance, overtime", bl, live=12)

def outlets(c, tab=None):
    r = latest("outlet"); out = (r or {}).get("out", {}); tiers = pd.Series(out.get("tiers", {})); d = decision(); health = (d or {}).get("ev", {}).get("health", {})
    o = Q("SELECT o.outlet_id, o.outlet_name, o.location, o.revenue, o.customer_satisfaction csat, o.staff_headcount hc, g.lat, g.lon FROM outlets o LEFT JOIN outlet_geo g USING(outlet_id)"); o["tier"] = o.outlet_id.map(tiers); o["health"] = o.outlet_id.map(health); o = o[o.outlet_id == c["scope"]] if c.get("scope") else o
    order = ["Excellent", "Good", "Needs Attention", "Critical"]; col = {"Excellent": TEAL, "Good": BLUE, "Needs Attention": AMBER, "Critical": RED}; tc = o.tier.value_counts().reindex(order).fillna(0)
    bl = [hero(c, "outlet", "Outlet performance agent", "KMeans (k=4) over revenue, orders and customers ranks every outlet. Drives 30% of the health score."),
          B.kpis(("Outlets", len(o)), ("Avg health", f"{o.health.mean():.1f}" if o.health.notna().any() else "…"), ("Critical tier", int((o.tier == "Critical").sum()), None, "Critical"), ("Silhouette", out.get("silhouette", "n/a")))]
    sc = {t: [{"x": a.revenue, "y": a.csat, "label": a.outlet_id, "size": 6 + a.hc / 4} for a in o[o.tier == t].itertuples()] for t in order if (o.tier == t).any()}
    bl.append(B.row(B.card("Outlets per KMeans tier", B.chart(B.bar(order, tc.values.astype(int).tolist(), itemcolors=[col[t] for t in order]), h=290, key="ou_t")), B.card("Revenue vs satisfaction (size = headcount · click a dot)", B.chart(B.scatter(sc, "revenue", "satisfaction", colors=col, sized=True), h=290, open={"scatter": True}, key="ou_s"))))
    bl.append(B.table(o.sort_values("health")[["outlet_id", "outlet_name", "location", "tier", "health", "csat", "hc"]], ["outlet_id", "outlet_name", "location", B.pill_col("tier"), B.num_col("health", "health", "bar", 100), B.num_col("csat", "csat", "num"), B.num_col("hc", "staff", "int")], title="All outlets · click a row for the drill-down", h=360, open={"col": "outlet_id"}, key="ou_tb", search=True))
    x = cross(c, o.sort_values("health").head(15).outlet_id, "outlet"); bl += [x] if x else []
    allo = o.sort_values("outlet_id").outlet_id.tolist(); picks = [p for p in ((c.get("params") or {}).get("cmp") or []) if p in allo][:5] or o.sort_values("health", ascending=False).outlet_id.tolist()[:2] + o.sort_values("health").outlet_id.tolist()[:1]
    names = ["outlet", "inventory", "staff", "marketing", "audit", "feedback"]; ser = {}; rows = []
    for p in picks:
        vals = [round(float(EN.latest_out(a).get("per_outlet", {}).get(p, 0) or 0), 1) for a in names]; ser[p] = vals; rr = o[o.outlet_id == p].iloc[0]; rows.append({"outlet": p, "location": rr.location, "health": health.get(p), "revenue": rr.revenue, "csat": rr.csat, "hc": rr.hc, **dict(zip(names, vals))})
    bl.append({"t": "form", "id": "cmp", "fields": [{"name": "cmp", "label": "Benchmark up to 5 outlets", "type": "multi", "options": allo, "value": picks}], "submit": "Compare", "url": "page:outlets", "title": None, "tone": "ghost", "inline": True})
    bl.append(B.row(B.card("Outlet benchmarking: score from each agent", B.chart(B.radar(names, ser), h=330, key="cmp_r")), B.card("Side by side", B.table(rows, ["outlet", "location", B.num_col("health", "health", "bar", 100)] + [B.num_col(a, a, "num") for a in names] + [B.num_col("revenue", "revenue", "money")], h=330, open={"col": "outlet"}, key="cmp_t")), ratio=[2, 3]))
    bl.append(ticket_block(c, "outlet", ["outlet_review", "health_alert"], title="Open tickets from this agent", key="tk_out")); return B.page("Outlets", "Agent 2 · KMeans tiers and the live health score", bl, live=12)

def inventory(c, tab=None):
    w, p = _sc(c); d = Q("SELECT outlet_id, sku_name, category, current_stock, reorder_threshold mn, weekly_demand wd, lead_time_days lead, stockout_risk_prob risk FROM inventory" + w, p)
    mult = LE.get_state("demand_mult", 1.0) or 1.0; d["cover_days"] = (d.current_stock / (d.wd.clip(lower=1) / 7 * mult)).round(1); d["short"] = d.cover_days < d.lead + 2
    po = Q("SELECT po_id, outlet_id, sku_name, qty, ts, eta_ts, status FROM purchase_orders" + w + " ORDER BY po_id DESC LIMIT 300", p)
    bl = [hero(c, "inventory", "Inventory agent", "Cover days = stock ÷ (forecast demand × live demand multiplier). Below lead time + 2 days raises a reorder and creates a purchase order."),
          B.kpis(("SKUs tracked", f"{len(d):,}"), ("Flagged short", int(d.short.sum()), None, "Warning"), ("POs in transit", int((po.status == "ordered").sum())), ("POs delivered", int((po.status == "delivered").sum())), ("Demand multiplier", f"x{mult}", "weather + holiday feed"))]
    cat = d.groupby("category").short.mean().mul(100).round(1)
    bl.append(B.row(B.card("Stock cover (days)", B.chart(B.hist(d.cover_days, 28, BLUE), h=270, key="iv_h")), B.card("% of SKUs short, by category", B.chart(B.bar(cat.index, cat.values, colors=[AMBER], pct=True), h=270, key="iv_c"))))
    bl.append(B.row(B.card("Most urgent SKUs", B.table(d[d.short].sort_values("cover_days").head(15)[["outlet_id", "sku_name", "current_stock", "cover_days", "lead", "risk"]], ["outlet_id", "sku_name", "current_stock", B.num_col("cover_days", "cover days", "num"), "lead", B.num_col("risk", "stock-out risk", "pct")], h=290, open={"col": "outlet_id"}, key="iv_u")),
                    B.card("Purchase orders created by this agent (they deliver into stock, closing the loop)", B.table(po.head(20), ["po_id", "outlet_id", "sku_name", "qty", B.pill_col("status"), "eta_ts"], h=290, open={"col": "outlet_id"}, key="iv_p"))))
    outs = sorted(d.outlet_id.unique()); sel = (c.get("params") or {}).get("o") or (outs[0] if outs else None); meth = (c.get("params") or {}).get("m") or "Prophet (if installed)"
    bl.append({"t": "form", "id": "inv_fc", "fields": [{"name": "o", "label": "Demand forecast for outlet", "type": "select", "options": outs, "value": sel}, {"name": "m", "label": "Method", "type": "select", "options": ["Prophet (if installed)", "Trend x weekday"], "value": meth}], "submit": "Forecast", "url": "page:inventory", "title": None, "tone": "ghost", "inline": True})
    if sel:
        h, f, how = dom.forecast_series(sel, 14, "trend" if meth.startswith("Trend") else None)
        if len(f): x = list(h.d.astype(str)) + list(f.d.astype(str)); bl.append(B.card(f"{sel}: daily sales and 14-day forecast ({how})", B.chart(B.line(x, {"actual": list(h.a) + [None] * len(f), "forecast": [None] * (len(h) - 1) + [h.a.iloc[-1]] + list(f.a)}, colors=[TEAL, AMBER], dashed=("forecast",), band=([None] * len(h) + list(f.lo), [None] * len(h) + list(f.hi))), h=290, key="iv_f")))
    x = cross(c, d[d.short].outlet_id.unique(), "inventory"); bl += [x] if x else []
    bl.append(ticket_block(c, "inventory", title="Open tickets from this agent", key="tk_inv")); return B.page("Inventory", "Agent 3 · cover days, reorders, forecast", bl, live=10)

def marketing(c, tab=None):
    w, p = _sc(c); m = Q("SELECT outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date FROM marketing" + w, p); today = datetime.date.today().isoformat()
    m["status"] = np.where(m.end_date >= today, "live", "ended / paused"); m["roi_pct"] = ((m.actual_roi - 1) * 100).round(0); r = (latest("marketing") or {}).get("out", {})
    bl = [hero(c, "marketing", "Marketing agent", "Campaign ROI = (revenue − budget) ÷ budget, plus KMeans (k=3) customer segments. Campaigns are paused when Inventory reports a shortage."),
          B.kpis(("Campaigns", len(m)), ("Live now", int((m.status == "live").sum())), ("Avg ROI", f"{m.roi_pct.mean():.0f}%"), ("Best channel", r.get("best_channel", "…")))]
    ch = m.groupby("channel").actual_roi.mean().sub(1).mul(100).round(0).sort_values(); seg = r.get("segments", {})
    bl.append(B.row(B.card("Average ROI % by channel", B.chart(B.bar(ch.index, ch.values, horizontal=True, colors=[TEAL], pct=True), h=270, key="mk_c")), B.card("Customer segments (KMeans k=3)", B.chart(B.pie(seg), h=270, key="mk_s")) if seg else B.callout("Segments appear after the first agent run.", "info")))
    bl.append(B.table(m[m.status == "live"].sort_values("roi_pct")[["outlet_id", "campaign_name", "channel", "budget", "roi_pct", "end_date"]].head(30), ["outlet_id", "campaign_name", "channel", B.num_col("budget", "budget", "money"), B.num_col("roi_pct", "ROI %", "num"), "end_date"], title="Live campaigns (lowest ROI first)", h=300, open={"col": "outlet_id"}, key="mk_l"))
    paused = Q("SELECT target, detail, ts FROM agent_actions WHERE type='pause_promo' ORDER BY action_id DESC LIMIT 8")
    if len(paused): bl.append(B.table(paused, ["target", "detail", "ts"], title="Promotions paused by the cross-agent rule (stock shortage + live campaign)", h=220, open={"col": "target"}, key="mk_p"))
    x = cross(c, m[m.status == "live"].outlet_id.unique(), "marketing"); bl += [x] if x else []
    bl.append(ticket_block(c, "marketing", title="Open tickets from this agent", key="tk_mk")); return B.page("Marketing", "Agent 4 · ROI, segments, campaign control", bl, live=12)

def sentiment(c, tab=None):
    w, p = _sc(c); f = Q("SELECT outlet_id, rating, comment, date, sentiment_score s FROM feedback" + w + " ORDER BY rowid DESC LIMIT 4000", p); r = (latest("feedback") or {}).get("out", {})
    bl = [hero(c, "feedback", "Sentiment agent", "VADER + star rating on every review, with issue categories. A review submitted here is analysed at once and picked up by the next agent cycle."),
          B.kpis(("Reviews analysed", len(f)), ("Avg rating", f"{f.rating.mean():.2f}" if len(f) else "n/a"), ("Outlets scoring < 45", sum(v < 45 for v in r.get("per_outlet", {}).values()), None, "Warning"))]
    iss = pd.Series(r.get("issues", {})).sort_values(); rd = f.rating.value_counts().sort_index()
    bl.append(B.row(B.card("Issue categories in reviews", B.chart(B.bar(iss.index, iss.values, horizontal=True, colors=[AMBER]), h=270, key="se_i")) if len(iss) else B.callout("Issue categories appear after the first run.", "info"), B.card("Rating distribution", B.chart(B.bar(rd.index.astype(int).astype(str), rd.values.astype(int), colors=[TEAL]), h=270, key="se_r"))))
    per = pd.Series(r.get("per_outlet", {})); per = per[per.index == c["scope"]] if c.get("scope") else per.sort_values().head(10)
    if len(per): bl.append(B.table(per.rename("score").reset_index().rename(columns={"index": "outlet"}), ["outlet", B.num_col("score", "score", "bar", 100)], title="Lowest sentiment scores (0-100)", h=250, open={"col": "outlet"}, key="se_l"))
    bl.append(B.table(f.head(14)[["outlet_id", "rating", "comment", "date"]], ["outlet_id", "rating", "comment", "date"], title="Latest reviews", h=300, open={"col": "outlet_id"}, key="se_t"))
    outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
    bl.append(B.form("review", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "rating", "label": "Rating", "type": "range", "min": 1, "max": 5, "value": 4}, {"name": "comment", "label": "Comment (any language, translated by NLLB)", "type": "textarea"}], "Submit review", "/api/franchise/review", title="Add a review", sub="Stored and analysed immediately; the agent includes it in its next cycle."))
    if can(c, "view"): bl.append(ticket_block(c, "feedback", title="Open tickets from this agent", key="tk_fb"))
    return B.page("Sentiment", "Agent 5 · reviews, VADER, issue categories", bl, live=15)

def audit(c, tab=None):
    w, p = _sc(c); a = Q("SELECT outlet_id, audit_date, score, violations, category, status, notes FROM audits" + w + " ORDER BY audit_date DESC, rowid DESC", p); last = a.drop_duplicates("outlet_id")
    bl = [hero(c, "audit", "Audit & compliance agent", "Latest inspection per outlet. Pass ≥ 80, Warning 60–79, Fail < 60. A fail opens a 72-hour corrective-action ticket and re-plans the Staff and Sentiment agents."),
          B.kpis(("Outlets audited", len(last)), ("Pass", int((last.score >= 80).sum()), None, "Low"), ("Warning", int(((last.score >= 60) & (last.score < 80)).sum()), None, "Warning"), ("Fail", int((last.score < 60).sum()), None, "Critical"))]
    cg = a.groupby("category").score.mean().round(1)
    bl.append(B.row(B.card("Latest audit scores", B.chart(B.hist(last.score, 20, TEAL), h=270, key="au_h")), B.card("Average score by category", B.chart(B.bar(cg.index, cg.values, colors=[BLUE]), h=270, key="au_c"))))
    bl.append(B.table(last.sort_values("score").head(15), ["outlet_id", "audit_date", B.num_col("score", "score", "bar", 100), "violations", "category", B.pill_col("status"), "notes"], title="Lowest-scoring outlets", h=320, open={"col": "outlet_id"}, key="au_l"))
    if can(c, "resolve"):
        outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
        bl.append(B.form("audit_log", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "score", "label": "Score", "type": "range", "min": 0, "max": 100, "value": 82}, {"name": "category", "label": "Category", "type": "select", "options": ["Food Safety", "Hygiene", "Documentation", "Equipment", "Staff Training"], "value": "Food Safety"}, {"name": "notes", "label": "Notes", "type": "text"}], "Save inspection", "/api/franchise/audit", title="Log an inspection", sub="Runs the Audit agent straight away; a fail opens a corrective-action ticket."))
    x = cross(c, last[last.score < 80].outlet_id, "audit"); bl += [x] if x else []
    bl.append(ticket_block(c, "audit", title="Open tickets from this agent", key="tk_au")); return B.page("Audit", "Agent 6 · inspections and compliance", bl, live=15)

PAGES = {"workforce": workforce, "outlets": outlets, "inventory": inventory, "marketing": marketing, "sentiment": sentiment, "audit": audit}


In [ ]:
%%writefile /content/app/views_freight.py
"""FreightQuote AI pages (server-driven blocks)."""
import os, re, json, datetime
import numpy as np, pandas as pd
import live_engine as LE, live_rbac as RB, entities as EN
import domain_freight as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FreightQuote AI"
MENU = [("copilot", "AI Copilot", "bot"), ("approvals", "Approvals", "check"), ("route", "Agent 1 · Route AI", "route"), ("quotes", "Agent 2 · Spot Quotes", "dollar"), ("carriers", "Agent 3 · Carriers", "truck"), ("weather", "Agent 4 · Weather Risk", "cloud"), ("margin", "Agent 5 · Margin", "trend"), ("customs", "Agent 6 · Customs", "file"),
        ("twin", "Digital Twin", "flask"), ("reports", "Reports", "chart"), ("notifications", "Notifications", "bell"), ("feeds", "Data Feed Center", "upload"), ("admin", "Admin", "shield"), ("profile", "My Profile", "user"), ("command", "Command Center", "radar")]
_ALL = {m[0] for m in MENU}; _OPS = _ALL - {"feeds", "admin"}
ROLE_MENU = {"Admin": _ALL, "Operations Manager": _OPS | {"feeds"}, "Freight Broker": _OPS, "Auditor": _OPS | {"admin"}, "Customer": {"copilot", "quotes", "weather", "notifications", "profile", "command"}}
SEVC = {2: RED, 1: AMBER, 0: BLUE, -1: TEAL}
FLEET_LEGEND = [("Vessel · no ticket", TEAL), ("Vessel · info ticket", BLUE), ("Vessel · warning ticket", AMBER), ("Vessel · critical ticket", RED), ("Port · low congestion (big dot)", "#8b8d98"), ("Port · congestion 3+", "#d98e04"), ("Port · congestion 4+", "#b4262b"), ("Storm cell (circle)", "#e5484d")]
def dport(ports, key, alt=0): return next((p for p in ports if key.lower() in p.lower()), ports[min(alt, len(ports) - 1)])

# ------------------------------------------------------------------ map layers
def fleet_layers(vessels=True, ports=True, storms=True, v=None):
    layers = []
    if ports:
        p = Q("SELECT port_name, lat, lon, congestion_index ci FROM ports"); layers.append({"type": "points", "id": "ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": "#b4262b" if r.ci >= 4 else "#d98e04" if r.ci >= 3 else "#8b8d98", "r": 3 + r.ci, "tip": f"<b>{r.port_name}</b><br>congestion {r.ci:.1f}", "open": r.port_name, "z": 1} for r in p.itertuples()]})
    if vessels:
        v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, v.progress, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)") if v is None else v
        tk = Q("SELECT target, MAX(CASE severity WHEN 'Critical' THEN 2 WHEN 'Warning' THEN 1 ELSE 0 END) sv FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").sv
        layers.append({"type": "points", "id": "vessels", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": SEVC[int(tk.get(r.shipment_id, -1))], "r": 5, "tip": f"<b>{r.shipment_id}</b><br>{r.o} → {r.d}<br>{r.route_name} · {r.progress:.0%}", "open": r.shipment_id, "z": 2} for r in v.itertuples()]})
    if storms: layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); oa = tk_df(c, limit=500); ex = dom.kpi_extra(); fl = Q("SELECT COUNT(*) n FROM vessels").n[0]
    mp = B.map_(fleet_layers(), h=510, view={"center": [20, 60], "zoom": 2}, key="ops_map", legend=FLEET_LEGEND)
    q = Q("SELECT quote_id, ROUND(final_price) price, margin_pct margin, status FROM freight_quotes ORDER BY rowid DESC LIMIT 7"); p = Q("SELECT port_name, ROUND(congestion_index,1) ci FROM ports ORDER BY congestion_index DESC LIMIT 7")
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; by = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int)
    bl = [B.kpis(("Fleet health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Vessels tracked", int(fl)), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), ("Active storms", len(dom.storms()), None, "Critical" if dom.storms() else None))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Live fleet · click a vessel or port", mp), B.col(B.card("Latest quotes", B.table(q, ["quote_id", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status")], h=230, open={"col": "quote_id"}, key="lq")), B.card("Most congested ports", B.table(p, ["port_name", B.num_col("ci", "index", "bar", 5)], h=230, open={"col": "port_name"}, key="cp"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=220, key="trend")), B.card("Open tickets by agent", B.chart(B.bar(by.index, by.values, horizontal=True, colors=[BLUE]), h=220, key="byag"))))
    if can(c, "admin"): bl.append(B.button("storm", "Inject a demo storm (tests the re-route loop)", "/api/freight/storm", "ghost", payload={}))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k == "shipment":
        v = Q("SELECT v.lat, v.lon, v.route_name, v.progress, v.chain, s.origin_port o, s.dest_port d, s.carrier, s.status, s.eta, s.weight_kg, s.hs_code, s.predicted_delay_risk risk FROM vessels v RIGHT JOIN shipments s USING(shipment_id) WHERE s.shipment_id=?", (e,))
        if not len(v): return {"title": e, "blocks": [B.callout("Unknown shipment.", "warn")]}
        r = v.iloc[0]; bl = []
        if pd.notna(r.lat):
            w = dom.weather_at(float(r.lat), float(r.lon), allow_live=False); ch = json.loads(r.chain) if r.chain else []
            bl.append(B.kpis(("Progress", f"{r.progress * 100:.0f}%"), ("Wind m/s", round(w["wind"], 1)), ("Wave m", round(w["wave"], 1)), ("Weather", w["level"], w["src"], TONE.get(w["level"]))))
            bl.append(B.map_([{"type": "lines", "lines": [{"pts": ch, "color": TEAL, "w": 3, "dash": "6 6"}]}, {"type": "points", "id": "me", "pts": [{"id": e, "lat": r.lat, "lon": r.lon, "color": "#1f2430", "r": 8, "tip": e}]}, {"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": "storm"} for s in dom.storms()]}], h=230, view={"center": [float(r.lat), float(r.lon)], "zoom": 3}, key="d_map", legend=[("This port", "#1f2430"), ("Planned voyage route", TEAL), ("Storm cell (red circle)", RED)]))
        else: bl.append(B.callout(f"Not at sea: status {r.status}.", "info"))
        bl.append(B.table([{"field": "Route", "value": f"{r.o} → {r.d}"}, {"field": "Carrier", "value": r.carrier}, {"field": "Status", "value": r.status}, {"field": "ETA", "value": r.eta}, {"field": "Cargo weight", "value": f"{(r.weight_kg or 0) / 1000:,.1f} t"}, {"field": "HS code", "value": r.hs_code}, {"field": "Predicted delay risk", "value": r.risk}], ["field", "value"], h=250, key="d_f"))
        bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
        try:
            import alts; po, pd_ = dom.port_xy(r.o), dom.port_xy(r.d); start = (float(r.lat), float(r.lon)) if pd.notna(r.lat) else (po[0], po[1]) if po else None
            if start and pd_:
                ao = alts.route_alternatives(dom, start, (pd_[0], pd_[1]), tonnes=float(r.weight_kg or 20000) / 1000, interp=dom.interp, dwell=(pd_[3] if pd_ else 0) * .5); bl += alts.route_blocks(ao, e, "Route alternatives" + (" from the vessel's position" if pd.notna(r.lat) else ""))
                if can(c, "approve") and pd.notna(r.lat) and ao: bl.append(B.form("rr_" + e, [{"name": "opt", "label": "Switch to", "type": "select", "options": [f"{i + 1}. {x['label']} · {x['name']}" for i, x in enumerate(ao)], "value": "1. " + ao[0]["label"] + " · " + ao[0]["name"]}, {"name": "shipment", "type": "hidden", "value": e}], "Re-route this voyage", "/api/freight/reroute", title="Apply a route", sub="Needs approve rights. The vessel is re-planned from its current position and the change is audited.", tone="ghost"))
        except Exception as ex: bl.append(B.callout(f"Route alternatives unavailable: {ex}", "info"))
        return {"title": e, "sub": f"{r.o} → {r.d}", "blocks": bl}
    if k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if not len(q): return {"title": e, "blocks": [B.callout("Unknown quote.", "warn")]}
        r = q.iloc[0]; cost = r.base_cost + r.fuel_surcharge + r.customs_fee + r.insurance
        return {"title": e, "sub": r.status, "blocks": [B.kpis(("Price", f"${r.final_price:,.0f}"), ("Margin", f"{r.margin_pct}%"), ("Status", r.status)), B.chart(B.waterfall(["Base (ML)", "Fuel", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel_surcharge, r.customs_fee, r.insurance, r.final_price - cost]), "How the price is built", h=260, key="d_wf"), ticket_block(c, target=e, title="Tickets", key="d_tk")]}
    p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
    if not len(p): return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    r = p.iloc[0]; w = Q("SELECT current_severity sev, wind_speed, wave_height FROM weather_risks WHERE port_name=?", (e,)); sh = Q("SELECT shipment_id, carrier, status FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 12", (e, e))
    bl = [B.kpis(("Congestion index", round(float(r.congestion_index), 2), None, "Critical" if r.congestion_index >= 4 else None), ("Avg dwell days", r.avg_dwell_days), ("Country", r.country)), B.table(sh, ["shipment_id", "carrier", B.pill_col("status")], title="Shipments via this port", h=250, open={"col": "shipment_id"}, key="d_sh")]
    try:
        import alts; bl += alts.port_alternatives(e)
    except Exception: pass
    if len(w): bl.insert(1, B.kpis(("Weather", w.sev[0], f"wind {w.wind_speed[0]:.0f} km/h · wave {w.wave_height[0]:.1f} m", TONE.get(w.sev[0]))))
    return {"title": e, "sub": f"{r.country} · port", "blocks": bl}

# ------------------------------------------------------------------ pages
def route(c, tab=None):
    v = Q("SELECT COUNT(*) n FROM vessels").n[0]; p = Q("SELECT port_name, country, region, congestion_index ci, avg_dwell_days dw FROM ports"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status").set_index("status").n
    ports = sorted(p.port_name); pr = c.get("params") or {}; o = pr.get("o") or dport(ports, "JNPT"); d = pr.get("d") or dport(ports, "Rotterdam", 1); cg = pr.get("cargo") or "Electronics"
    bl = [hero(c, "route_watch", "Route & congestion agent", "Compares Suez and Cape routings with a RandomForest congestion classifier and a live weather check along the lane. Watches every port."),
          B.kpis(("Vessels tracked", int(v)), ("Ports monitored", len(p)), ("Congested (≥ 4)", int((p.ci >= 4).sum()), None, "Warning"), ("Active storms", len(dom.storms())))]
    bl.append(B.card("Live fleet", B.map_(fleet_layers(), h=400, view={"center": [20, 60], "zoom": 2}, key="rt_map", legend=FLEET_LEGEND)))
    top = p.sort_values("ci", ascending=False).head(12)
    bl.append(B.row(B.card("Most congested ports (click)", B.chart(B.bar(top.port_name, top.ci.round(2), horizontal=True, itemcolors=[RED if x >= 4 else AMBER for x in top.ci], max_=5), h=330, open={"bar": "y"}, key="rt_c")), B.card("Shipments by status", B.chart(B.bar(sh.index, sh.values.astype(int), colors=[BLUE]), h=330, key="rt_s"))))
    bl.append(B.form("rt_plan", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": o}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": d}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": cg}], "Plan route", "page:route", title="Route planner", sub="Same route, weather and congestion logic as the quote pipeline.", tone="ghost"))
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if po and pd_ and o != d:
        opts = dom.route_options(po[:2], pd_[:2]); cong = dom.predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, cg); rows, lines = [], []
        for i, op in enumerate(opts):
            ws = [dom.weather_at(*dom.interp(op["chain"], f), allow_live=False) for f in (0, .25, .5, .75, 1)]; worst = max(ws, key=lambda w: w["wind"]); days = op["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
            rows.append({"route": op["name"], "distance_nm": op["nm"], "transit_days": round(days, 1), "weather_risk": worst["level"], "max_wind_ms": worst["wind"]}); lines.append({"pts": op["chain"], "color": TEAL if i == 0 else AMBER, "w": 4, "dash": None if i == 0 else "8 8", "tip": op["name"]})
        bl.append(B.row(B.card("Route options", B.table(rows, ["route", B.num_col("distance_nm", "nm", "int"), B.num_col("transit_days", "days", "num"), B.pill_col("weather_risk"), B.num_col("max_wind_ms", "max wind m/s", "num")], h=150, key="rt_o"), B.callout(f"Predicted congestion level {cong}/3 (RandomForest: port congestion index, dwell time, month, weekday, cargo).", "info")),
                        B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "ends", "pts": [{"id": o, "lat": po[0], "lon": po[1], "color": TEAL, "r": 7, "tip": o}, {"id": d, "lat": pd_[0], "lon": pd_[1], "color": BLUE, "r": 7, "tip": d}]}], h=300, key="rt_pm", view={"bounds": [[min(p[0] for ln in lines[:1] for p in ln["pts"]) - 4, min(p[1] for ln in lines[:1] for p in ln["pts"]) - 4], [max(p[0] for ln in lines[:1] for p in ln["pts"]) + 4, max(p[1] for ln in lines[:1] for p in ln["pts"]) + 4]]}, legend=[("Recommended (sea route)", TEAL), ("Alternative", AMBER)])))
    x = cross(c, p[p.ci >= 4].port_name.tolist(), "route_watch"); bl += [x] if x else []
    bl.append(ticket_block(c, "route_watch", title="Open tickets from this agent", key="tk_rt")); return B.page("Route AI", "Agent 1 · routing, congestion, lane planning", bl, live=10)

def _docs_all(): return sorted({d for s in Q("SELECT docs FROM customs_rules").docs for d in str(s).split(";") if d})
def quote_form(c):
    ports = Q("SELECT port_name FROM ports ORDER BY port_name").port_name.tolist(); cust = Q("SELECT customer_id FROM customers").customer_id.tolist(); alld = _docs_all(); ch, base, *_ = dom._required_docs("HS-8501")
    return B.form("quote", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": dport(ports, "JNPT")}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": dport(ports, "Rotterdam", 1)}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": "Electronics"},
                                  {"name": "weight", "label": "Weight (tons)", "type": "number", "value": 120, "min": 1, "max": 5000}, {"name": "hs", "label": "HS code", "type": "text", "value": "HS-8501"}, {"name": "customer", "label": "Customer", "type": "select", "options": cust, "value": cust[0]},
                                  {"name": "docs", "label": "Documents you are submitting (tick all that apply; the customs agent checks them against the rules for the HS code)", "type": "checks", "options": alld, "value": base}], "Generate quote", "/api/freight/quote", title="New quote", sub="Runs the 7-agent pipeline: route → weather → (re-plan loop) → pricing (RandomForest) → customs → margin → documents → alerts.")
def quote_result(rid):
    r = Q("SELECT quote_id, result FROM quote_requests WHERE req_id=?", (rid,))
    if not len(r) or not r.result[0]: return [B.callout("The pipeline did not return a result.", "warn")]
    res = json.loads(r.result[0]); bl = [B.callout(res.get("narrative", ""), "success", "Pipeline result"), B.kpis(("Price (USD)", f"{res.get('price', 0):,.0f}"), ("Decision", res.get("decision")), ("Margin", f"{res.get('margin_pct')}%"), ("Transit", f"{res.get('days')} days"))]
    if res.get("missing"): bl.append(B.callout("Missing documents: " + ", ".join(res["missing"]) + ". Attach them on the Customs page and the quote re-runs.", "warn"))
    tr = Q("SELECT agent, risk, score, latency_ms ms, summary FROM agent_runs WHERE cycle_id LIKE ? ORDER BY run_id", (f"Q{rid}-%",)); bl.append(B.table(tr, ["agent", B.pill_col("risk"), B.num_col("score", "score", "num"), B.num_col("ms", "ms", "int"), "summary"], title="Agent trace for this quote", h=260, key="q_tr"))
    ddir = os.path.join(dom.DATA_DIR, "docs"); files = [p for p in (sorted(os.listdir(ddir)) if os.path.isdir(ddir) else []) if p.startswith(r.quote_id[0])]
    for p in files: bl.append(B.button("dl_" + p, "Download " + p, f"/api/doc/{p}", "ghost", payload={"download": True}))
    return bl
def quotes(c, tab=None):
    bl = [hero(c, "quote_watch", "Spot quotes agent", "Every quote goes through one pipeline: ML price (RandomForest) + 12% fuel + port fees + 15% operating cost, target margin ~20%. Quotes above $50,000 need admin approval.")]
    if can(c, "quote"): bl.append(quote_form(c))
    if can(c, "view"):
        q = Q("SELECT quote_id, customer_id, carrier, final_price price, margin_pct margin, status, created_at FROM freight_quotes ORDER BY created_at DESC, quote_id DESC LIMIT 400"); st = q.status.value_counts()
        bl.append(B.kpis(("Quotes", len(q)), ("Approved", int((q.status == "Approved").sum()), None, "Low"), ("Pending approval", int((q.status == "Pending Approval").sum()), None, "Warning"), ("On hold", int((q.status == "On Hold").sum()), None, "Critical")))
        bl.append(B.row(B.card("Quote decisions", B.chart(B.pie(st.to_dict(), colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=280, key="qt_p")), B.card("Margin % across quotes", B.chart(B.hist(q.margin, 24, BLUE), h=280, key="qt_h"))))
        bl.append(B.table(q.head(30), ["quote_id", "customer_id", "carrier", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status"), "created_at"], title="Latest quotes · click a row", h=340, open={"col": "quote_id"}, key="qt_t", search=True))
        bl.append(ticket_block(c, "quote_watch", title="Open tickets from this agent", key="tk_qt"))
    return B.page("Spot Quotes", "Agent 2 · the 7-agent quote pipeline", bl, live=10)

def carriers(c, tab=None):
    k = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); k["score"] = (k.rating / 5 * 40 + k.on_time_pct * .4 - k.avg_cost_index * 12 - (k.risk_level == "High") * 8).round(1)
    s = Q("SELECT carrier name, COUNT(*) shipments, SUM(status LIKE '%Delay%' OR status='Customs Hold') problems, AVG(predicted_delay_risk) delay_risk FROM shipments GROUP BY carrier"); k = k.merge(s, on="name", how="left").fillna({"shipments": 0, "problems": 0})
    top = k.sort_values("score", ascending=False).head(12); rc = {"Low": TEAL, "Medium": AMBER, "High": RED}
    sc = {t: [{"x": a.on_time_pct, "y": a.avg_cost_index, "label": a.name, "size": 6 + a.shipments ** .5} for a in k[k.risk_level == t].itertuples()] for t in ["Low", "Medium", "High"] if (k.risk_level == t).any()}
    bl = [hero(c, "route_watch", "Carrier intelligence", "Score = rating × 40% + on-time % × 0.4 − cost-index penalty − 8 points for High risk. Shipments and delays come from the live shipments table."),
          B.kpis(("Carriers", len(k)), ("High-risk carriers", int((k.risk_level == "High").sum()), None, "Warning"), ("Avg on-time", f"{k.on_time_pct.mean():.1f}%"), ("Best score", k.sort_values("score").name.iloc[-1])),
          B.row(B.card("Top carriers by agent score", B.chart(B.bar(top.name, top.score, horizontal=True, colors=[TEAL], sort=True), h=360, key="ca_t")), B.card("On-time vs cost index (size = shipments)", B.chart(B.scatter(sc, "on-time %", "cost index", colors=rc, sized=True), h=360, key="ca_s"))),
          B.table(k.sort_values("score", ascending=False), ["name", B.num_col("score", "score", "bar", 100), "rating", B.num_col("on_time_pct", "on-time %", "num"), B.num_col("avg_cost_index", "cost idx", "num"), B.pill_col("risk_level", "risk"), B.num_col("shipments", "shipments", "int"), B.num_col("problems", "problems", "int")], title="Carrier league table", h=340, key="ca_tb", search=True)]
    return B.page("Carriers", "Agent 3 · ranking and risk", bl, live=15)

METRICS = {"wind": ("Wind speed", [("Calm / light  < 8 m/s", TEAL), ("Strong  8–15 m/s", AMBER), ("Storm-force  > 15 m/s", RED)]),
           "waves": ("Wave height", [("Slight  < 2.5 m", TEAL), ("Rough  2.5–4 m", AMBER), ("Very rough  > 4 m", RED)]),
           "rain": ("Rain now", [("Dry", "#8b8d98"), ("Light rain  < 2.5 mm/h", BLUE), ("Heavy rain  ≥ 2.5 mm/h", VIOLET)]),
           "temp": ("Temperature", [("Cold  < 10 °C", BLUE), ("Mild  10–30 °C", TEAL), ("Hot  30–35 °C", AMBER), ("Very hot  > 35 °C", RED)])}
def _wxcolor(metric, r):
    if metric == "wind": return RED if r.wind_ms > 15 else AMBER if r.wind_ms >= 8 else TEAL
    if metric == "waves": w = r.wave_m if r.wave_m == r.wave_m and r.wave_m is not None else 0; return RED if w > 4 else AMBER if w >= 2.5 else TEAL
    if metric == "rain": p = r.precip_mm or 0; return VIOLET if p >= 2.5 else BLUE if p > 0 else "#8b8d98"
    t = r.temp_c if r.temp_c is not None else 25; return RED if t > 35 else AMBER if t >= 30 else TEAL if t >= 10 else BLUE
def wx_layers(metric, hi):
    import weather as WX
    t = WX.table(); layers = []
    if len(t):
        pts = []
        for r in t.itertuples():
            w = r.wave_m if r.wave_m is not None and r.wave_m == r.wave_m else None
            pts.append({"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": _wxcolor(metric, r), "r": 4 + min(9, r.wind_ms / 2.2), "z": 1, "open": r.port_name,
                        "tip": f"<b>{r.port_name}</b> · {WX.WMO.get(r.wcode, '')}<br>wind {r.wind_ms:.1f} m/s ({r.wind_ms * 3.6:.0f} km/h), gusts {r.gust_ms or 0:.0f} m/s<br>waves {'-' if w is None else f'{w:.1f} m'} · rain {r.precip_mm or 0:.1f} mm/h · {r.temp_c:.0f} °C<br><i>real reading {r.ts[11:16]} UTC</i>"})
        layers.append({"type": "points", "id": "wx_ports", "pts": pts})
    else:
        w = Q("SELECT w.port_name, w.current_severity sev, w.wind_speed, w.wave_height, w.temperature, p.lat, p.lon FROM weather_risks w JOIN ports p USING(port_name)")
        layers.append({"type": "points", "id": "wx_ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": RED if r.sev == "High" else AMBER if r.sev == "Medium" else TEAL, "r": 4 + min(9, r.wind_speed / 8), "z": 1, "open": r.port_name, "tip": f"<b>{r.port_name}</b><br>wind {r.wind_speed:.0f} km/h · waves {r.wave_height:.1f} m<br><i>simulated, not a real reading</i>"} for r in w.itertuples()]})
    v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)"); hs = set(hi)
    layers.append({"type": "points", "id": "wx_v", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": "#1f2430" if r.shipment_id in hs else "#6b7080", "r": 4.5 if r.shipment_id in hs else 3, "z": 2, "open": r.shipment_id, "tip": f"<b>{r.shipment_id}</b> {r.o} → {r.d}" + ("<br><b>in storm-force wind</b>" if r.shipment_id in hs else "")} for r in v.itertuples()]})
    layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm cell · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers
def weather(c, tab=None):
    import weather as WX
    p = c.get("params") or {}; metric = p.get("metric") if p.get("metric") in METRICS else "wind"; st = WX.status()
    r = (latest("weather_watch") or {}).get("out", {}); hi = r.get("vessels_high", []); w = Q("SELECT port_name, current_severity sev, wind_speed, wave_height, temperature, forecast FROM weather_risks")
    src = (f"Real · Open-Meteo, {st['ok']}/{st['total']} ports, updated {st['ts'][11:16]} UTC ({st['age_min']:.0f} min ago)" if st["live"] else "Simulated · Open-Meteo not reachable" + (f" ({st['err']})" if st["err"] else "") + ". Press Refresh to retry.")
    lg = list(METRICS[metric][1]) + [("Big dot = port (size = wind)", "#8b8d98"), ("Black dot = vessel in storm-force wind", "#1f2430"), ("Grey dot = other vessel", "#6b7080"), ("Red circle = storm cell", "#e5484d")]
    pick = {"t": "form", "id": "wx_pick", "fields": [{"name": "metric", "label": "Colour ports by", "type": "select", "options": list(METRICS), "value": metric}], "submit": "Show", "url": "page:weather", "title": None, "tone": "ghost", "inline": True}
    hot = w.sort_values("wind_speed", ascending=False)
    bl = [B.callout(src, "success" if st["live"] else "warning", "Weather data source"), hero(c, "weather_watch", "Weather risk agent", "Real wind, wave, rain and temperature from Open-Meteo for every port (refreshed every 20 min) plus injected storm cells. Vessels in High risk (wind > 15 m/s) get a re-route ticket; the system re-plans the voyage once it is approved."),
          B.kpis(("Vessels in High risk", len(hi), None, "Critical" if hi else None), ("Active storms", len(dom.storms())), ("Ports with weather data", len(w)), ("Ports in High risk", int((w.sev == "High").sum()), None, "Warning"), ("Data", "Real" if st["live"] else "Simulated", None, "Low" if st["live"] else "Warning")),
          pick, B.card(f"Port weather · coloured by {METRICS[metric][0].lower()} · click a port or vessel", B.map_(wx_layers(metric, hi), h=470, view={"center": [20, 60], "zoom": 2}, key="wx_map", legend=lg)),
          B.row(B.card("Strongest port winds (km/h)", B.chart(B.bar(hot.head(12).port_name, hot.head(12).wind_speed.round(0), horizontal=True, colors=[AMBER], sort=True), h=330, open={"bar": "y"}, key="wx_w")), B.card("Port weather risk levels", B.chart(B.pie({str(k): int(v) for k, v in w.sev.value_counts().items()}, colors={"High": RED, "Medium": AMBER, "Low": TEAL}), h=330, key="wx_d")))]
    if hi: bl.append(B.table(Q("SELECT shipment_id, lat, lon, route_name FROM vessels WHERE shipment_id IN (%s)" % ",".join("?" * len(hi[:50])), hi[:50]), ["shipment_id", B.num_col("lat", "lat", "num"), B.num_col("lon", "lon", "num"), "route_name"], title="Vessels in High-risk weather now · click to inspect", h=240, open={"col": "shipment_id"}, key="wx_hv"))
    bl.append(B.table(w.sort_values("wind_speed", ascending=False).head(60).rename(columns={"wind_speed": "wind km/h", "wave_height": "waves m", "temperature": "temp °C"}), ["port_name", B.pill_col("sev"), "wind km/h", "waves m", "temp °C", "forecast"], title="Port weather (real where the forecast says Open-Meteo live)", h=300, open={"col": "port_name"}, key="wx_t"))
    if can(c, "run"): bl.append(B.button("wx_ref", "Refresh real weather now", "/api/weather/refresh", "ghost", payload={}))
    bl.append(B.table(pd.DataFrame(dom.feed_status(), columns=["feed", "status"]), title="Data sources", h=190, key="wx_f"))
    if can(c, "admin"):
        from twin import PRESET_STORMS; bl.append(B.form("storm", [{"name": "area", "label": "Area", "type": "select", "options": list(PRESET_STORMS), "value": "Red Sea"}, {"name": "wind", "label": "Wind m/s", "type": "range", "min": 15, "max": 35, "value": 24}], "Inject storm", "/api/freight/storm", title="Inject a storm", sub="Adds a storm cell for testing the re-route loop; it does not change the real readings."))
    bl.append(ticket_block(c, "weather_watch", title="Open tickets from this agent", key="tk_wx")); return B.page("Weather Risk", "Agent 4 · real wind, waves, rain, storms", bl, live=10)

def margin(c, tab=None):
    q = Q("SELECT quote_id, customer_id, base_cost, fuel_surcharge fuel, customs_fee, insurance, final_price price, margin_pct margin, status FROM freight_quotes WHERE base_cost>0 ORDER BY created_at DESC, quote_id DESC LIMIT 600")
    bl = [hero(c, "quote_watch", "Margin & approvals agent", "Margin = (price − cost) ÷ price with tiered targets (Platinum 18%, Gold 20%, Silver/Bronze 22–24%). Quotes > $50,000 or < 10% margin wait for an approver."),
          B.kpis(("Average margin", f"{q.margin.mean():.1f}%"), ("Below 10% margin", int((q.margin < 10).sum()), None, "Warning"), ("Over $50,000", int((q.price > 50000).sum())), ("Revenue in quotes", f"${q.price.sum():,.0f}"))]
    sc = {s: [{"x": a.base_cost, "y": a.price, "label": a.quote_id} for a in q[q.status == s].head(250).itertuples()] for s in q.status.unique()}; cm = q.groupby("customer_id").margin.mean().round(1).sort_values().head(10)
    bl.append(B.row(B.card("Price vs base cost (click a dot)", B.chart(B.scatter(sc, "base cost", "price", colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=330, open={"scatter": True}, key="mg_s")), B.card("Lowest-margin customers", B.chart(B.bar(cm.index, cm.values, horizontal=True, colors=[AMBER]), h=330, key="mg_c"))))
    if len(q):
        sel = (c.get("params") or {}).get("q") or q.quote_id.iloc[0]; r = q[q.quote_id == sel]; r = r.iloc[0] if len(r) else q.iloc[0]; cost = r.base_cost + r.fuel + r.customs_fee + r.insurance
        bl.append({"t": "form", "id": "mg_pick", "fields": [{"name": "q", "label": "Cost build-up for quote", "type": "select", "options": q.quote_id.tolist()[:200], "value": r.quote_id}], "submit": "Show", "url": "page:margin", "title": None, "tone": "ghost", "inline": True})
        bl.append(B.card(f"{r.quote_id}: how the price is built", B.chart(B.waterfall(["Base (ML)", "Fuel 12%", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel, r.customs_fee, r.insurance, r.price - cost]), h=300, key="mg_w")))
    bl.append(ticket_block(c, "quote_watch", ["quote_approval", "margin_review"], title="Approval queue", key="tk_mg")); return B.page("Margin & Approvals", "Agent 5 · margin, tiers, approvals", bl, live=10)

def customs(c, tab="customs"):
    tabs = [{"id": "customs", "label": "Customs & tariff"}, {"id": "docs", "label": "Documents (OCR)"}]
    if tab == "docs":
        ddir = os.path.join(dom.DATA_DIR, "docs"); files = sorted(os.listdir(ddir), reverse=True) if os.path.isdir(ddir) else []
        bl = [B.callout("PDFs are generated by the Document agent for every quote (Bill of Lading, Packing List). Upload a document to read its fields and attach it to the matching quote.", "info"), B.kpis(("Generated documents", len(files)))]
        bl += [B.button("dl_" + f, "Download " + f, f"/api/doc/{f}", "ghost", payload={"download": True}) for f in files[:10]]
        bl.append(B.form("readdoc", [{"name": "file", "label": "PDF or TXT", "type": "file"}], "Read document", "/api/freight/readdoc", title="Read an uploaded document (OCR / text extraction)", tone="primary")); return B.page("Customs", "Agent 6 · documents", bl, tabs, live=0)
    h = Q("SELECT shipment_id, hs_code, origin_port, dest_port, carrier FROM shipments WHERE status='Customs Hold' LIMIT 200"); rules = Q("SELECT * FROM customs_rules ORDER BY chapter"); rows = []
    for r in h.head(30).itertuples():
        ch, docs, rs, duty, goods = dom._required_docs(r.hs_code); miss = [d for d in docs if d not in dom._attached(r.shipment_id)]; rows.append({"shipment": r.shipment_id, "hs": r.hs_code, "goods": goods, "duty_pct": duty, "missing": ", ".join(miss), "route": f"{r.origin_port} → {r.dest_port}"})
    bl = [hero(c, "customs_watch", "Customs & tariff agent", "Requirements come from the customs_rules table by HS chapter. A hold lifts as soon as every required document is attached: the quote re-runs and the shipment clears."),
          B.kpis(("Shipments on hold", len(h), None, "Warning" if len(h) else None), ("Quotes on hold", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='On Hold'").n[0])), ("Rules in force", len(rules))),
          B.table(rows, ["shipment", "hs", "goods", B.num_col("duty_pct", "duty %", "num"), "missing", "route"], title="Held shipments and what each is missing · click a row", h=300, open={"col": "shipment"}, key="cu_h"), B.table(rules, list(rules.columns), title="Customs rules", h=230, key="cu_r")]
    if can(c, "resolve") or can(c, "quote"):
        held = Q("SELECT shipment_id ref FROM shipments WHERE status='Customs Hold' UNION SELECT quote_id FROM freight_quotes WHERE status='On Hold' LIMIT 200").ref.tolist(); alld = _docs_all()
        if held: bl.append(B.form("attach", [{"name": "ref", "label": "Shipment or quote on hold", "type": "select", "options": held, "value": held[0]}, {"name": "doc", "label": "Document", "type": "select", "options": alld, "value": alld[0]}], "Attach document", "/api/freight/attach", title="Attach a document", sub="The Customs agent re-checks on its next cycle; quotes re-run immediately."))
    if can(c, "view"): bl.append(ticket_block(c, "customs_watch", title="Open tickets from this agent", key="tk_cu"))
    return B.page("Customs", "Agent 6 · rules, holds, documents", bl, tabs, live=12)

PAGES = {"route": route, "quotes": quotes, "carriers": carriers, "weather": weather, "margin": margin, "customs": customs}


In [ ]:
%%writefile /content/app/views_misc.py
"""Shared pages: Notifications, Digital Twin, Data Feed Center, Admin, Copilot (chat, knowledge base, document translator)."""
import os, re, json, time
import numpy as np, pandas as pd
import live_engine as LE, live_rbac as RB, data_feeder as FD, twin as TW, gpu, rag, authx
import blocks as B
from blocks import Q, clean, TEAL, BLUE, AMBER, RED, VIOLET
from views_core import can, latest, ticket_block, tk_df, scoped
from config import DB_PATH

# ------------------------------------------------------------------ Notifications
def notifications(c, tab="alerts"):
    dom, kind = c["dom"], c["kind"]; tabs = [{"id": "alerts", "label": "Alert feed"}, {"id": "chan", "label": "Channels"}] + ([{"id": "digest", "label": "Weekly digest"}] if kind == "Franchise" else [])
    sub = "Alerts and delivery channels. Critical tickets go to Slack, Teams, Gmail and SMS when configured; overdue tickets escalate automatically (see Approvals)."
    if tab == "chan":
        ch = [("Slack", "SLACK_WEBHOOK_URL"), ("Microsoft Teams", "TEAMS_WEBHOOK_URL"), ("Gmail SMTP", "EMAIL_ID"), ("SMS (Twilio)", "TWILIO_SID")]
        bl = [B.table([{"channel": n, "status": "configured" if os.environ.get(k) else "not configured", "environment variable": k} for n, k in ch], ["channel", B.pill_col("status"), "environment variable"], h=220, key="chan")]
        if can(c, "admin"): bl.append(B.button("tn", "Send a test Critical alert", "/api/notify_test", "ghost"))
        return B.page("Notifications", sub, bl, tabs, live=0)
    if tab == "digest" and kind == "Franchise":
        r = latest("digest"); txt = (r or {}).get("out", {}).get("digest")
        bl = [B.hero("digest", "Digest agent", "Weekly executive digest built from every other agent's latest findings.", r, can_run=can(c, "run")), B.callout(txt, "info", "Latest digest") if txt else B.callout("No digest yet. Press Run on the card above.", "info")]
        bl.append(ticket_block(c, "digest", title="Digest tickets", key="tk_dg")); return B.page("Notifications", sub, bl, tabs, live=0)
    ec = dom.ENTITY_COL; a = Q(f"SELECT alert_id id, {ec} target, severity, category, message, date FROM alerts ORDER BY alert_id DESC LIMIT 300"); a = scoped(a, c)
    bl = [B.kpis(("Alerts logged", len(a)), ("Critical", int((a.severity == "Critical").sum()), None, "Critical"), ("Open tickets", int(Q("SELECT COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed')").n[0]))),
          B.table(a, ["id", B.pill_col("severity"), "target", "category", "message", "date"], h=470, open={"col": "target"}, key="alerts", search=True)]
    return B.page("Notifications", sub, bl, tabs, live=10)

# ------------------------------------------------------------------ Digital twin
def _rng(n, l, lo, hi, v, st=None):
    d = {"name": n, "label": l, "type": "range", "min": lo, "max": hi, "value": v}
    if st: d["step"] = st
    return d
def _outs(kind): return Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
def _multi(kind): return {"name": "outlets", "label": "Outlets (empty = all)", "type": "multi", "options": _outs(kind), "value": []}
def _carriers(): return Q("SELECT carrier, COUNT(*) n FROM shipments WHERE carrier IS NOT NULL GROUP BY carrier ORDER BY n DESC LIMIT 30").carrier.tolist()
AREAS = ["Red Sea", "Arabian Sea", "Strait of Malacca", "South China Sea", "Bay of Biscay", "Cape of Good Hope", "North Atlantic"]
SIMDEF = {   # page -> (title, description, fields(kind), agents re-run, agent scenario builder)
    "Franchise": {
        "inventory": ("Inventory simulator", "100 simulated 30-day futures per SKU: fill rate, lost sales, the best safety buffer, and what to order now.", lambda: [_rng("pct", "Demand change %", -50, 100, 30), _rng("days", "Supplier delay (extra lead-time days)", 0, 14, 4), _rng("cv", "Demand volatility %", 10, 80, 30), _multi("Franchise")], ["inventory"]),
        "workforce": ("Workforce simulator", "Overtime, temp cover, extra leavers and revenue at risk when staff are absent.", lambda: [_rng("pct", "Staff absent %", 0, 60, 25), _rng("ot_cap", "Overtime cap (hours per person per week)", 0, 20, 10), _rng("temp", "Share of the gap filled by temps %", 0, 100, 0), _multi("Franchise")], ["staff"]),
        "outlets": ("Outlet P&L simulator", "Profit, margin, loss-making outlets and what moved profit, under demand, price and cost changes.", lambda: [_rng("pct", "Demand change %", -50, 50, -10), _rng("price", "Price change %", -20, 20, 0), _rng("cost", "Cost inflation %", -20, 40, 5), _multi("Franchise")], ["outlet", "anomaly"]),
        "marketing": ("Marketing budget simulator", "Conversions with diminishing returns per channel, and the best way to split the same budget.", lambda: [_rng("pct", "Total budget change %", -50, 100, 20), _rng("alpha", "Diminishing returns (higher = closer to linear)", 40, 90, 70)], ["marketing"]),
        "sentiment": ("Reviews & rating simulator", "How a wave of negative reviews moves each outlet's rating, the revenue at risk, and the good reviews needed to recover.", lambda: [_rng("pct", "Share of new reviews that are negative %", 5, 100, 40), _rng("n_new", "New reviews per outlet", 5, 200, 30), _multi("Franchise")], ["feedback"]),
        "audit": ("Audit readiness simulator", "Probability each outlet fails the next inspection, expected fines and remediation load.", lambda: [_rng("raise", "Stricter pass mark (points)", 0, 15, 5), _rng("pct", "More violations found %", 0, 100, 10), {"name": "fine", "label": "Fine per violation", "type": "number", "value": 5000}, _rng("days", "Remediation days", 1, 60, 14), _multi("Franchise")], ["audit"])},
    "Freight": {
        "route": ("Route simulator", "Re-plans real sea routes for your busiest lanes when chokepoints close, storms hit or ports congest: extra nm, days and cost.", lambda: [{"name": "closures", "label": "Closed chokepoints", "type": "multi", "options": ["suez", "panama", "malacca", "hormuz", "bosphorus"], "value": ["suez"]}, {"name": "area", "label": "Storm area", "type": "select", "options": ["None"] + AREAS, "value": "None"}, _rng("wind", "Storm wind m/s", 15, 40, 25), _rng("add", "Port congestion increase", 0, 3, 0, .5), {"name": "delay_cost", "label": "Cost of a shipment-day waiting ($)", "type": "number", "value": 120}], ["route_watch", "weather_watch"]),
        "quotes": ("Quote cost simulator", "How fuel, carrier-rate and currency moves hit quote margins, and the price that maximises expected profit.", lambda: [_rng("pct", "Fuel price change %", -30, 80, 20), _rng("carrier", "Carrier rate change %", -10, 30, 0), _rng("fx", "Currency move %", -10, 15, 0), _rng("elasticity", "Customer price sensitivity", 0.5, 4, 2, .5)], ["quote_watch"]),
        "margin": ("Pricing & margin simulator", "Test a discount or a price rise: expected profit and win probability across all open quotes.", lambda: [_rng("pct", "Fuel price change %", -30, 80, 0), _rng("carrier", "Carrier rate change %", -10, 30, 0), _rng("discount", "Discount %", 0, 20, 5), _rng("elasticity", "Customer price sensitivity", 0.5, 4, 2, .5)], ["quote_watch", "anomaly_watch"]),
        "carriers": ("Carrier outage simulator", "A carrier fails: where its shipments go, what it costs and who has no capacity.", lambda: [{"name": "carrier", "label": "Carrier", "type": "select", "options": _carriers(), "value": _carriers()[0]}, _rng("loss", "Capacity lost %", 10, 100, 100), {"name": "delay_cost", "label": "Cost of a shipment-day waiting ($)", "type": "number", "value": 120}], ["route_watch", "anomaly_watch"]),
        "weather": ("Storm simulator", "Vessels inside the storm, delay days and demurrage cost, and how cost climbs with wind.", lambda: [{"name": "area", "label": "Storm area", "type": "select", "options": AREAS, "value": "Red Sea"}, _rng("wind", "Wind m/s", 12, 40, 28), _rng("radius", "Storm radius (nm)", 300, 2000, 900), {"name": "demurrage", "label": "Delay cost per shipment-day ($)", "type": "number", "value": 900}], ["weather_watch"]),
        "customs": ("Customs simulator", "Extra holds, dwell days and duty changes: held shipments, storage cost and the hit to open quotes.", lambda: [_rng("hold", "Inbound shipments put on hold %", 0, 60, 20), _rng("dwell", "Extra dwell days", 1, 14, 4), {"name": "storage", "label": "Storage cost per shipment-day ($)", "type": "number", "value": 180}, _rng("duty", "Duty change (points)", -5, 10, 0)], ["customs_watch"])}}
SIM = {k: {p: (v[3], []) for p, v in d.items()} for k, d in SIMDEF.items()}
def _agent_mut(kind, pid, a):
    if kind == "Franchise":
        cat = TW.franchise_scenarios(); o = a.get("outlets") or []
        return {"inventory": lambda: [cat["Demand change"][0](_nf(a, "pct"), o), cat["Supplier lead-time delay"][0](int(_nf(a, "days")), o)], "workforce": lambda: [cat["Staff absenteeism"][0](_nf(a, "pct"), o)], "outlets": lambda: [cat["Demand change"][0](_nf(a, "pct"), o)],
                "sentiment": lambda: [cat["Negative review wave"][0](_nf(a, "pct"), o)], "audit": lambda: [cat["Failed inspection"][0](o or _outs(kind)[:2])], "marketing": lambda: []}.get(pid, lambda: [])()
    cat = TW.freight_scenarios()
    return {"route": lambda: ([cat["Storm on a sea area"][0](a.get("area"), _nf(a, "wind", 25))] if a.get("area") not in (None, "None") else []) + ([cat["Port congestion surge"][0](_nf(a, "add"), "All")] if _nf(a, "add") else []), "quotes": lambda: [cat["Fuel price move"][0](_nf(a, "pct"))], "margin": lambda: [cat["Fuel price move"][0](_nf(a, "pct"))],
            "weather": lambda: [cat["Storm on a sea area"][0](a.get("area") or "Red Sea", _nf(a, "wind", 25))], "customs": lambda: [cat["Customs crackdown"][0](int(_nf(a, "hold") / 100 * 200))], "carriers": lambda: []}.get(pid, lambda: [])()
def _nf(a, k, d=0.0):
    try: return float(a.get(k, d))
    except Exception: return float(d)
def simulator(c, pid):
    kind = c["kind"]; title, desc, flds, agents = SIMDEF[kind][pid]
    fields = flds() + [{"name": "page", "type": "hidden", "value": pid}, {"name": "scenario", "type": "hidden", "value": "sim"}, {"name": "agents", "label": "Also re-run the real agents on the copy (slower)", "type": "select", "options": ["no", "yes"], "value": "no"}]
    bl = [B.callout(desc + " Every run copies the live data; nothing live changes. Assumptions are listed in the results.", "info"), B.form("sim_" + pid, fields, "Run simulation", "/api/sim/run", title=title)]
    return B.page("Simulator", "Per-agent what-if on a private copy of the data", bl, [], live=0)
def sim_run(c, pid, sc, args):
    kind, dom = c["kind"], c["dom"]; title, desc, flds, agents = SIMDEF[kind][pid]
    import sim_fr, sim_fq; mod = sim_fr if kind == "Franchise" else sim_fq; r = mod.RUN[pid](args); bl = r["blocks"]
    if args.get("agents") == "yes":
        muts = _agent_mut(kind, pid, args)
        if muts:
            base = TW.run_agents(dom, agents); sc_res = TW.run_agents(dom, agents, (lambda cn: [f(cn) for f in muts])); rows = [{"agent": n, "before": base[n].score, "after": sc_res[n].score, "delta": round(sc_res[n].score - base[n].score, 1), "finding": sc_res[n].summary} for n in agents]
            bk = {(a["type"], str(a["target"])) for n in agents for a in base[n].actions}; new = [a for n in agents for a in sc_res[n].actions if (a["type"], str(a["target"])) not in bk]
            bl.append(B.card("What the real agent says on the copy", B.callout(" | ".join(x["finding"] for x in rows), "info"), B.table(rows, ["agent", B.num_col("before", "before", "num"), B.num_col("after", "after", "num"), B.num_col("delta", "Δ", "num")], h=160, key="sim_ag"),
                             *([B.table(pd.DataFrame(new)[["type", "target", "severity", "detail"]].head(20), ["type", "target", B.pill_col("severity"), "detail"], title=f"{len(new)} tickets it would raise", h=260, key="sim_na")] if new else [])))
        else: bl.append(B.callout("No data-changing scenario maps to this agent, so only the model above applies.", "info"))
    import mc_sims; bl = bl + mc_sims.blocks(kind, pid, args)
    return bl
def twin(c, tab=None):
    import twin2 as T2; kind = c["kind"]; tabs = [{"id": "lab", "label": "Scenario lab"}, {"id": "saved", "label": "Saved scenarios"}, {"id": "assume", "label": "Assumptions"}]; tab = tab if tab in ("lab", "saved", "assume") else "lab"
    if tab == "assume":
        import sim_fr, sim_fq; A = sim_fr.ASSUME if kind == "Franchise" else sim_fq.ASSUME
        return B.page("Digital Twin", "What the twin assumes", [B.callout("Every number the twin shows comes from these models and settings. They are explicit so you can challenge them; edit sim_fr.py / sim_fq.py to match your business.", "info")] + [B.card(k.title(), B.md(v)) for k, v in A.items()], tabs, live=0)
    if tab == "saved":
        s = T2.saved(); rows = [{"name": r.name, "saved": r.ts[:16], "by": r.by, "total_impact": (json.loads(r.result).get("kpi", {}).get("TOTAL", "")) if r.result else ""} for r in s.itertuples()]; names = [r["name"] for r in rows]
        bl = [B.table(rows, ["name", "saved", "by", "total_impact"], title="Saved scenarios", h=260, key="t2_s")] if rows else [B.callout("Nothing saved yet. In the Scenario lab, type a name before running to save the scenario.", "info")]
        if len(names) >= 1: bl += [B.form("t2_run", [{"name": "a", "label": "Run saved scenario", "type": "select", "options": names, "value": names[0]}], "Run again on today's data", "/api/twin/saved", title=None, tone="ghost")]
        if len(names) >= 2: bl += [B.form("t2_cmp", [{"name": "a", "label": "Scenario A", "type": "select", "options": names, "value": names[0]}, {"name": "b", "label": "Scenario B", "type": "select", "options": names, "value": names[1]}], "Compare side by side", "/api/twin/compare", title="Compare two scenarios", tone="ghost")]
        return B.page("Digital Twin", "Saved what-ifs", bl, tabs, live=0)
    fields = T2.fields(kind) + [{"name": "agents", "label": "Also re-run the real agents on the copy (slower)", "type": "select", "options": ["no", "yes"], "value": "no"}, {"name": "name", "label": "Save this scenario as (optional)", "type": "text"}]
    bl = [B.callout("Stack as many shocks as you like. The twin runs every simulator together, shows the money impact with uncertainty bands, ranks the best responses, and can send one to the approval queue. The live database is never touched.", "info"), B.form("twin_lab", fields, "Run the twin", "/api/twin/run", title="Scenario lab")]
    return B.page("Digital Twin", "Stacked what-if on a private copy of the live data", bl, tabs, live=0)
def twin_run(c, args):
    import twin2 as T2; args = dict(args); name = (args.pop("name", "") or "").strip(); bl, summ = T2.run(c, args)
    if name: T2.save(name, args, c.get("email", "user"), summ); bl.insert(0, B.callout(f"Saved as '{name}'.", "info"))
    return bl
def twin_compare(c, an, bn):
    import twin2 as T2; s = T2.saved().set_index("name"); out = []
    for n in (an, bn):
        a = json.loads(s.loc[n].args); _, sm = T2.run(c, a); out.append((n, sm))
    keys = list(dict.fromkeys(list(out[0][1]["kpi"]) + list(out[1][1]["kpi"]))); rows = [{"measure": k, an: out[0][1]["kpi"].get(k, ""), bn: out[1][1]["kpi"].get(k, "")} for k in keys]
    return [B.table(rows, ["measure", an, bn], title="Side by side (re-run on today's data)", h=min(80 + 38 * len(rows), 360), key="t2_cmpt")]
# ------------------------------------------------------------------ Data Feed Center
def feeds(c, tab="upload"):
    kind = c["kind"]; tabs = [{"id": "upload", "label": "Upload"}, {"id": "api", "label": "Live API push"}, {"id": "url", "label": "URL feeds"}, {"id": "inbox", "label": "Folder inbox"}, {"id": "quality", "label": "Quality & rejects"}]
    sub = "Real data in, agents re-run, entities with real data stop being simulated."
    if not can(c, "feeder"): return B.page("Data Feed Center", sub, [B.callout("Your role cannot feed data. Ask an Admin or a manager.", "warn")], [], live=0)
    FD.init(); hist = Q("SELECT ts, source, target, rows_in, rows_ok, rows_bad FROM feed_log ORDER BY id DESC LIMIT 15"); real = FD.real_entities()
    foot = [B.row(B.card("Ingestion history", B.table(hist, ["ts", "source", "target", "rows_in", "rows_ok", "rows_bad"], h=300, key="fh")), B.card("Entities on a real feed right now", B.kpis(("Real-fed entities", len(real), "no longer simulated")), B.table([{"entity": e} for e in sorted(real)[:60]], ["entity"], h=200, key="rf")))]
    if tab == "api":
        k = os.environ.get("INGEST_KEY", "(set INGEST_KEY in the launcher first)"); ex = '[{"outlet_id":"OUT-001","ts":"2026-01-01T10:00:00","amount":1520.5,"tx_count":6}]' if kind == "Franchise" else '[{"shipment_id":"SHP-L00000001","lat":18.9,"lon":72.8}]'; tb = "sales" if kind == "Franchise" else "vessels"
        return B.page("Data Feed Center", sub, [B.md(f"**POST rows as JSON** from a POS terminal, an AIS gateway or any system:\n\n```\ncurl -X POST $BASE_URL/ingest/{tb} -H 'X-API-Key: {k}' -H 'Content-Type: application/json' -d '{ex}'\n```\nAdd `-H 'Idempotency-Key: <unique id>'` so a retried request is never applied twice; send up to 50,000 rows as a JSON array or NDJSON. Rows are validated, auto-mapped and upserted. The matching entities leave the simulator for 10 minutes after their last real record."), *([B.form("ais_map", [{"name": "pairs", "label": "One 'MMSI,shipment_id' per line, so AIS pushes (POST /ingest/ais with mmsi, lat, lon, speed_kn) update the right vessel", "type": "textarea"}], "Save links", "/api/feed/ais_map", title="AIS vessel identifiers")] if kind == "Freight" else []), *foot], tabs, live=15)
    if tab == "url":
        u = Q("SELECT id, url, target, every_s FROM feed_urls")
        return B.page("Data Feed Center", sub, [B.form("fd_url", [{"name": "url", "label": "CSV or JSON URL", "type": "text"}, {"name": "target", "label": "Into table", "type": "select", "options": FD.ALLOWED[kind], "value": FD.ALLOWED[kind][0]}, {"name": "every", "label": "Refresh every (seconds, min 30)", "type": "number", "value": 300}, {"name": "auth_env", "label": "Auth: NAME of the environment variable holding the Bearer token (never the token itself)", "type": "text"}, {"name": "json_path", "label": "JSON path to the rows, e.g. data.items (optional)", "type": "text"}], "Register feed", "/api/feed/url", title="Register a REST / CSV / JSON feed"), B.table(Q("SELECT id, url, target, every_s, auth_env, last_status FROM feed_urls"), ["id", "url", "target", "every_s", "auth_env", "last_status"], title="Registered feeds", h=200, key="fu"), *foot], tabs, live=0)
    if tab == "quality":
        rj = Q("SELECT ts, source, target, row_no, field, reason, value FROM feed_rejects ORDER BY id DESC LIMIT 200"); sm = Q("SELECT target, reason, COUNT(*) n FROM feed_rejects GROUP BY target, reason ORDER BY n DESC LIMIT 12")
        bl = [B.kpis(("Rejected rows stored", int(Q("SELECT COUNT(*) n FROM feed_rejects").n[0]), "kept 90 days", None), ("Feeds registered", int(Q("SELECT COUNT(*) n FROM feed_urls").n[0]))),
              B.md("Every feed is validated row by row: numeric ranges, valid dates, allowed values and foreign keys (an outlet or port must exist). Bad rows are quarantined here with the reason; good rows still load. [Download all rejected rows (CSV)](/api/feed/rejects.csv)"),
              B.table(sm, ["target", "reason", B.num_col("n", "rows", "int")], title="Most common problems", h=220, key="fq_s"), B.table(rj, ["ts", "source", "target", B.num_col("row_no", "row", "int"), "field", "reason", "value"], title="Latest rejected rows", h=320, key="fq_r"), *foot[:0]]
        return B.page("Data Feed Center", sub, bl, tabs, live=10)
    if tab == "inbox":
        files = sorted(os.listdir(FD.INBOX)); return B.page("Data Feed Center", sub, [B.md(f"Drop files named `<table>.csv` or `<table>__anything.csv` into `{FD.INBOX}` (for example `staff.csv`, `ports__2026.xlsx`). They are ingested automatically within seconds."), B.table([{"file": f} for f in files], ["file"], title="Waiting in the inbox", h=160, key="fi"), *foot], tabs, live=10)
    return B.page("Data Feed Center", sub, [B.form("fd_up", [{"name": "table", "label": "Target table", "type": "select", "options": FD.ALLOWED[kind], "value": "sales" if kind == "Franchise" else "vessels"}, {"name": "file", "label": "CSV, XLSX or JSON", "type": "file"}], "Validate and ingest", "/api/feed/upload", title="Upload a file", sub="Columns are auto-matched to the table (exact, alias and fuzzy). Rows that fail validation are rejected and counted."), *foot], tabs, live=0)

# ------------------------------------------------------------------ Admin
def admin(c, tab="runtime"):
    dom, kind = c["dom"], c["kind"]; tabs = [{"id": "runtime", "label": "Runtime"}, {"id": "gpu", "label": "GPU & models"}, {"id": "decisions", "label": "Decisions"}, {"id": "users", "label": "Users & access"}, {"id": "audit", "label": "Audit log"}, {"id": "insights", "label": "Copilot insights"}, {"id": "teach", "label": "Teach Copilot"}, {"id": "check", "label": "System check"}, {"id": "gov", "label": "Data governance"}, {"id": "data", "label": "Data"}]
    if not (can(c, "admin") or c["role"] == "Auditor"): return B.page("Admin", "", [B.callout("Admin or Auditor access only.", "warn")], [], live=0)
    sub = "Platform health, access control and the audit trail"
    if tab == "gpu":
        g = gpu.status(); m = g["models"]; bl = []
        bl.append(B.kpis(("Device", g["gpu"] or "CPU only", f"CUDA {'available' if g['cuda'] else 'NOT available'} · torch {g['torch']}", None if g["cuda"] else "Critical"), ("VRAM used", f"{g['vram_used_gb']} GB", f"reserved {g['vram_reserved_gb']} of {g['vram_gb']} GB"), ("LLM speed", f"{g['stats']['tok_s'] or '–'} tok/s", f"{g['stats']['llm_calls']} generations"), ("Translations", g["stats"]["translations"], f"last batch {g['stats']['translate_ms'] or '–'} ms")))
        if not g["cuda"]: bl.append(B.callout("This runtime has no GPU. In Colab choose Runtime > Change runtime type > T4 GPU, then run the notebook again. Models fall back to CPU or stay unloaded.", "danger"))
        bl.append(B.table([{"model": k, "name": v["name"], "state": v["state"], "device": v["device"], "load_s": v["sec"], "download_gb": (f"{v.get('done_gb')}/{v.get('total_gb')}" if v.get("total_gb") else None), "precision": v.get("quant"), "error": v["err"]} for k, v in m.items()], ["model", "name", B.pill_col("state"), "download_gb", "device", "load_s", "precision", "error"], title="Models on the GPU", h=190, key="gm"))
        return B.page("Admin", sub, bl, tabs, live=3)
    if tab == "insights":
        import copilot_pro as CPX; I = CPX.insights(); tot = I["total"] or 0; rt = {r[0]: r[1] for r in I["routes"]}; ex = rt.get("exact", 0) + rt.get("exact+llm", 0); un = rt.get("unanswered", 0); fbn = I["up"] + I["down"]
        bl = [B.kpis(("Questions", tot, "all time, this database"), ("Answered exactly (SQL)", f"{round(100 * ex / tot) if tot else 0}%", f"{ex} question(s)", None), ("Unanswered", f"{round(100 * un / tot) if tot else 0}%", f"{un} question(s)", "Critical" if tot and un / tot > .25 else ("Warning" if un else None)), ("Thumbs up / down", f"{I['up']} / {I['down']}", f"{fbn} ratings"), ("Avg exact answer", f"{round(I['avg_ms'])} ms" if I["avg_ms"] else "–"))]
        bl.append(B.table([{"route": r[0], "questions": r[1]} for r in I["routes"]], ["route", B.num_col("questions", "questions", "int")], title="How questions were answered", h=190, key="ci_r"))
        bl.append(B.table([{"question": r[0], "times": r[1], "last_asked": str(r[2]).replace("T", " ")} for r in I["unanswered"]], ["question", B.num_col("times", "times", "int"), "last_asked"], title="Top unanswered questions: add an intent or synonym for these", h=300, key="ci_u"))
        bl.append(B.row(B.card("Thumbs-down answers", B.table([{"when": str(r[0]).replace("T", " "), "question": r[1]} for r in I["down_q"]], ["when", "question"], h=260, key="ci_d")), B.card("Recent questions", B.table([{"when": str(r[0]).replace("T", " "), "question": r[1], "route": r[2], "ms": r[3]} for r in I["recent"]], ["when", "question", B.pill_col("route"), B.num_col("ms", "ms", "int")], h=260, key="ci_c"))))
        return B.page("Admin", sub, bl, tabs, live=0)
    if tab == "teach":
        import copilot_pro as CPX; S = CPX.teach_stats(); GOAL = 50; bk = CPX.backlog(300); tr = CPX.taught_rows()
        ex_q = [r["question"] for r in bk[:40]]
        bl = [B.kpis(("Taught so far", f"{S['taught']} / {GOAL}", "goal: 50+ questions", None if S["taught"] >= GOAL else "Warning"), ("Unanswered found", S["found"], "distinct questions"), ("Still open", S["open"], "not taught yet", "Critical" if S["open"] > 20 else ("Warning" if S["open"] else None)))]
        bl.append(B.callout("How this works: 1) Find questions the Copilot cannot answer: ask it in AI Copilot, or paste a list below and test them. 2) Teach it: pick the question, write the correct answer (and keywords). 3) Ask again: it now answers, also for rephrased versions. Aim for 50+ taught questions from real, useful domain questions.", "info"))
        bl.append(B.row(
            B.form("probe", [{"name": "questions", "label": "Questions to test (one per line, up to 25)", "type": "textarea", "placeholder": "What is the refund policy?\nWhich outlets need a service recovery visit?\n..."}], "Test them now", "/api/copilot/probe", title="1. Find unanswered questions", sub="Runs each question through the real Copilot (SQL, agents, RAG, LLM). Unanswered ones go to the backlog."),
            B.form("teach", [{"name": "question", "label": "Question", "type": ("select" if ex_q else "text"), **({"options": ex_q} if ex_q else {})}, {"name": "answer", "label": "Correct answer (a few lines; include the numbers or the policy)", "type": "textarea"}, {"name": "keywords", "label": "Keywords / other ways to ask it (comma separated)", "type": "text"}], "Teach the Copilot", "/api/copilot/teach", title="2. Teach it", sub="Pick a question from the backlog list (filled after you test questions or ask the Copilot).")))
        bl.append(B.row(
            B.form("teach_q2", [{"name": "question", "label": "Or type any question", "type": "text"}, {"name": "answer", "label": "Correct answer", "type": "textarea"}, {"name": "keywords", "label": "Keywords (optional)", "type": "text"}], "Save", "/api/copilot/teach", title="Add a question directly"),
            B.form("teach_imp", [{"name": "file", "label": "CSV with the columns: question, answer, keywords", "type": "file"}], "Import CSV", "/api/copilot/teach/import", title="Bulk add from a CSV", sub="Best for adding many at once. Download the current list first to see the format."),))
        bl.append(B.table(bk, ["question", B.num_col("times", "times asked", "int"), "last_asked"], title="Backlog: questions the Copilot could not answer", h=300, key="tc_bk"))
        bl.append(B.table([{"id": r["id"], "question": r["question"], "answer": r["answer"][:160], "keywords": r["keywords"], "by": r["by"], "when": r["when"].replace("T", " ")} for r in tr], ["id", "question", "answer", "keywords", "by", "when"], title="Taught questions", h=320, key="tc_tr"))
        bl.append(B.row(B.button("tc_exp", "Download taught questions (CSV)", "/api/copilot/teach/export", "ghost", payload={"download": True}), B.form("teach_del", [{"name": "id", "label": "Remove a taught question (enter its id)", "type": "number"}], "Remove", "/api/copilot/teach/delete", tone="ghost")))
        return B.page("Admin", sub, bl, tabs, live=0)
    if tab == "check":
        import datetime as _dt, copilot_pro as CPX, qa as QAX
        from config import DATA_DIR
        chk = []
        def add(name, ok, detail, warn=False): chk.append({"check": name, "status": "OK" if ok else ("Warning" if warn else "Critical"), "detail": detail})
        n = _dt.datetime.now(); add("Server clock (IST)", True, n.strftime("%A %d %B %Y, %I:%M:%S %p") + " · TZ=" + (os.environ.get("TZ") or "system"))
        db = LE.conn()
        try: LE._steps_ddl(db); db.commit()
        except Exception: pass
        try:
            import rag as _rg0; _rg0._seed()
        except Exception: pass
        for t in ["outlets", "staff", "inventory", "sales"] if kind == "Franchise" else ["shipments", "ports", "carriers", "freight_quotes", "customs_rules"]:
            try: k = int(db.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]); add("Table " + t, k > 0, f"{k} rows")
            except Exception as e: add("Table " + t, False, str(e)[:80])
        def age(sql):
            try:
                r = db.execute(sql).fetchone(); return None if not r or not r[0] else (n - _dt.datetime.fromisoformat(str(r[0])[:19].replace(" ", "T"))).total_seconds()
            except Exception: return None
        for ag in dom.AGENTS:
            if ag == "digest":
                a = age("SELECT MAX(ts) FROM agent_runs WHERE agent='digest'"); add("Agent digest (daily)", True, "runs once a day on its schedule; " + ("not run yet today" if a is None else f"last run {int(a // 3600)} h ago")); continue
            a = age("SELECT MAX(ts) FROM agent_runs WHERE agent=?".replace("?", "'" + ag + "'")); add("Agent " + ag, a is not None and a < 900, "not run yet: the worker starts about a minute after the app" if a is None else f"last run {int(a // 60)} min ago", warn=True)
        a = age("SELECT MAX(ts) FROM decisions"); add("Intelligence Engine decision", a is not None and a < 900, "none yet: first cycle runs about a minute after start" if a is None else f"{int(a // 60)} min ago", warn=True)
        try:
            k = int(db.execute("SELECT COUNT(*) FROM kb_chunks").fetchone()[0]); add("Knowledge base (RAG)", k > 0, f"{k} chunks indexed", warn=True)
        except Exception: add("Knowledge base (RAG)", False, "not built yet: it builds in the background after start", warn=True)
        import rag as _rg
        try:
            _d = _rg.drive_dir(); _mt = os.path.isdir(_d); add("Google Drive (RAG builder output)", _mt, _d if _mt else "Drive not mounted or folder missing. In the launcher cell keep USE_DRIVE = True, allow Drive access, run the RAG notebook first, then use AI Copilot > Knowledge base > Import from Drive.", warn=True)
            _n = int(db.execute("SELECT COUNT(*) FROM kb_chunks WHERE builtin=2").fetchone()[0]); add("Chunks imported from Drive", _n > 0, f"{_n} chunks (your RAG notebook output)" if _n else "none imported yet", warn=True)
        except Exception as e: add("Google Drive (RAG builder output)", False, str(e)[:80], warn=True)
        try:
            import persist as _ps; _ok, _dt = _ps.status(); add("Database backup on Google Drive (chats, users, tickets)", bool(_ok), _dt, warn=True)
        except Exception as e: add("Database backup on Google Drive", False, str(e)[:80], warn=True)
        L = gpu.S["models"]["llm"]; add("LLM", L["state"] == "ready", f"{L['state']} on {L['device'] or '-'} {L['err'] or ''}".strip(), warn=True)
        N = gpu.S["models"]["nllb"]; add("Translation (NLLB)", N["state"] == "ready", f"{N['state']} on {N['device'] or '-'}", warn=True)
        ok = 0; probes = ["Top 5 outlets by revenue", "How many staff are there?"] if kind == "Franchise" else ["How many shipments are on customs hold?", "Which port has the highest congestion?"]
        for p in probes:
            try: r = QAX.answer(kind, p, None, True); good = bool(r and r.get("answer")); ok += good; add("Copilot: " + p, good, (r or {}).get("answer", "no exact answer")[:90] if good else "no exact answer")
            except Exception as e: add("Copilot: " + p, False, repr(e)[:90])
        try:
            r = db.execute("SELECT COUNT(*) FROM agent_actions WHERE status IN ('proposed','open')").fetchone()[0]; s2 = db.execute("SELECT COUNT(*) FROM approval_steps WHERE status='pending'").fetchone()[0]; add("Approvals pipeline", True, f"{r} open tickets · {s2} pending steps")
        except Exception as e: add("Approvals pipeline", False, str(e)[:80])
        try: t = os.path.join(DATA_DIR, ".w"); open(t, "w").write("x"); os.remove(t); add("Data folder writable", True, DATA_DIR)
        except Exception as e: add("Data folder writable", False, str(e)[:80])
        bad = sum(1 for x in chk if x["status"] == "Critical"); wr = sum(1 for x in chk if x["status"] == "Warning")
        return B.page("Admin", sub, [B.kpis(("Checks", len(chk)), ("Critical", bad, None, "Critical" if bad else None), ("Warnings", wr, None, "Warning" if wr else None), ("IST now", n.strftime("%I:%M %p"), n.strftime("%d %b %Y"))), B.table(chk, ["check", B.pill_col("status"), "detail"], title="System check (run again by reopening this tab)", h=560, key="sc")], tabs, live=0)
    if tab == "gov":
        import privacy as PV; pol = PV.policy(); rows = PV.purge(True, "view"); last = LE.get_state("purge_last", {}) or {}
        bl = [B.kpis(("Rows past retention", sum(r["older_rows"] for r in rows), "removed by the nightly job", None), ("Last purge", last.get("day", "never"))),
              B.table(rows, ["table", B.num_col("keep_days", "keep (days)", "int"), B.num_col("older_rows", "past retention", "int")], title="Retention policy (operational history only; master data is never purged)", h=300, key="gv_r"),
              B.form("gv_set", [{"name": "table", "label": "Table", "type": "select", "options": list(pol), "value": list(pol)[0]}, {"name": "days", "label": "Keep for (days, min 7)", "type": "number", "value": 365}], "Set retention", "/api/gov/retention", title="Change a retention period", tone="ghost"),
              B.form("gv_purge", [{"name": "mode", "label": "Mode", "type": "select", "options": ["dry", "run"], "value": "dry"}], "Run purge", "/api/gov/purge", title="Purge now", sub="Start with a dry run: it only counts."),
              B.table([{"table": t, "field": f, "what": w, "handling": h} for t, f, w, h in PV.PII[kind]], ["table", "field", "what", "handling"], title="Personal data inventory", h=260, key="gv_p"),
              B.form("gv_dsar", [{"name": "subject", "label": "Staff id or login email", "type": "text"}, {"name": "action", "label": "Action", "type": "select", "options": ["export", "erase"], "value": "export"}], "Run request", "/api/gov/dsar", title="Data-subject request (access / erasure)", sub="Export returns everything held about the person. Erase anonymises or deletes it and is recorded in the audit log."),
              B.callout("Free-text feedback has emails and phone numbers replaced before storage. Names and salaries are hidden from roles that are not Admin, Owner or Operations Manager in the Copilot. This is a technical baseline, not legal advice: set the retention periods your counsel requires.", "info")]
        return B.page("Admin", sub, bl, tabs, live=0)
    if tab == "decisions": return B.page("Admin", sub, [B.table(Q("SELECT ts, trigger_src, score, risk, narrative FROM decisions ORDER BY decision_id DESC LIMIT 60"), ["ts", "trigger_src", B.num_col("score", "score", "num"), B.pill_col("risk"), "narrative"], h=520, key="dc")], tabs, live=10)
    if tab == "users":
        us = pd.DataFrame(authx.users()); bl = [B.table(us, ["email", "username", B.pill_col("role"), B.pill_col("status"), "failed", "last_login"], title="Accounts", h=260, key="us")]
        if can(c, "admin"):
            em = us.email.tolist(); bl.append(B.row(B.form("u_role", [{"name": "email", "label": "User", "type": "select", "options": em, "value": em[0]}, {"name": "role", "label": "New role", "type": "select", "options": authx.ROLES, "value": authx.ROLES[-1]}], "Update role", "/api/user/role", title="Change a role"),
                                                  B.form("u_unlock", [{"name": "email", "label": "Locked user", "type": "select", "options": em, "value": em[0]}], "Unlock account", "/api/user/unlock", title="Unlock", tone="ghost")))
            bl.append(B.row(B.form("u_inv", [{"name": "email", "label": "New user's e-mail", "type": "text"}, {"name": "username", "label": "User name", "type": "text"}, {"name": "role", "label": "Role", "type": "select", "options": authx.ROLES, "value": authx.ROLES[-2]}], "Create and invite", "/api/user/invite", title="Invite a user", sub="Creates the account and e-mails a temporary password."),
                            B.form("u_off", [{"name": "email", "label": "Account", "type": "select", "options": em, "value": em[-1]}, {"name": "on", "label": "Action", "type": "select", "options": ["Deactivate", "Reactivate"], "value": "Deactivate"}], "Apply", "/api/user/deactivate", title="Deactivate or reactivate", tone="ghost")))
            if kind == "Franchise": bl.append(B.form("u_scope", [{"name": "email", "label": "Account (Store Manager / Staff)", "type": "select", "options": em, "value": em[0]}, {"name": "outlet", "label": "Outlet", "type": "select", "options": Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist(), "value": "OUT-001"}], "Assign outlet", "/api/user/scope", title="Scope an account to one outlet"))
        caps = ["view", "resolve", "approve", "run", "feeder", "admin", "quote"]; bl.append(B.table([{"role": r, **{k: ("✓" if k in v else "") for k in caps}} for r, v in RB.CAPS.items()], ["role"] + caps, title="Capability matrix", h=300, key="cap")); return B.page("Admin", sub, bl, tabs, live=0)
    if tab == "audit":
        a = Q("SELECT ts, actor, event, target, detail FROM audit_log ORDER BY audit_id DESC LIMIT 400"); return B.page("Admin", sub, [B.table(a, ["ts", "actor", B.pill_col("event"), "target", "detail"], title="Audit log", h=520, key="au", search=True), B.button("csv", "Export CSV", "/api/audit.csv", "ghost", payload={"download": True})], tabs, live=10)
    if tab == "data":
        tbls = [r[0] for r in LE.conn().execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()]; rows = []
        for t in tbls:
            try: rows.append({"table": t, "rows": int(LE.conn().execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])})
            except Exception: pass
        return B.page("Admin", sub, [B.kpis(("Database size", f"{os.path.getsize(DB_PATH) / 2**20:.1f} MB"), ("Tables", len(rows))), B.button("bk", "Download database backup", "/api/backup", "ghost", payload={"download": True}), B.callout("The database lives in " + os.path.dirname(DB_PATH) + (". Colab erases it when the runtime resets: set PERSIST = True in the notebook's start cell to keep it on Google Drive, or download a backup here." if "/content/" in DB_PATH else "."), "info"), B.table(sorted(rows, key=lambda r: -r["rows"]), ["table", B.num_col("rows", "rows", "int")], h=480, key="dt")], tabs, live=15)
    s = LE.get_state("runtime", {}) or {}; perf = Q("SELECT agent, COUNT(*) runs, ROUND(AVG(latency_ms)) avg_ms, ROUND(AVG(score),1) avg_score, MAX(ts) last_run FROM agent_runs GROUP BY agent ORDER BY runs DESC"); lat = Q("SELECT substr(ts,12,8) t, agent, latency_ms ms FROM agent_runs ORDER BY run_id DESC LIMIT 200")
    g = {a: [{"x": i, "y": r.ms, "label": r.t} for i, r in enumerate(lat[lat.agent == a].itertuples())] for a in lat.agent.unique()}
    bl = [B.kpis(("Ticks", s.get("ticks", 0)), ("Cycles", s.get("cycles", 0)), ("Last cycle", (s.get("last_cycle") or "starting")[11:]), ("Open tickets", int(Q("SELECT COUNT(*) n FROM agent_actions WHERE status='open'").n[0]))),
          B.row(B.card("Agent performance", B.table(perf, ["agent", B.num_col("runs", "runs", "int"), B.num_col("avg_ms", "avg ms", "int"), B.num_col("avg_score", "avg score", "num"), "last_run"], h=300, key="perf")), B.card("Recent agent latency (ms)", B.chart(B.scatter(g, "run", "ms"), h=300, key="latc"))),
          B.table(Q("SELECT ts, etype, entity FROM live_events ORDER BY event_id DESC LIMIT 15"), ["ts", "etype", "entity"], title="Live events", h=260, key="ev")]
    return B.page("Admin", sub, bl, tabs, live=6)

# ------------------------------------------------------------------ Copilot
def copilot(c, tab="chat"):
    kind = c["kind"]; tabs = [{"id": "chat", "label": "Ask the agents"}, {"id": "kb", "label": "Knowledge base"}, {"id": "tr", "label": "Document translator"}]; sub = "Agents run first, then the GPU LLM answers from their fresh evidence, in any of 37 languages (NLLB-200)."
    ex = ["Which outlets are at highest risk and why?", "How is inventory holding up this week?", "Why is OUT-007 flagged?"] if kind == "Franchise" else ["Which vessels are in danger from weather right now?", "What quotes need approval?", "How congested are the ports on the Suez route?"]
    if tab == "kb":
        d = rag.docs(c["email"]); seen = {}
        for x in d: seen.setdefault((x["title"].split(" (part")[0], x["source"]), 0); seen[(x["title"].split(" (part")[0], x["source"])] += 1
        mine = sum(1 for x in d if x["owner"] == c["email"]); ms = gpu.S["models"]
        st = f"Retrieval: BM25 always on · dense embeddings {ms['embed']['state']} · cross-encoder reranker {ms['rerank']['state']}"
        return B.page("AI Copilot", sub, [B.callout(st + f" · {len(d)} passages ({mine} yours). The sample policies are illustrative; upload your own PDFs or import a corpus built in the RAG notebook.", "info"),
            B.row(B.form("kb_up", [{"name": "file", "label": "PDF or text file", "type": "file"}], "Add to knowledge base", "/api/kb/upload", title="Add a document", sub="Chunked, indexed for BM25, embedded on the GPU. Your uploads are private to you."),
                  B.form("kb_q", [{"name": "q", "label": "Search the knowledge base", "type": "text"}, {"name": "mode", "label": "Method", "type": "select", "options": ["Hybrid + rerank", "BM25 only", "Dense only"], "value": "Hybrid + rerank"}, {"name": "live", "label": "Also search live tickets / reviews", "type": "select", "options": ["no", "yes"], "value": "no"}], "Search", "/api/kb/search", title="Search", tone="ghost")),
            B.row(B.form("kb_imp", [{"name": "file", "label": "kb_export.json (from the RAG builder notebook or an earlier export)", "type": "file"}], "Import corpus", "/api/kb/import", title="Import a corpus", tone="ghost"),
                  B.card("Export, evaluate, sync", B.button("kb_exp", "Download my corpus (JSON)", "/api/kb/export", "ghost", payload={"download": True}), B.button("kb_eval", "Evaluate retrieval (recall@k)", "/api/kb/eval", "primary", payload={}), B.button("kb_drv", "Reload my Drive RAG index", "/api/kb/drive", "ghost", payload={}))),
            B.table([{"document": k[0], "source": k[1], "chunks": v} for k, v in seen.items()], ["document", "source", B.num_col("chunks", "chunks", "int")], title=f"Documents available to you · {len(seen)}", h=300, key="kbd")], tabs, live=0)
    if tab == "tr":
        langs = [k for k in gpu.NLLB_LANGS]; return B.page("AI Copilot", sub, [B.form("trf", [{"name": "text", "label": "Text (any language, auto-detected)", "type": "textarea"}, {"name": "file", "label": "or a PDF / text file", "type": "file"}, {"name": "lang", "label": "Translate into", "type": "select", "options": langs, "value": langs[1]}], "Translate with NLLB-200", "/api/translate_doc", title="Document translator", sub="Facebook NLLB-200 on the GPU, sentence-chunked so long documents are translated completely.")], tabs, live=0)
    return B.page("AI Copilot", sub, [{"t": "chat", "examples": ex, "langs": list(gpu.NLLB_LANGS)}], tabs, live=0)


# ------------------------------------------------------------------ Reports
import reports as RP
def reports(c, tab="pdf"):
    kind = c["kind"]; tabs = [{"id": "pdf", "label": "PDF reports"}, {"id": "csv", "label": "Data exports (CSV)"}]; sub = "Print-ready reports built from live data and the agents' latest findings"
    if tab == "csv":
        bl = [B.callout("Each file is a live snapshot of the table, exactly as the agents see it." + (f" You only get rows for {c['scope']}." if c.get("scope") else ""), "info")]
        bl += [B.button("csv_" + t, f"Download {t}.csv", f"/api/export/{t}.csv", "ghost", payload={"download": True}) for t in RP.EXPORTS[kind]]
        return B.page("Reports", sub, bl, tabs, live=0)
    em = c["email"]; bl = [B.callout(f"PDFs are generated on demand from the current database. Use \"E-mail me\" to receive a copy at {em}" + ("" if __import__("mailer").configured() else " (e-mail is not configured on this server yet, so only Download works)") + ".", "info")]; cards = []
    for rid, title, desc, _, arg in RP.REPORTS[kind]:
        if c.get("scope") and rid in ("fq_exec",): continue
        fields = []
        if arg == "outlet": fields = [{"name": "arg", "label": "Outlet", "type": "select", "options": ([c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()), "value": c.get("scope")}]
        if arg == "quote": fields = [{"name": "arg", "label": "Quote", "type": "select", "options": Q("SELECT quote_id FROM freight_quotes ORDER BY created_at DESC LIMIT 300").quote_id.tolist()}]
        cards.append(B.card(title, B.md(desc), B.form("dl_" + rid, fields, "Download PDF", "get:/api/report/" + rid, tone="primary", inline=True) if False else {"t": "form", "id": "dl_" + rid, "fields": fields, "submit": "Download PDF", "url": "get:/api/report/" + rid, "title": None, "tone": "primary", "inline": True},
                            {"t": "form", "id": "em_" + rid, "fields": fields + [{"name": "rid", "type": "hidden", "value": rid}], "submit": "E-mail me", "url": "/api/report/email", "title": None, "tone": "ghost", "inline": True}))
    for i in range(0, len(cards), 2): bl.append(B.row(*cards[i:i + 2]))
    return B.page("Reports", sub, bl, tabs, live=0)

# ------------------------------------------------------------------ My profile
def profile(c, tab="me"):
    p = authx.profile(c["email"]); pr = p["prefs"]; tabs = [{"id": "me", "label": "Profile"}, {"id": "security", "label": "Security"}, {"id": "activity", "label": "My activity"}]; sub = f"{p.get('username', '')} · {c['role']}"
    if tab == "security":
        return B.page("My profile", sub, [B.form("pw", [{"name": "old", "label": "Current password", "type": "password"}, {"name": "new", "label": "New password", "type": "password"}], "Change password", "/api/profile/password", title="Change password"),
                                           B.callout("Sessions last 10 hours and are stored in an HttpOnly cookie. Five wrong passwords lock the account until an Admin unlocks it.", "info")], tabs, live=0)
    if tab == "activity":
        a = Q("SELECT ts, event, target, detail FROM audit_log WHERE actor=? ORDER BY audit_id DESC LIMIT 200", (c["email"],)); return B.page("My profile", sub, [B.table(a, ["ts", B.pill_col("event"), "target", "detail"], title="Everything you did on the platform (from the audit log)", h=520, key="myact", search=True)], tabs, live=0)
    langs = list(gpu.NLLB_LANGS)
    return B.page("My profile", sub, [B.kpis(("E-mail", p.get("email")), ("Role", p.get("role")), ("Member since", (p.get("created") or "")[:10]), ("Last sign-in", (p.get("last_login") or "-")[:16].replace("T", " "))),
        B.row(B.form("pf", [{"name": "username", "label": "User name", "type": "text", "value": p.get("username")}, {"name": "lang", "label": "Preferred language (NLLB-200)", "type": "select", "options": langs, "value": pr.get("lang") if pr.get("lang") in langs else "English"}], "Save profile", "/api/profile", title="Your details"),
              B.form("pp", [{"name": "email_alerts", "label": "E-mail me Critical alerts", "type": "select", "options": ["on", "off"], "value": "on" if pr.get("email_alerts") else "off"}, {"name": "daily_digest", "label": "Daily digest with the executive PDF", "type": "select", "options": ["off", "on"], "value": "on" if pr.get("daily_digest") else "off"}], "Save preferences", "/api/profile/prefs", title="Notifications", sub="The digest is e-mailed once a day to " + str(p.get("email")) + " (needs e-mail configured)." ))], tabs, live=0)


In [ ]:
%%writefile /content/app/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.now().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.now() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.now() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


In [ ]:
%%writefile /content/app/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os as _os, time as _time
_os.environ["TZ"] = "Asia/Kolkata"
try: _time.tzset()
except Exception: pass          # all times in this app are India Standard Time (IST)
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    import mailer, authx, reports as RP, datetime, threading
    def digest_loop():
        every = float(os.environ.get("DIGEST_EVERY_H", 24)) * 3600
        while True:
            time.sleep(60)
            try:
                if not mailer.configured(): continue
                for u in authx.users():
                    p = authx.profile(u["email"])["prefs"]
                    if not p.get("daily_digest"): continue
                    last = p.get("last_digest"); due = (not last) or (datetime.datetime.now() - datetime.datetime.fromisoformat(last)).total_seconds() >= every
                    if not due: continue
                    try: scope = LE_scope(u)
                    except Exception: scope = None
                    name, data = RP.build_pdf(config.KIND, "fr_exec" if config.KIND == "Franchise" else "fq_exec", {"scope": scope})
                    ok, m = mailer.send(u["email"], f"{dom.NAME} daily digest", "Your daily digest: the executive report is attached.", [(name, data, "application/pdf")])
                    if ok: authx.set_prefs(u["email"], last_digest=datetime.datetime.now().isoformat(timespec="seconds")); LE.audit(u["email"], "digest_sent", u["email"])
            except Exception: pass
    def LE_scope(u):
        import live_rbac as RB
        return RB.scope_for(u["email"]) if u["role"] in RB.SCOPED else None
    threading.Thread(target=digest_loop, daemon=True).start()
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


In [ ]:
%%writefile /content/app/db.py
import sqlite3, os
import pandas as pd
from config import DB_PATH

def get_conn():
    """Establishes and returns a connection to the SQLite database with WAL enabled."""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    return conn

def save_chat_message(username, role, message):
    """Saves a user or assistant chat message to the history ledger."""
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception:
        pass

def load_chat_history(username=None, limit=100):
    """Loads chat history for a specific user or the global system up to the specified limit."""
    try:
        with get_conn() as conn:
            if username:
                df = pd.read_sql("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", conn, params=(username, limit))
                if df.empty:
                    df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))
            else:
                df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))

            res = []
            for _, r in df.iterrows():
                content_val = str(r.get("message") or r.get("content") or "")
                res.append({
                    "role": str(r.get("role", "assistant")),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception:
        return []

def clear_chat_history(username=None):
    """Clears stored chat history entries."""
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception:
        pass

def init_db():
    """Initializes all database tables and executes migrations for existing database schemas."""
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            role TEXT,
            security_question TEXT,
            security_answer TEXT,
            profile_picture TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until DATETIME DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)

        # Safe migration for DBs created before these columns existed
        existing_cols = {r[1] for r in conn.execute("PRAGMA table_info(users);").fetchall()}
        migrations = {
            "username": "ALTER TABLE users ADD COLUMN username TEXT;",
            "security_question": "ALTER TABLE users ADD COLUMN security_question TEXT;",
            "security_answer": "ALTER TABLE users ADD COLUMN security_answer TEXT;",
            "profile_picture": "ALTER TABLE users ADD COLUMN profile_picture TEXT;",
            "failed_attempts": "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0;",
            "lock_until": "ALTER TABLE users ADD COLUMN lock_until DATETIME DEFAULT NULL;",
            "account_status": "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active';",
        }
        for col, ddl in migrations.items():
            if col not in existing_cols:
                try:
                    conn.execute(ddl)
                except Exception:
                    pass

        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.commit()

In [ ]:
%%writefile /content/app/domain_franchise.py
"""FranchiseOps AI live domain: simulator, 5 agents, intelligence engine, action handlers."""
import os, re, json, math, random, time, datetime, uuid
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit

KIND = "Franchise"; NAME = "FranchiseOps AI"; ENTITY_COL = "outlet_id"; SMS = True; ESCALATE_TO = "VP Operations"
OWNER = {"reorder": "Store Manager", "pause_promo": "Marketing Manager", "corrective_action": "Regional Manager", "retention_plan": "HR Lead",
         "outlet_review": "Regional Manager", "staffing_shortage": "Store Manager", "reallocate_budget": "Marketing Manager"}
ACTION_AGENT = {"reorder": "inventory", "pause_promo": "inventory", "corrective_action": "audit", "retention_plan": "staff",
                "outlet_review": "outlet", "staffing_shortage": "staff", "reallocate_budget": "marketing"}
SIM_DAY_SEC = 60          # one simulated day = 60 s of wall-clock (5 s ticks -> 12 ticks per day)
TICKS_PER_DAY = 12
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- bootstrap / simulator
def bootstrap():
    from scale_data import scale_franchise; scale_franchise()
    try:
        import live_rbac; live_rbac.set_scope("manager@infosys.com", "OUT-003")
    except Exception: pass
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS sales(sale_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, ts TEXT, amount REAL, tx_count INT, source TEXT);
        CREATE TABLE IF NOT EXISTS attendance(att_id INTEGER PRIMARY KEY AUTOINCREMENT, staff_id TEXT, outlet_id TEXT, ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS purchase_orders(po_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, sku_name TEXT, qty INT, ts TEXT, eta_ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE INDEX IF NOT EXISTS ix_sales ON sales(outlet_id, ts);
        """)
        if c.execute("SELECT COUNT(*) FROM sales").fetchone()[0] == 0:       # 90 days of seeded history so baselines exist
            rng = np.random.default_rng(7); today = datetime.date.today()
            for oid, rev in c.execute("SELECT outlet_id, revenue FROM outlets").fetchall():
                base = (rev or 100000) / 30
                for d in range(90, 0, -1):
                    dt = today - datetime.timedelta(days=d)
                    c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'seed')", (oid, dt.isoformat(), float(base * rng.normal(1, .07) * (1.15 if dt.weekday() >= 5 else 1)), int(base / 250)))
        active = c.execute("SELECT COUNT(*) FROM marketing WHERE end_date>=date('now','localtime')").fetchone()[0]
        if active == 0:                                                       # launch a live promo set (seed campaigns are all expired)
            ids = [r[0] for r in c.execute("SELECT campaign_id FROM marketing ORDER BY RANDOM() LIMIT 14").fetchall()]
            for i in ids: c.execute("UPDATE marketing SET start_date=date('now','localtime','-3 day'), end_date=date('now','localtime','+14 day') WHERE campaign_id=?", (i,))
        c.execute("UPDATE inventory SET weekly_demand=MAX(weekly_demand,1)")
        for rid, st_, wk, lead in c.execute("SELECT record_id, current_stock, weekly_demand, lead_time_days FROM inventory").fetchall():   # (re)start from a healthy, varied stock position
            if st_ / max(wk / 7, 1) < lead + 1.5: c.execute("UPDATE inventory SET current_stock=? WHERE record_id=?", (int(wk / 7 * (lead + random.uniform(1.5, 14))), rid))

_incident = {}; _tickn = [0]
def tick():
    reasons = []; _tickn[0] += 1; n = _tickn[0]; mult = get_state("demand_mult", 1.0) or 1.0
    ts = now()
    with _DB_LOCK, conn() as c:
        outlets = c.execute("SELECT outlet_id, revenue, operating_costs, customer_satisfaction FROM outlets").fetchall()
        import data_feeder as _FD; real = _FD.real_entities()
        for oid, rev, cost, csat in outlets:
            if oid in real: continue
            if oid not in _incident and random.random() < 0.002: _incident[oid] = random.randint(8, 14)
            f = 0.35 if oid in _incident else 1.0
            if oid in _incident:
                _incident[oid] -= 1
                if _incident[oid] <= 0: _incident.pop(oid)
            base = 3000000 / 30 / TICKS_PER_DAY if not rev else (rev / 30 / TICKS_PER_DAY)
            amt = float(base * random.uniform(.8, 1.2) * mult * f)
            c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'sim')", (oid, ts, amt, max(1, int(amt / 250))))
            c.execute("UPDATE outlets SET revenue=revenue+?, operating_costs=operating_costs+? WHERE outlet_id=?", (amt, amt * (cost / rev if rev else .65), oid))
        # stock drawdown + deliveries
        for rid, oid, sku, stock, thr, wk, lead in c.execute("SELECT record_id,outlet_id,sku_name,current_stock,reorder_threshold,weekly_demand,lead_time_days FROM inventory").fetchall():
            use = max(0, int(round(wk / 7 / TICKS_PER_DAY * mult * random.uniform(.6, 1.6))))
            new = max(0, stock - use); cover = new / max(wk / 7, 1)
            risk = float(min(1, max(0, 1 - cover / (lead + 3))))
            c.execute("UPDATE inventory SET current_stock=?, stockout_risk_prob=? WHERE record_id=?", (new, round(risk, 2), rid))
            if stock / max(wk / 7, 1) >= lead + 2 > cover: reasons.append(f"stock-low:{oid}")
        for po, oid, sku, qty in c.execute("SELECT po_id,outlet_id,sku_name,qty FROM purchase_orders WHERE status='ordered' AND eta_ts<=?", (ts,)).fetchall():
            c.execute("UPDATE inventory SET current_stock=current_stock+? WHERE outlet_id=? AND sku_name=?", (qty, oid, sku))
            c.execute("UPDATE purchase_orders SET status='delivered' WHERE po_id=?", (po,)); emit("po_delivered", oid, {"sku": sku, "qty": qty})
        # staff: attendance, overtime drift, attrition recompute (same heuristic the app uses)
        for sid, oid, sal, sat, ot in c.execute("SELECT staff_id,outlet_id,salary,job_satisfaction,overtime_hrs FROM staff ORDER BY RANDOM() LIMIT 40").fetchall():
            st = random.choices(["Present", "Late", "Absent"], [.86, .09, .05 + (.25 if oid in _incident else 0)])[0]
            c.execute("INSERT INTO attendance(staff_id,outlet_id,ts,status) VALUES(?,?,?,?)", (sid, oid, ts, st))
            ot2 = float(min(45, max(0, ot + random.uniform(-1.5, 2) + (1.5 if st == "Absent" else 0))))
            c.execute("UPDATE staff SET overtime_hrs=? WHERE staff_id=?", (round(ot2, 1), sid))
        # feedback + csat moving average
        for oid, csat in random.sample([(o[0], o[3]) for o in outlets], 6):
            r = int(min(5, max(1, round(random.gauss(csat if csat else 4, .9) - (1.5 if oid in _incident else 0)))))
            c.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)",
                      ("FB-L"+uuid.uuid4().hex[:10], oid, r, "Live feedback", datetime.date.today().isoformat(), (r - 3) / 2))
            c.execute("UPDATE outlets SET customer_satisfaction=ROUND(0.95*customer_satisfaction+0.05*?,3) WHERE outlet_id=?", (r, oid))
        if random.random() < 0.05:
            oid = random.choice(outlets)[0]; sc = float(np.clip(random.gauss(84, 12), 40, 100))
            c.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)",
                      ("AUD-L"+uuid.uuid4().hex[:10], oid, datetime.date.today().isoformat(), sc, int((100 - sc) / 6), "Food Safety", "Pass" if sc >= 85 else "Conditional Pass" if sc >= 70 else "Fail", "Live inspection"))
            if sc < 70: reasons.append(f"audit-fail:{oid}")
    return list(dict.fromkeys(reasons))[:5]

def process(): pass
def feed(): _external()
DEFAULT_POLICY = {"reorder": "approval", "pause_promo": "approval", "win_back_campaign": "approval", "reallocate_budget": "approval", "retention_plan": "approval",
                  "staffing_shortage": "approval", "service_recovery": "approval", "corrective_action": "approval", "outlet_review": "approval"}
# who must sign off, in order, for each agent action that needs a human (Admin can change any chain on Approvals > Autonomy policy)
DEFAULT_CHAINS = {"reorder": ["Operations Manager"], "pause_promo": ["Operations Manager"], "win_back_campaign": ["Operations Manager", "Franchise Owner"],
                  "reallocate_budget": ["Operations Manager", "Franchise Owner"], "retention_plan": ["Operations Manager", "Franchise Owner"],
                  "staffing_shortage": ["Operations Manager"], "service_recovery": ["Operations Manager"], "corrective_action": ["Operations Manager", "Admin"], "outlet_review": ["Franchise Owner", "Admin"]}
# things a person can ask for (Approvals > Request approval) and who signs each one off
REQUEST_KINDS = {"Stock transfer / emergency purchase": ["Operations Manager"], "Discount or price exception": ["Operations Manager", "Franchise Owner"], "Extra staffing or overtime": ["Operations Manager"],
                 "New hire or replacement": ["Operations Manager", "Admin"], "Leave or shift exception": ["Operations Manager"], "Refund or customer compensation": ["Operations Manager"],
                 "New marketing campaign": ["Operations Manager", "Franchise Owner"], "Budget or spend": ["Operations Manager", "Franchise Owner"], "Equipment repair or replacement": ["Operations Manager", "Admin"],
                 "Policy exception": ["Franchise Owner", "Admin"], "Outlet temporary closure": ["Franchise Owner", "Admin"], "Other": ["Operations Manager"]}
CHAIN_PRESETS = ["Operations Manager", "Operations Manager → Admin", "Operations Manager → Franchise Owner", "Operations Manager → Franchise Owner → Admin", "Franchise Owner → Admin", "Admin"]

def _external():
    """Real feeds: Open-Meteo weather (city list from the app), Nager.Date public holidays. Falls back silently when offline."""
    mult, feeds, rain = 1.0, {}, []
    try:
        from weather_context import CITY_COORDS
        for city, (la, lo) in list(CITY_COORDS.items())[:5]:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={la}&longitude={lo}&current=temperature_2m,precipitation,wind_speed_10m", timeout=4).json()["current"]
            rain.append(r.get("precipitation", 0) or 0); feeds[city] = {"temp": r["temperature_2m"], "rain": r["precipitation"], "wind": r["wind_speed_10m"]}
        if np.mean(rain) > 1.5: mult *= .92
        feeds_ok = True
    except Exception: feeds_ok = False
    hol = False
    try:
        hol = requests.get("https://date.nager.at/api/v3/IsTodayPublicHoliday/IN", timeout=4).status_code == 200
        if hol: mult *= 1.15
    except Exception: pass
    set_state("demand_mult", round(mult, 3)); set_state("weather", feeds); set_state("feeds", {"weather": feeds_ok, "holiday": hol, "ts": now()})

def feed_status():
    f = get_state("feeds", {}) or {}
    return [("Open-Meteo weather", "Real · live" if f.get("weather") else "Offline · no demand adjustment"),
            ("Public holidays (Nager.Date)", "Real · today is a holiday" if f.get("holiday") else "Real · no holiday today" if f else "Waiting"),
            ("Outlet POS, stock, attendance, feedback", "Simulated feed (replace via Data Feed Center)")]

# ---------------------------------------------------------------- agents (PRS v3: 7 agents + Intelligence Engine)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, silhouette_score
TIER_BASE = {"Excellent": 95, "Good": 80, "Needs Attention": 58, "Critical": 35}
def _reg_model(name, metrics, active=1):
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE ml_models SET is_active=0 WHERE model_name=?", (name,))
        c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,?)", (name, json.dumps(metrics), now(), active))

def outlet_agent(s):
    """Agent 1: KMeans (k=4) on 30-day revenue/orders/customers, ranked into Excellent/Good/Needs Attention/Critical."""
    d = Q("SELECT outlet_id, source, AVG(amount) rev, AVG(tx_count) orders FROM sales GROUP BY outlet_id, source")
    d.loc[d.source == "sim", ["rev", "orders"]] *= TICKS_PER_DAY
    d = d.sort_values("source").drop_duplicates("outlet_id", keep="first")            # sim rows preferred once present ('sim' < 'seed')
    d["cust"] = d.orders * .8; X = StandardScaler().fit_transform(d[["rev", "orders", "cust"]])
    km = KMeans(4, n_init=10, random_state=42).fit(X); d["cl"] = km.labels_
    order = d.groupby("cl").rev.mean().sort_values(ascending=False).index.tolist(); tmap = dict(zip(order, ["Excellent", "Good", "Needs Attention", "Critical"]))
    d["tier"] = d.cl.map(tmap); dist = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1); d["conf"] = (1 / (1 + dist)).round(2)
    d["score"] = d.tier.map(TIER_BASE) + d.groupby("tier").rev.rank(pct=True) * 8 - 4
    sil = float(silhouette_score(X, km.labels_)) if len(d) > 4 else 0
    _reg_model("kmeans_outlet_clusters", {"silhouette": round(sil, 3), "n": len(d)})
    crit = d[d.tier == "Critical"]
    return AgentResult("outlet", round(float(d.score.mean()), 1), risk_label(d.score.mean()), f"KMeans tiers: " + ", ".join(f"{t} {int((d.tier == t).sum())}" for t in TIER_BASE),
        {"per_outlet": d.set_index("outlet_id").score.round(1).to_dict(), "tiers": d.set_index("outlet_id").tier.to_dict(), "silhouette": round(sil, 3), "weak": crit.outlet_id.tolist()},
        [{"type": "outlet_review", "target": r.outlet_id, "detail": f"KMeans tier Critical (revenue {r.rev:,.0f}/day, confidence {r.conf})", "severity": "Warning"} for r in crit.itertuples()])

def forecast_series(outlet, horizon=14, method=None):
    """Demand forecast for one outlet. Prophet when installed and USE_PROPHET != 0 (daily + weekly seasonality), else linear trend x weekday-seasonal index."""
    h = Q("SELECT substr(ts,1,10) d, SUM(amount) a FROM sales WHERE outlet_id=? AND source='seed' GROUP BY d ORDER BY d", (outlet,))
    if len(h) < 21: return h.assign(kind="actual"), pd.DataFrame(), "insufficient history"
    h["d"] = pd.to_datetime(h.d); fut_d = pd.date_range(h.d.iloc[-1] + pd.Timedelta(days=1), periods=horizon)
    if (method or os.environ.get("USE_PROPHET", "1")) not in ("0", "trend"):
        try:
            from prophet import Prophet
            m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False); m.fit(h.rename(columns={"d": "ds", "a": "y"}))
            f = m.predict(pd.DataFrame({"ds": fut_d})); return h, pd.DataFrame({"d": f.ds, "a": f.yhat, "lo": f.yhat_lower, "hi": f.yhat_upper}), "Prophet"
        except Exception: pass
    y = h.a.values; x = np.arange(len(y)); k, b = np.polyfit(x, y, 1); trend = k * x + b; ratio = pd.Series(y / np.maximum(trend, 1)).groupby(h.d.dt.weekday.values).mean()
    xf = np.arange(len(y), len(y) + horizon); fa = (k * xf + b) * pd.Series(fut_d.weekday).map(ratio).values; sd = float(np.std(y - trend * h.d.dt.weekday.map(ratio).values))
    return h, pd.DataFrame({"d": fut_d, "a": fa, "lo": fa - 1.28 * sd, "hi": fa + 1.28 * sd}), "trend x weekday seasonality"

def _forecast_ratio(outlet):
    """Agent 2 forecast: Prophet when installed (USE_PROPHET=1), otherwise weekday-seasonal linear trend. Returns next-7-day / last-7-day demand ratio."""
    h, f, _ = forecast_series(outlet, 7, method="trend")                 # the fleet-wide loop stays on the fast method; Prophet is used on the Inventory page per outlet
    if not len(f): return 1.0
    return float(np.clip(f.a.mean() / max(h.a.iloc[-7:].mean(), 1), .8, 1.3))

def inventory_agent(s):
    mult = get_state("demand_mult", 1.0) or 1.0
    d = Q("SELECT outlet_id, sku_name, current_stock, reorder_threshold mn, weekly_demand, lead_time_days lead FROM inventory")
    fr = get_state("forecast_ratio", {}) or {}
    if not fr or now()[:13] != (get_state("forecast_hour") or ""):
        fr = {o: _forecast_ratio(o) for o in d.outlet_id.unique()}; set_state("forecast_ratio", fr); set_state("forecast_hour", now()[:13])
    d["daily"] = d.weekly_demand.clip(lower=1) / 7 * mult * d.outlet_id.map(fr).fillna(1.0); d["cover"] = d.current_stock / d.daily
    po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status='ordered'"); open_po = set(zip(po.outlet_id, po.sku_name))
    d["short"] = (d.current_stock <= d.mn) | (d.cover < d.lead + 2)
    per = (100 - d.groupby("outlet_id").short.mean() * 200).clip(0, 100).round(1)
    acts = []
    for r in d[d.short].sort_values("cover").itertuples():
        if (r.outlet_id, r.sku_name) in open_po: continue
        qty = int(max(r.mn * 3, math.ceil(r.daily * (r.lead + 7)) - r.current_stock))               # PRS: reorder_qty = min_stock x 3 (or forecast need if larger)
        acts.append({"type": "reorder", "target": r.outlet_id, "detail": f"{r.sku_name}: {r.cover:.1f} days cover, order {qty} units", "severity": "Critical" if r.cover < r.lead else "Warning", "sku": r.sku_name, "qty": qty, "lead": int(r.lead)})
    n = int(d.short.sum())
    return AgentResult("inventory", round(float(per.mean()), 1), risk_label(per.mean()), f"{n} SKUs flagged of {len(d)}, demand x{mult}", {"per_outlet": per.to_dict(), "outlets": d[d.short].outlet_id.unique().tolist(), "demand_mult": mult, "forecast": "weekday-seasonal trend (Prophet optional)"}, acts[:40])

_LR = {}
def _attrition_model():
    """Agent 3 ML: LogisticRegression. Uses a fed HR dataset (table hr_training, e.g. IBM HR Analytics) when present; otherwise a bootstrap model on the staff table labelled by the rule score."""
    d = Q("SELECT st.staff_id, st.outlet_id, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; feats = ["salary", "overtime_hrs", "sat", "ten", "wlb", "age", "att"]
    src = "bootstrap (labels from rule score)"
    try:
        h = Q("SELECT * FROM hr_training")
        if len(h) >= 300 and "attrition" in h.columns:
            src = "fed HR dataset"; tr = h.rename(columns={"job_satisfaction": "sat", "tenure_years": "ten", "work_life_balance": "wlb", "monthly_income": "salary", "overtime_hrs": "overtime_hrs"})
            for f in feats:
                if f not in tr: tr[f] = d[f].median()
            X, y = tr[feats].fillna(0), tr.attrition.astype(int)
    except Exception: pass
    if src.startswith("boot"): X, y = d[feats], (d.p + np.random.default_rng(1).normal(0, .05, len(d)) > .5).astype(int)
    sc = StandardScaler().fit(X); Xtr, Xte, ytr, yte = train_test_split(sc.transform(X), y, test_size=.2, random_state=42, stratify=y if y.nunique() > 1 else None)
    lr = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    m = {"accuracy": round(accuracy_score(yte, lr.predict(Xte)), 3), "auc": round(roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]), 3) if yte.nunique() > 1 else None, "source": src}
    _reg_model("logreg_attrition", m); _LR.update(model=lr, sc=sc, feats=feats, metrics=m, trained=time.time()); return d

def _staff_frame():
    d = Q("SELECT st.staff_id, st.outlet_id, st.name, st.role, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; return d

def attrition_table(scope=None, n=25):
    """Per-staff attrition probability with the single biggest driver (coefficient x standardised value): what the Workforce page shows."""
    if not _LR: return pd.DataFrame()
    d = _staff_frame()
    if scope: d = d[d.outlet_id == scope]
    if not len(d): return d
    X = _LR["sc"].transform(d[_LR["feats"]]); d["prob"] = _LR["model"].predict_proba(X)[:, 1].round(3)
    contrib = X * _LR["model"].coef_[0]; d["driver"] = [_LR["feats"][i] for i in contrib.argmax(1)]
    return d.sort_values("prob", ascending=False).head(n)
def attrition_importance():
    if not _LR: return {}
    return dict(zip(_LR["feats"], _LR["model"].coef_[0].round(3).tolist()))

def staff_agent(s):
    if not _LR or time.time() - _LR["trained"] > 900: _attrition_model()
    d = _staff_frame()
    d["prob"] = _LR["model"].predict_proba(_LR["sc"].transform(d[_LR["feats"]]))[:, 1].round(3)
    with _DB_LOCK, conn() as c: c.executemany("UPDATE staff SET predicted_attrition_prob=? WHERE staff_id=?", list(zip(d.prob.tolist(), d.staff_id)))
    d["perf"] = d.sat * 20; hi = d[d.prob > .6]; low = d[(d.perf < 60) | (d.att < 75)]
    per = (100 - (d.groupby("outlet_id").prob.mean() * 100 * .6 + (100 - d.groupby("outlet_id").att.mean()) * .4)).clip(0, 100).round(1)
    a = [{"type": "retention_plan", "target": o, "detail": f"{n} staff above 60% predicted attrition (LogReg {_LR['metrics']['source']})", "severity": "Warning"} for o, n in hi.groupby("outlet_id").size().items() if n >= 4]
    a += [{"type": "staffing_shortage", "target": o, "detail": f"{n} staff below 75% attendance or performance 60", "severity": "Warning"} for o, n in low.groupby("outlet_id").size().items() if n >= 6]
    return AgentResult("staff", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(hi)} high attrition risk, {len(low)} low performers or attendance (attrition model AUC {_LR['metrics'].get('auc', 0):.2f})", {"per_outlet": per.to_dict(), "short_staffed": [x["target"] for x in a if x["type"] == "staffing_shortage"], "model": _LR["metrics"]}, a[:25])

def marketing_agent(s):
    """Agent 4: ROI = (revenue-budget)/budget; KMeans (k=3) customer segments: High-Value, Frequent, Churned."""
    m = Q("SELECT outlet_id, campaign_name, channel, actual_roi, end_date FROM marketing"); m["roi_pct"] = (m.actual_roi - 1) * 100
    cu = Q("SELECT outlet_id, total_spend_90d sp, visit_freq vf, avg_order_value aov, days_since_last_visit dl FROM customers_fr")
    churn = {}; seg_counts = {}
    if len(cu) > 30:
        X = StandardScaler().fit_transform(cu[["sp", "vf", "aov", "dl"]]); km = KMeans(3, n_init=10, random_state=42).fit(X); cu["cl"] = km.labels_
        g = cu.groupby("cl").agg(sp=("sp", "mean"), dl=("dl", "mean")); names = {g.sp.idxmax(): "High-Value", g.dl.idxmax(): "Churned"}
        cu["seg"] = cu.cl.map(lambda k: names.get(k, "Frequent")); churn = cu.assign(ch=cu.seg == "Churned").groupby("outlet_id").ch.mean().to_dict(); seg_counts = cu.seg.value_counts().to_dict()
    per = (m.groupby("outlet_id").roi_pct.mean().clip(0, 300) / 3).round(1); best = m.groupby("channel").actual_roi.mean().idxmax()
    live = m[m.end_date >= datetime.date.today().isoformat()]; low = live[live.actual_roi < 2.0]
    acts = [{"type": "reallocate_budget", "target": r.outlet_id, "detail": f"{r.campaign_name}: ROI {r.roi_pct:.0f}% below 100%; shift budget to {best}", "severity": "Info"} for r in low.head(10).itertuples()]
    acts += [{"type": "win_back_campaign", "target": o, "detail": f"{v*100:.0f}% of customers in Churned segment", "severity": "Warning"} for o, v in churn.items() if v > .5][:10]
    return AgentResult("marketing", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(live)} live campaigns, best channel {best}, {sum(v > .5 for v in churn.values())} outlets with high churn", {"per_outlet": per.to_dict(), "active_outlets": live.outlet_id.unique().tolist(), "best_channel": best, "segments": seg_counts, "churn_share": {k: round(v, 3) for k, v in churn.items()}}, acts)

def audit_agent(s):
    """Agent 5: latest audit per outlet. Pass >= 80, Warning >= 60, Fail < 60. SOP hint from the FAISS/keyword RAG when available."""
    d = Q("SELECT outlet_id, score, category FROM audits ORDER BY audit_date DESC, rowid DESC").drop_duplicates("outlet_id")
    per = d.set_index("outlet_id").score.clip(0, 100).round(1); fail = d[d.score < 60]; warn = d[(d.score >= 60) & (d.score < 80)]
    hint = ""
    try:
        from rag_engine import answer_with_citation
        ctx, _ = answer_with_citation("food safety corrective action procedure"); hint = (str(ctx)[:140] + "...") if ctx else ""
    except Exception: pass
    acts = [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit FAIL {r.score:.0f} ({r.category}): corrective plan within 72 h. {hint}", "severity": "Critical", "sla_h": 72} for r in fail.itertuples()]
    acts += [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit WARNING {r.score:.0f} ({r.category}): re-check in 14 days", "severity": "Warning", "sla_h": 336} for r in warn.itertuples()][:15]
    return AgentResult("audit", round(float(per.mean()), 1), risk_label(per.mean()), f"Pass {int((d.score >= 80).sum())}, Warning {len(warn)}, Fail {len(fail)}", {"per_outlet": per.to_dict(), "failed": fail.outlet_id.tolist()}, acts)

POS = {"great", "good", "clean", "tasty", "friendly", "fast", "love", "excellent", "value", "fresh"}; NEG = {"slow", "rude", "dirty", "cold", "bad", "long", "wait", "stale", "expensive", "late"}
ISSUES = {"food quality": ("cold", "stale", "taste", "food"), "staff behavior": ("rude", "staff"), "wait time": ("slow", "wait", "long", "late"), "hygiene": ("dirty", "clean", "hygiene"), "value": ("expensive", "value", "price")}
def feedback_agent(s):
    """Agent 7: VADER when installed, else lexicon + rating blend; keyword issue categorisation."""
    f = Q("SELECT outlet_id, rating, comment FROM feedback ORDER BY rowid DESC LIMIT 6000")
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; v = SentimentIntensityAnalyzer(); f["sent"] = f.comment.map(lambda t: v.polarity_scores(str(t))["compound"]); method = "VADER"
    except Exception:
        f["sent"] = f.comment.map(lambda t: (sum(w in POS for w in str(t).lower().split()) - sum(w in NEG for w in str(t).lower().split())) / 2); method = "lexicon"
    f["sent"] = (.6 * (f.rating - 3) / 2 + .4 * f.sent.clip(-1, 1)); per = ((f.groupby("outlet_id").sent.mean() + 1) / 2 * 100).round(1)
    cat = {k: int(f.comment.str.lower().str.contains("|".join(v)).sum()) for k, v in ISSUES.items()}
    low = per[per < 45]
    return AgentResult("feedback", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(low)} outlets with poor sentiment ({method}); top issue {max(cat, key=cat.get)}", {"per_outlet": per.to_dict(), "issues": cat},
        [{"type": "service_recovery", "target": o, "detail": f"Sentiment score {v:.0f}/100; main issue: {max(cat, key=cat.get)}", "severity": "Warning"} for o, v in low.items()][:15])

def digest_agent(s):
    """Agent 6: weekly digest + immediate alerts. Runs on the weekly trigger or on request; stores the digest and emails the owner when SMTP is set."""
    r = s["results"]; per = r.get("outlet").findings["per_outlet"] if "outlet" in r else {}
    top = sorted(per, key=per.get, reverse=True)[:3]; bot = sorted(per, key=per.get)[:3]
    txt = f"Weekly digest {now()[:10]}: top {top}, bottom {bot}. " + " ".join(f"{k}: {v.summary}." for k, v in r.items() if k != "digest")
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS digests(digest_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, body TEXT)"); c.execute("INSERT INTO digests(ts,body) VALUES(?,?)", (now(), txt))
    return AgentResult("digest", 100, "Low", "weekly digest generated", {"digest": txt}, [{"type": "digest_sent", "target": "OWNER", "detail": txt[:300], "severity": "Info"}])


def anomaly_agent(s):
    """Live anomaly detection: IsolationForest over each outlet's recent sales-vs-baseline ratio, volatility and feedback sentiment."""
    from sklearn.ensemble import IsolationForest
    sl = Q("SELECT outlet_id, amount FROM sales WHERE source='sim' ORDER BY sale_id DESC LIMIT 4000")
    if len(sl) < 600: return AgentResult("anomaly", 100, "Low", "collecting live data (needs a few minutes of ticks)", {"flagged": []})
    base = Q("SELECT outlet_id, AVG(amount)/? b FROM sales WHERE source='seed' GROUP BY outlet_id", (TICKS_PER_DAY,)).set_index("outlet_id").b
    g = sl.groupby("outlet_id").amount; d = pd.DataFrame({"ratio": g.agg(lambda x: x.iloc[:5].mean()) / base, "vol": g.std() / g.mean()}).dropna()
    if len(d) < 10: return AgentResult("anomaly", 100, "Low", "waiting for enough outlets with both live and baseline sales", {"flagged": []})
    fb = Q("SELECT outlet_id, AVG(sentiment_score) s FROM (SELECT * FROM feedback ORDER BY rowid DESC LIMIT 1500) GROUP BY outlet_id").set_index("outlet_id").s
    d["sent"] = fb.reindex(d.index).fillna(0); iso = IsolationForest(contamination=.05, random_state=42).fit(d); d["flag"] = iso.predict(d) == -1
    bad = d[d.flag | (d.ratio < .6)]; sc = 100 - len(bad) / max(len(d), 1) * 300
    return AgentResult("anomaly", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} outlets with anomalous sales/feedback pattern (IsolationForest)", {"flagged": bad.index.tolist(), "detail": bad.round(2).reset_index().to_dict("records")[:30], "all": d.round(3).reset_index().to_dict("records")},
        [{"type": "anomaly_review", "target": o, "detail": f"Unusual pattern: sales at {r.ratio*100:.0f}% of baseline, volatility {r.vol:.2f}, sentiment {r.sent:.2f}", "severity": "Critical" if r.ratio < .6 else "Warning"} for o, r in bad.iterrows()][:20])

AGENTS = {"outlet": outlet_agent, "inventory": inventory_agent, "staff": staff_agent, "marketing": marketing_agent, "audit": audit_agent, "feedback": feedback_agent, "anomaly": anomaly_agent, "digest": digest_agent}
REPLAN = {"inventory": lambda r: ["marketing"] if r.risk != "Low" else [], "audit": lambda r: ["staff", "feedback"] if r.risk == "High" else [], "outlet": lambda r: ["inventory", "staff"] if r.risk != "Low" else []}
KW = {"outlet": ["outlet", "store", "revenue", "sales", "performance", "health"], "inventory": ["stock", "inventory", "reorder", "demand"], "staff": ["staff", "attrition", "workforce", "attendance", "employee"],
      "marketing": ["campaign", "roi", "marketing", "promo", "customer"], "audit": ["audit", "compliance", "violation", "sop"], "feedback": ["feedback", "sentiment", "review", "complaint"], "digest": ["digest", "weekly report"], "anomaly": ["anomal", "unusual", "spike", "fraud", "outlier"]}
ACTION_AGENT.update({"digest_sent": "digest", "anomaly_review": "anomaly", "win_back_campaign": "marketing", "service_recovery": "feedback", "health_alert": "outlet"}); OWNER.update({"anomaly_review": "Regional Manager", "win_back_campaign": "Marketing Manager", "service_recovery": "Store Manager", "health_alert": "Regional Manager", "digest_sent": "Franchise Owner"})
W = {"outlet": .30, "inventory": .20, "staff": .20, "marketing": .15, "audit": .15}     # PRS v3 weighted health score
def plan(req):
    q = (req or "").lower(); picked = [a for a, ks in KW.items() if any(re.search(r"\b" + k, q) for k in ks)]
    base = [a for a in AGENTS if a != "digest"]
    return (picked + (["digest"] if "digest" in picked and len(picked) == 1 else [])) if picked else base

def engine(s):
    r = s["results"]; used = [k for k in W if k in r and "per_outlet" in r[k].findings]
    outlets = set().union(*[set(r[k].findings.get("per_outlet", {})) for k in used]) if used else set()
    health = {o: round(sum(W[k] * r[k].findings["per_outlet"].get(o, r[k].score) for k in used) / sum(W[k] for k in used), 1) for o in outlets}
    score = float(np.mean(list(health.values()))) if health else 0.0
    actions = [a for res in r.values() for a in res.actions]
    for o, h in health.items():
        if h < 60: actions.append({"type": "health_alert", "target": o, "detail": f"Outlet health score {h} below 60 (perf, stock, staff, ROI, audit)", "severity": "Critical"})
    if "inventory" in r and "marketing" in r:
        for o in set(r["inventory"].findings.get("outlets", [])) & set(r["marketing"].findings.get("active_outlets", [])):
            actions.append({"type": "pause_promo", "target": o, "detail": "Stock shortage during active campaign: promo paused", "severity": "Critical"})
    if "staff" in r and "inventory" in r:
        for o in set(r["staff"].findings.get("short_staffed", [])) & set(r["inventory"].findings.get("outlets", [])):
            actions.append({"type": "staffing_shortage", "target": o, "detail": "Short-staffed while stock is low: add cover shift", "severity": "Critical"})
    order = {"Critical": 0, "Warning": 1, "Info": 2}; actions.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"franchise_health": round(score, 1), "outlets": len(health), "outlets_below_60": sum(h < 60 for h in health.values()), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()},
          "critical": sum(a.get("severity") == "Critical" for a in actions), "total_actions": len(actions)}
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": _narrate(ev, actions), "actions": actions, "agents": ev["agents"], "health": health, "evidence": ev}

def _narrate(ev, actions):
    tmpl = f"Franchise health {ev['franchise_health']}/100 across {ev['outlets']} outlets, {ev['outlets_below_60']} below 60. " + " ".join(f"{k.title()}: {v['summary']}." for k, v in ev["agents"].items()) + f" {ev['critical']} critical of {ev['total_actions']} actions."
    try:
        from llm_engine import is_llm_loaded, generate_text
        if not is_llm_loaded(): return tmpl
        txt = generate_text([{"role": "system", "content": "Write a 3-line executive recommendation for a franchise owner. Use only numbers present in the JSON."}, {"role": "user", "content": json.dumps(ev)}], max_new_tokens=160, temperature=0.1)
        ok = all(n in json.dumps(ev) for n in re.findall(r"\d+(?:\.\d+)?", txt or ""))
        return txt.strip() if txt and ok else tmpl
    except Exception: return tmpl

def apply(a, c):
    if a["type"] == "reorder":
        eta = (datetime.datetime.now() + datetime.timedelta(seconds=SIM_DAY_SEC * a.get("lead", 3))).isoformat(timespec="seconds")
        cur = c.execute("INSERT INTO purchase_orders(outlet_id,sku_name,qty,ts,eta_ts,status) VALUES(?,?,?,?,?,'ordered')", (a["target"], a["sku"], a["qty"], now(), eta))
        return f"PO-{cur.lastrowid} created, ETA {eta[11:19]}"
    if a["type"] == "pause_promo":
        n = c.execute("UPDATE marketing SET end_date=date('now','localtime','-1 day') WHERE outlet_id=? AND end_date>=date('now','localtime')", (a["target"],)).rowcount
        return f"{n} campaign(s) paused"
    return ""

# ---------------------------------------------------------------- UI hooks
def kpi_extra():
    s = Q("SELECT COALESCE(SUM(amount),0) v FROM sales WHERE source='sim'").v[0]; po = Q("SELECT COUNT(*) n FROM purchase_orders WHERE status='ordered'").n[0]
    return [("Live sales since start", f"Rs {s:,.0f}"), ("Purchase orders in transit", int(po))]


In [ ]:
%%writefile /content/app/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    """Safely converts a pandas DataFrame to Markdown format."""
    if df is None or df.empty:
        return ""
    try:
        return df.to_markdown(index=False)
    except Exception:
        headers = list(df.columns)
        lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
        lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
        for _, row in df.iterrows():
            vals = [str(v) if v is not None else "" for v in row.values]
            lines.append("| " + " | ".join(vals) + " |")
        return "\n".join(lines)

# Use meaningful phrases/whole words. Avoid tiny substrings such as
# "ad", "item", and "location" accidentally hijacking unrelated questions.
INTENT_MAP = {
    "staff": ["staff", "employee", "employees", "attrition", "salary", "workforce", "headcount", "hiring", "hire"],
    "outlet": ["outlet", "store", "branch", "revenue", "margin", "sales", "profit", "csat", "customer satisfaction"],
    "inventory": ["inventory", "stock", "stockout", "sku", "reorder", "demand", "supply", "safety stock"],
    "marketing": ["marketing", "campaign", "roi", "conversion", "promotion", "budget", "channel", "advertising", "advertisement", "ads"],
    "audit": ["audit", "compliance", "violation", "inspection", "policy", "standard", "fssai", "food safety"],
    "feedback": ["feedback", "review", "sentiment", "rating", "complaint", "customer review"],
}

def _contains_term(q, term):
    return bool(re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", q))

def classify_intent(query):
    """Score meaningful whole-word/phrase matches and return one primary domain."""
    q = query.lower().strip()
    scores = {}
    for intent, keywords in INTENT_MAP.items():
        score = 0
        for kw in keywords:
            if _contains_term(q, kw):
                score += 2 if " " in kw else 1
        scores[intent] = score
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"

def handle_franchise_intent(query):
    """Return only data for the most relevant franchise domain."""
    q_low = query.lower()
    intent = classify_intent(query)

    try:
        with get_conn() as conn:
            if intent == "outlet":
                total = conn.execute("SELECT COUNT(*) FROM outlets;").fetchone()[0]
                revenue = conn.execute("SELECT SUM(revenue) FROM outlets;").fetchone()[0] or 0
                csat = conn.execute("SELECT AVG(customer_satisfaction) FROM outlets;").fetchone()[0] or 0

                if any(_contains_term(q_low, k) for k in ["margin", "profit"]):
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(operating_costs, 0) AS costs_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY net_margin_pct DESC LIMIT 10;"""
                else:
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY revenue_rs DESC LIMIT 10;"""
                df = pd.read_sql(sql, conn)
                return f"### Franchise Outlet Data\nActive outlets: **{total}** | Total revenue: **Rs. {revenue:,.0f}** | Average CSAT: **{csat:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Outlets DB"

            if intent == "staff":
                total = conn.execute("SELECT COUNT(*) FROM staff;").fetchone()[0]
                high_risk = conn.execute("SELECT COUNT(*) FROM staff WHERE predicted_attrition_prob > 0.6;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, role, COUNT(*) AS staff_count,
                                    ROUND(AVG(salary), 0) AS avg_salary_rs,
                                    ROUND(AVG(predicted_attrition_prob), 2) AS avg_attrition_risk
                                    FROM staff GROUP BY outlet_id, role
                                    ORDER BY avg_attrition_risk DESC LIMIT 10;""", conn)
                return f"### Workforce Data\nTotal staff: **{total}** | High attrition risk (>60%): **{high_risk}**\n\n{df_to_markdown_safe(df)}", "Staff DB"

            if intent == "inventory":
                total_skus = conn.execute("SELECT COUNT(*) FROM inventory;").fetchone()[0]
                risk_items = conn.execute("SELECT COUNT(*) FROM inventory WHERE stockout_risk_prob > 0.7;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, sku_name, category, current_stock,
                                    reorder_threshold, stockout_risk_prob
                                    FROM inventory ORDER BY stockout_risk_prob DESC LIMIT 12;""", conn)
                return f"### Inventory Data\nTracked SKUs: **{total_skus}** | SKUs above 70% stockout risk: **{risk_items}**\n\n{df_to_markdown_safe(df)}", "Inventory DB"

            if intent == "marketing":
                avg_roi = conn.execute("SELECT AVG(actual_roi) FROM marketing;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT channel, COUNT(*) AS campaigns,
                                    ROUND(AVG(actual_roi), 2) AS avg_roi,
                                    SUM(conversions) AS conversions
                                    FROM marketing GROUP BY channel ORDER BY avg_roi DESC;""", conn)
                return f"### Marketing Data\nAverage campaign ROI: **{avg_roi:.2f}x**\n\n{df_to_markdown_safe(df)}", "Marketing DB"

            if intent == "feedback":
                avg_rating = conn.execute("SELECT AVG(rating) FROM feedback;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT outlet_id, rating, comment, sentiment_score, date
                                    FROM feedback ORDER BY sentiment_score ASC LIMIT 10;""", conn)
                return f"### Customer Feedback Data\nAverage rating: **{avg_rating:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Feedback DB"

            if intent == "audit":
                avg_score = conn.execute("SELECT AVG(score) FROM audits;").fetchone()[0] or 0
                open_items = conn.execute("SELECT COUNT(*) FROM audits WHERE status != 'Pass';").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, audit_date, score, violations, category, status
                                    FROM audits ORDER BY score ASC LIMIT 10;""", conn)
                return f"### Audit & Compliance Data\nAverage score: **{avg_score:.1f}/100** | Non-pass audits: **{open_items}**\n\n{df_to_markdown_safe(df)}", "Audits DB"

    except Exception:
        return None

    return None

def handle_port_intent(query):
    try:
        with get_conn() as conn:
            tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
            df_ports = pd.read_sql("""SELECT port_name AS 'Port Name', country AS 'Country',
                                      region AS 'Region',
                                      congestion_index AS 'Congestion (1-5)',
                                      avg_dwell_days AS 'Avg Dwell Days'
                                      FROM ports ORDER BY congestion_index ASC LIMIT 10;""", conn)
        return f"### Global Ports Data ({tot_ports} monitored)\n\n{df_to_markdown_safe(df_ports)}", f"Ports Ledger DB ({tot_ports} Hubs)"
    except Exception:
        return None, None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_franchise as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    """Retrieve one relevant evidence set. Do not mix unrelated data sources."""
    q_low = query.lower()

    franchise = handle_franchise_intent(query)
    if franchise:
        return franchise

    if any(_contains_term(q_low, k) for k in ["port", "ports", "harbor", "terminal", "congestion"]):
        return handle_port_intent(query)

    if any(_contains_term(q_low, k) for k in ["shipment", "shipments", "carrier"]):
        try:
            with get_conn() as conn:
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("""SELECT shipment_id, origin_port, dest_port, carrier,
                                         status, predicted_delay_risk
                                         FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 10;""", conn)
            return f"### Active Shipments ({tot_shipments})\n\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"
        except Exception:
            pass

    # Only invoke RAG for questions that clearly look like document/SOP/policy questions.
    rag_terms = ["sop", "document", "manual", "policy", "procedure", "fssai", "hygiene", "food safety", "guideline"]
    if any(_contains_term(q_low, k) for k in rag_terms):
        try:
            from rag_engine import answer_with_citation
            ctx, src = answer_with_citation(query)
            if ctx and str(ctx).strip():
                return ctx, src
        except Exception:
            pass

    return None, None

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)

In [ ]:
%%writefile /content/app/report_generator.py
import os, io
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

def generate_franchise_pdf_report(outlet_name, location, revenue, csat, audit_score, filename="franchise_audit_report.pdf"):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
    story = []
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontSize=22, textColor=colors.HexColor('#1e3a8a'), spaceAfter=12)
    sub_style = ParagraphStyle('DocSub', parent=styles['Normal'], fontSize=11, textColor=colors.HexColor('#475569'), spaceAfter=18)
    body_style = ParagraphStyle('DocBody', parent=styles['Normal'], fontSize=10, textColor=colors.HexColor('#0f172a'), spaceAfter=10)

    story.append(Paragraph("🏢 INFOSYS ENTERPRISE FRANCHISE AUDIT REPORT", title_style))
    story.append(Paragraph(f"Official Compliance & Operational Performance Briefing — {outlet_name}", sub_style))
    story.append(Spacer(1, 12))

    data = [
        ["Metric Parameter", "Telemetry Value", "Compliance Status"],
        ["Outlet Location", str(location), "Verified 🟢"],
        ["Monthly Revenue", f"₹{revenue:,.2f}", "Above Target 🟢"],
        ["Customer CSAT", f"{csat:.1f} / 5.0", "Optimal 🟢" if csat>=4.0 else "Needs Review 🟡"],
        ["Audit Compliance Score", f"{audit_score:.1f}%", "Pass 🟢" if audit_score>=80 else "Conditional Pass 🟡"]
    ]

    t = Table(data, colWidths=[200, 180, 150])
    t.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1e3a8a')),
        ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,0), 11),
        ('BOTTOMPADDING', (0,0), (-1,0), 8),
        ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#f8fafc')),
        ('GRID', (0,0), (-1,-1), 1, colors.HexColor('#cbd5e1')),
        ('ALIGN', (0,0), (-1,-1), 'LEFT')
    ]))
    story.append(t)
    story.append(Spacer(1, 24))

    story.append(Paragraph("<b>Executive Compliance Note:</b> This document certifies that the operational telemetry, food safety adherence, and staff workforce metrics for this outlet have been verified by the Multi-Agent Autonomous AI Auditor.", body_style))

    doc.build(story)
    buffer.seek(0)
    return buffer.getvalue()

In [ ]:
%%writefile /content/app/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas")
]

def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # 2. Demo accounts — one per role, INCLUDING Admin, all seeded the same guaranteed way.
        # Admin uses your ADMIN_EMAIL_ID/ADMIN_PASSWORD secrets if set, else falls back to a fixed default
        # (admin@infosys.com / Admin@123) — same predictable pattern as the other 3 roles.
        try:
            import hashlib
            from auth import hash_password
            from config import ADMIN_EMAIL_ID, ADMIN_PASSWORD

            demo_pw = hash_password("Demo@1234")
            demo_ans = hashlib.sha256("infosys".encode("utf-8")).hexdigest()
            admin_pw = hash_password(ADMIN_PASSWORD)
            admin_ans = hashlib.sha256("infosys".encode("utf-8")).hexdigest()

            demo_users = [
                (1, "admin",    ADMIN_EMAIL_ID,        "Admin",            admin_pw),
                (2, "owner",    "owner@infosys.com",   "Franchise Owner",  demo_pw),
                (3, "manager",  "manager@infosys.com", "Store Manager",    demo_pw),
                (4, "staff",    "staff@infosys.com",   "Staff",             demo_pw),
                (5, "customer", "customer@infosys.com","Customer",          demo_pw),
            ]

            # Upsert demo accounts. This is intentionally limited to the
            # authentication fields so stale passwords/lock states cannot
            # prevent non-admin users from logging in.
            for uid, uname, uemail, urole, pw_hash in demo_users:
                safe_exec(
                    conn,
                    "INSERT INTO users (id, username, email, password_hash, role, security_question, "
                    "security_answer, failed_attempts, lock_until, account_status) "
                    "VALUES (?, ?, ?, ?, ?, 'What is your favorite book/movie?', ?, 0, NULL, 'active') "
                    "ON CONFLICT(id) DO UPDATE SET "
                    "username=excluded.username, email=excluded.email, password_hash=excluded.password_hash, "
                    "role=excluded.role, security_question=excluded.security_question, "
                    "security_answer=excluded.security_answer, failed_attempts=0, lock_until=NULL, "
                    "account_status='active';",
                    (uid, uname, uemail, pw_hash, urole,
                     admin_ans if urole == "Admin" else demo_ans)
                )

        except Exception:
            pass

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"
            weight = round(random.uniform(500.0, 45000.0), 1)
            dist = round(random.uniform(800.0, 18000.0), 1)
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            co2 = round(weight * dist * 0.00012, 1)
            margin = round(random.uniform(8.5, 28.0), 1)
            dwell = random.randint(1, 8)
            cargo = random.choice(cargos)
            hs = f"HS-{random.randint(8400, 8900)}"

            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, weight_kg, distance_km, weather_severity, customs_hold_prob, congestion_index, predicted_delay_risk, co2_emissions_kg, freight_margin, port_dwell_days, cargo_type, hs_code) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), weight, dist, sev, ch_prob, cong, d_risk, co2, margin, dwell, cargo, hs)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")
        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()

In [ ]:
%%writefile /content/app/weather_context.py
import requests

CITY_COORDS = {
    "Mumbai": (19.0760, 72.8777), "Delhi": (28.61, 77.21), "Bangalore": (12.97, 77.59),
    "Chennai": (13.0827, 80.2707), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86),
    "Ahmedabad": (23.03, 72.57), "Jaipur": (26.91, 75.79), "Kolkata": (22.5726, 88.3639), "Surat": (21.1702, 72.8311)
}

def get_weather_report(city):
    try:
        lat, lon = CITY_COORDS.get(city, (19.08, 72.88))
        r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true", timeout=5)
        if r.status_code == 200:
            cw = r.json().get("current_weather", {})
            return {"temp": cw.get("temperature", 28), "wind": cw.get("windspeed", 10), "city": city}
    except: pass
    return {"temp": 28, "wind": 10, "city": city}

def get_city_weather(city):
    return get_weather_report(city)

def get_route_weather_multiplier(origin, dest):
    try:
        w1 = get_weather_report(origin)
        w2 = get_weather_report(dest)
        avg_wind = (w1["wind"] + w2["wind"]) / 2
        return 1.0 + (avg_wind / 100)
    except:
        return 1.0

In [ ]:
%%writefile /content/app/static/index.html
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Operations Platform</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%235b5bd6'/%3E%3Cpath d='M8 20l5-6 4 4 7-9' stroke='%23fff' stroke-width='3' fill='none' stroke-linecap='round'/%3E%3C/svg%3E">
<link rel="stylesheet" href="/static/vendor/leaflet.css"><link rel="stylesheet" href="/static/style.css"></head>
<body><div id="app"></div><div id="toast"></div><div id="drawer"><div class="dh"><b id="dt"></b><button class="x" onclick="UI.closeDrawer()">✕</button></div><div id="db"></div></div>
<script src="/static/vendor/echarts.min.js"></script><script src="/static/vendor/leaflet.js"></script><script src="/static/vendor/topojson-client.min.js"></script><script src="/static/app.js"></script></body></html>


In [ ]:
%%writefile /content/app/static/style.css
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-wght-normal.woff2) format("woff2");unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+2000-206F,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD}
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-ext-wght-normal.woff2) format("woff2");unicode-range:U+0100-02BA,U+02BD-02C5,U+02C7-02CC,U+02CE-02D7,U+02DD-02FF,U+0304,U+0308,U+0329,U+1D00-1DBF,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20C0,U+2113,U+2C60-2C7F,U+A720-A7FF}
:root{--bg:#f6f7fb;--card:#fff;--line:#e6e8f0;--line2:#d9dce8;--tx:#1c2024;--tx2:#454a58;--mut:#6b7080;--pri:#5b5bd6;--pri2:#4a4ac4;--pri-bg:#eef0ff;--teal:#12a594;--teal-bg:#e6f7f4;--blue:#3e63dd;--blue-bg:#edf2fe;--amber:#f5a524;--amber-bg:#fff6e0;--red:#e5484d;--red-bg:#feebec;--vio:#8e4ec6;--vio-bg:#f5eefb;--green:#30a46c;--r:14px;--sh:0 1px 2px rgba(20,24,40,.04),0 4px 14px rgba(20,24,40,.05)}
*{box-sizing:border-box}html,body{margin:0;height:100%;background:var(--bg);color:var(--tx);font:14px/1.5 "Inter",ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;font-feature-settings:"cv11","ss01";-webkit-font-smoothing:antialiased}
button,input,select,textarea{font:inherit;color:inherit}a{color:var(--pri);text-decoration:none}a:hover{text-decoration:underline}
#app{display:flex;height:100vh}
.side{width:248px;flex:none;background:#fff;border-right:1px solid var(--line);display:flex;flex-direction:column;padding:14px 12px;overflow:auto}
.brand{display:flex;gap:11px;align-items:center;padding:4px 6px 14px;font-weight:700;font-size:15px;letter-spacing:-.01em}.brand i{width:34px;height:34px;border-radius:11px;background:linear-gradient(135deg,var(--pri),#8b8bf0);display:grid;place-items:center;color:#fff;box-shadow:0 4px 12px rgba(91,91,214,.35)}.brand i svg{width:18px;height:18px;stroke:#fff;fill:none;stroke-width:2}
.brand small{display:block;color:var(--mut);font-weight:500;font-size:11px;max-width:150px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.ksearch{display:flex;align-items:center;gap:8px;margin:0 2px 10px;padding:8px 10px;border:1px solid var(--line);border-radius:10px;color:var(--mut);cursor:pointer;font-size:12.5px;background:var(--bg)}.ksearch:hover{border-color:var(--line2)}.ksearch kbd{margin-left:auto;font:600 10.5px "Inter";border:1px solid var(--line2);border-radius:5px;padding:1px 5px;background:#fff;color:var(--mut)}
.ngrp{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:12px 10px 4px}
.nav{display:flex;align-items:center;gap:10px;padding:8px 10px;border-radius:10px;color:var(--tx2);cursor:pointer;border:1px solid transparent;margin-bottom:1px;user-select:none;font-weight:500;transition:background .12s}
.nav:hover{background:#f1f2f8}.nav.on{background:var(--pri-bg);color:var(--pri2);font-weight:600}
.nav svg{width:17px;height:17px;flex:none;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round;opacity:.85}.nav .bd{margin-left:auto;background:var(--red);color:#fff;border-radius:99px;font-size:10.5px;padding:1px 7px;font-weight:700}.nav .bd.a{background:var(--amber);color:#3d2600}
.nav.cmd{margin-top:10px;background:linear-gradient(135deg,#5b5bd6,#7a6be0);color:#fff;font-weight:600;box-shadow:0 6px 16px rgba(91,91,214,.28)}.nav.cmd:hover{filter:brightness(1.05)}.nav.cmd.on{background:linear-gradient(135deg,#4a4ac4,#6a5bd8);color:#fff}
.main{flex:1;min-width:0;display:flex;flex-direction:column}
.top{display:flex;align-items:center;gap:12px;padding:12px 24px;border-bottom:1px solid var(--line);background:rgba(255,255,255,.85);backdrop-filter:blur(10px);flex-wrap:wrap}
.top h1{font-size:19px;margin:0;letter-spacing:-.015em}.top .sub{color:var(--mut);font-size:12.5px;margin:0}.sp{flex:1}
.chip{display:inline-flex;gap:6px;align-items:center;padding:4px 11px;border-radius:99px;border:1px solid var(--line);background:#fff;font-size:12px;white-space:nowrap;color:var(--tx2)}
.dot{width:8px;height:8px;border-radius:50%;background:#b4b8c8}.dot.g{background:var(--green);box-shadow:0 0 0 3px rgba(48,164,108,.18)}.dot.r{background:var(--red);box-shadow:0 0 0 3px rgba(229,72,77,.16)}.dot.a{background:var(--amber)}
.live{animation:pl 1.6s infinite}@keyframes pl{50%{opacity:.4}}
.content{flex:1;overflow:auto;padding:20px 24px 70px}
.tabs{display:flex;gap:2px;border-bottom:1px solid var(--line);margin-bottom:16px;flex-wrap:wrap}.tabs b{padding:9px 14px;cursor:pointer;color:var(--mut);border-bottom:2px solid transparent;font-weight:600;margin-bottom:-1px}.tabs b:hover{color:var(--tx)}.tabs b.on{color:var(--pri2);border-color:var(--pri)}
.blocks>*{margin-bottom:16px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:16px;box-shadow:var(--sh);min-width:0}.card>h3{margin:0 0 2px;font-size:14px;letter-spacing:-.005em}.card>.sub,.sub{color:var(--mut);font-size:12.5px;margin-bottom:8px}
.row{display:grid;gap:16px}.row>*{min-width:0}.col>*{margin-bottom:16px}.col>*:last-child{margin-bottom:0}
@media(max-width:1000px){.row{grid-template-columns:1fr!important}.side{width:64px}.nav span,.brand div,.side .foot,.ngrp,.ksearch{display:none}}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px}.kpi{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:14px 16px;position:relative;overflow:hidden;box-shadow:var(--sh)}
.kpi:before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:3px;border-radius:0 3px 3px 0;background:var(--teal)}.kpi.Critical:before,.kpi.High:before{background:var(--red)}.kpi.Warning:before,.kpi.Medium:before{background:var(--amber)}.kpi.Info:before{background:var(--blue)}
.kpi .l{color:var(--mut);font-size:11.5px;font-weight:600;text-transform:uppercase;letter-spacing:.05em}.kpi .v{font-size:26px;font-weight:700;margin-top:3px;letter-spacing:-.02em}.kpi .s{color:var(--mut);font-size:12px}
.pill{display:inline-block;padding:1px 9px;border-radius:99px;font-size:11.5px;font-weight:600;border:1px solid}
.tbl{overflow:auto;border:1px solid var(--line);border-radius:12px;background:#fff}.tbl table{width:100%;border-collapse:collapse;font-size:12.8px}.tbl th{position:sticky;top:0;background:#fafbfe;text-align:left;padding:9px 12px;color:var(--mut);font-weight:600;cursor:pointer;white-space:nowrap;text-transform:capitalize;z-index:1;border-bottom:1px solid var(--line)}
.tbl td{padding:8px 12px;border-top:1px solid #eff0f6;max-width:380px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.tbl tr.cl{cursor:pointer}.tbl tr.cl:hover td{background:var(--pri-bg)}
.bar{display:flex;align-items:center;gap:8px;min-width:110px}.bar i{display:block;height:6px;border-radius:4px;flex:1;background:#eceef6;overflow:hidden}.bar i b{display:block;height:100%;border-radius:4px}
.srch{margin:0 0 8px;padding:7px 11px;border-radius:9px;border:1px solid var(--line2);background:#fff;width:240px}
.callout{padding:11px 14px;border-radius:12px;border:1px solid;display:flex;gap:10px;color:var(--tx2)}.callout.info{border-color:#cfd9fb;background:var(--blue-bg)}.callout.warn,.callout.warning{border-color:#f6d98f;background:var(--amber-bg)}.callout.error,.callout.critical,.callout.bad{border-color:#f5bcbe;background:var(--red-bg)}.callout.good,.callout.ok,.callout.success{border-color:#b6e3da;background:var(--teal-bg)}
.md{line-height:1.65}.md code,.md pre{background:#f1f2f8;border-radius:6px;padding:1px 5px;font-family:ui-monospace,"JetBrains Mono",monospace;font-size:12px}.md pre{padding:10px;overflow:auto;white-space:pre-wrap}.md h1,.md h2,.md h3{margin:.7em 0 .3em;letter-spacing:-.01em}.md ul{margin:.3em 0;padding-left:20px}.md table{border-collapse:collapse;margin:.5em 0}.md td,.md th{border:1px solid var(--line);padding:5px 10px}
.btn{border:1px solid var(--line2);background:#fff;padding:7px 14px;border-radius:10px;cursor:pointer;font-weight:600;color:var(--tx2);transition:all .12s}.btn:hover{background:#f4f5fb;border-color:#c9cde0}
.btn.primary{background:var(--pri);color:#fff;border-color:var(--pri);box-shadow:0 2px 8px rgba(91,91,214,.3)}.btn.primary:hover{background:var(--pri2)}.btn.danger{background:var(--red-bg);border-color:#f5bcbe;color:#c62a2f}.btn.ok{background:var(--teal-bg);border-color:#b6e3da;color:#0b7a6c}.btn.ghost{background:transparent}.btn.sm{padding:3px 10px;font-size:12px}.btn:disabled{opacity:.5;cursor:wait}
form.f{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;align-items:end}form.f.inline{grid-template-columns:repeat(auto-fit,minmax(160px,max-content))}
.fld label{display:block;color:var(--mut);font-size:12px;font-weight:500;margin-bottom:4px}.fld input,.fld select,.fld textarea,.in{width:100%;padding:8px 11px;border-radius:10px;border:1px solid var(--line2);background:#fff;transition:border .12s,box-shadow .12s}.fld input:focus,.fld select:focus,.fld textarea:focus,.in:focus,.srch:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}.fld textarea{min-height:80px;grid-column:1/-1}.fld.wide{grid-column:1/-1}.fld input[type=range]{padding:0;accent-color:var(--pri)}
.chk{display:flex;flex-wrap:wrap;gap:6px}.chk label{display:flex;gap:6px;align-items:center;margin:0;color:var(--tx2);padding:3px 10px;border:1px solid var(--line2);border-radius:99px;font-size:12px;background:#fff}
.res{margin-top:12px}
.tk{border:1px solid var(--line);border-left:4px solid var(--blue);border-radius:12px;padding:12px 14px;margin-bottom:10px;background:#fff;box-shadow:var(--sh)}.tk.Critical{border-left-color:var(--red)}.tk.Warning{border-left-color:var(--amber)}.tk.proposed{background:var(--vio-bg)}
.tk .h{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.tk .d{margin:6px 0;color:var(--tx2)}.tk .m{color:var(--mut);font-size:11.8px;display:flex;gap:10px;flex-wrap:wrap}.tk .a{display:flex;gap:6px;margin-top:8px;flex-wrap:wrap;align-items:center}.tk .cm{margin-top:8px;border-top:1px dashed var(--line2);padding-top:6px;font-size:12.5px}
.filters{display:flex;gap:6px;margin-bottom:12px;flex-wrap:wrap;align-items:center}.filters .chip{cursor:pointer}.filters .chip:hover{border-color:var(--pri)}.filters .chip.on{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2);font-weight:600}
.hero{display:flex;gap:18px;align-items:center;background:linear-gradient(120deg,#eef0ff,#e8f6f4);border:1px solid #d9def7;border-radius:16px;padding:16px 20px}.hero .sc{font-size:36px;font-weight:800;min-width:84px;text-align:center;letter-spacing:-.03em}.hero h2{margin:0;font-size:16px}.hero p{margin:2px 0;color:var(--mut);font-size:12.5px}
.map{border-radius:12px;overflow:hidden;border:1px solid var(--line);background:#cfe5f7}.leaflet-container{background:#cfe5f7;font:inherit}.lg{display:flex;gap:12px;font-size:12px;color:var(--mut);margin-top:8px;flex-wrap:wrap}.lg i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
.leaflet-tooltip{background:#fff;color:var(--tx);border:1px solid var(--line);border-radius:8px;box-shadow:var(--sh)}
#toast{position:fixed;right:18px;bottom:18px;display:flex;flex-direction:column;gap:8px;z-index:99}#toast div{background:#1c2024;color:#fff;padding:10px 16px;border-radius:12px;max-width:380px;box-shadow:0 10px 30px rgba(20,24,40,.25);animation:up .25s}#toast div.bad{background:#c62a2f}@keyframes up{from{transform:translateY(10px);opacity:0}}
#drawer{z-index:2000!important;position:fixed;right:0;top:0;bottom:0;width:min(720px,96vw);background:#fff;border-left:1px solid var(--line);transform:translateX(105%);transition:.25s;z-index:50;display:flex;flex-direction:column;box-shadow:-20px 0 60px rgba(20,24,40,.12)}#drawer.on{transform:none}
#drawer .dh{display:flex;justify-content:space-between;padding:14px 18px;border-bottom:1px solid var(--line);font-size:16px}#db{padding:16px;overflow:auto;flex:1;background:var(--bg)}.x{background:none;border:0;color:var(--mut);cursor:pointer;font-size:16px}
.login{display:grid;grid-template-columns:1.1fr 1fr;width:100%;min-height:100vh;background:#fff}.lp{background:radial-gradient(900px 500px at 20% 0%,#7c7cf0 0%,transparent 60%),linear-gradient(145deg,#4a4ac4,#5b5bd6 55%,#12a594);color:#fff;padding:56px;display:flex;flex-direction:column;justify-content:center;gap:18px}.lp h2{font-size:34px;line-height:1.15;margin:0;letter-spacing:-.025em}.lp p{margin:0;color:rgba(255,255,255,.85);max-width:460px}.lp ul{list-style:none;margin:8px 0 0;padding:0;display:grid;gap:10px}.lp li{display:flex;gap:10px;align-items:center;color:rgba(255,255,255,.92)}.lp li:before{content:"";width:8px;height:8px;border-radius:50%;background:#7ff3df;flex:none}
.auth{margin:auto;width:min(440px,92vw);padding:26px;border:0;box-shadow:none}.auth h1{margin:0 0 4px;letter-spacing:-.02em}.auth .fld{margin-bottom:12px}.demo{display:flex;gap:6px;flex-wrap:wrap;margin-top:14px}.demo .chip{cursor:pointer}.demo .chip:hover{border-color:var(--pri);color:var(--pri2)}
@media(max-width:860px){.login{grid-template-columns:1fr}.lp{display:none}}
.boot{text-align:center;padding:60px;color:var(--mut)}.spin{width:28px;height:28px;border:3px solid var(--line);border-top-color:var(--pri);border-radius:50%;animation:sp .8s linear infinite;margin:0 auto 12px}@keyframes sp{to{transform:rotate(360deg)}}
.warnbar{background:var(--amber-bg);border-bottom:1px solid #f6d98f;color:#7a4b00;padding:7px 24px;font-size:12.5px}
.chat{display:flex;flex-direction:column;gap:12px}.msgs{display:flex;flex-direction:column;gap:12px;min-height:280px}.m{max-width:88%;padding:11px 15px;border-radius:16px;border:1px solid var(--line)}.m.u{align-self:flex-end;background:var(--pri);color:#fff;border-color:var(--pri);border-bottom-right-radius:5px}.m.a{background:#fff;box-shadow:var(--sh);border-bottom-left-radius:5px}.st{color:var(--mut);font-size:12px}.st:before{content:"› "}.cites{margin:4px 0;font-size:12px;color:var(--mut)}.cites summary{cursor:pointer;color:var(--pri2);font-weight:600}
.cin{display:flex;gap:8px}.cin input{flex:1;padding:11px 15px;border-radius:12px;border:1px solid var(--line2);background:#fff}.cin input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}
.kpal{position:fixed;inset:0;background:rgba(20,24,40,.35);backdrop-filter:blur(2px);z-index:90;display:flex;justify-content:center;padding-top:12vh}.kpal>div{width:min(620px,94vw);background:#fff;border-radius:16px;box-shadow:0 30px 80px rgba(20,24,40,.3);overflow:hidden;align-self:flex-start}.kpal input{width:100%;border:0;border-bottom:1px solid var(--line);padding:16px 18px;font-size:15px;outline:0}.kpal .it{display:flex;gap:10px;padding:10px 18px;cursor:pointer;align-items:center;color:var(--tx2)}.kpal .it.on{background:var(--pri-bg);color:var(--pri2)}.kpal .it small{margin-left:auto;color:var(--mut)}.kpal .lst{max-height:50vh;overflow:auto;padding:6px 0}.kpal .gl{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:8px 18px 2px}
::-webkit-scrollbar{width:9px;height:9px}::-webkit-scrollbar-thumb{background:#d5d8e6;border-radius:9px}
@media(max-width:760px){#app{flex-direction:column-reverse}.side{width:100%;flex-direction:row;padding:4px;overflow-x:auto;border-right:0;border-top:1px solid var(--line);position:sticky;bottom:0;z-index:20;gap:2px}.side .brand,.side .foot,.side .sp,.ngrp,.ksearch{display:none}#nav{display:flex;gap:2px}.nav{flex-direction:column;gap:2px;padding:6px 10px;font-size:10px;min-width:62px;text-align:center;margin:0}.nav.cmd{margin-top:0}.nav span{display:block!important;white-space:nowrap;max-width:70px;overflow:hidden;text-overflow:ellipsis}.nav .bd{position:absolute;margin:0}.nav{position:relative}.content{padding:12px 10px 20px}.top{padding:8px 10px}.top h1{font-size:15px}.kpis{grid-template-columns:repeat(2,1fr)}.hero{flex-direction:column;align-items:flex-start}#drawer{width:100vw}.tbl td{max-width:200px}.srch{width:100%}.chip{font-size:11px}}
.top>div:first-child{min-width:0;flex:1 1 280px}.top .sub{white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:100%}

.leaflet-container{isolation:isolate;overflow:hidden!important}.mapw,.map,.leafmap{isolation:isolate;position:relative;z-index:0}#drawer.on{z-index:2000}.kpal{z-index:3000!important}#toast{z-index:3500!important}


In [ ]:
%%writefile /content/app/static/app.js
"use strict";
const $=(s,r=document)=>r.querySelector(s);
const esc=s=>String(s==null?"":s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
function h(tag,attrs,...kids){const e=document.createElement(tag);for(const k in(attrs||{})){const v=attrs[k];if(v==null||v===false)continue;if(k==="class")e.className=v;else if(k==="html")e.innerHTML=v;else if(k.startsWith("on"))e.addEventListener(k.slice(2),v);else e.setAttribute(k,v===true?"":v)}for(const c of kids.flat())if(c!=null&&c!==false)e.append(c.nodeType?c:document.createTextNode(c));return e}
const TONE={Critical:"#cd2b31",High:"#cd2b31",Fail:"#cd2b31",rejected:"#cd2b31","On Hold":"#cd2b31",overdue:"#cd2b31",Warning:"#ab6400",Medium:"#ab6400","Pending Approval":"#ab6400",proposed:"#793aaf",Info:"#3451b2",open:"#3451b2",Low:"#0b7a6c",Pass:"#0b7a6c",Approved:"#0b7a6c",approved:"#0b7a6c",resolved:"#0b7a6c","auto-closed":"#6b7080",live:"#0b7a6c",Healthy:"#0b7a6c",Watch:"#ab6400",Rejected:"#6b7080"};
const tone=v=>TONE[v]||(/crit|high|fail|bad|reject|hold/i.test(v)?"#cd2b31":/warn|med|pend/i.test(v)?"#ab6400":/ok|pass|low|good|appr|done|clear|live/i.test(v)?"#0b7a6c":"#6b7080");
const ICON={bot:"M12 8V4H8M4 12a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2zM2 14h2M20 14h2M9 15v1M15 15v1",users:"M16 21v-2a4 4 0 0 0-4-4H6a4 4 0 0 0-4 4v2M9 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8M22 21v-2a4 4 0 0 0-3-3.9M16 3.1a4 4 0 0 1 0 7.8",store:"M3 9l1-5h16l1 5M3 9v11h18V9M3 9a3 3 0 0 0 6 0 3 3 0 0 0 6 0 3 3 0 0 0 6 0M9 20v-6h6v6",boxes:"M21 8l-9-5-9 5 9 5zM3 8v8l9 5 9-5V8M12 13v8",megaphone:"M3 11v2a1 1 0 0 0 1 1h3l8 5V5L7 10H4a1 1 0 0 0-1 1zM19 8a5 5 0 0 1 0 8",messages:"M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z",clipboard:"M9 4h6v3H9zM8 5H6a2 2 0 0 0-2 2v13a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V7a2 2 0 0 0-2-2h-2M9 13l2 2 4-4",bell:"M18 8a6 6 0 0 0-12 0c0 7-3 9-3 9h18s-3-2-3-9M13.7 21a2 2 0 0 1-3.4 0",network:"M12 2v6M12 16v6M5 6l5 4M19 6l-5 4M5 18l5-4M19 18l-5-4M12 8a4 4 0 1 0 0 8 4 4 0 0 0 0-8",flask:"M10 2v7L4 20a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1l-6-11V2M8 2h8M7 15h10",upload:"M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4M17 8l-5-5-5 5M12 3v12",shield:"M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10",radar:"M12 12l7-7M12 2a10 10 0 1 0 10 10M12 6a6 6 0 1 0 6 6M12 10a2 2 0 1 0 2 2",route:"M6 19a3 3 0 1 0 0-6 3 3 0 0 0 0 6M18 11a3 3 0 1 0 0-6 3 3 0 0 0 0 6M9 16h6a3 3 0 0 0 0-6H9a3 3 0 0 1 0-6h3",dollar:"M12 2v20M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6",truck:"M1 3h15v13H1zM16 8h4l3 3v5h-7M5.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5M18.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5",cloud:"M17.5 19a4.5 4.5 0 1 0-1.4-8.8A6 6 0 1 0 5 15a4 4 0 0 0 1 4z",trend:"M3 17l6-6 4 4 8-8M15 7h6v6",chart:"M3 3v18h18M7 15l4-4 3 3 5-6",check:"M22 11.1V12a10 10 0 1 1-5.9-9.1M22 4L12 14l-3-3",user:"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8",file:"M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M9 13h6M9 17h6"};
const icon=n=>{const s=document.createElementNS("http://www.w3.org/2000/svg","svg");s.setAttribute("viewBox","0 0 24 24");const p=document.createElementNS("http://www.w3.org/2000/svg","path");p.setAttribute("d",ICON[n]||ICON.file);s.append(p);return s};
const S={me:null,page:null,tab:null,params:{},lang:localStorage.getItem("lang")||"English",pulse:null,tree:null,timer:null,seq:0,boot:null};
const LANGS_CODE={};
async function api(path,opt={}){const o={credentials:"same-origin",...opt};if(opt.json!==undefined){o.method=o.method||"POST";o.headers={"Content-Type":"application/json"};o.body=JSON.stringify(opt.json)}
 const r=await fetch(path,o);let d=null;try{d=await r.json()}catch(e){}
 if(r.status===401&&S.me&&!path.includes("login")){S.me=null;render();return{ok:false,msg:"Session ended"}}
 if(!r.ok&&d&&!d.msg)d.msg=d.detail||("Error "+r.status);if(!d)d={ok:false,msg:"Error "+r.status};return d}
function toast(msg,bad){const t=h("div",{class:bad?"bad":""},msg);$("#toast").append(t);setTimeout(()=>t.remove(),bad?7000:4500)}
const fmt={num:v=>v==null?"–":(typeof v==="number"?(Math.abs(v)>=1000?v.toLocaleString(undefined,{maximumFractionDigits:1}):+v.toFixed(2)):v),int:v=>v==null?"–":Math.round(v).toLocaleString(),money:v=>v==null?"–":"$"+Math.round(v).toLocaleString()};
function md(t){let s=esc(t||"");s=s.replace(/```([\s\S]*?)```/g,(_,c)=>"<pre>"+c.trim()+"</pre>").replace(/`([^`]+)`/g,"<code>$1</code>").replace(/\*\*([^*]+)\*\*/g,"<b>$1</b>").replace(/\[(\d{1,2})\]/g,'<sup style="font-weight:600;color:#2b6cb0">[$1]</sup>');const L=s.split("\n");let o="",inl=false,tb=null;const row=l=>l.trim().replace(/^\||\|$/g,"").split("|").map(x=>x.trim());const endT=()=>{if(tb){o+='<div class="tbl" style="margin:8px 0"><table><thead><tr>'+row(tb[0]).map(x=>"<th>"+x+"</th>").join("")+"</tr></thead><tbody>"+tb.slice(2).map(r=>"<tr>"+row(r).map(x=>"<td>"+x+"</td>").join("")+"</tr>").join("")+"</tbody></table></div>";tb=null}};for(let i=0;i<L.length;i++){const l=L[i];if(/^\s*\|.*\|\s*$/.test(l)){if(inl){o+="</ul>";inl=false}(tb=tb||[]).push(l);continue}endT();const m=l.match(/^\s*[-*•]\s+(.*)/),nm=l.match(/^\s*(\d{1,2})\.\s+(.*)/);if(nm){if(inl){o+="</ul>";inl=false}o+='<div style="margin:2px 0 2px 4px"><b>'+nm[1]+".</b> "+nm[2]+"</div>"}else if(m){if(!inl){o+="<ul>";inl=true}o+="<li>"+m[1]+"</li>"}else{if(inl){o+="</ul>";inl=false}const hd=l.match(/^(#{1,3})\s+(.*)/);o+=hd?`<h${hd[1].length}>${hd[2]}</h${hd[1].length}>`:(l.trim()?l+"<br>":"")}}endT();if(inl)o+="</ul>";return o}

// ---------------------------------------------------------------- block renderers (each returns {el, update(b)})
function mount(b,key){const f=R[b.t];if(!f){return{el:h("div",{class:"card"},"Unsupported block: "+b.t),update(){}}}const m=f(b,key);m.t=b.t;m.k=b.key||b.id||"";return m}
function mountList(parent,blocks,key){const hs=blocks.map((b,i)=>mount(b,key+"/"+i));hs.forEach(x=>parent.append(x.el));return hs}
function updList(parent,hs,blocks,key){if(hs.length!==blocks.length||hs.some((x,i)=>x.t!==blocks[i].t||x.k!==(blocks[i].key||blocks[i].id||""))){parent.replaceChildren();return mountList(parent,blocks,key)}hs.forEach((x,i)=>x.update(blocks[i]));return hs}
const R={};
R.kpis=b=>{const el=h("div",{class:"kpis"});const up=b=>{el.replaceChildren(...b.items.map(i=>h("div",{class:"kpi "+(i.tone||"")},h("div",{class:"l"},i.label),h("div",{class:"v"},typeof i.value==="number"?fmt.num(i.value):i.value),i.sub?h("div",{class:"s"},i.sub):null)))};up(b);return{el,update:up}};
R.callout=b=>{const el=h("div");const up=b=>{el.className="callout "+(b.tone||"info");el.replaceChildren(h("div",null,b.title?h("b",null,b.title+" "):null,h("span",{html:md(b.text)})))};up(b);return{el,update:up}};
R.md=b=>{const el=h("div",{class:"md"});const up=b=>el.innerHTML=md(b.text);up(b);return{el,update:up}};
R.row=(b,k)=>{const el=h("div",{class:"row"});const sty=r=>el.style.gridTemplateColumns=(r||b.cols.map(()=>1)).map(x=>x+"fr").join(" ");sty(b.ratio);let hs=[];const wrap=[];const up=nb=>{sty(nb.ratio);if(hs.length!==nb.cols.length||hs.some((x,i)=>x.t!==nb.cols[i].t)){el.replaceChildren();hs=nb.cols.map((c,i)=>{const m=mount(c,k+"/"+i);el.append(m.el);return m})}else hs.forEach((x,i)=>x.update(nb.cols[i]))};up(b);return{el,update:up}};
R.col=(b,k)=>{const el=h("div",{class:"col"});let hs=mountList(el,b.blocks,k);return{el,update:nb=>{hs=updList(el,hs,nb.blocks,k)}}};
R.card=(b,k)=>{const el=h("div",{class:"card"});const hd=h("h3"),sub=h("div",{class:"sub"}),body=h("div");el.append(hd,sub,body);let hs=[];const up=(nb,first)=>{hd.textContent=nb.title||"";hd.style.display=nb.title?"":"none";sub.textContent=nb.sub||"";sub.style.display=nb.sub?"":"none";hs=first?mountList(body,nb.blocks,k):updList(body,hs,nb.blocks,k)};up(b,true);return{el,update:nb=>up(nb)}};
R.hero=b=>{const el=h("div",{class:"hero"});const up=b=>{const c=b.risk?tone(b.risk):"#8b8d98";el.replaceChildren(h("div",{class:"sc",style:"color:"+c},b.score==null?"–":Math.round(b.score)),h("div",{style:"flex:1"},h("h2",null,b.title," ",b.risk?h("span",{class:"pill",style:`color:${c};border-color:${c}`},b.risk):null),h("p",null,b.blurb),h("p",{style:"color:#454a58"},b.summary||"No run yet."),h("p",null,b.ts?`last run ${b.ts.slice(11,19)} · ${b.ms} ms`:"")),b.can_run?h("button",{class:"btn primary",onclick:async e=>{e.target.disabled=true;const r=await api("/api/run_agent",{json:{agent:b.agent}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage()}},"Run now"):null)};up(b);return{el,update:up}};
R.brief=b=>{const el=h("div",{class:"card"});const out=h("div",{class:"md"});const up=b=>{out.dataset.def=b.text;if(!out.dataset.ai)out.innerHTML=md(b.text)};el.append(h("h3",null,"Executive briefing"),out,b.can_brief?h("button",{class:"btn sm",style:"margin-top:8px",onclick:async e=>{e.target.disabled=true;out.dataset.ai="1";out.textContent="";let acc="";await sse("/api/brief",{lang:S.lang},ev=>{if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}if(ev.ev==="final")out.innerHTML=md(ev.text)});e.target.disabled=false}},"Write with the GPU model"):null);up(b);return{el,update:up}};
R.button=b=>{const el=h("button",{class:"btn "+(b.tone||"primary"),onclick:async()=>{if(b.payload&&b.payload.download){window.open(b.url,"_blank");return}if(b.confirm&&!confirm(b.confirm))return;el.disabled=true;const r=await api(b.url,{json:b.payload||{}});toast(r.msg||"Done",!r.ok);el.disabled=false;if(r.refresh)reloadPage()}},b.label);return{el,update(){}}};
R.table=b=>{const el=h("div"),title=h("h3",{style:"margin:0 0 6px;font-size:13.5px"}),sub=h("div",{class:"sub"}),srch=h("input",{class:"srch",placeholder:"Search…",oninput:()=>{st.q=srch.value.toLowerCase();draw()}}),box=h("div",{class:"tbl"}),st={sort:null,dir:1,q:""};let cur=b;el.append(title,sub,srch,box);
 const draw=()=>{const b=cur;title.textContent=b.title||"";title.style.display=b.title?"":"none";sub.textContent=b.sub||"";sub.style.display=b.sub?"":"none";srch.style.display=b.search?"":"none";if(b.h)box.style.maxHeight=b.h+"px";
  let rows=b.rows;if(st.q)rows=rows.filter(r=>Object.values(r).some(v=>String(v).toLowerCase().includes(st.q)));if(st.sort)rows=[...rows].sort((x,y)=>{const a=x[st.sort],c=y[st.sort];return(a==null)-(c==null)||(a>c?1:a<c?-1:0)*st.dir});
  const mx={};b.cols.forEach(c=>{if(c.type==="pct"){mx[c.k]=Math.max(...b.rows.map(r=>Math.abs(r[c.k]||0)))<=1.0001?100:1}});
  const th=h("tr",null,...b.cols.map(c=>h("th",{onclick:()=>{st.dir=st.sort===c.k?-st.dir:1;st.sort=c.k;draw()}},c.label+(st.sort===c.k?(st.dir>0?" ▲":" ▼"):""))));
  const tb=rows.slice(0,400).map(r=>{const tr=h("tr",{class:b.open?"cl":""},...b.cols.map(c=>{const v=r[c.k];const t=c.type||"text";let n;
    if(t==="pill"&&v!=null&&v!=="")n=h("span",{class:"pill",style:`color:${tone(v)};border-color:${tone(v)}55;background:${tone(v)}18`},v);
    else if(t==="bar"){const m=c.max||100,p=Math.max(0,Math.min(100,(v||0)/m*100)),col=c.k.match(/risk|prob|congest|ci|index/i)?(p>66?"#e5484d":p>40?"#f5a524":"#12a594"):(p<60?"#e5484d":p<80?"#f5a524":"#12a594");n=h("div",{class:"bar"},h("span",null,v==null?"–":fmt.num(v)),h("i",null,h("b",{style:`width:${p}%;background:${col}`})))}
    else if(t==="pct")n=document.createTextNode(v==null?"–":(+(v*mx[c.k]).toFixed(1))+"%");else if(t==="money")n=document.createTextNode(fmt.money(v));else if(t==="int")n=document.createTextNode(fmt.int(v));else if(t==="num")n=document.createTextNode(fmt.num(v));else n=document.createTextNode(v==null?"":v);
    return h("td",{title:typeof v==="string"&&v.length>40?v:null},n)}));if(b.open)tr.onclick=()=>UI.openEntity(r[b.open.col]);return tr});
  const sc=box.scrollTop;box.replaceChildren(h("table",null,h("thead",null,th),h("tbody",null,...tb)));box.scrollTop=sc;if(!rows.length)box.append(h("div",{class:"sub",style:"padding:12px"},"No rows."))};
 draw();return{el,update:nb=>{cur=nb;draw()}}};
R.tickets=b=>{const el=h("div"),st={f:"all",open:new Set()};let cur=b;const body=h("div"),fl=h("div",{class:"filters"});el.append(h("h3",{style:"margin:0 0 6px;font-size:13.5px"},b.title||"Tickets"),h("div",{class:"sub"},b.sub||""),fl,body);
 const act=async(id,a,json)=>{const r=await api(`/api/ticket/${id}/${a}`,{json:json||{}});toast(r.msg,!r.ok);if(document.activeElement)document.activeElement.blur();st.sig=null;reloadPage(true)};
 const draw=()=>{const items=cur.items;st.sig=JSON.stringify(items)+st.f+[...st.open];const cnt=k=>k==="all"?items.length:k==="proposed"?items.filter(i=>i.status==="proposed").length:k==="overdue"?items.filter(i=>i.overdue).length:items.filter(i=>i.sev===k).length;
  fl.replaceChildren(...["all","proposed","Critical","Warning","Info","overdue"].map(k=>h("span",{class:"chip"+(st.f===k?" on":""),onclick:()=>{st.f=k;draw()}},(k==="proposed"?"awaiting approval":k)+" · "+cnt(k))));
  const list=items.filter(i=>st.f==="all"||(st.f==="proposed"?i.status==="proposed":st.f==="overdue"?i.overdue:i.sev===st.f));
  body.replaceChildren(...list.slice(0,80).map(i=>{const c=tone(i.sev);const t=h("div",{class:"tk "+i.sev+" "+i.status},
   h("div",{class:"h"},h("b",null,"#"+i.id+" "+i.type.replace(/_/g," ")),h("span",{class:"pill",style:`color:${c};border-color:${c}66`},i.sev),i.status==="proposed"?h("span",{class:"pill",style:"color:#8e4ec6;border-color:#8e4ec666"},"awaiting approval"):null,i.overdue?h("span",{class:"pill",style:"color:#e5484d;border-color:#e5484d66"},"overdue"):null,h("a",{href:"#",onclick:e=>{e.preventDefault();UI.openEntity(i.target)}},i.target)),
   h("div",{class:"d"},i.detail),h("div",{class:"m"},h("span",null,"agent: "+i.agent),h("span",null,"owner: "+(i.owner||"–")),h("span",null,"assignee: "+(i.assignee||"unassigned")),h("span",null,"due "+(i.due||"–")),h("span",null,i.ts)),
   i.chain&&i.chain.length?h("div",{class:"m"},h("span",null,"approval chain: "),...i.chain.map(s=>h("span",{class:"pill",style:`color:${s.status==="approved"?"#1a9e6e":s.status==="rejected"?"#e5484d":"#8e4ec6"};border-color:currentColor`},"L"+s.step+" "+s.role+(s.status==="approved"?" ✓ "+s.by:s.status==="rejected"?" ✗":" …")))):null,
   i.status==="proposed"&&i.turn&&!i.can_approve?h("div",{class:"m"},"Waiting for a "+i.turn+" to decide"):null,
   i.note?h("div",{class:"m"},"note: "+i.note):null,
   h("div",{class:"a"},i.status==="proposed"&&i.can_approve?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve & execute"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.quote&&i.can_approve&&i.status!=="proposed"?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve quote"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.can_resolve?[h("button",{class:"btn sm",onclick:()=>act(i.id,"resolve")},"Resolve"),(()=>{const a=h("input",{class:"in",style:"width:170px",placeholder:"assign to email",value:i.assignee||""});return h("span",{style:"display:flex;gap:4px"},a,h("button",{class:"btn sm",onclick:()=>a.value&&act(i.id,"assign",{assignee:a.value})},"Assign"))})()]:null,
    h("button",{class:"btn sm",onclick:()=>{st.open.has(i.id)?st.open.delete(i.id):st.open.add(i.id);draw()}},"Comments ("+i.comments.length+")")),
   st.open.has(i.id)?h("div",{class:"cm"},...i.comments.map(c=>h("div",null,h("b",null,c.by),` ${c.ts}: ${c.body}`)),(()=>{const t=h("input",{class:"in",placeholder:"Write a comment and press Enter",onkeydown:e=>{if(e.key==="Enter"&&t.value.trim())act(i.id,"comment",{body:t.value})}});return t})()):null);return t}));if(!list.length)body.append(h("div",{class:"sub"},"Nothing here."))};
 draw();return{el,update:nb=>{if(el.contains(document.activeElement)&&document.activeElement.tagName==="INPUT"){cur=nb;return}if(JSON.stringify(nb.items)+st.f+[...st.open]===st.sig)return;cur=nb;draw()}}};
// ---- charts
const FN={size3:v=>Math.max(8,Math.min(34,Math.sqrt(Math.abs(v[3]||10))*1.6)),scatterTip:p=>`<b>${esc(p.value[2]||"")}</b><br>${esc(p.seriesName||"")}<br>${p.value[0]}, ${p.value[1]}`,nodeTip:p=>p.dataType==="node"?`<b>${esc(p.name)}</b><br>${esc(p.data.tip||"")}`:"",nodeLabel:p=>p.data.showLabel?p.name:""};
function revive(o){if(Array.isArray(o))return o.map(revive);if(o&&typeof o==="object"){if(o.__fn)return FN[o.__fn];const r={};for(const k in o)r[k]=revive(o[k]);return r}return o}
echarts.registerTheme("lt",{color:["#5b5bd6","#12a594","#f5a524","#8e4ec6","#e5484d","#30a46c","#d6409f","#3e63dd","#8b8d98"],backgroundColor:"transparent",textStyle:{fontFamily:"Inter, system-ui, sans-serif",color:"#454a58"},legend:{textStyle:{color:"#454a58"}},categoryAxis:{axisLine:{lineStyle:{color:"#d9dce8"}},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},valueAxis:{axisLine:{show:false},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},tooltip:{backgroundColor:"#fff",borderColor:"#e6e8f0",textStyle:{color:"#1c2024"},extraCssText:"box-shadow:0 8px 24px rgba(20,24,40,.12);border-radius:10px"}});
const CH=[];function lazy(box,fn){let d=0;const t=()=>{if(d)return;if(box.isConnected&&box.clientWidth>0){d=1;fn()}else setTimeout(t,60)};t()}
R.chart=b=>{const el=h("div"),box=h("div",{style:`height:${b.h||290}px`});el.append(h("h3",{style:"margin:0 0 4px;font-size:13.5px;display:"+(b.title?"block":"none")},b.title||""),box);let ch=null,cur=b;
 const init=()=>{ch=echarts.init(box,"lt",{renderer:"canvas"});CH.push(ch);ch.setOption(revive(cur.opt),true);
  ch.on("click",p=>{const o=cur.open;if(!o)return;let id=null;if(o.bar)id=p.name;else if(o.scatter)id=p.value&&p.value[2];else if(o.graph)id=p.data&&p.data.open;else if(p.data&&p.data.open)id=p.data.open;if(id)UI.openEntity(id)});
  if(cur.open)box.style.cursor="pointer"};
 lazy(box,init);
 return{el,update:nb=>{cur=nb;if(ch){const r=JSON.stringify(nb.opt);if(box._h!==r){box._h=r;ch.setOption(revive(nb.opt),{replaceMerge:["series","xAxis","yAxis"]})}}},};};
window.addEventListener("resize",()=>CH.forEach(c=>{try{c.resize()}catch(e){}}));
// ---- map
function shiftGeo(g,sh){const mv=c=>typeof c[0]==="number"?[c[0]+sh,c[1]]:c.map(mv);return{type:"FeatureCollection",features:g.features.map(f=>({type:"Feature",properties:f.properties,geometry:{type:f.geometry.type,coordinates:mv(f.geometry.coordinates)}}))}}
let WORLD=null;async function world(){if(WORLD)return WORLD;try{const t=await(await fetch("/static/vendor/world.topo.json")).json();WORLD=topojson.feature(t,t.objects.countries)}catch(e){WORLD={type:"FeatureCollection",features:[]}}return WORLD}
R.map=b=>{const el=h("div"),box=h("div",{class:"map",style:`height:${b.h||470}px`}),lg=h("div",{class:"lg"});el.append(box,lg);let M=null,grp=null,cur=b;
 const draw=()=>{grp.clearLayers();const SH=[-360,0,360];cur.layers.forEach(L=>{if(L.type==="lines")L.lines.forEach(l=>{if(l.pts.length<2)return;SH.forEach(sh=>{const p=L2.polyline(l.pts.map(q=>[q[0],q[1]+sh]),{color:l.color||"#5b5bd6",weight:l.w||2,dashArray:l.dash||null,opacity:.9,lineJoin:"round",renderer:M._r});if(l.tip)p.bindTooltip(l.tip,{sticky:true});grp.addLayer(p)})});
   else if(L.type==="circles")L.items.forEach(c=>SH.forEach(sh=>{const k=L2.circle([c.lat,c.lon+sh],{radius:c.r_m,color:c.color||"#e5484d",weight:1,fillOpacity:.18,renderer:M._r});if(c.tip)k.bindTooltip(c.tip);grp.addLayer(k)}));
   else if(L.type==="points")[...L.pts].sort((a,c)=>(a.z||0)-(c.z||0)).forEach(p=>SH.forEach(sh=>{const m=L2.circleMarker([p.lat,p.lon+sh],{radius:p.r||5,color:"#ffffff",weight:1.2,fillColor:p.color||"#5b5bd6",fillOpacity:.95,renderer:M._r});if(p.tip)m.bindTooltip(p.tip,{sticky:true});const o=p.open||(cur.open?p.id:null)||p.id;if(o)m.on("click",()=>UI.openEntity(o));grp.addLayer(m)}))});
  lg.replaceChildren(...(cur.legend||[]).map(([t,c])=>h("span",null,h("i",{style:"background:"+c}),t)))};
 const L2=L;const go=async()=>{M=L2.map(box,{zoomControl:true,worldCopyJump:true,attributionControl:false,minZoom:1,preferCanvas:true});M._r=L2.canvas({padding:.5});const w=await world();[-360,0,360].forEach(sh=>L2.geoJSON(sh?shiftGeo(w,sh):w,{renderer:M._r,style:{color:"#b9b2a0",weight:.8,fillColor:"#f3efe6",fillOpacity:1},interactive:false}).addTo(M));grp=L2.layerGroup().addTo(M);
  const v=cur.view||{};if(v.bounds)M.fitBounds(v.bounds);else M.setView(v.center||[20,0],v.zoom||2);draw();setTimeout(()=>M.invalidateSize(),50)};
 lazy(box,go);
 return{el,update:nb=>{cur=nb;if(M&&grp)draw()}}};
// ---- forms
R.form=b=>{const el=h("div",{class:b.title||b.inline?"":""});const res=h("div",{class:"res"});const f=h("form",{class:"f"+(b.inline?" inline":"")});let ins={};
 b.fields.forEach(fl=>{const w=h("div",{class:"fld"+(fl.type==="textarea"||fl.type==="checks"||fl.type==="multi"?" wide":"")});if(fl.type!=="hidden")w.append(h("label",null,fl.label||fl.name));let i;
  if(fl.type==="select")i=h("select",{name:fl.name},...fl.options.map(o=>h("option",{value:o,selected:String(o)===String(fl.value)},o)));
  else if(fl.type==="multi"){i=h("select",{name:fl.name,multiple:true,size:5},...fl.options.map(o=>h("option",{value:o,selected:(fl.value||[]).includes(o)},o)))}
  else if(fl.type==="checks"){i=h("div",{class:"chk"},...fl.options.map(o=>h("label",null,h("input",{type:"checkbox",value:o,checked:(fl.value||[]).includes(o)}),o)));i.get=()=>[...i.querySelectorAll("input:checked")].map(x=>x.value)}
  else if(fl.type==="textarea")i=h("textarea",{name:fl.name,placeholder:fl.placeholder||""},fl.value||"");
  else if(fl.type==="range"){const v=h("span",{style:"color:#12a594"},fl.value);i=h("input",{type:"range",name:fl.name,min:fl.min,max:fl.max,step:fl.step||1,value:fl.value,oninput:e=>v.textContent=e.target.value});w.firstChild.append(" ",v)}
  else if(fl.type==="file")i=h("input",{type:"file",name:fl.name,accept:fl.accept||null});
  else if(fl.type==="hidden"){i=h("input",{type:"hidden",name:fl.name,value:fl.value});w.style.display="none"}
  else i=h("input",{type:fl.type==="number"?"number":fl.type==="date"?"date":fl.type==="password"?"password":"text",name:fl.name,value:fl.value==null?"":fl.value,min:fl.min,max:fl.max,step:fl.step||(fl.type==="number"?"any":null),placeholder:fl.placeholder||""});
  i.fl=fl;ins[fl.name]=i;w.append(i);f.append(w)});
 const btn=h("button",{class:"btn "+(b.tone||"primary"),type:"submit"},b.submit||"Submit");f.append(h("div",{class:"fld"},btn));
 f.addEventListener("submit",async e=>{e.preventDefault();const v={};let fd=null;for(const n in ins){const i=ins[n],fl=i.fl;if(fl.type==="file"){if(i.files[0]){fd=fd||new FormData();fd.append(n,i.files[0])}}else if(fl.type==="checks")v[n]=i.get();else if(fl.type==="multi")v[n]=[...i.selectedOptions].map(o=>o.value);else if(fl.type==="number"||fl.type==="range")v[n]=i.value===""?null:+i.value;else v[n]=i.value}
  if(b.url.startsWith("get:")){const q=new URLSearchParams();for(const n in v)if(v[n]!==""&&v[n]!=null)q.set(n,v[n]);window.open(b.url.slice(4)+"?"+q,"_blank");return}
  if(b.url.startsWith("page:")){const[,id,tab]=b.url.split(":");UI.go(id,tab||S.tab,v);return}
  btn.disabled=true;const t0=btn.textContent;btn.textContent="Working…";let r;
  if(fd){for(const n in v)fd.append(n,Array.isArray(v[n])?v[n].join(","):v[n]);r=await api(b.url,{method:"POST",body:fd})}else r=await api(b.url,{json:v});
  btn.disabled=false;btn.textContent=t0;toast(r.msg||(r.ok?"Done":"Failed"),!r.ok);res.replaceChildren();if(r.blocks){const hs=mountList(res,r.blocks,"res");}if(r.refresh)reloadPage(true)});
 if(b.title||b.sub){const c=h("div",{class:"card"},b.title?h("h3",null,b.title):null,b.sub?h("div",{class:"sub"},b.sub):null,f,res);el.append(c)}else el.append(f,res);return{el,update(){}}};
// ---- chat
async function sse(url,json,on){const r=await fetch(url,{method:"POST",credentials:"same-origin",headers:{"Content-Type":"application/json"},body:JSON.stringify(json)});if(!r.ok){on({ev:"final",text:"Error "+r.status});return}const rd=r.body.getReader(),dec=new TextDecoder();let buf="";for(;;){const{value,done}=await rd.read();if(done)break;buf+=dec.decode(value,{stream:true});let i;while((i=buf.indexOf("\n\n"))>=0){const m=buf.slice(0,i);buf=buf.slice(i+2);if(m.startsWith("data: ")){try{on(JSON.parse(m.slice(6)))}catch(e){}}}}}
R.chat=b=>{let cid=null,hist=[],sid="s"+Math.random().toString(36).slice(2,10),qh=[],qi=-1;const el=h("div",{class:"card chat"}),msgs=h("div",{class:"msgs"});
 const inp=h("input",{placeholder:"Ask anything: the exact SQL engine answers first, then the agents, knowledge base and GPU model for open questions",autocomplete:"off",list:"cpl",onkeydown:e=>{if(e.key==="Enter"){send(inp.value)}else if(e.key==="ArrowUp"&&qh.length){qi=Math.min(qi+1,qh.length-1);inp.value=qh[qh.length-1-qi];e.preventDefault()}else if(e.key==="ArrowDown"){qi=Math.max(qi-1,-1);inp.value=qi<0?"":qh[qh.length-1-qi];e.preventDefault()}},oninput:()=>suggest()});
 const dl=h("datalist",{id:"cpl"});let st=null;
 const suggest=()=>{clearTimeout(st);st=setTimeout(async()=>{const v=inp.value.trim();if(v.length<2){dl.replaceChildren();return}const r=await api("/api/copilot/suggest?q="+encodeURIComponent(v));dl.replaceChildren(...(r.items||[]).map(x=>h("option",{value:x})))},160)};
 const lang=h("select",{class:"in",style:"width:auto",onchange:()=>{S.lang=lang.value;localStorage.setItem("lang",S.lang)}},...b.langs.map(l=>h("option",{value:l,selected:l===S.lang},l)));
 const hsel=h("select",{class:"in",style:"width:auto;max-width:260px",onchange:()=>{if(hsel.value)openChat(hsel.value)}},h("option",{value:""},"Previous chats…"));
 const listChats=async()=>{const r=await api("/api/chats");if(!r.chats)return;hsel.replaceChildren(h("option",{value:""},r.chats.length?`Previous chats (${r.chats.length})`:"No saved chats yet"),...r.chats.map(c=>h("option",{value:c.id,selected:c.id===cid},c.title+" · "+(c.ts||"").replace("T"," ").slice(5,16))));return r.chats};
 const chipRow=items=>h("div",{class:"filters",style:"margin:8px 0 0"},...items.map(e=>h("span",{class:"chip",onclick:()=>send(e)},e)));
 const tools=(m,q)=>{const row=h("div",{class:"st",style:"display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin-top:6px"});
  const cp=h("a",{class:"lnk",style:"cursor:pointer",onclick:()=>{navigator.clipboard&&navigator.clipboard.writeText(m.text);cp.textContent="Copied"}},"Copy");row.append(cp);
  if(m.csv)row.append(h("a",{class:"lnk",href:"/api/copilot/csv?sid="+encodeURIComponent(m.sid||sid),download:"copilot_result.csv"},"Download CSV"));
  const fb=v=>async()=>{await api("/api/copilot/feedback",{json:{q:q||"",helpful:v}});up.remove();dn.remove();row.append(h("span",null,v?"Thanks, noted":"Noted, logged to improve"))};
  const up=h("a",{class:"lnk",style:"cursor:pointer",title:"Helpful",onclick:fb(true)},"👍"),dn=h("a",{class:"lnk",style:"cursor:pointer",title:"Not helpful",onclick:fb(false)},"👎");row.append(up,dn);return row};
 const sqlBox=sql=>sql?h("details",{class:"cites"},h("summary",null,"Show the exact query"),h("pre",{style:"white-space:pre-wrap;font-size:11px"},sql)):null;
 const enToggle=(m,o,host)=>{if(!m.en)return;const tg=h("a",{class:"lnk",style:"cursor:pointer;font-size:12px",onclick:()=>{const on=tg._e=!tg._e;o.innerHTML=md(on?m.en:m.text);tg.textContent=on?"Show "+(m.lang||"translation"):"Show English original"}},"Show English original");host.append(h("div",{class:"st"},"Translated with NLLB-200 ",tg))};
 const showMsg=(m,i)=>{if(m.role==="user"){msgs.append(h("div",{class:"m u"},m.text));return}const o=h("div",{class:"md"});o.innerHTML=md(m.text);const a=h("div",{class:"m a"},o);enToggle(m,o,a);const sb=sqlBox(m.sql);if(sb)a.append(sb);a.append(tools(m,(hist[i-1]||{}).text));msgs.append(a)};
 const welcome=()=>{if(hist.length)return;msgs.append(h("div",{class:"m a"},h("div",{class:"md"},h("b",null,"Hi, I am your operations copilot."),h("div",{class:"st",style:"margin-top:4px"},"Exact answers come from SQL (the query is shown). Open questions use the agents, the knowledge base and the GPU model. I understand typos, follow-ups and 37 languages. Pick one to start:")),chipRow(["Briefing","What can you do?",...b.examples.slice(0,3)])))};
 const openChat=async id=>{const r=await api("/api/chats/"+encodeURIComponent(id));if(!r.msgs)return;cid=id;hist=r.msgs;msgs.replaceChildren();hist.forEach(showMsg);msgs.lastChild&&msgs.lastChild.scrollIntoView({block:"nearest"});listChats()};
 const newChat=()=>{cid=null;hist=[];msgs.replaceChildren();hsel.value="";sid="s"+Math.random().toString(36).slice(2,10);welcome();inp.focus()};
 const delChat=async()=>{if(!cid||!confirm("Delete this chat?"))return;await api("/api/chats/"+encodeURIComponent(cid),{method:"DELETE"});newChat();listChats()};
 const saveChat=async()=>{const r=await api("/api/chats/save",{json:{id:cid,msgs:hist}});if(r.id){cid=r.id;listChats()}};
 listChats().then(cs=>{if(cs&&cs.length&&!hist.length)openChat(cs[0].id);else welcome()});
 const send=async q=>{q=q.trim();if(!q)return;inp.value="";dl.replaceChildren();qh.push(q);qi=-1;hist.push({role:"user",text:q});msgs.append(h("div",{class:"m u"},q));
  const a=h("div",{class:"m a"}),stg=h("div"),tr=h("div"),out=h("div",{class:"md"}),ch=h("div"),foot=h("div");a.append(stg,tr,out,ch,foot);msgs.append(a);let acc="",base="";const sc=()=>{msgs.lastChild.scrollIntoView({block:"nearest"})};
  await sse("/api/copilot",{q,lang:S.lang,sid},ev=>{
   if(ev.ev==="stage"){stg.append(h("div",{class:"st"},ev.text))}
   else if(ev.ev==="note")stg.append(h("div",{class:"st"},"✎ "+ev.text));
   else if(ev.ev==="cites"){const d=h("details",{class:"cites"},h("summary",null,`Sources (${ev.rows.length})`),...ev.rows.map(r=>h("div",{class:"st",title:r.text},h("b",null,`[${r.n}] `),`${r.title} · ${r.source} · ${r.why||r.score}`)));stg.append(d)}
   else if(ev.ev==="trace"){tr.replaceChildren(h("div",{class:"tbl",style:"margin:6px 0"},h("table",null,h("tbody",null,...ev.rows.map(r=>h("tr",null,h("td",null,r.agent),h("td",null,h("span",{class:"pill",style:`color:${tone(r.risk)};border-color:${tone(r.risk)}55`},r.risk)),h("td",{style:"white-space:normal"},r.finding)))))))}
   else if(ev.ev==="partial"){base=ev.text;out.innerHTML=md(base)}
   else if(ev.ev==="draft"){if(!acc&&!base)out.innerHTML=md(ev.text)}
   else if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(base?base+"\n\n"+acc:acc)}
   else if(ev.ev==="layers"){const f=(ok,t,tip)=>h("span",{class:"chip",style:"cursor:default;margin-right:4px;"+(ok?"":"opacity:.55"),title:tip||""},(ok?"✓ ":"✗ ")+t);stg.append(h("div",{class:"filters",style:"margin:4px 0"},f(ev.sql,"SQL"),f(ev.agents>0,"Agents"+(ev.agents?" "+ev.agents:""),"Findings of the latest agent cycle"),f(ev.rag>0||ev.kb_hit,"RAG"+(ev.rag?" "+ev.rag:""),ev.drive?"Google Drive knowledge connected":"Google Drive not mounted: only the built-in knowledge pack"),f(ev.llm,"LLM",ev.llm?"":"LLM state: "+ev.llm_state)))}
   else if(ev.ev==="chart"){const c=R.chart(ev.block);ch.append(c.el||c)}
   else if(ev.ev==="final"){out.innerHTML=md(ev.text);const m={role:"assistant",text:ev.text,en:ev.en,lang:ev.lang,sql:ev.sql,csv:ev.csv,sid};hist.push(m);saveChat();enToggle(m,out,stg);const sb=sqlBox(ev.sql);if(sb)foot.append(sb);foot.append(h("div",{class:"st"},`${ev.rows>0?ev.rows+" rows · ":""}done in ${(ev.ms/1000).toFixed(1)}s`+(ev.tok_s?` · ${ev.tok_s} tok/s`:"")),tools(m,q));if(ev.sugg&&ev.sugg.length)foot.append(chipRow(ev.sugg))}
   sc()})};
 el.append(h("div",{style:"display:flex;gap:8px;align-items:center;flex-wrap:wrap"},h("button",{class:"btn sm",onclick:newChat},"+ New chat"),hsel,h("button",{class:"btn sm ghost",onclick:delChat,title:"Delete this chat"},"Delete"),h("b",{style:"margin-left:8px"},"Answer language"),lang,h("span",{class:"sub",style:"margin:0"},"NLLB-200 translates in and out · ↑ recalls your last questions")),msgs,h("div",{class:"cin"},inp,dl,(()=>{const SR=window.SpeechRecognition||window.webkitSpeechRecognition;if(!SR)return null;const VL={Hindi:"hi-IN",Tamil:"ta-IN",Telugu:"te-IN",Kannada:"kn-IN",Malayalam:"ml-IN",Bengali:"bn-IN",Gujarati:"gu-IN",Marathi:"mr-IN",Punjabi:"pa-IN",Urdu:"ur-IN",Arabic:"ar-SA",French:"fr-FR",German:"de-DE",Spanish:"es-ES",Portuguese:"pt-PT",Chinese:"zh-CN",Japanese:"ja-JP",Korean:"ko-KR",Russian:"ru-RU",Indonesian:"id-ID",Turkish:"tr-TR",Italian:"it-IT",Dutch:"nl-NL",Thai:"th-TH",Vietnamese:"vi-VN"};let rec=null;const mic=h("button",{class:"btn",title:"Speak your question (browser speech recognition)",onclick:()=>{if(rec){rec.stop();return}rec=new SR();const k=Object.keys(VL).find(x=>S.lang.startsWith(x));rec.lang=k?VL[k]:"en-IN";rec.interimResults=true;rec.onresult=e=>{inp.value=[...e.results].map(r=>r[0].transcript).join(" ")};rec.onend=()=>{rec=null;mic.textContent="🎤";if(inp.value.trim())send(inp.value)};mic.textContent="⏺";try{rec.start()}catch(e){rec=null;mic.textContent="🎤"}}},"🎤");return mic})(),h("button",{class:"btn primary",onclick:()=>send(inp.value)},"Ask")));return{el,update(){}}};


// ---------------------------------------------------------------- shell
const UI={};
UI.logout=async()=>{try{await Promise.race([api("/api/logout",{json:{}}),new Promise(r=>setTimeout(r,2500))])}catch(e){}try{localStorage.clear();sessionStorage.clear()}catch(e){}S.me=null;location.replace(location.pathname)};
UI.openEntity=async id=>{if(!id)return;$("#dt").textContent=id;$("#db").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"));$("#drawer").classList.add("on");const d=await api("/api/detail/"+encodeURIComponent(id));if(d.title===undefined){$("#db").replaceChildren(h("div",{class:"callout warn"},d.msg||"Not available"));return}$("#dt").textContent=d.title||id;$("#db").replaceChildren();mountList($("#db"),d.blocks||[],"d");translateUI($("#db"))};
UI.closeDrawer=()=>$("#drawer").classList.remove("on");
UI.go=(id,tab,params)=>{S.page=id;S.tab=tab||null;S.params=params||{};history.replaceState(null,"","#"+id+(S.tab?":"+S.tab:""));loadPage(false);renderNav()};
async function loadPage(silent){const seq=++S.seq;clearTimeout(S.timer);const q=new URLSearchParams();if(S.tab)q.set("tab",S.tab);if(Object.keys(S.params).length)q.set("p",JSON.stringify(S.params));
 if(!silent){S.tree=null;$("#blocks").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"))}
 const t=performance.now();const d=await api(`/api/page/${S.page}?`+q);if(seq!==S.seq)return;
 if(d.blocks===undefined){$("#blocks").replaceChildren(h("div",{class:"callout error"},d.msg||"Could not load"));return}
 S.ms=Math.round(performance.now()-t);$("#ttl").textContent=d.title;$("#sub").textContent=d.sub||"";document.title=d.title+" · "+S.me.name;
 const tabs=$("#tabs");tabs.replaceChildren(...(d.tabs||[]).map(x=>h("b",{class:(S.tab||d.tabs[0].id)===x.id?"on":"",onclick:()=>{S.params={};UI.go(S.page,x.id)}},x.label)));tabs.style.display=d.tabs?"":"none";
 $("#acts").replaceChildren(...(d.actions||[]).map(a=>h("button",{class:"btn "+(a.tone||"primary"),onclick:async e=>{e.target.disabled=true;const r=await api(a.url,{json:{}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage(true)}},a.label)));
 const box=$("#blocks");const focus=document.activeElement&&box.contains(document.activeElement)&&["INPUT","TEXTAREA","SELECT"].includes(document.activeElement.tagName);
 if(!S.tree||silent===false||!S.tree.ok){box.replaceChildren();S.tree={hs:mountList(box,d.blocks,"p"),ok:1,id:S.page,tab:S.tab};translateUI(box)}else S.tree.hs=updList(box,S.tree.hs,d.blocks,"p");
 S.live=d.live||0;sched()}
function sched(){clearTimeout(S.timer);if(S.live&&S.lang==="English"||S.live&&true)S.timer=setTimeout(()=>{if(!document.hidden)loadPage(true);else sched()},S.live*1000)}
function reloadPage(soft){loadPage(!!soft)}
async function translateUI(root){if(S.lang==="English")return;const nodes=[];const w=document.createTreeWalker(root,NodeFilter.SHOW_TEXT);let n;while((n=w.nextNode())){const t=n.nodeValue.trim();if(t.length>2&&/[A-Za-z]{3}/.test(t)&&!n.parentElement.closest("script,style,pre,code,.m.u,.mono,.noTr,input,select,option")&&!n._tr&&!(n.parentElement.closest("td")&&t.split(/\s+/).length<4)&&!/^[A-Z]{2,6}-?[A-Z]?\d+$/.test(t))nodes.push(n)}
 const list=nodes.slice(0,300);if(!list.length)return;const r=await api("/api/translate_batch",{json:{texts:list.map(n=>n.nodeValue.trim()),lang:S.lang}});if(r.out)list.forEach((n,i)=>{if(r.out[i]){n.nodeValue=r.out[i];n._tr=1}})}
const GROUPS=[["Assistant",["copilot","approvals"]],["Agents",["workforce","outlets","inventory","marketing","sentiment","audit","route","quotes","carriers","weather","margin","customs"]],["Insights",["twin","reports","notifications"]],["Platform",["feeds","admin","profile"]]];
function renderNav(){const nav=$("#nav");if(!nav)return;const items=[];const used=new Set();const mk=m=>h("div",{class:"nav"+(m.id===S.page?" on":"")+(m.id==="command"?" cmd":""),"data-id":m.id,onclick:()=>UI.go(m.id)},icon(m.icon),h("span",null,m.label.replace(/^Agent \d+ · /,"")),h("em",{class:"bd",style:"display:none"}));
 GROUPS.forEach(([g,ids])=>{const ms=S.me.menu.filter(m=>ids.includes(m.id));if(!ms.length)return;items.push(h("div",{class:"ngrp"},g));ms.forEach(m=>{used.add(m.id);items.push(mk(m))})});
 S.me.menu.filter(m=>!used.has(m.id)&&m.id!=="command").forEach(m=>items.push(mk(m)));const c=S.me.menu.find(m=>m.id==="command");if(c)items.push(mk(c));
 nav.replaceChildren(...items);updBadges()}
// ---- command palette (Ctrl/Cmd+K): pages, entities, ask the copilot
UI.palette=()=>{if($(".kpal"))return;let sel=0,rows=[];const inp=h("input",{placeholder:"Jump to a page, open an entity (e.g. OUT-007, QTE-0012, SHP-…), or ask the Copilot…"}),lst=h("div",{class:"lst"}),ov=h("div",{class:"kpal",onclick:e=>{if(e.target===ov)close()}},h("div",null,inp,lst));
 const close=()=>{ov.remove();document.removeEventListener("keydown",kd,true)};
 const build=()=>{const q=inp.value.trim(),ql=q.toLowerCase();rows=[];S.me.menu.filter(m=>!ql||m.label.toLowerCase().includes(ql)||m.id.includes(ql)).forEach(m=>rows.push({g:"Pages",t:m.label.replace(/^Agent \d+ · /,""),s:"go to",f:()=>UI.go(m.id)}));
  if(/^[A-Za-z]{2,5}-?[A-Za-z]?\d+$/.test(q))rows.unshift({g:"Open",t:q.toUpperCase(),s:"entity",f:()=>UI.openEntity(q.toUpperCase())});
  if(q.length>2)rows.push({g:"Ask",t:"Ask the Copilot: "+q,s:"agents + RAG",f:()=>{UI.go("copilot");setTimeout(()=>{const i=document.querySelector(".cin input");if(i){i.value=q;i.dispatchEvent(new KeyboardEvent("keydown",{key:"Enter"}))}},700)}});
  rows.push({g:"Account",t:"Sign out",s:"",f:()=>UI.logout()});if(sel>=rows.length)sel=rows.length-1;
  let last="";lst.replaceChildren(...rows.flatMap((r,i)=>{const e=[];if(r.g!==last){last=r.g;e.push(h("div",{class:"gl"},r.g))}e.push(h("div",{class:"it"+(i===sel?" on":""),onclick:()=>{close();r.f()}},r.t,h("small",null,r.s)));return e}))};
 const kd=e=>{if(e.key==="Escape"){close()}else if(e.key==="ArrowDown"){sel=Math.min(rows.length-1,sel+1);build();e.preventDefault()}else if(e.key==="ArrowUp"){sel=Math.max(0,sel-1);build();e.preventDefault()}else if(e.key==="Enter"&&document.activeElement===inp){const r=rows[sel];if(r){close();r.f()}}};
 document.addEventListener("keydown",kd,true);inp.addEventListener("input",()=>{sel=0;build()});document.body.append(ov);build();inp.focus()};
document.addEventListener("keydown",e=>{if((e.ctrlKey||e.metaKey)&&e.key.toLowerCase()==="k"&&S.me){e.preventDefault();UI.palette()}});
function updBadges(){const p=S.pulse;if(!p)return;const cc=$('.nav[data-id="command"] .bd');if(cc){const n=p.critical;cc.style.display=n?"":"none";cc.textContent=n;cc.className="bd"}const ap=$('.nav[data-id="approvals"] .bd');if(ap){const n=p.proposed;ap.style.display=n?"":"none";ap.textContent=n;ap.className="bd a"}
 const g=p.gpu||{};$("#gpu").innerHTML=`<span class="dot ${g.cuda?"g":"r"}"></span>${g.cuda?esc(g.name||"GPU")+" · "+g.used+"/"+g.total+" GB":"NO GPU"} · LLM ${g.llm==="downloading"?"downloading "+(g.llm_done||0)+"/"+(g.llm_total||"?")+" GB":g.llm} · NLLB ${g.nllb}${g.tok_s?" · "+g.tok_s+" tok/s":""}`;
 $("#hl").innerHTML=`<span class="dot g live"></span>live · health ${p.health==null?"–":Math.round(p.health)} · open ${p.open} · crit ${p.critical} · cycle ${p.cycles}`;$("#warn").style.display=g.cuda?"none":"";if(!p.worker_alive)$("#hl").innerHTML+=' · <b style="color:#e5484d">worker down</b>'}
function openStream(){if(S.es)S.es.close();const es=new EventSource("/api/stream");S.es=es;es.onmessage=e=>{const p=JSON.parse(e.data);if(p.error)return;const old=S.pulse;S.pulse=p;updBadges();if(old&&p.dec!==old.dec&&S.live)loadPage(true)};es.onerror=()=>{}}
// ---------------------------------------------------------------- auth screens
async function render(){const app=$("#app");if(S.es){S.es.close();S.es=null}
 const pub=await api("/api/public");S.pub=pub;
 if(!S.me){const m=await api("/api/me");if(m.user)S.me=m}
 if(!S.me)return authView(pub);
 app.replaceChildren(h("div",{class:"side"},h("div",{class:"brand"},h("i",null,icon("radar")),h("div",null,S.me.name,h("small",null,S.me.user.role+" · "+S.me.user.email))),h("div",{class:"ksearch",onclick:()=>UI.palette()},"Search or ask…",h("kbd",null,"Ctrl K")),h("div",{id:"nav"}),h("div",{class:"sp"}),h("div",{class:"foot sub",style:"padding:8px"},h("button",{class:"btn sm",onclick:()=>UI.logout()},"Sign out"))),
  h("div",{class:"main"},h("div",{id:"warn",class:"warnbar",style:"display:none"},"No CUDA GPU detected: models run on CPU and will be slow. In Colab choose Runtime → Change runtime type → T4 GPU, then restart."),
   h("div",{class:"top"},h("div",null,h("h1",{id:"ttl"}),h("div",{class:"sub",id:"sub"})),h("div",{class:"sp"}),h("div",{id:"acts",style:"display:flex;gap:8px"}),h("span",{class:"chip",id:"hl"},"connecting…"),h("span",{class:"chip",id:"gpu"}),h("span",{class:"chip",id:"ist",title:"India Standard Time"}),(()=>{const s=h("select",{class:"in",style:"width:auto;padding:4px 8px",onchange:()=>{S.lang=s.value;localStorage.setItem("lang",S.lang);loadPage(false)}},...S.me.langs.map(l=>h("option",{value:l,selected:l===S.lang},l)));return s})()),
   h("div",{class:"content"},h("div",{class:"tabs",id:"tabs"}),h("div",{id:"pg"},h("div",{class:"blocks",id:"blocks"})))));

 renderNav();const hs=(location.hash||"").slice(1).split(":");const ok=S.me.menu.find(m=>m.id===hs[0]);S.page=ok?hs[0]:S.me.menu[0].id;S.tab=ok&&hs[1]||null;S.params={};
 loadPage(false);openStream();api("/api/pulse").then(p=>{if(!p.error){S.pulse=p;updBadges()}})}
function authView(pub){const app=$("#app");let mode="login";const box=h("div",{class:"card auth"});app.replaceChildren(h("div",{class:"login"},h("div",{class:"lp"},h("h2",null,pub.name),h("p",null,"Agents watch your data, raise tickets, and wait for a human to approve anything that matters."),h("ul",null,h("li",null,"Live agents, simulators and a digital twin"),h("li",null,"Hybrid RAG with citations over your policies and live data"),h("li",null,"37 languages, translated on the GPU with NLLB-200"),h("li",null,"Role-based access and a full audit trail"))),box));
 const draw=()=>{const err=h("div",{class:"sub",style:"color:#e5484d;min-height:18px"});const F={};const fld=(k,l,t="text",o)=>{const w=h("div",{class:"fld"},h("label",null,l));F[k]=o?h("select",null,...o.map(x=>h("option",{value:x},x))):h("input",{type:t,autocomplete:"off"});w.append(F[k]);return w};
  let fields,btn,go;
  if(mode==="login"){fields=[fld("login","Email or username"),fld("pw","Password","password")];btn="Sign in";go=async()=>{const r=await api("/api/login",{json:{login:F.login.value,password:F.pw.value}});if(!r.ok){err.textContent=r.msg;return}S.me=null;render()}}
  else if(mode==="register"){fields=[fld("u","Username"),fld("e","Email","email"),fld("pw","Password","password"),fld("role","Role",0,["Customer","Staff"]),fld("sq","Security question",0,pub.questions),fld("sa","Answer")];btn="Create account";if(pub.email_enabled){fields.push(fld("otp","6-digit code from your e-mail"),h("button",{class:"btn sm",type:"button",onclick:async()=>{const r=await api("/api/otp",{json:{email:F.e.value,purpose:"register"}});err.textContent=r.msg;err.style.color=r.ok?"#12a594":"#e5484d"}},"E-mail me a code"))}go=async()=>{const r=await api("/api/register",{json:{username:F.u.value,email:F.e.value,password:F.pw.value,role:F.role.value,sq:F.sq.value,sa:F.sa.value,otp:F.otp?F.otp.value:""}});err.textContent=r.msg;err.style.color=r.ok?"#12a594":"#e5484d";if(r.ok)setTimeout(()=>{mode="login";draw()},900)}}
  else{fields=[fld("e","Email"),fld("sa","Answer to your security question"),fld("pw","New password","password")];const q=h("div",{class:"sub"});F.e.addEventListener("change",async()=>{q.textContent=(await api("/api/question?email="+encodeURIComponent(F.e.value))).q||""});fields.splice(1,0,q);if(pub.email_enabled){fields.push(fld("otp","or the 6-digit code from your e-mail"),h("button",{class:"btn sm",type:"button",onclick:async()=>{const r=await api("/api/otp",{json:{email:F.e.value,purpose:"reset"}});err.textContent=r.msg;err.style.color=r.ok?"#12a594":"#e5484d"}},"E-mail me a code"))}btn="Reset password";go=async()=>{const r=await api("/api/reset",{json:{email:F.e.value,answer:F.sa.value,password:F.pw.value,otp:F.otp?F.otp.value:""}});err.textContent=r.msg;err.style.color=r.ok?"#12a594":"#e5484d"}}
  const stage=pub.stage!=="ready"?h("div",{class:"callout info"},h("div",{class:"spin",style:"margin:0;width:18px;height:18px"}),"Starting up: "+pub.stage+" (you can sign in already)"):null;
  box.replaceChildren(h("h1",null,pub.name),h("div",{class:"sub"},"Agentic operations platform · every action is audited"),stage,...fields,err,h("div",{style:"display:flex;gap:8px;align-items:center"},h("button",{class:"btn primary",onclick:go},btn),h("span",{class:"sp"}),mode!=="login"?h("a",{href:"#",onclick:e=>{e.preventDefault();mode="login";draw()}},"Back to sign in"):[h("a",{href:"#",onclick:e=>{e.preventDefault();mode="register";draw()}},"Register"),h("span",null," · "),h("a",{href:"#",onclick:e=>{e.preventDefault();mode="reset";draw()}},"Forgot password")]),
   mode==="login"&&pub.demo.length?h("div",{class:"demo"},h("span",{class:"sub",style:"width:100%"},"Demo accounts (click to fill)"),...pub.demo.map(d=>h("span",{class:"chip",onclick:()=>{F.login.value=d.email;F.pw.value=d.pw}},d.role))):null);box.querySelectorAll("input").forEach(i=>i.addEventListener("keydown",e=>{if(e.key==="Enter")go()}))};draw()}
window.UI=UI;render();

;(()=>{const f=new Intl.DateTimeFormat("en-IN",{timeZone:"Asia/Kolkata",weekday:"short",day:"2-digit",month:"short",hour:"2-digit",minute:"2-digit",hour12:true});const t=()=>{const e=document.getElementById("ist");if(e)e.textContent="🕐 "+f.format(new Date())+" IST"};t();setInterval(t,1000)})();


In [ ]:
%%writefile /content/app/mc.py
"""Monte Carlo panel shared by every simulator and the Digital Twin: distribution, risk numbers, exceedance curve, fan chart, tornado and a two-factor heat map.
Everything is plain ECharts options so it needs no extra front-end code."""
import numpy as np
import blocks as B
from blocks import TEAL, BLUE, AMBER, RED, VIOLET, GREY, L, clean
GRID = {"left": 8, "right": 18, "top": 30, "bottom": 6, "containLabel": True}
SPLIT = {"lineStyle": {"color": "#eceef6"}}
def fmt(x, unit="money"):
    x = float(x)
    if unit == "money":
        s = "-" if x < 0 else ""; x = abs(x); return s + ("₹%.2f Cr" % (x / 1e7) if x >= 1e7 else "₹%.2f L" % (x / 1e5) if x >= 1e5 else "₹{:,.0f}".format(x))
    if unit == "pct": return f"{x:.1%}"
    return f"{x:,.1f}" if abs(x) < 1000 else f"{x:,.0f}"
def stats(v, bad="high"):
    v = np.asarray(v, float); v = v[np.isfinite(v)]; p = lambda q: float(np.percentile(v, q))
    tail = v[v >= p(95)] if bad == "high" else v[v <= p(5)]
    return {"mean": float(v.mean()), "sd": float(v.std()), "p5": p(5), "p10": p(10), "p50": p(50), "p90": p(90), "p95": p(95), "p5lo": p(5), "var95": p(95) if bad == "high" else p(5), "cvar95": float(tail.mean()) if len(tail) else p(95)}
def density(v, bins=36):
    v = np.asarray(v, float); h, e = np.histogram(v, bins=bins); c = (e[:-1] + e[1:]) / 2; return c, h / max(h.sum(), 1)
def hist_opt(v, st, unit, color=BLUE, base=None):
    c, p = density(v); idx = lambda x: int(np.argmin(np.abs(c - x)))
    marks = [{"xAxis": idx(st["p10"]), "name": "P10"}, {"xAxis": idx(st["p50"]), "name": "P50"}, {"xAxis": idx(st["p90"]), "name": "P90"}]
    if base is not None: marks.append({"xAxis": idx(base), "name": "today"})
    ml = {"symbol": "none", "label": {"formatter": "{b}", "fontSize": 10}, "lineStyle": {"type": "dashed", "color": GREY}, "data": marks}
    o = B._base(grid=GRID); o.update({"tooltip": {"trigger": "axis", "confine": True}, "xAxis": {"type": "category", "data": [fmt(x, unit) for x in c], "axisLabel": {"hideOverlap": True, "fontSize": 10}}, "yAxis": {"type": "value", "name": "probability", "axisLabel": {"formatter": "{value}"}, "splitLine": SPLIT},
        "series": [{"type": "bar", "data": [round(float(x), 4) for x in p], "itemStyle": {"color": color, "borderRadius": [3, 3, 0, 0]}, "barCategoryGap": "6%", "markLine": ml},
                   {"type": "line", "data": [round(float(x), 4) for x in np.convolve(np.pad(p, 2, mode="edge"), np.ones(5) / 5, "valid")[:len(p)]], "smooth": True, "showSymbol": False, "lineStyle": {"color": RED, "width": 2}}]}); return o
def exceed_opt(v, unit, bad="high"):
    v = np.sort(np.asarray(v, float)); n = len(v); pr = 1 - np.arange(1, n + 1) / n if bad == "high" else np.arange(1, n + 1) / n; step = max(1, n // 120)
    o = B._base(grid=GRID); o.update({"xAxis": {"type": "category", "data": [fmt(x, unit) for x in v[::step]], "axisLabel": {"hideOverlap": True, "fontSize": 10}}, "yAxis": {"type": "value", "min": 0, "max": 1, "name": "chance outcome is worse" if bad == "high" else "chance outcome is lower", "axisLabel": {"formatter": "{value}"}, "splitLine": SPLIT},
        "series": [{"type": "line", "data": [round(float(x), 3) for x in pr[::step]], "showSymbol": False, "lineStyle": {"color": RED, "width": 2}, "areaStyle": {"color": "rgba(229,72,77,.12)"}, "markLine": {"symbol": "none", "silent": True, "lineStyle": {"type": "dashed", "color": GREY}, "label": {"formatter": "{b}", "fontSize": 10}, "data": [{"yAxis": .5, "name": "50%"}, {"yAxis": .1, "name": "10%"}, {"yAxis": .05, "name": "5%"}]}}]}); return o
def fan_opt(x, M, color=BLUE, unit="money"):
    """M: (paths, steps) matrix; bands P5-95, P25-75 and the median."""
    M = np.asarray(M, float); q = lambda p: np.percentile(M, p, axis=0); p5, p25, p50, p75, p95 = q(5), q(25), q(50), q(75), q(95)
    band = lambda lo, hi, a: [{"type": "line", "data": L(lo), "stack": f"s{a}", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": L(hi - lo), "stack": f"s{a}", "lineStyle": {"opacity": 0}, "areaStyle": {"color": color, "opacity": a}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    o = B._base(grid=GRID); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": {"type": "value", "scale": True, "splitLine": SPLIT},
        "series": band(p5, p95, .12) + band(p25, p75, .25) + [{"type": "line", "name": "median", "data": L(p50), "showSymbol": False, "smooth": True, "lineStyle": {"color": color, "width": 2.5}}]}); return o
def tornado_opt(rows, unit="money"):
    """rows: [(label, low_delta, high_delta)] change of the outcome when the input moves down/up."""
    rows = sorted(rows, key=lambda r: max(abs(r[1]), abs(r[2])))
    o = B._base(legend=True, grid={"left": 8, "right": 18, "top": 30, "bottom": 6, "containLabel": True}); o.update({"tooltip": {"trigger": "axis", "axisPointer": {"type": "shadow"}, "confine": True},
        "yAxis": {"type": "category", "data": [r[0] for r in rows], "axisLabel": {"fontSize": 11}}, "xAxis": {"type": "value", "splitLine": SPLIT, "name": "change in outcome"},
        "series": [{"type": "bar", "name": "input lower", "stack": "t", "data": [round(float(r[1]), 2) for r in rows], "itemStyle": {"color": BLUE}, "barMaxWidth": 20}, {"type": "bar", "name": "input higher", "stack": "t", "data": [round(float(r[2]), 2) for r in rows], "itemStyle": {"color": AMBER}, "barMaxWidth": 20}]}); return o
def heat_opt(xl, yl, Z, xname, yname, unit="money", good_low=True):
    Z = np.asarray(Z, float); data = [[i, j, round(float(Z[j, i]), 2)] for j in range(len(yl)) for i in range(len(xl))]; lo, hi = float(Z.min()), float(Z.max())
    cols = ["#12a594", "#f5a524", "#e5484d"] if good_low else ["#e5484d", "#f5a524", "#12a594"]
    o = B._base(); o.update({"tooltip": {"confine": True}, "grid": {"left": 8, "right": 14, "top": 8, "bottom": 52, "containLabel": True}, "xAxis": {"type": "category", "data": xl, "name": xname, "nameLocation": "middle", "nameGap": 26, "splitArea": {"show": True}}, "yAxis": {"type": "category", "data": yl, "name": yname, "splitArea": {"show": True}},
        "visualMap": {"min": lo, "max": hi if hi > lo else lo + 1, "calculable": False, "orient": "horizontal", "left": "center", "bottom": 0, "inRange": {"color": cols}, "itemWidth": 12, "itemHeight": 120, "text": ["high", "low"]},
        "series": [{"type": "heatmap", "data": data, "label": {"show": len(xl) * len(yl) <= 36, "fontSize": 10}, "emphasis": {"itemStyle": {"shadowBlur": 6}}}]}); return o
def panel(metric, samples, unit="money", bad="high", today=None, fan=None, tornado=None, heat=None, n_label=None, note=None, key="mc"):
    """Returns a list of blocks. samples: outcomes of many simulated futures. bad: which tail is the risk ('high' = costs/losses, 'low' = profit/conversions)."""
    s = stats(samples, bad); n = len(samples); w = "worst-case" if bad == "high" else "bad-case"
    prob = None
    if today is not None: prob = float(np.mean(np.asarray(samples) > today)) if bad == "high" else float(np.mean(np.asarray(samples) < today))
    k = [(f"Expected {metric}", fmt(s["mean"], unit), f"P10–P90 {fmt(s['p10'], unit)} – {fmt(s['p90'], unit)}"), ("Median (P50)", fmt(s["p50"], unit)), (f"{w} 1-in-20 (VaR 95)", fmt(s["var95"], unit), "95% of futures are better than this"), (f"Average of worst 5% (CVaR)", fmt(s["cvar95"], unit), "what a really bad run looks like")]
    if prob is not None: k.append(("Chance of being worse than today", f"{prob:.0%}", "across all simulated futures", "Critical" if prob > .6 else None))
    bl = [B.callout(f"Monte Carlo: {n:,} simulated futures with random demand, timing and model uncertainty. {note or ''}".strip(), "info", "How to read these charts"), B.kpis(*k)]
    r1 = [B.card(f"Distribution of {metric}", B.chart(hist_opt(samples, s, unit, base=today), h=250, key=key + "_h"), sub="Bars = how likely each outcome is; dashed lines mark P10 / P50 / P90"),
          B.card("Exceedance curve", B.chart(exceed_opt(samples, unit, bad), h=250, key=key + "_e"), sub="Read across: how likely is it to end up beyond a given value")]
    bl.append(B.row(*r1))
    r2 = []
    if fan is not None: r2.append(B.card("Fan chart over time", B.chart(fan_opt(fan[0], fan[1], unit=unit), h=260, key=key + "_f"), sub="Dark band = middle 50% of futures, light band = 90%; line = median"))
    if tornado: r2.append(B.card("What drives the result (tornado)", B.chart(tornado_opt(tornado, unit), h=260, key=key + "_t"), sub="Each input moved down/up on its own; the longest bar matters most"))
    if r2: bl.append(B.row(*r2))
    if heat: bl.append(B.card(heat["title"], B.chart(heat_opt(heat["x"], heat["y"], heat["Z"], heat["xname"], heat["yname"], unit, good_low=(bad == "high")), h=280, key=key + "_hm"), sub=heat.get("sub")))
    return bl, s
def sweep_tornado(fn, base, spec):
    """fn(params)->float. spec: {label: (key, low, high)}; returns rows (label, d_low, d_high) relative to fn(base)."""
    b0 = fn(base); out = []
    for lab, (k, lo, hi) in spec.items():
        a = dict(base); a[k] = lo; c = dict(base); c[k] = hi; out.append((lab, fn(a) - b0, fn(c) - b0))
    return out
def grid(fn, base, kx, xs, ky, ys):
    Z = np.zeros((len(ys), len(xs)))
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            p = dict(base); p[kx] = x; p[ky] = y; Z[j, i] = fn(p)
    return Z


In [ ]:
%%writefile /content/app/mc_sims.py
"""Monte Carlo layer for every simulator. Each function returns blocks (distribution, risk numbers, exceedance, tornado, heat map, fan chart where there is a time axis).
Uncertain inputs are sampled around the values you set; the spreads are stated in each panel's note and are the first thing to calibrate with your own history."""
import numpy as np, pandas as pd
import blocks as B, mc
from blocks import Q
N = 1500
def _rng(): return np.random.default_rng(11)
def _norm(r, mu, sd, lo=None, hi=None): return np.clip(r.normal(mu, sd, N), lo, hi)
def _fan(x, M): return (x, M)
# ============================================================================== FRANCHISE
def fr_inventory(a):
    import sim_fr as S
    d = S._inv_frame(); outs = a.get("outlets") or []
    if outs: d = d[d.outlet_id.isin(outs)].reset_index(drop=True)
    if not len(d): return []
    dp, dl, buf, cv = S.g(a, "pct", 0) / 100, S.g(a, "days", 0), S.g(a, "buffer", 0) / 100, S.g(a, "cv", 30) / 100; uv = d.uv.values; gmm = float(np.average(d.gm.values, weights=uv * d.weekly_demand.values))
    tot = lambda r: (r["lost"] * uv[:, None]).sum(0) * gmm
    base = S._inv_core(d, 0, 0, 0, cv, R=300, seed=21); sc = S._inv_core(d, dp, dl, buf, cv, R=300, seed=21); smp = tot(sc); today = float(tot(base).mean())
    fn = lambda p: float(tot(S._inv_core(d, p["dp"], p["dl"], p["buf"], p["cv"], R=50, seed=5)).mean()); p0 = {"dp": dp, "dl": dl, "buf": buf, "cv": cv}
    torn = mc.sweep_tornado(fn, p0, {"Demand ±15%": ("dp", dp - .15, dp + .15), "Supplier delay ±3 days": ("dl", max(0, dl - 3), dl + 3), "Demand volatility ±15 pts": ("cv", max(.05, cv - .15), cv + .15), "Safety buffer 0% / +40%": ("buf", 0, .4)})
    xs, ys = [-10, 0, 10, 25, 40], [0, 2, 4, 7]; Z = mc.grid(fn, p0, "dp", [x / 100 for x in xs], "dl", ys)
    T = sc["day_lost"].shape[0]; paths = (np.cumsum(sc["day_lost"], 0) * gmm).T
    bl, _ = mc.panel("lost margin, next 30 days", smp, "money", "high", today=today, fan=_fan([f"D{i + 1}" for i in range(T)], paths), tornado=torn, key="mci",
                     heat={"title": "Expected lost margin: demand change × supplier delay", "x": [f"{x:+d}%" for x in xs], "y": [f"+{y}d" for y in ys], "Z": Z, "xname": "demand change", "yname": "supplier lead-time delay", "sub": "Darker red = more money lost; find the combination where the business breaks"},
                     note="300 futures: daily demand is random per SKU with a shared demand shock; reorders arrive after the lead time.")
    return bl
def _model_panel(metric, fn, p0, sd, spec, grid_spec, bad, today=None, unit="money", note=""):
    r = _rng(); keys = list(p0)
    S = {k: sd[k](r) for k in keys}; out = np.array([fn({k: S[k][i] for k in keys}) for i in range(N)])
    torn = mc.sweep_tornado(fn, p0, spec); kx, xs, xl, ky, ys, yl, tx, ty, title = grid_spec; Z = mc.grid(fn, p0, kx, xs, ky, ys)
    bl, _ = mc.panel(metric, out, unit, bad, today=today, tornado=torn, heat={"title": title, "x": xl, "y": yl, "Z": Z, "xname": tx, "yname": ty}, note=note, key="mc" + metric[:3].replace(" ", ""))
    return bl
def fr_workforce(a):
    import sim_fr as S; r = S.run_workforce(a); m = r["m"]
    if not m.get("model"): return []
    ab, cap, temp = S.g(a, "pct", 0) / 100, S.g(a, "ot_cap", 10), S.g(a, "temp", 0) / 100; f = lambda p: m["model"](float(np.clip(p["ab"], 0, .8)), float(max(0, p["cap"])), float(np.clip(p["temp"], 0, 1)))
    return _model_panel("monthly people cost", f, {"ab": ab, "cap": cap, "temp": temp}, {"ab": lambda r: _norm(r, ab, max(.03, ab * .3), 0, .8), "cap": lambda r: _norm(r, cap, 2, 0, 20), "temp": lambda r: _norm(r, temp, .1, 0, 1)},
                        {"Absenteeism ±10 pts": ("ab", max(0, ab - .1), ab + .1), "Overtime cap ±5 h": ("cap", max(0, cap - 5), cap + 5), "Temp cover 0% / 100%": ("temp", 0, 1)},
                        ("ab", [0, .1, .2, .3, .4], ["0%", "10%", "20%", "30%", "40%"], "temp", [0, .25, .5, .75, 1], ["0%", "25%", "50%", "75%", "100%"], "absenteeism", "temp cover", "Monthly cost: absenteeism × temp cover"), "high", today=f({"ab": 0, "cap": cap, "temp": 0}),
                        note="Absenteeism, overtime cap and temp cover are sampled around your settings (absenteeism spread = 30% of the value, at least 3 points).")
def fr_pnl(a):
    import sim_fr as S; r = S.run_pnl(a); m = r["m"]
    if not m.get("model"): return []
    dp, pp, cp = S.g(a, "pct", 0) / 100, S.g(a, "price", 0) / 100, S.g(a, "cost", 0) / 100; f = lambda p: m["model"](p["dp"], p["pp"], p["cp"])
    return _model_panel("profit change per month", f, {"dp": dp, "pp": pp, "cp": cp}, {"dp": lambda r: _norm(r, dp, .06), "pp": lambda r: _norm(r, pp, .02), "cp": lambda r: _norm(r, cp, .03)},
                        {"Demand ±10 pts": ("dp", dp - .1, dp + .1), "Price ±5 pts": ("pp", pp - .05, pp + .05), "Costs ±5 pts": ("cp", cp - .05, cp + .05)},
                        ("dp", [-.2, -.1, 0, .1, .2], ["-20%", "-10%", "0", "+10%", "+20%"], "cp", [-.05, 0, .05, .1, .2], ["-5%", "0", "+5%", "+10%", "+20%"], "demand change", "cost inflation", "Profit change: demand × cost inflation"), "low", today=0.0,
                        note="Demand, price and cost moves are sampled around your settings (spreads 6, 2 and 3 points).")
def fr_marketing(a):
    import sim_fr as S; mm = S._mk_frame()
    if not len(mm): return []
    bp, al = S.g(a, "pct", 0) / 100, S.g(a, "alpha", 70) / 100; B0 = mm.B.values; C0 = mm.C.values
    def f(p):
        alp = float(np.clip(p["al"], .3, .95)); k = C0 / B0 ** alp; return float((k * (B0 * (1 + p["bp"])) ** alp).sum())
    rr = _rng()
    S_ = {"bp": _norm(rr, bp, .05), "al": _norm(rr, al, .08, .3, .95)}; out = np.array([f({"bp": S_["bp"][i], "al": S_["al"][i]}) * rr.lognormal(0, .08) for i in range(N)])
    p0 = {"bp": bp, "al": al}; torn = mc.sweep_tornado(f, p0, {"Budget ±20 pts": ("bp", bp - .2, bp + .2), "Diminishing returns α ±0.15": ("al", al - .15, al + .15)})
    Z = mc.grid(f, p0, "bp", [-.3, -.15, 0, .25, .5], "al", [.5, .6, .7, .8, .9]); bl, _ = mc.panel("conversions", out, "num", "low", today=float(C0.sum()), tornado=torn, key="mcmk",
        heat={"title": "Conversions: budget change × diminishing-returns strength", "x": ["-30%", "-15%", "0", "+25%", "+50%"], "y": ["α .5", "α .6", "α .7", "α .8", "α .9"], "Z": Z, "xname": "total budget change", "yname": "returns exponent α"}, note="Budget change, α and per-run channel noise (8%) are sampled around your settings.")
    return bl
def fr_sentiment(a):
    import sim_fr as S; f_ = Q("SELECT outlet_id, COUNT(*) n, AVG(rating) r FROM feedback GROUP BY outlet_id"); o = Q("SELECT outlet_id, revenue FROM outlets"); d = f_.merge(o, on="outlet_id"); outs = a.get("outlets") or []
    if outs: d = d[d.outlet_id.isin(outs)]
    if not len(d): return []
    neg, nn = S.g(a, "pct", 25) / 100, S.g(a, "n_new", 30); n, rr_, rev = d.n.values, d.r.values, d.revenue.values
    def f(p):
        nv = max(0, p["nn"]); ng = float(np.clip(p["neg"], 0, 1)); mix = ng * 1.5 + (1 - ng) * 4.3; rn = (n * rr_ + nv * mix) / (n + nv); return float((np.maximum(0, rr_ - rn) * p["st"] * rev).sum())
    r = _rng(); sm = {"neg": _norm(r, neg, .08, 0, 1), "nn": _norm(r, nn, max(3, nn * .25), 0), "st": _norm(r, .07, .02, .01)}; out = np.array([f({k: v[i] for k, v in sm.items()}) for i in range(N)])
    p0 = {"neg": neg, "nn": nn, "st": .07}; torn = mc.sweep_tornado(f, p0, {"Negative share ±15 pts": ("neg", max(0, neg - .15), neg + .15), "New reviews ±40%": ("nn", nn * .6, nn * 1.4), "Revenue per star 3% / 11%": ("st", .03, .11)})
    Z = mc.grid(f, p0, "neg", [0, .1, .25, .5, .75], "nn", [10, 30, 60, 100]); bl, _ = mc.panel("revenue at risk per month", out, "money", "high", tornado=torn, key="mcse", heat={"title": "Revenue at risk: negative share × number of new reviews", "x": ["0%", "10%", "25%", "50%", "75%"], "y": ["10", "30", "60", "100"], "Z": Z, "xname": "share of new reviews that are negative", "yname": "new reviews"},
                note="Revenue sensitivity per star (7%, spread 2 points) is the biggest unknown; replace it with your own estimate.")
    return bl
def fr_audit(a):
    import sim_fr as S; raise_, extra, fine = S.g(a, "raise", 0), S.g(a, "pct", 0) / 100, S.g(a, "fine", 5000); outs = a.get("outlets") or []
    d = Q("SELECT outlet_id, score, violations FROM audits WHERE audit_id IN (SELECT audit_id FROM audits a2 WHERE a2.outlet_id=audits.outlet_id ORDER BY audit_date DESC LIMIT 1)")
    if outs: d = d[d.outlet_id.isin(outs)]
    if not len(d): return []
    sc, v = d.score.values, d.violations.values; pf = lambda s, mark: 1 / (1 + np.exp((s - mark) / 6)); r = _rng()
    p1 = pf(sc - extra * 20, 65 + raise_); fails = r.random((N, len(d))) < p1; out = (fails * (v * (1 + extra))[None, :] * fine * r.lognormal(0, .15, (N, 1))).sum(1)
    f = lambda p: float((pf(sc - p["ex"] * 20, 65 + p["rs"]) * v * (1 + p["ex"]) * p["fi"]).sum()); p0 = {"ex": extra, "rs": raise_, "fi": fine}
    torn = mc.sweep_tornado(f, p0, {"Stricter pass mark 0 / +10": ("rs", 0, 10), "Extra violations ±10 pts": ("ex", max(0, extra - .1), extra + .1), "Fine per violation ±40%": ("fi", fine * .6, fine * 1.4)})
    Z = mc.grid(f, p0, "rs", [0, 5, 10, 15], "ex", [0, .1, .2, .3]); nf = fails.sum(1)
    bl, _ = mc.panel("audit fines", out, "money", "high", tornado=torn, key="mcau", heat={"title": "Expected fines: pass mark × extra violations", "x": ["+0", "+5", "+10", "+15"], "y": ["0%", "+10%", "+20%", "+30%"], "Z": Z, "xname": "pass mark raised by", "yname": "extra violations"},
                note=f"Each outlet fails or passes by chance (logistic on its last score). Failing outlets: median {int(np.median(nf))}, 95% of runs ≤ {int(np.percentile(nf, 95))}.")
    return bl
# ============================================================================== FREIGHT
def fq_route(a):
    import sim_fq as S
    L = getattr(S, "LAST", {}).get("route")
    if L is None or not len(L): return []
    dist, dly = float(L.dist_cost.sum()), float(L.delay_cost.sum()); f = lambda p: p["m1"] * dist + p["m2"] * dly; r = _rng()
    out = np.array([f({"m1": m1, "m2": m2}) for m1, m2 in zip(r.lognormal(0, .18, N), r.lognormal(0, .35, N))]) * r.lognormal(0, .1, N)
    p0 = {"m1": 1.0, "m2": 1.0}; torn = mc.sweep_tornado(f, p0, {"Freight rate ±30%": ("m1", .7, 1.3), "Delay cost / waiting ±50%": ("m2", .5, 1.5)}); Z = mc.grid(f, p0, "m1", [.7, .85, 1, 1.15, 1.3], "m2", [.5, .75, 1, 1.5, 2])
    bl, _ = mc.panel("extra cost of the re-route", out, "money", "high", tornado=torn, key="mcrt", heat={"title": "Extra cost: freight rate × delay cost", "x": ["-30%", "-15%", "base", "+15%", "+30%"], "y": ["×0.5", "×0.75", "×1", "×1.5", "×2"], "Z": Z, "xname": "freight rate vs assumption", "yname": "delay cost vs assumption"},
                note="Distances come from the sea-route model; the rate per tonne-mile (spread 18%) and the cost of waiting (spread 35%) are uncertain and sampled.")
    return bl
def fq_quote(a):
    import sim_fq as S; fuel, car, fx, disc = S.g(a, "pct", 0) / 100, S.g(a, "carrier", 0) / 100, S.g(a, "fx", 0) / 100, S.g(a, "discount", 0) / 100
    q = Q("SELECT base_cost, customs_fee, fuel_surcharge, final_price, margin_pct FROM freight_quotes WHERE final_price>0 ORDER BY created_at DESC LIMIT 400")
    if not len(q): return []
    p0v = q.final_price.values; c0 = p0v * (1 - q.margin_pct.values / 100); bc, cf, fs = q.base_cost.values, q.customs_fee.values, q.fuel_surcharge.values
    f = lambda p: float((p0v - (c0 + bc * ((1 + p["car"]) * (1 + p["fx"]) - 1) + cf * p["fx"] + fs * p["fu"])).sum()); r = _rng()
    sm = {"car": _norm(r, car, .03), "fx": _norm(r, fx, .03), "fu": _norm(r, fuel, .08)}; out = np.array([f({k: v[i] for k, v in sm.items()}) for i in range(N)]); p0 = {"car": car, "fx": fx, "fu": fuel}
    torn = mc.sweep_tornado(f, p0, {"Fuel ±15 pts": ("fu", fuel - .15, fuel + .15), "Carrier rates ±8 pts": ("car", car - .08, car + .08), "Currency ±8 pts": ("fx", fx - .08, fx + .08)}); Z = mc.grid(f, p0, "fu", [-.1, 0, .1, .2, .3], "car", [-.05, 0, .05, .1, .2])
    bl, _ = mc.panel("total margin if prices are held", out, "money", "low", today=float((p0v - c0).sum()), tornado=torn, key="mcqt", heat={"title": "Margin: fuel move × carrier rate move", "x": ["-10%", "0", "+10%", "+20%", "+30%"], "y": ["-5%", "0", "+5%", "+10%", "+20%"], "Z": Z, "xname": "fuel move", "yname": "carrier rate move"},
                note="Fuel (spread 8 points), carrier rate and currency (3 points each) are sampled around your settings; prices on open quotes stay as quoted.")
    return bl
def fq_weather(a):
    import sim_fq as S; area = a.get("area") or "Red Sea"; la, lo = S.STORM_AREAS.get(area, S.STORM_AREAS["Red Sea"]); wind, rad, dem = S.g(a, "wind", 25), S.g(a, "radius", 900), S.g(a, "demurrage", 900)
    v = Q("SELECT lat, lon FROM vessels");
    if not len(v): return []
    dist = S.hav(v.lat.values, S._wrap(v.lon.values), la, lo)
    def f(p):
        rd = max(50, p["rad"]); h = dist[dist <= rd]; dl = min(7.0, max(0.0, .25 * (p["w"] - 12))); return float((dl * (1 - h / rd * .5) * p["dem"]).sum())
    r = _rng(); sm = {"w": _norm(r, wind, 4, 8), "rad": _norm(r, rad, rad * .15, 100), "dem": r.lognormal(np.log(max(dem, 1)), .25, N)}; out = np.array([f({k: v_[i] for k, v_ in sm.items()}) for i in range(N)])
    p0 = {"w": wind, "rad": rad, "dem": dem}; torn = mc.sweep_tornado(f, p0, {"Wind ±8 m/s": ("w", wind - 8, wind + 8), "Storm radius ±300 nm": ("rad", max(100, rad - 300), rad + 300), "Demurrage ±40%": ("dem", dem * .6, dem * 1.4)}); Z = mc.grid(f, p0, "w", [14, 20, 26, 32, 38], "rad", [400, 700, 900, 1200, 1600])
    bl, _ = mc.panel("storm delay + demurrage cost", out, "money", "high", tornado=torn, key="mcwx", heat={"title": "Cost: wind speed × storm radius", "x": [f"{x} m/s" for x in (14, 20, 26, 32, 38)], "y": [f"{y} nm" for y in (400, 700, 900, 1200, 1600)], "Z": Z, "xname": "wind speed", "yname": "storm radius"},
                note="Wind (±4 m/s), radius (±15%) and demurrage rate (±25%) are uncertain because storm forecasts are; they are sampled around your settings.")
    return bl
def fq_customs(a):
    import sim_fq as S; hp, dw, sc = S.g(a, "hold", 20) / 100, S.g(a, "dwell", 4), S.g(a, "storage", 180)
    s = Q("SELECT shipment_id FROM shipments WHERE status IN ('In Transit','Port Congestion Delay','Anchorage Pending','Customs Hold')"); n0 = len(s)
    if not n0: return []
    f = lambda p: float(round(n0 * float(np.clip(p["hp"], 0, 1))) * max(0, p["dw"]) * p["sc"]); r = _rng()
    out = np.array([r.binomial(n0, float(np.clip(h, 0, 1))) * max(0, d) * c for h, d, c in zip(_norm(r, hp, .05, 0, 1), r.gamma(4, max(dw, .1) / 4, N), r.lognormal(np.log(max(sc, 1)), .15, N))])
    p0 = {"hp": hp, "dw": dw, "sc": sc}; torn = mc.sweep_tornado(f, p0, {"Hold rate ±10 pts": ("hp", max(0, hp - .1), hp + .1), "Extra dwell ±2 days": ("dw", max(0, dw - 2), dw + 2), "Storage cost ±$60": ("sc", max(10, sc - 60), sc + 60)}); Z = mc.grid(f, p0, "hp", [.05, .1, .2, .3, .5], "dw", [1, 2, 4, 6, 8])
    bl, _ = mc.panel("customs hold cost", out, "money", "high", tornado=torn, key="mccu", heat={"title": "Cost: hold rate × dwell days", "x": ["5%", "10%", "20%", "30%", "50%"], "y": ["1d", "2d", "4d", "6d", "8d"], "Z": Z, "xname": "share of inbound held", "yname": "extra dwell days"}, note="Which shipments get held is random (binomial); dwell days follow a gamma distribution around your setting.")
    return bl
def fq_carrier(a):
    import sim_fq as S; m = S.LAST.get("carrier") if hasattr(S, "LAST") else None
    if m is None or not len(m): return []
    ok = m[m.to != "NO CAPACITY"]; n = len(m); base_ex = float((ok.cost_index * ok.tonnes * 400).sum()) + float(ok.delay_cost.sum()); r = _rng()
    # capacity of alternatives is uncertain: each re-placement succeeds with p, failures pay a spot-market premium of 35% of $400/tonne equivalent
    pl = len(ok) / max(n, 1); out = []
    for _ in range(N):
        cap = np.clip(r.normal(1, .25), .2, 1.8); k = r.binomial(n, float(np.clip(pl * cap, 0, 1))); spot = (n - k) * float(m.tonnes.mean() or 1) * 400 * .35; out.append(base_ex * r.lognormal(0, .2) + spot)
    out = np.array(out); f = lambda p: base_ex * p["m"] + (n - round(n * min(1, pl * p["cap"]))) * float(m.tonnes.mean() or 1) * 400 * .35; p0 = {"m": 1.0, "cap": 1.0}
    torn = mc.sweep_tornado(f, p0, {"Alternative capacity ±40%": ("cap", .6, 1.4), "Cost index change ±50%": ("m", .5, 1.5)}); Z = mc.grid(f, p0, "cap", [.4, .7, 1, 1.3], "m", [.5, 1, 1.5, 2])
    bl, _ = mc.panel("cost of the carrier outage", out, "money", "high", tornado=torn, key="mccr", heat={"title": "Cost: spare capacity × cost increase", "x": ["×0.4", "×0.7", "×1", "×1.3"], "y": ["×0.5", "×1", "×1.5", "×2"], "Z": Z, "xname": "spare capacity at alternatives", "yname": "cost-index penalty"},
                note="Spare capacity at the alternative carriers is the big unknown (±25%); shipments that cannot be placed pay a 35% spot-market premium.")
    return bl
MC = {"Franchise": {"inventory": fr_inventory, "workforce": fr_workforce, "outlets": fr_pnl, "marketing": fr_marketing, "sentiment": fr_sentiment, "audit": fr_audit},
      "Freight": {"route": fq_route, "quotes": fq_quote, "margin": fq_quote, "weather": fq_weather, "customs": fq_customs, "carriers": fq_carrier}}
def blocks(kind, pid, a):
    f = MC.get(kind, {}).get(pid)
    if not f: return []
    try:
        bl = f(a)
    except Exception as e:
        import traceback, sys; traceback.print_exc(file=sys.stderr); return [B.callout(f"Monte Carlo panel skipped: {e}", "warn")]
    return [B.md("### Monte Carlo risk analysis")] + bl if bl else []


In [ ]:
%%writefile /content/app/copilot_pro.py
"""Copilot Pro - the SQL-only assistant, upgraded. No LLM, no RAG: every number comes from a read-only query.
Layers (each is plain Python you can read and extend):
  1. small talk      greetings, thanks, goodbye, "who are you", "what can you do"
  2. typo repair     fuzzy-corrects words against the vocabulary of this app's tables, columns, places and names
  3. conversation    follow-ups ("and OUT-010?", "what about Mumbai", "top 10 instead", "its staff") use the previous turn
  4. smart intents   daily briefing, compare two entities, entity profile, export, show SQL
  5. exact engine    analyst.py (explanations) then qa.py (whitelisted NL->SQL)
  6. presentation    auto chart, follow-up suggestions, "interpreted as" note, CSV export, type-ahead, thumbs feedback
Every question that could not be answered is logged (table copilot_unanswered) so you can see what to teach it next."""
import re, io, time, json, difflib, sqlite3, threading, collections, datetime
import pandas as pd
import qa as QA, analyst as AN, blocks as B
from blocks import Q
from config import DB_PATH

ID_RE = re.compile(r"\b((?:OUT|STF|EMP|SHP|QTE|PORT|CAR|CUST|CMP|CAMP|AUD|FB)-[A-Z0-9]*\d[A-Z0-9-]*)\b", re.I)
GREET = re.compile(r"^\W*(hi+|hello+|hey+|hola|namaste|yo|good (morning|afternoon|evening|day)|greetings|howdy|sup)\b[\w\s!.,?]*$", re.I)
THANKS = re.compile(r"^\W*(thanks|thank you|thx|ty|ok thanks|great|cool|nice|awesome|perfect|got it|ok|okay)\b[\w\s!.,]*$", re.I)
BYE = re.compile(r"^\W*(bye|goodbye|see you|good night|cya|later)\b[\w\s!.,]*$", re.I)
HOWARE = re.compile(r"\b(how are you|how r u|how's it going|whats up|what's up)\b", re.I)
WHO = re.compile(r"\b(who are you|what are you|your name|who made you|who built you)\b", re.I)
TIME = re.compile(r"\b(what('?s| is) the (time|date|day)|current time|today'?s date|what day is it|time now|date today|what time is it)\b", re.I)
HELP = re.compile(r"^\W*(help|what can you do|what do you do|how (do|can) i use|commands|capabilities|what can i ask|examples?)\b", re.I)
BRIEF = re.compile(r"\b(briefing|brief me|overview|summary|summari[sz]e|status report|dashboard|health check|what needs (my )?attention|what'?s happening|big picture|key numbers|kpis?|situation)\b", re.I)
COMPARE = re.compile(r"\b(compare|versus|vs\.?|difference between)\b", re.I)
SHOWSQL = re.compile(r"\b(show|give|what'?s?) (me )?(the )?(sql|query)\b|how did you (get|calculate|work out)", re.I)
FOLLOW = re.compile(r"^\W*(and|what about|how about|same for|now|also|then|what of)\b\s*", re.I)
MORE = re.compile(r"^\W*(more|show more|next|go on|continue|another|show all|expand)\W*$", re.I)
TOPN = re.compile(r"\b(?:top|bottom|first|last)\s+(\d+)\b", re.I)
PRON = re.compile(r"\b(its|their|it|them|this (?:outlet|store|branch|shipment|quote|carrier|port|customer|campaign|employee|member)|that (?:outlet|store|branch|shipment|quote|carrier|port|customer|campaign|employee|member)|the same (?:outlet|store|shipment|quote|carrier|port))\b", re.I)

STOP = set("""about above after again against also always among another answer any are around ask asked asking available because been before being below between both brief but can cannot could current currently data database did does doing done down during each either else enough ever every explain find first following for from further get give given going good great had has have having here highest how however into its itself just keep know last least less let like list little look made make many may maybe might more most much must need never next none nor not now number off often once only other ought our out over own per please put rather really same say see seems shall she should show since some still such take tell than that the their them then there these they thing think this those though three through thus tight time today together too top toward under until upon use used using very want was way well were what when where whether which while who whom whose why will with within without would write year years yes you your yours""".split())
EXTRA_WORDS = """highest lowest average total number count revenue satisfaction customer inventory staff employees outlets stores campaigns marketing sentiment feedback reviews audit audits compliance violations stockout reorder threshold attrition overtime salary headcount tickets shipments quotes carriers ports customers weather congestion congested delay delayed margin price customs duty vessels route routes delivery rating ratings largest smallest biggest cheapest riskiest weakest strongest compare location locations region regions country tier category channel status severity critical briefing overview summary""".split()

# ------------------------------------------------------------------ vocabulary / typo repair
_V = {"t": 0, "words": set(), "list": [], "corpus": [], "places": []}
def _vocab():
    if time.time() - _V["t"] < 600 and _V["words"]: return _V
    kind = _kind(); w = set(EXTRA_WORDS)
    for t, (syn, idc, cols, _m) in QA.TABLES[kind].items():
        w.add(t); w.update(re.findall(r"[a-z]{3,}", " ".join(syn + cols).lower()))
    for c, syns in QA.CSYN.items(): w.update(re.findall(r"[a-z]{3,}", " ".join(syns + [c.replace("_", " ")]).lower()))
    vals = []; places = set()
    for t, (syn, idc, cols, _m) in QA.TABLES[kind].items():
        for c in cols:
            if c in ("location", "tier", "role", "category", "channel", "region", "country", "origin_port", "dest_port", "carrier", "port_name", "industry", "name", "outlet_name", "sku_name", "campaign_name", "risk_level", "priority_tier", "status"):
                try:
                    got = Q(f"SELECT DISTINCT {c} v FROM {t} WHERE {c} IS NOT NULL LIMIT 400").v.astype(str).tolist(); vals += got
                    if c in ("location", "port_name", "country", "region", "origin_port", "dest_port"): places.update(got)
                except Exception: pass
    for v in vals: w.update(re.findall(r"[a-z]{3,}", v.lower()))
    pv = set(places)
    for p0 in places:
        pv.add(re.sub(r"\s*\(.*?\)", "", p0).strip()); pv.update(re.findall(r"\(([^)]+)\)", p0)); pv.add(re.sub(r"\s+(Port|Terminal|Harbou?r)$", "", p0).strip())
    _V.update(t=time.time(), words=w, list=sorted(w), corpus=sorted(set(vals)), places=sorted({x for x in pv if len(x) >= 4}, key=len, reverse=True))
    return _V
def _kind():
    import config; return config.KIND
def correct(q):
    V = _vocab(); out, changed = [], []
    def fix(m):
        tok = m.group(0); low = tok.lower()
        if len(low) < 5 or low in V["words"] or low in STOP or any(len(low) - k >= 4 and low[:-k] in V["words"] for k in (1, 2, 3)): return tok
        cand = [c for c in difflib.get_close_matches(low, V["list"], n=3, cutoff=0.84) if c[0] == low[0] and abs(len(c) - len(low)) <= 2]
        if not cand: return tok
        changed.append((tok, cand[0])); return cand[0].capitalize() if tok[0].isupper() else cand[0]
    new = re.sub(r"(?<![A-Za-z0-9-])[A-Za-z]{5,}(?![A-Za-z0-9-])", fix, q)
    return new, changed

# ------------------------------------------------------------------ conversation memory
_S = collections.OrderedDict(); _L = threading.Lock()
def session(sid):
    with _L:
        s = _S.get(sid)
        if s is None:
            s = _S[sid] = {"q": "", "ids": [], "df": None, "sql": "", "turns": 0}
            while len(_S) > 300: _S.popitem(last=False)
        _S.move_to_end(sid); return s
def last_csv(sid):
    s = _S.get(sid); df = s and s.get("df")
    return df.to_csv(index=False) if df is not None and len(df) else None

def _places_in(t):
    hits = []
    for p in _vocab()["places"]:
        for m in re.finditer(r"(?<![A-Za-z])" + re.escape(p) + r"(?![A-Za-z])", t, re.I):
            if not any(m.start() < e and m.end() > b for b, e, _ in hits): hits.append((m.start(), m.end(), p))
    return sorted(hits)
def _place_in(t):
    h = _places_in(t); return h[-1][2] if h else None
def _swap_place(old_q, new_place):
    h = _places_in(old_q)
    if not h: return None
    b, e, _ = h[-1]; return old_q[:b] + new_place + old_q[e:]
def contextualize(q, s):
    """Rewrite a follow-up into a self-contained question, using the previous turn."""
    ql = q.strip(); note = None; ids = [i.upper() for i in ID_RE.findall(q)]
    if MORE.match(ql) and s["q"]:
        m = TOPN.search(s["q"]); n = min(50, (int(m.group(1)) * 2) if m else 20)
        base = TOPN.sub(f"top {n}", s["q"]) if m else s["q"] + f" top {n}"; return base, f"showing more: {base}"
    m = TOPN.search(ql)
    if m and s["q"] and len(ql.split()) <= 4 and not re.search(r"\b(outlet|store|staff|shipment|quote|carrier|port|campaign|customer|audit|item|sku)", ql):
        base = TOPN.sub(m.group(0), s["q"]) if TOPN.search(s["q"]) else s["q"] + " " + m.group(0); return base, f"{base}"
    if FOLLOW.match(ql) and s["q"]:
        rest = FOLLOW.sub("", ql).strip(" ?.!")
        if ids and s["ids"] and ID_RE.search(s["q"]): new = ID_RE.sub(ids[0], s["q"], count=1)
        elif ids and re.search(r"\b(staff|inventory|revenue|risk|why|quote|customs|weather|margin|campaign|audit|sentiment|feedback)\b", rest, re.I) is None: new = s["q"] + " " + rest
        elif ids: new = (rest if len(rest.split()) > 2 else rest + " " + s["q"])
        elif _place_in(rest) and _place_in(s["q"]): new = _swap_place(s["q"], _place_in(rest))
        elif re.match(r"^(in|at|for|from|to)\b", rest, re.I): new = s["q"] + " " + rest
        elif len(rest.split()) <= 3: new = s["q"] + " in " + rest
        else: new = rest
        return new, new
    if not ids and s["ids"] and PRON.search(ql):
        new = PRON.sub(s["ids"][0], ql, count=1); return new, new
    return q, None

# ------------------------------------------------------------------ smart intents
def _num(sql, p=()):
    try:
        r = Q(sql, p); return None if not len(r) else r.iloc[0, 0]
    except Exception: return None
def _f(v, d=1): return "n/a" if v is None or v != v else (f"{v:,.{d}f}" if isinstance(v, float) else f"{int(v):,}")

def briefing(kind, dec, dom):
    rows, sqls = [], []
    def add(label, sql, d=1, suffix=""):
        v = _num(sql); rows.append((label, _f(v, d) + suffix)); sqls.append(sql)
    if kind == "Franchise":
        add("Outlets", "SELECT COUNT(*) FROM outlets", 0); add("Total revenue", "SELECT SUM(revenue) FROM outlets", 0)
        add("Avg customer satisfaction", "SELECT AVG(customer_satisfaction) FROM outlets", 2); add("Staff", "SELECT COUNT(*) FROM staff", 0)
        add("Staff with attrition risk > 60%", "SELECT COUNT(*) FROM staff WHERE predicted_attrition_prob>0.6", 0)
        add("Items below reorder level", "SELECT COUNT(*) FROM inventory WHERE current_stock<reorder_threshold", 0)
        add("Avg audit score", "SELECT AVG(score) FROM audits", 1); add("Avg review rating", "SELECT AVG(rating) FROM feedback", 2)
        add("Open agent tickets", "SELECT COUNT(*) FROM agent_actions WHERE status IN ('open','proposed')", 0)
        focus_q = "Which outlets are at highest risk?"
    else:
        add("Shipments", "SELECT COUNT(*) FROM shipments", 0); add("Avg delay risk", "SELECT AVG(predicted_delay_risk) FROM shipments", 2)
        add("Quotes", "SELECT COUNT(*) FROM freight_quotes", 0); add("Avg margin %", "SELECT AVG(margin_pct) FROM freight_quotes", 1)
        add("Avg port congestion", "SELECT AVG(congestion_index) FROM ports", 2); add("Avg carrier on-time %", "SELECT AVG(on_time_pct) FROM carriers", 1)
        add("Weather points above 0.5 severity", "SELECT COUNT(*) FROM weather_risks WHERE current_severity>0.5", 0)
        add("Open agent tickets", "SELECT COUNT(*) FROM agent_actions WHERE status IN ('open','proposed')", 0)
        focus_q = "Which ports are most congested?"
    text = "**Briefing: the numbers right now**\n\n" + "\n".join(f"- {a}: **{b}**" for a, b in rows)
    ex = None
    try: ex = AN.answer(kind, focus_q, None, dec, dom) or QA.answer(kind, focus_q, None, None)
    except Exception: pass
    out = {"answer": text, "df": None, "sql": ";\n".join(sqls), "sugg": [focus_q]}
    if ex and ex.get("answer"): out["answer"] += "\n\n**Needs attention:** " + ex["answer"]; out["df"] = ex.get("df"); out["extra"] = ex.get("extra")
    return out

def _entity_row(e):
    for t, (_, idc, cols, _m) in QA.TABLES[_kind()].items():
        if t == "agent_actions": continue
        try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
        except Exception: continue
        if len(df): return t, df.iloc[0]
    return None, None

def compare(ids):
    cols, t0 = {}, None; sqls = []
    for e in ids[:3]:
        t, r = _entity_row(e)
        if r is None: return {"answer": f"I could not find {e} in the data.", "df": None, "sql": ""}
        if t0 and t != t0: return {"answer": f"{ids[0]} and {e} are different kinds of records ({t0} vs {t}), so I cannot compare them side by side.", "df": None, "sql": ""}
        t0 = t; cols[e] = r; sqls.append(f"SELECT * FROM {t} WHERE id='{e}'")
    df = pd.DataFrame({e: {k: QA._fmt(v) for k, v in r.items() if str(v) not in ("None", "nan")} for e, r in cols.items()}).reset_index().rename(columns={"index": "field"})
    return {"answer": f"Side-by-side: **{' vs '.join(ids[:3])}** ({t0}).", "df": df, "sql": "; ".join(sqls), "nochart": True}

def profile(kind, e):
    t, r = _entity_row(e)
    if r is None: return None
    base = QA.lookup(kind, e)
    extra, sqls = [], []
    def m(label, sql, p=(), d=1):
        v = _num(sql, p); sqls.append(sql)
        if v is not None and v == v: extra.append(f"- {label}: **{_f(v, d)}**")
    if kind == "Franchise" and e.startswith("OUT-"):
        m("Staff on record", "SELECT COUNT(*) FROM staff WHERE outlet_id=?", (e,), 0); m("Avg attrition risk", "SELECT AVG(predicted_attrition_prob) FROM staff WHERE outlet_id=?", (e,), 2)
        m("Items below reorder level", "SELECT COUNT(*) FROM inventory WHERE outlet_id=? AND current_stock<reorder_threshold", (e,), 0)
        m("Avg review rating", "SELECT AVG(rating) FROM feedback WHERE outlet_id=?", (e,), 2); m("Avg audit score", "SELECT AVG(score) FROM audits WHERE outlet_id=?", (e,), 1)
        m("Active campaigns", "SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (e,), 0)
    if kind == "Freight" and e.startswith("SHP-"):
        m("Quotes on this shipment", "SELECT COUNT(*) FROM freight_quotes WHERE shipment_id=?", (e,), 0)
    if not base: return None
    out = dict(base); out["answer"] = base["answer"] + (("\n\n**Context**\n" + "\n".join(extra)) if extra else ""); out["sql"] = (base.get("sql") or "") + ("; " + "; ".join(sqls) if sqls else "")
    return out

FKEY = {"OUT": "outlet_id", "SHP": "shipment_id", "CUST": "customer_id", "STF": "staff_id", "QTE": "quote_id", "EMP": "staff_id"}
def scoped(kind, e, q):
    """'staff in OUT-010', 'its inventory', 'quotes for CUST-001': the rows of one table that belong to one entity."""
    col = FKEY.get(e.split("-")[0]); ql = q.lower()
    if not col: return None
    own = {"OUT": "outlets", "SHP": "shipments", "CUST": "customers", "STF": "staff", "QTE": "freight_quotes"}.get(e.split("-")[0])
    best = None
    for t, (syn, idc, show, met) in QA.TABLES[kind].items():
        if t == own or t == "agent_actions" and not re.search(r"\b(ticket|action|task|alert)", ql): continue
        if not any(QA._w(ql, x.strip()) for x in syn): continue
        try: cols = QA._cols(t)
        except Exception: continue
        if col not in cols: continue
        best = (t, show, met, cols); break
    if not best: return None
    t, show, met, cols = best; show = [c for c in show if c in cols] or cols[:8]
    n = _num(f"SELECT COUNT(*) FROM {t} WHERE {col}=?", (e,))
    order = f" ORDER BY {met} DESC" if met and met in cols else ""
    sql = f"SELECT {', '.join(show)} FROM {t} WHERE {col}='{e}'{order} LIMIT 15"
    df = Q(f"SELECT {', '.join(show)} FROM {t} WHERE {col}=?{order} LIMIT 15", (e,))
    label = t.replace("_", " ")
    return {"answer": f"**{e}** has **{_f(n, 0)}** {label} records" + (f" (showing the top 15 by {met.replace('_', ' ')})." if met and met in cols and (n or 0) > 15 else "."), "df": df, "sql": sql + f"; SELECT COUNT(*) FROM {t} WHERE {col}='{e}'", "sugg": [f"Tell me about {e}"]}

# ------------------------------------------------------------------ teach me: plain intents (pattern -> SQL). Add yours here.
# (regex on the lower-cased question, answer title, SQL with an optional {id} placeholder bound as a parameter, needs_id)
TICKETS = ("Open agent tickets, critical first.", "SELECT action_id, type, target, severity, status, detail FROM agent_actions WHERE status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 10")
UNSUPPORTED = {
 "Franchise": [(r"\b(expired|expiry|expiring|shelf[- ]?life|best before)\b", "The inventory data has no expiry dates, so I cannot answer that. I can show items at or below reorder level, or with less than 7 days of stock cover.")],
 "Freight": [],
}
INTENTS = {
 "Franchise": [
  (r"(which|what|top).*(outlets?|stores?|branch\w*).*(tickets?|issues?|alerts?|actions?)|(most|highest|largest|top).*(open )?(tickets?|issues?|alerts?)", "Outlets with the most open tickets.", "SELECT target AS outlet_id, COUNT(*) AS open_tickets, SUM(severity = 'Critical') AS critical FROM agent_actions WHERE status IN ('open', 'proposed') AND target LIKE 'OUT-%' GROUP BY target ORDER BY open_tickets DESC, critical DESC LIMIT 10", False),
  (r"(revenue|sales|csat|satisfaction|outlets?|performance).*\bby tier\b|\btier[- ]?wise\b|\bper tier\b|\beach tier\b", "Outlets by tier.", "SELECT tier, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat, SUM(staff_headcount) AS staff FROM outlets GROUP BY tier ORDER BY revenue DESC", False),
  (r"(revenue|sales|csat|satisfaction).*\b(by|per|each) (city|location|town)\b|\bcity[- ]?wise\b", "Outlets by city.", "SELECT location AS city, COUNT(*) AS outlets, ROUND(SUM(revenue), 0) AS revenue, ROUND(AVG(customer_satisfaction), 2) AS avg_csat FROM outlets GROUP BY location ORDER BY revenue DESC LIMIT 12", False),
  (r"overtime.*(outlets?|stores?|branch\w*)|(outlets?|stores?|branch\w*).*overtime", "Overtime hours by outlet.", "SELECT outlet_id, ROUND(SUM(overtime_hrs), 0) AS overtime_hours, ROUND(AVG(overtime_hrs), 1) AS avg_per_staff, COUNT(*) AS staff FROM staff GROUP BY outlet_id ORDER BY overtime_hours DESC LIMIT 10", False),
  (r"(stock|inventory).*(value|worth|units|quantity|total)|total (stock|inventory)|stock by category", "Units in stock by category (the inventory table holds no prices, so a money value cannot be computed).", "SELECT category, COUNT(*) AS sku_records, SUM(current_stock) AS units_in_stock, SUM(CASE WHEN current_stock <= reorder_threshold THEN 1 ELSE 0 END) AS at_or_below_reorder FROM inventory GROUP BY category ORDER BY units_in_stock DESC", False),
  (r"\b(absent|absence|absentees?|on leave|leave|attendance|late arrivals?|punctuality)\b", "Absences and late arrivals by outlet (the data records Present, Late and Absent; there is no separate leave type).", "SELECT outlet_id, SUM(status = 'Absent') AS absent, SUM(status = 'Late') AS late, COUNT(*) AS records, ROUND(100.0 * SUM(status = 'Absent') / COUNT(*), 1) AS absent_pct FROM attendance GROUP BY outlet_id ORDER BY absent DESC LIMIT 10", False),
  (r"(sales|revenue|takings|turnover).*\btoday\b|\btoday'?s (sales|revenue|takings)", "Sales so far today (IST), by outlet.", "SELECT outlet_id, ROUND(SUM(amount), 0) AS sales_today, SUM(tx_count) AS transactions FROM sales WHERE substr(ts, 1, 10) = date('now', 'localtime') GROUP BY outlet_id ORDER BY sales_today DESC LIMIT 10", False),
  (r"(sales|revenue|takings|turnover).*\byesterday\b|\byesterday'?s (sales|revenue|takings)", "Sales yesterday (IST), by outlet.", "SELECT outlet_id, ROUND(SUM(amount), 0) AS sales_yesterday, SUM(tx_count) AS transactions FROM sales WHERE substr(ts, 1, 10) = date('now', 'localtime', '-1 day') GROUP BY outlet_id ORDER BY sales_yesterday DESC LIMIT 10", False),
  (r"(sales|revenue|takings|turnover).*(this week|past week|last week|last 7 days|past 7 days)", "Sales in the last 7 days (IST), by outlet.", "SELECT outlet_id, ROUND(SUM(amount), 0) AS sales_7d, SUM(tx_count) AS transactions FROM sales WHERE substr(ts, 1, 10) >= date('now', 'localtime', '-6 day') GROUP BY outlet_id ORDER BY sales_7d DESC LIMIT 10", False),
  (r"(likely|expected|predicted|going|about) to (leave|quit|resign)|attrition|churn|resign", "Staff most likely to leave (highest predicted attrition risk).", "SELECT staff_id, name, role, outlet_id, predicted_attrition_prob, overtime_hrs, salary FROM staff ORDER BY predicted_attrition_prob DESC LIMIT 10", False),
  (r"(negative|bad|poor|worst|unhappy|angry|low).*(review|feedback|comment|complaint)", "The most negative customer feedback.", "SELECT feedback_id, outlet_id, rating, sentiment_score, date, comment FROM feedback ORDER BY sentiment_score ASC, rating ASC LIMIT 10", False),
  (r"(positive|best|good|happy|great).*(review|feedback|comment)", "The most positive customer feedback.", "SELECT feedback_id, outlet_id, rating, sentiment_score, date, comment FROM feedback ORDER BY sentiment_score DESC, rating DESC LIMIT 10", False),
  (r"(average|avg|mean) (review |customer )?rating.*", "Average review rating for {id}.", "SELECT ROUND(AVG(rating), 2) FROM feedback WHERE outlet_id=?", True),
  (r"overtime.*(most|highest|top|worst)|(most|highest|top|worst).*overtime", "Staff with the most overtime hours.", "SELECT staff_id, name, role, outlet_id, overtime_hrs, salary FROM staff ORDER BY overtime_hrs DESC LIMIT 10", False),
  (r"(losing money|negative roi|worst campaign|lowest roi|poor(ly)? performing campaign)", "Campaigns with the lowest return on investment.", "SELECT campaign_id, campaign_name, outlet_id, channel, budget, actual_roi, conversions FROM marketing ORDER BY actual_roi ASC LIMIT 10", False),
  (r"^(show |list |what are |which are )?(the )?(open |pending |critical |latest )+(tickets?|actions?|alerts?)\W*$", *TICKETS, False),
 ],
 "Freight": [
  (r"(quotes?|pricing).*(need|needs|needing|awaiting|pending|waiting|require|requiring).*(approval|review|sign[- ]?off|attention)|(approval|review).*quotes?|quotes? (to|for) (approve|review)", "Quotes that need a decision: drafts, submitted quotes and quotes with an open anomaly, hold or approval ticket.", "SELECT q.quote_id, q.shipment_id, q.customer_id, q.carrier, q.final_price, q.margin_pct, q.status FROM freight_quotes q WHERE q.status IN ('Draft', 'Submitted', 'On Hold') OR q.quote_id IN (SELECT target FROM agent_actions WHERE status IN ('open', 'proposed') AND type IN ('quote_approval', 'quote_anomaly', 'quote_hold')) ORDER BY q.margin_pct LIMIT 15", False),
  (r"(arriv|eta|due|deliver|reach|land).*\btoday\b", "Shipments due today (IST).", "SELECT shipment_id, origin_port, dest_port, carrier, status, eta, predicted_delay_risk FROM shipments WHERE substr(eta, 1, 10) = date('now', 'localtime') ORDER BY predicted_delay_risk DESC LIMIT 15", False),
  (r"(arriv|eta|due|deliver|reach|land).*\btomorrow\b", "Shipments due tomorrow (IST).", "SELECT shipment_id, origin_port, dest_port, carrier, status, eta, predicted_delay_risk FROM shipments WHERE substr(eta, 1, 10) = date('now', 'localtime', '+1 day') ORDER BY predicted_delay_risk DESC LIMIT 15", False),
  (r"(arriv|eta|due|deliver|reach|land).*(this week|next 7 days|coming week|next week)", "Shipments due in the next 7 days (IST).", "SELECT shipment_id, origin_port, dest_port, carrier, status, eta, predicted_delay_risk FROM shipments WHERE substr(eta, 1, 10) BETWEEN date('now', 'localtime') AND date('now', 'localtime', '+6 day') ORDER BY eta, predicted_delay_risk DESC LIMIT 15", False),
  (r"^(show |list |what are |give me )?(the |all )?customs (rules|duties|duty|tariffs?)\W*$", "Customs rules, highest duty first.", "SELECT chapter, goods, duty_pct, docs, restricted FROM customs_rules ORDER BY duty_pct DESC LIMIT 10", False),
  (r"^(which|list|show|what).*(delayed|late|behind schedule)", "Shipments with the highest delay risk.", "SELECT shipment_id, origin_port, dest_port, carrier, status, eta, predicted_delay_risk FROM shipments WHERE status LIKE '%elay%' OR predicted_delay_risk>=0.5 ORDER BY predicted_delay_risk DESC LIMIT 10", False),
  (r"(unreliable|least reliable|worst carrier|late carrier|poor carrier)", "Carriers with the lowest on-time performance.", "SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers ORDER BY on_time_pct ASC LIMIT 10", False),
  (r"(risky|risk|bad credit|poor credit).*(customer|client)|(customer|client).*(risk|credit)", "Customers with the highest credit risk.", "SELECT customer_id, name, industry, priority_tier, credit_risk FROM customers ORDER BY credit_risk DESC LIMIT 10", False),
  (r"(storm|rough|dangerous|bad weather|severe)", "Ports with the roughest conditions.", "SELECT port_name, current_severity, forecast, wind_speed, wave_height FROM weather_risks ORDER BY wave_height DESC LIMIT 10", False),
  (r"^(show |list |what are |which are )?(the )?(open |pending |critical |latest )+(tickets?|actions?|alerts?)\W*$", *TICKETS, False),
 ]}
def intent(kind, q, ids, scope=None):
    ql = q.lower().strip()
    for pat, msg in UNSUPPORTED.get(kind, []):
        if re.search(pat, ql): return {"answer": msg, "df": None, "sql": "", "complete": True}
    for pat, title, sql, needs_id in INTENTS.get(kind, []):
        if not re.search(pat, ql): continue
        if needs_id and not ids: continue
        params = (ids[0],) if needs_id else ()
        if scope:                                   # a scoped user (one outlet) only sees their own rows
            if needs_id:
                if ids[0] != scope: continue
            elif "outlet_id" in sql and " WHERE " not in sql and " ORDER BY" in sql: sql = sql.replace(" ORDER BY", " WHERE outlet_id=? ORDER BY", 1); params = (scope,)
            else: continue
        try: df = Q(sql, params)
        except Exception: continue
        if df is None or not len(df):
            if "(IST)" in title: return {"answer": "No records: " + title.rstrip(".").replace(" (IST)", "").lower() + ": none found for today's IST date.", "df": None, "sql": sql, "complete": True}
            continue
        if needs_id: return {"answer": title.format(id=ids[0]) + f" **{df.iloc[0, 0]}**", "df": None, "sql": sql.replace("?", f"'{ids[0]}'")}
        return {"answer": title, "df": df, "sql": sql.replace("?", f"'{params[0]}'") if params else sql}
    return None

def goods_duty(kind, q):
    """Freight: 'customs duty on pharmaceuticals' -> the matching customs rule rows."""
    if kind != "Freight": return None
    m = re.search(r"\b(?:duty|duties|tariff|customs|restricted|documents?|docs)\b.*?\b(?:on|for|of|about)\s+([a-z][a-z ]{3,40})", q.lower())
    if not m: return None
    term = re.sub(r"\b(the|a|an|my|our|goods|cargo|shipments?|items?)\b", "", m.group(1)).strip(" ?.!")
    if len(term) < 4: return None
    like = "%" + term[:7] + "%"
    try: df = Q("SELECT chapter, goods, duty_pct, docs, restricted FROM customs_rules WHERE LOWER(goods) LIKE ? LIMIT 10", (like,))
    except Exception: return None
    if df is None or not len(df): return None
    return {"answer": f"Customs rules matching **{term}**: {len(df)} found.", "df": df, "sql": f"SELECT chapter, goods, duty_pct, docs, restricted FROM customs_rules WHERE LOWER(goods) LIKE '{like}'"}

SMALL = {
 "greet": "Hello! I'm your data copilot. I answer from live database queries, never from guesses. Ask a question, or say **briefing** for the key numbers.",
 "thanks": "You're welcome. Ask me anything else about your data.",
 "bye": "Goodbye! Come back any time.",
 "how": "I'm running well and ready for questions. What would you like to know about your data?",
 "who": "I'm the data copilot of this platform. I turn your question into a read-only SQL query on the live tables and show you the exact rows and the query. I do not use a language model, so I never invent numbers.",
}
def samples(kind):
    try:
        if kind == "Franchise": o = Q("SELECT outlet_id v FROM outlets ORDER BY outlet_id LIMIT 3").v.tolist(); return {"a": o[0], "b": o[1], "c": o[2]}
        sh = Q("SELECT shipment_id v FROM shipments ORDER BY shipment_id LIMIT 1").v.tolist(); q = Q("SELECT quote_id v FROM freight_quotes ORDER BY quote_id LIMIT 2").v.tolist()
        return {"a": sh[0], "b": q[0], "c": q[1]}
    except Exception: return {"a": "OUT-007", "b": "OUT-010", "c": "OUT-020"}
def help_text(kind):
    sm = samples(kind)
    if kind == "Franchise":
        g = {"Rank": ["Top 5 outlets by revenue", "Which stores have the lowest customer satisfaction?", "Worst audit scores"], "Count and average": ["How many staff work in Hyderabad?", "Average salary by role", "Total budget of social media campaigns"],
             "Explain": [f"Why is {sm['a']} flagged?", f"Tell me about {sm['b']}", f"Compare {sm['a']} and {sm['b']}"], "Operations": ["Which items are below reorder level?", "Which staff are most likely to leave?", "Open tickets by severity"], "Overview": ["Briefing", "How is inventory holding up?"]}
    else:
        g = {"Rank": ["Top 5 carriers by on-time", "Which ports are most congested?", "Cheapest carriers"], "Count and average": ["How many shipments are delayed?", "Average margin by carrier", "Total quote value"],
             "Explain": [f"Tell me about {sm['a']}", f"Compare {sm['b']} and {sm['c']}", "Mumbai to Singapore 50 t electronics: days, cost and best route"], "Operations": ["Quotes with margin below 10%", "Where is the weather worst?", "Customs duty for electronics"], "Overview": ["Briefing", "Which shipments are at highest delay risk?"]}
    return "**What I can do**\n\n" + "\n".join(f"- **{k}:** " + " · ".join(f"*{x}*" for x in v) for k, v in g.items()) + "\n\nI understand typos, follow-ups (*and OUT-010?*, *top 10 instead*, *what about Mumbai*), and I can show the SQL behind any answer."

# ------------------------------------------------------------------ presentation
def chart_block(df):
    if df is None or len(df) < 2 or len(df) > 40: return None
    lab = next((c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])), None)
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and not re.search(r"(^id$|_id$|lat|lon|^rank$)", c)]
    if not lab or not num: return None
    c = num[0]; d = df[[lab, c]].dropna().head(15)
    if len(d) < 2: return None
    d = d.iloc[::-1]
    opt = B.bar(d[lab].astype(str).tolist(), {c.replace("_", " "): d[c].astype(float).round(3).tolist()}, horizontal=True)
    return B.chart(opt, title=c.replace("_", " ").title() + " by " + lab.replace("_", " "), h=max(180, 34 * len(d) + 60), open={"bar": True} if "id" in lab else None, key="cp_" + c)

def suggestions(kind, q, res, ids):
    df = res.get("df"); out = list(res.get("sugg") or [])
    first = None
    if df is not None and len(df):
        fcol = None
        for c in df.columns:
            if re.search(r"(^|_)id$", c) and str(df[c].iloc[0]).upper().startswith(("OUT-", "SHP-", "QTE-", "CAR-", "PORT-", "CUST-")): first = str(df[c].iloc[0]); fcol = c; break
        if first and kind == "Franchise" and first.startswith("OUT-"): out += [f"Why is {first} flagged?", f"Tell me about {first}"]
        elif first: out += [f"Tell me about {first}"]
        if len(df) >= 2 and first and len(df) > 1:
            second = str(df[fcol].iloc[1]); out.append(f"Compare {first} and {second}")
        if not TOPN.search(q) and len(df) >= 5: out.append("Top 10 instead")
    elif ids and kind == "Franchise" and ids[0].startswith("OUT-"): out += [f"Why is {ids[0]} flagged?", f"Which items are below reorder level in {ids[0]}?"]
    if not out: out = ["Briefing"]
    seen, r = set(), []
    for x in out:
        if x not in seen and x.lower() != q.lower(): seen.add(x); r.append(x)
    return r[:4]

# ------------------------------------------------------------------ logging / feedback
def _db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;")
    c.execute("CREATE TABLE IF NOT EXISTS copilot_unanswered(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, q TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS copilot_feedback(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, q TEXT, helpful INT)")
    c.execute("CREATE TABLE IF NOT EXISTS copilot_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, q TEXT, route TEXT, ms INT)"); return c
def _log_unanswered(q):
    try:
        with _db() as c: c.execute("INSERT INTO copilot_unanswered(ts,q) VALUES(datetime('now'),?)", (q[:300],))
    except Exception: pass
def log_turn(q, route, ms=0):
    """Every question is logged with the route that answered it: chat, exact (SQL), agents+llm, unanswered. Feeds the Copilot insights page."""
    try:
        with _db() as c: c.execute("INSERT INTO copilot_log(ts,q,route,ms) VALUES(?,?,?,?)", (datetime.datetime.now().isoformat(timespec="seconds"), (q or "")[:300], route, int(ms)))
    except Exception: pass
def insights():
    with _db() as c:
        one = lambda sql: (c.execute(sql).fetchone() or [0])[0] or 0
        out = {"total": one("SELECT COUNT(*) FROM copilot_log"), "avg_ms": one("SELECT AVG(ms) FROM copilot_log WHERE route='exact'"), "up": one("SELECT COUNT(*) FROM copilot_feedback WHERE helpful=1"), "down": one("SELECT COUNT(*) FROM copilot_feedback WHERE helpful=0")}
        out["routes"] = c.execute("SELECT route, COUNT(*) n FROM copilot_log GROUP BY route ORDER BY n DESC").fetchall()
        out["unanswered"] = c.execute("SELECT q, COUNT(*) n, MAX(ts) last FROM copilot_log WHERE route='unanswered' GROUP BY q ORDER BY n DESC, last DESC LIMIT 25").fetchall()
        out["recent"] = c.execute("SELECT ts, q, route, ms FROM copilot_log ORDER BY id DESC LIMIT 40").fetchall()
        out["down_q"] = c.execute("SELECT ts, q FROM copilot_feedback WHERE helpful=0 ORDER BY id DESC LIMIT 20").fetchall()
    return out
def feedback(q, helpful):
    with _db() as c: c.execute("INSERT INTO copilot_feedback(ts,q,helpful) VALUES(datetime('now'),?,?)", (q[:300], 1 if helpful else 0))
def unanswered(n=50):
    with _db() as c: return [{"ts": a, "q": b, "times": k} for a, b, k in c.execute("SELECT max(ts), q, COUNT(*) FROM copilot_unanswered GROUP BY q ORDER BY COUNT(*) DESC, max(ts) DESC LIMIT ?", (n,))]

def suggest(prefix, examples, n=7):
    p = prefix.strip().lower()
    if len(p) < 2: return []
    V = _vocab(); kind = _kind(); pool = list(examples)
    for t, (syn, idc, cols, met) in QA.TABLES[kind].items():
        if t == "agent_actions": continue
        nm = syn[0]; pool += [f"How many {nm}s are there?", f"Top 10 {nm}s"]
        if met: pool += [f"Which {nm} has the highest {met.replace('_', ' ')}?", f"Which {nm} has the lowest {met.replace('_', ' ')}?", f"Average {met.replace('_', ' ')}"]
    pool += ["Briefing", "What can you do?", f"Compare {samples(kind)['b' if kind != 'Franchise' else 'a']} and {samples(kind)['c' if kind != 'Franchise' else 'b']}"]
    words = p.split(); sc = []
    for x in pool:
        xl = x.lower(); s = (3 if xl.startswith(p) else 0) + sum(1 for w in words if w in xl)
        if s >= max(1, len(words) - 1) and (p in xl or s == len(words)): sc.append((-s, len(x), x))
    r = [x for _, _, x in sorted(set(sc))][:n]
    if len(words) == 1 and len(p) >= 3:
        for v in V["corpus"]:
            if v.lower().startswith(p) and len(r) < n: r.append("Tell me about " + v)
    return r

# ------------------------------------------------------------------ main entry (used by the full app)
SENS = ("salary", "age")
def _mask(df, mask):
    if df is None or not mask: return df
    drop = [c for c in df.columns if c in ("salary", "age")] + (["name"] if "staff_id" in df.columns else [])
    return df.drop(columns=drop, errors="ignore")
def prepare(kind, raw, sid, dom, dec_fn, examples, scope=None, mask=False):
    """Everything that happens BEFORE the heavy pipeline.
    Returns (direct, q2, res, notes, ids): direct = an instant reply (small talk, help, show-SQL) or None;
    q2 = the question after spelling repair and follow-up resolution; res = an exact SQL answer from a smart intent or None."""
    s = session(sid or "default"); raw = (raw or "").strip()[:800]
    def D(text, **k): return {"text": text, "sql": k.pop("sql", ""), **k}
    if not raw: return D("Type a question about your data, or say **briefing**.", sugg=["Briefing", "What can you do?"]), raw, None, [], []
    if GREET.match(raw): return D(SMALL["greet"], sugg=["Briefing", "What can you do?"] + examples[:1]), raw, None, [], []
    if BYE.match(raw): return D(SMALL["bye"]), raw, None, [], []
    if THANKS.match(raw): return D(SMALL["thanks"], sugg=["Briefing"]), raw, None, [], []
    if HOWARE.search(raw) and len(raw.split()) < 7: return D(SMALL["how"], sugg=["Briefing"]), raw, None, [], []
    if WHO.search(raw): return D(SMALL["who"], sugg=["What can you do?", "Briefing"]), raw, None, [], []
    if TIME.search(raw) and len(raw.split()) < 8:
        n = datetime.datetime.now(); return D(f"It is **{n:%A, %d %B %Y}**, **{n:%I:%M %p} IST** (India Standard Time).", sugg=["Briefing"]), raw, None, [], []
    if HELP.match(raw): return D(help_text(kind), sugg=["Briefing"] + examples[:2]), raw, None, [], []
    if SHOWSQL.search(raw) and s["sql"]: return D("This is the exact query behind my last answer:\n\n```sql\n" + s["sql"] + "\n```", sql=s["sql"]), raw, None, [], []
    try: _t = taught_match(raw)
    except Exception: _t = None
    if _t: return D(_t["answer"] + "\n\n**Source:** taught answer, added by " + str(_t["by"]) + " on " + str(_t["when"])[:10], sugg=["Briefing"] + examples[:1], route="taught"), raw, None, [], []
    fixed, changes = correct(raw); q2, fnote = contextualize(fixed, s); notes = []
    if changes: notes.append("Spelling: " + ", ".join(f"{a} -> {b}" for a, b in changes))
    if fnote and fnote != fixed: notes.append("Follow-up understood as: " + fnote)
    ids = [i.upper() for i in ID_RE.findall(q2)]; res = None
    if mask and kind == "Franchise" and re.search(r"\b(salary|salaries|wage|wages|payroll|pay|age)\b", q2, re.I) and re.search(r"\b(staff|employees?|workers?|people|team|role)\b", q2, re.I):
        return D("Salary and personal details are visible to the Admin, Franchise Owner and Operations roles only.", sugg=["Briefing"]), raw, None, notes, ids
    if scope:                                              # a scoped user (one outlet): only simple intents, filtered to their outlet; everything else uses the scope-aware engine
        try: r0 = intent(kind, q2, ids, scope)
        except Exception: r0 = None
        if r0 is not None: r0["df"] = _mask(r0.get("df"), mask); r0["complete"] = True
        return None, q2, r0, notes, ids
    if mask and any(i.startswith(("STF", "EMP")) for i in ids): return None, q2, None, notes, ids
    try:
        if ids: res = intent(kind, q2, ids)
        if res is not None: pass
        elif BRIEF.search(q2) and not ids and len(q2.split()) < 9: res = briefing(kind, dec_fn(), dom)
        elif len(ids) >= 2 and COMPARE.search(q2): res = compare(ids)
        elif len(ids) == 1 and scoped(kind, ids[0], q2) is not None and not re.search(r"\b(why|flag|risk|reason|explain)\b", q2, re.I): res = scoped(kind, ids[0], q2)
        elif len(ids) == 1 and len(q2.split()) <= 6 and not re.search(r"\b(why|flag|risk|reason|explain)\b", q2, re.I): res = profile(kind, ids[0])
        if res is None and not ids: res = intent(kind, q2, ids) or goods_duty(kind, q2)
    except Exception as e:
        import sys; print("copilot_pro failed:", repr(e), file=sys.stderr); res = None
    if res is not None:
        res["df"] = _mask(res.get("df"), mask); res["complete"] = True
        if mask and res.get("answer") and "salary=" in res["answer"]: res["answer"] = re.sub(r"salary=[^;\n]*;?\s*", "", res["answer"])
    return None, q2, res, notes, ids

def decorate(kind, sid, q2, res, ids):
    """After an exact answer exists: remember it (for follow-ups, CSV and show-SQL) and build the chart and the suggestion chips."""
    s = session(sid or "default"); df = res.get("df")
    s["q"] = q2; s["ids"] = ids or s["ids"]; s["df"] = df; s["sql"] = res.get("sql", ""); s["turns"] += 1
    cb = None if res.get("nochart") else chart_block(df)
    return (B.clean(cb) if cb else None), suggestions(kind, q2, res, ids), bool(df is not None and len(df))

_KQ = re.compile(r"\b(what is|what are|what's|how do|how to|how should|how can|policy|procedure|rules?|explain|define|meaning|difference|steps|guidelines?|requirements?|process|checklist|standards?|why do|who is responsible)\b", re.I)
_SW = set("what is are the a an of to in on for and or how do does should can i we you my our your with about this that these those why who when which be it its into per".split())
def _sentences(text):
    t = re.sub(r"\s+", " ", text or "").strip()
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", t) if len(x.strip()) > 25]
def _short(x, n=28):
    w = x.split(); return x if len(w) <= n else " ".join(w[:n]).rstrip(",;:") + "..."
def src_label(h):
    s = str(h.get("source") or "").replace("\\", "/"); s = s.rsplit("/", 1)[-1] if "/" in s and not s.startswith("http") else s
    return f"{h.get('title', 'Document')}" + (f" · {s[:60]}" if s and s != h.get("title") else "")
def sources_block(hits, cited=None):
    """Numbered source list that matches the [n] markers in the answer."""
    ids = sorted(cited) if cited else list(range(1, min(len(hits), 3) + 1))
    rows = [f"[{n}] {src_label(hits[n - 1])}" + (f" (match {hits[n - 1]['score']:.2f})" if isinstance(hits[n - 1].get("score"), (int, float)) else "") for n in ids if 1 <= n <= len(hits)]
    return ("**Sources**\n" + "\n".join(rows)) if rows else ""
def finish(ans, hits, max_points=3):
    """Tidy an LLM answer: Answer line, up to 3 numbered points, optional Next step; every cited [n] is listed in a Sources block."""
    ans = (ans or "").strip()
    if not ans: return ans
    lines = [l.rstrip() for l in ans.split("\n") if l.strip()]
    head = [l for l in lines if not re.match(r"\s*(\d{1,2}[.)]|[-*•])\s", l) and not l.lower().lstrip("* ").startswith("next step")][:2]
    pts = [l for l in lines if re.match(r"\s*(\d{1,2}[.)]|[-*•])\s", l)][:max_points]
    nxt = [l for l in lines if l.lower().lstrip("* ").startswith("next step")][:1]
    pts = [re.sub(r"^\s*(\d{1,2}[.)]|[-*•])\s*", f"{i}. ", l) for i, l in enumerate(pts, 1)]
    body = "\n".join(head + pts + nxt)
    cited = {int(x) for x in re.findall(r"\[(\d{1,2})\]", body) if 1 <= int(x) <= len(hits or [])}
    sb = sources_block(hits, cited) if hits and cited else ""
    return body + (("\n\n" + sb) if sb else "")

def kb_answer(q, hits, max_points=3):
    """No GPU model needed: a short, ordered answer built from the best sentences of the best passages, every line cited, then the Sources block. None when the passages do not match."""
    if not hits or not (_KQ.search(q) or is_term_q(q)): return None
    terms = {w for w in re.findall(r"[a-z]{3,}", q.lower()) if w not in _SW and w not in _S3}
    if not terms: return None
    key0 = {w for w in terms if w not in _GEN and w.rstrip("s") not in _GEN}
    _gen = {"policy", "policies", "customer", "customers", "rule", "rules", "process", "standard", "guide", "tell", "about", "explain", "describe", "limit", "limits"}
    key = {w.rstrip("s") for w in terms - _gen} or terms
    _how = bool(re.match(r"\s*(how|what (should|do|to)|steps)", q.lower()))
    cand = []
    for n, h in enumerate(hits[:3], 1):
        body = (h.get("title", "") + " " + h.get("text", "")).lower()
        if sum(1 for w in (key0 or terms) if w in body) < min(2, len(key0 or terms)): continue
        for k, sen in enumerate(_sentences(h.get("text", ""))):
            sc = sum(2 for w in key if w in sen.lower()) + sum(0.5 for w in terms - set(key) if w in sen.lower()) + (0.4 if re.search(r"\d", sen) else 0) + (1.5 if _how and re.search(r"\b(must|should|escalat|recover|call-?back|review|within|trigger|approv|follow)", sen.lower()) else 0) - k * 0.15 - (n - 1) * 0.4
            if sc > 0: cand.append((sc, n, k, sen))
    if not cand: return None
    cand.sort(key=lambda x: -x[0]); picked = sorted(cand[: max_points + 1], key=lambda x: (x[1], x[2]))
    lead = cand[0]; rest = [c for c in picked if c is not lead and c[0] >= 2][:max_points]
    out = [f"**Answer:** {_short(lead[3], 34)} [{lead[1]}]"] + [f"{i}. {_short(c[3], 26)} [{c[1]}]" for i, c in enumerate(rest, 1)]
    cited = {lead[1]} | {c[1] for c in rest}
    return "\n".join(out) + "\n\n" + sources_block(hits, cited)

def relevant(q, hits, need=2):
    """Keep only retrieved passages that really share content words with the question (so a plain SQL question does not show unrelated sources)."""
    terms = {w for w in re.findall(r"[a-z]{3,}", q.lower()) if w not in _SW and w not in _S3}
    if not terms: return []
    key = {w for w in terms if w not in _GEN and w.rstrip("s") not in _GEN}
    if _KQ.search(q) and key: terms, need = key, min(2, len(key))      # generic words (policy, process, outlet...) do not count as evidence
    else: need = 2 if _KQ.search(q) else 3
    out = []
    for h in hits:
        txt = (h.get("title", "") + " " + h.get("text", "")).lower()
        if sum(1 for w in terms if w in txt) >= min(need, len(terms)): out.append(h)
    return out

# ------------------------------------------------------------------ teaching: answers added by the admin / interns for questions the Copilot could not answer
def _tdb():
    c = _db(); c.execute("CREATE TABLE IF NOT EXISTS copilot_taught(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, q TEXT, answer TEXT, keywords TEXT, by TEXT)"); return c
def _tk(x):
    return {w[:-1] if len(w) > 4 and w.endswith("s") else w for w in re.findall(r"[a-z0-9]{3,}", (x or "").lower()) if w not in _SW and w not in {"the", "and", "for", "are", "you", "can", "how", "what", "who", "why", "this", "that", "with", "does", "from", "our", "any"}}
_S3 = {"the", "for", "and", "are", "you", "can", "its", "get", "any", "has", "was", "not", "our", "all", "out", "off", "use", "new", "low", "high", "per", "why", "who", "how"}
_GEN = set("outlet outlets franchise franchises staff store stores employee employees shipment shipments freight carrier carriers port ports quote quotes customs route routes company new need needed policy policies process procedure rules steps guidelines guideline requirements requirement checklist standard standards explain define meaning difference tell about give show list which have there their each every between using used make made take work works working please currently current right data highest lowest best worst most least average total count many much more less than want know like also only just over under after before".split())
def weak_match(q, res):
    """True when none of the question's key words appear anywhere in the answer, the SQL or the table headings: a confident but unrelated answer."""
    kt = [w for w in re.findall(r"[a-z]{3,}", (q or "").lower()) if w not in _SW and w not in _GEN and w not in {"the", "for", "and", "are", "you", "can", "its", "get", "any", "has", "was", "not", "our", "all", "out", "off", "use"}]
    if not kt: return False
    df = res.get("df"); hay = str(res.get("answer", "")) + " " + str(res.get("sql", ""))
    try: hay += " " + " ".join(map(str, df.columns)) + " " + df.head(5).to_string()
    except Exception: pass
    toks = set(re.findall(r"[a-z]+", hay.lower()))
    def seen(w): return w in toks or w.rstrip("s") in toks or (w == "csat" and "satisfaction" in toks) or (len(w) >= 6 and any(t.startswith(w[:5]) for t in toks))
    return not any(seen(w) for w in kt)

def teach(q, answer, keywords="", by="admin"):
    """Save a question + the answer you want. It is matched (also for rephrased versions) before the other layers, and indexed into the knowledge base."""
    q = (q or "").strip()[:300]; answer = (answer or "").strip()[:3000]
    if len(q) < 4 or len(answer) < 10: return None
    with _tdb() as c:
        old = c.execute("SELECT id FROM copilot_taught WHERE lower(q)=lower(?)", (q,)).fetchone()
        if old: c.execute("UPDATE copilot_taught SET answer=?, keywords=?, by=?, ts=? WHERE id=?", (answer, (keywords or "")[:300], by, datetime.datetime.now().isoformat(timespec="seconds"), old[0])); tid = old[0]
        else: tid = c.execute("INSERT INTO copilot_taught(ts,q,answer,keywords,by) VALUES(?,?,?,?,?)", (datetime.datetime.now().isoformat(timespec="seconds"), q, answer, (keywords or "")[:300], by)).lastrowid
    try:
        import rag as _rg; _rg.delete_doc("Taught: " + q[:80], None); _rg.add("Taught: " + q[:80], "Taught answer (added by " + by + ")", q + "\n" + answer + ("\nKeywords: " + keywords if keywords else ""), None)
    except Exception: pass
    return tid
def taught_rows():
    with _tdb() as c: return [{"id": a, "when": b, "question": q, "answer": an, "keywords": k, "by": by} for a, b, q, an, k, by in c.execute("SELECT id, ts, q, answer, keywords, by FROM copilot_taught ORDER BY id DESC")]
def taught_del(i):
    with _tdb() as c:
        r = c.execute("SELECT q FROM copilot_taught WHERE id=?", (i,)).fetchone(); c.execute("DELETE FROM copilot_taught WHERE id=?", (i,))
    if r:
        try: import rag as _rg; _rg.delete_doc("Taught: " + r[0][:80], None)
        except Exception: pass
    return bool(r)
def taught_match(q):
    qt = _tk(q)
    if not qt: return None
    best, bs = None, 0.0
    try: rows = taught_rows()
    except Exception: return None
    for r in rows:
        if r["question"].strip().lower() == (q or "").strip().lower(): return r
        tt = _tk(r["question"]) | _tk(r["keywords"]); inter = len(qt & tt)
        if inter < min(2, len(qt)): continue
        sc = inter / len(qt | _tk(r["question"]))
        if sc > bs: best, bs = r, sc
    return best if best and bs >= 0.6 else None
def backlog(n=300):
    """Distinct questions that went unanswered (not yet taught), most asked first."""
    with _db() as c: rows = c.execute("SELECT q, COUNT(*) n, MAX(ts) last FROM (SELECT q, ts FROM copilot_log WHERE route IN ('unanswered','weak') UNION ALL SELECT q, ts FROM copilot_feedback WHERE helpful=0) GROUP BY lower(q) ORDER BY n DESC, last DESC LIMIT ?", (n,)).fetchall()
    return [{"question": q, "times": k, "last_asked": str(l).replace("T", " ")} for q, k, l in rows if not taught_match(q)]
def teach_stats():
    with _db() as c:
        tot = c.execute("SELECT COUNT(DISTINCT lower(q)) FROM (SELECT q FROM copilot_log WHERE route IN ('unanswered','weak') UNION ALL SELECT q FROM copilot_feedback WHERE helpful=0)").fetchone()[0]
    return {"found": tot, "open": len(backlog(500)), "taught": len(taught_rows())}

def sane(t):
    """False for degenerate model output (only quotes/newlines, repeated tokens, foreign-script noise)."""
    t = (t or "").strip()
    if len(t) < 8: return False
    core = re.sub(r"\s", "", t)
    if sum(c.isascii() and c.isalnum() for c in core) < 0.55 * len(core): return False
    ws = re.findall(r"[A-Za-z]{2,}", t)
    if len(ws) < 3 or len(set(w.lower() for w in ws)) < 3: return False
    if re.search(r"(.{1,6})\1{5,}", t): return False
    return True
_DATA_WORDS = re.compile(r"\b(outlets?|stores?|sales|revenue|stock|inventory|staff|attrition|shipments?|ports?|vessels?|quotes?|carriers?|customers?|campaigns?|audits?|tickets?|approvals?|agents?|risk|health|top|bottom|how many|count|average|total|list|show)\b", re.I)
def is_term_q(q):
    """A bare term or very short phrase with no data words (FSSAI, HACCP, demurrage): look it up in the documents, skip the agents."""
    w = re.findall(r"[A-Za-z][A-Za-z0-9\-]*", q or "")
    return 1 <= len(w) <= 4 and not re.search(r"\d", q or "") and not _DATA_WORDS.search(q or "")
def is_knowledge_q(q): return bool(_KQ.search(q))

def unanswered_note(raw): _log_unanswered(raw)

def respond(kind, q, sid, dom, dec_fn, examples):
    """Standalone version (used by the lite app): prepare, then the exact engine only."""
    t0 = time.time(); ms = lambda: round((time.time() - t0) * 1000)
    direct, q2, res, notes, ids = prepare(kind, q, sid, dom, dec_fn, examples)
    if direct: log_turn(q, "chat", ms()); yield {"ev": "final", "ms": ms(), **direct}; return
    if notes: yield {"ev": "note", "text": " · ".join(notes)}
    try:
        if res is None: res = goods_duty(kind, q2) or AN.answer(kind, q2, None, dec_fn(), dom) or QA.answer(kind, q2, None, None)
    except Exception as e:
        import sys; print("copilot_pro failed:", repr(e), file=sys.stderr); res = None
    if not res or not res.get("answer"):
        _log_unanswered(q); log_turn(q, "unanswered", ms())
        yield {"ev": "final", "text": "I could not compute an exact answer to that from the tables. I would rather say so than guess.\n\nYou could try:\n" + "\n".join("- " + e for e in examples[:4]) + "\n\nOr say **what can you do**.", "sql": "", "ms": ms(), "sugg": ["What can you do?", "Briefing"]}; return
    cb, sg, has = decorate(kind, sid, q2, res, ids); df = res.get("df"); log_turn(q, "exact", ms())
    yield {"ev": "exact", "sql": res.get("sql", ""), "rows": 0 if df is None else len(df)}
    if cb: yield {"ev": "chart", "block": cb}
    tbl = (QA.md(df) if df is not None else "") + (("\n\n" + res["extra"]) if res.get("extra") else "")
    yield {"ev": "final", "text": res["answer"] + (("\n\n" + tbl) if tbl else ""), "sql": res.get("sql", ""), "ms": ms(), "rows": 0 if df is None else len(df), "csv": has, "sugg": sg}


In [ ]:
%%writefile /content/app/persist.py
"""Keeps the app database (users, tickets, approvals and every saved Copilot chat) safe on Google Drive.
The live SQLite file stays on the fast local disk; a consistent copy is written to Drive (SQLite online-backup, temp file then atomic rename)
every few minutes, a few seconds after each chat is saved, and at shutdown. On the next start, if the local database is gone (Colab runtime reset),
the Drive copy is restored before the app opens it. Set DRIVE_DB_DIR to the Drive folder; unset = disabled."""
import os, shutil, sqlite3, threading, time, json
import config

DIR = os.environ.get("DRIVE_DB_DIR", "").strip()
EVERY = int(os.environ.get("DB_BACKUP_EVERY", "180"))
_state = {"last": None, "ok": None, "err": "", "size": 0, "restored": False, "n": 0}
_lock = threading.Lock(); _timer = [None]; _stop = threading.Event()
_EXTRA = (".jwt_secret", ".admin_initial")

def enabled(): return bool(DIR)
def _target(): return os.path.join(DIR, os.path.basename(config.DB_PATH))
def _sig():
    try: return tuple(os.stat(config.DB_PATH + s).st_mtime_ns if os.path.exists(config.DB_PATH + s) else 0 for s in ("", "-wal"))
    except Exception: return None

def restore():
    """Call before the database is opened. Copies the Drive backup back when the local file is missing."""
    if not enabled(): return False
    try:
        t = _target()
        if os.path.exists(t) and not os.path.exists(config.DB_PATH):
            os.makedirs(os.path.dirname(config.DB_PATH), exist_ok=True)
            shutil.copy2(t, config.DB_PATH)
            for x in _EXTRA:
                s = os.path.join(DIR, x)
                if os.path.exists(s) and not os.path.exists(os.path.join(config.DATA_DIR, x)): shutil.copy2(s, os.path.join(config.DATA_DIR, x))
            _state["restored"] = True; print("persist: restored database from", t)
            return True
    except Exception as e: _state["err"] = repr(e)[:160]; print("persist restore failed:", repr(e))
    return False

def backup(force=False):
    if not enabled() or not os.path.exists(config.DB_PATH): return False
    with _lock:
        sig = _sig()
        if not force and sig is not None and sig == _state.get("sig"): return True
        try:
            os.makedirs(DIR, exist_ok=True); t = _target(); tmp = t + ".tmp"
            if os.path.exists(t) and (not os.path.exists(t + ".prev") or time.time() - os.path.getmtime(t + ".prev") > 1800):
                try: shutil.copy2(t, t + ".prev"); os.utime(t + ".prev", None)        # an older generation (refreshed at most every 30 min), in case the latest copy is ever bad
                except Exception: pass
            src = sqlite3.connect(config.DB_PATH, timeout=30); dst = sqlite3.connect(tmp)
            try: src.backup(dst)
            finally: dst.close(); src.close()
            os.replace(tmp, t)
            for x in _EXTRA:
                s = os.path.join(config.DATA_DIR, x)
                if os.path.exists(s): shutil.copy2(s, os.path.join(DIR, x))
            _state.update(last=time.strftime("%Y-%m-%d %H:%M:%S"), ok=True, err="", size=os.path.getsize(t), sig=sig, n=_state["n"] + 1)
            return True
        except Exception as e:
            _state.update(ok=False, err=repr(e)[:160]); print("persist backup failed:", repr(e)); return False

def soon(delay=5):
    """Debounced backup (used after a chat is saved)."""
    if not enabled(): return
    if _timer[0] and _timer[0].is_alive(): return
    _timer[0] = threading.Timer(delay, lambda: backup()); _timer[0].daemon = True; _timer[0].start()

def _loop():
    while not _stop.wait(EVERY): backup()

def start():
    if not enabled(): return
    threading.Thread(target=_loop, daemon=True).start(); threading.Timer(15, lambda: backup(True)).start()

def stop(): _stop.set(); backup()

def status():
    """(ok, detail) for the System check page."""
    if not enabled(): return None, "Not enabled. In the launcher cell keep PERSIST = True so the database (chats, users, tickets) is copied to Google Drive."
    if _state["ok"] is False: return False, "Last backup failed: " + _state["err"]
    if not _state["last"]: return None, f"Waiting for the first backup to {DIR} (runs ~15 s after start)."
    return True, f"{_state['n']} backup(s), last at {_state['last']} IST, {_state['size'] / 2**20:.1f} MB in {DIR}" + (" (restored from Drive at start)" if _state["restored"] else "")


In [ ]:
# Front-end libraries (ECharts, Leaflet, TopoJSON, world map), bundled so nothing loads from a third-party CDN
import base64, gzip, os
V={
  'world.topo.json': '...',
  'echarts.min.js': '...',
  'topojson-client.min.js': 'H4sIAH622WoC/81ZzXLjuBG+71PIOqiIFSRLM7mENOzabG1yyVS2KrmxWFMwDUrcoQEFBHfXsbn3PEheLE+SrwH+yvLMpCqHXCwSBBqN7q+/brSvrxdH5051fH19KN2xud/m5vHamZP5qTZ6eNjkVam0W/z8frvf7hbfm9OTLQ9Ht3i32/9+8aH8pBZ/MLUz+advropG5640OlLcseeluf9J5W4phHs6KVMs1K8nY129Wi0b/aCKUquH5VX/8dE8NJW6c1E3i8XLXtwoIaxarcLvVj4+3IXHKF1265YZ9o5dFCmhXl5qVRVs259FqOHx5eW5ZW3kjmXNR7WhdFOrRe1sCcWT/sPC0SerXGP1QrXDsKXhsoh0U1VCKNbNcMnP0i4s19xgxzqXlUp3GZfDyz7jJelipa4r6fzXfDawz5JO2Nym7uUlskKLHfN7NOIdL7CwUvrgjrwWWv2y+M5a+RQVLCmMjWrIFpFdC9qEfWvWJa8hXUSahvYYkus8aW6KhNVpk2Gsyfh63fTb1+14XG8g2tZxLXD2oC92eWQ46f56hyManGxjcJqNGc3X0EI4hAQw6HFjVqvIBJU4/b0tMVAOA/vsRmJABg39wG2OgTwMjBoVJLj+pXT5kdQBSNhzLmu1/JMyj8rZp+9NVakAolhtD2G0VPUWWv8gsQpmurdKfkr8sh9Nqd0yhr4IBmMfSg1n1LMpH5rKld282axBZMNgM5jeLUqgZSttPtluAjWaE6xpxWbPm8GPyXptb5qERY4sltqM26nZnLeSG83WDwxmc95sbjSbH2hhyW2IyZrBcv1z6jLWuTs1XPKS59loYhOA5xEt3LZ8gO/d9v7e/AqfB+C77cmak7IOhr17buPpO+AgvYQeUGGJXa3Cg757JrfFyz8qie9qyce1seGdx57iso3fWlA+xPbtVRdnk/qxfnPReHo5nv4C5INzk/PZrvPjagVLmFMUIjFY0KTqZnf3m4oVEcIO5rG91+VNCc9LhkVNfYx0ZFOJSYwlWLJazYlgkMj1GP9wyMYl5maz0QmzCBaTIXjMek1xrelZZ8KC9Xg5CaJyQm4EzPFLPoOpSAFEqDxumNgbTWBlMsB09HJvgpt33gh0ng6l7ZwVRum0WTKs+13CpusGuWex33Py9lGeKPDOSHOhoh67HraEhaTjC/0ZqghiAnQ0H2kD0J5wCO0Jc82ow4oycl9FHhRNU/ogaeVs9p+R2P6KrfSBZueQS3B7LXA+L8zy4vIzcqueDv50onhT1mTSRNDAksi2EtPiSTC3c2NNjhTbFlibI23kkucWLsEfQ38kgQuZY59MwTcFuvZsGhYbD2coUymnFi4129pJ6zLejXTv46vSD9xcJGE7xqPYgyFlAJ0Bhw9QfrUOYOoU8UmdzJRaEmNjmyWyx/D71epKgnkB4P5pn3UUvl6XiJe0RDziB2FpiJ8v7DXgF8QsXo/qfn9H+7sYRgALYMdkwlV3qBnSHUeVod8wwi4LFQK3UDFUBi1jAIFOdXeczViT0F6p5SaLkTBsRk5Gxi9pfYMfTERRRFyXZwxPvRtSS47AHl1gM+4HRAMratQd7LmbqVPZedRXOYWQAinjDtZFxOhcukiyRKdFmAQGDZOxYeEFSr+PKFpV1SRtMiGoIGziP3k1/dajmuN0qNfo+lgWLugadsupyqGTPQ/nCvt5XetBV9nraknXulstR1Vqr2unUP1FXXVKXkSu7vUYpoiGWB0uiAyAiR8UnhexNMf7y0sH9lT54kC289hD1RESHwfMyaPSHppH3AXqDhG3e6bFNC1NVlA8U1An8zQTqL5XIYkkDoSSln4wmbGg0HMZK36IdXuei9RwpmkCa+af8jfSQM/7wnH3pVrRXaoV1Zvs/HXcPGfh5sscfDX1nJoWmP1aT7Agrf5TV13dTReaYFQK7y3MFs/gEIYPKA3UGOfbA1utzpah3mmjnfdxwCMRs/HlwHjp6NmwE8VQi0gUCIahsgRck2mqeGUfTitjyhGatTPvniWNNPP4St7CxpQlLzMqSisnBsakO9UcgMjqdGI9vE20cUPV8ZliflaUIVnBqFTu+coetZnDPYnGUTPJta/fvzWgzQ3V6XjaDUz7QbrjVt7XILw2ovKys10PkWAzKN+yaTFBJdPb+WsWDv+LUBgAm38Z025+IUKwekNf4ipQzpXafuxZA26H71Xmq2mMiz03fSZJcLnsau2+blR03bkkVn2JGN9Y9RE4CdsOuPAwaT9zhC5FYBmmTHE/d6AJVdZZdg80Sszr/ouDfPl4Y2lupnCHhiBHxDSFoEEodEEMmu/RLXnOCxigFk1kfF1e+BtAwRD6NFQghG5rf+ek75z+UFqmpyJD7kR+HIp5A9AWZeWUnava1/Rdltm1Uzoozu4/ry8kfY1o1/r29naf0DXoxt2hHlnvYy1MHxh2FFr/XzdyeCU8D1jT6IfIM/KmZNfI8cezD6CQnF3LSeun2lgwUhWaPseNhrGOn2/3KH+5F5qiJ1yaxVz1MHVZB9YeenMOXkdqn3YV+CVCEaK7iJ1d5CdTeLfxhfvWNEZ6GHm1AJK4e4Lmj8oepnoPk4lDm608naqn0Pwbiho2LvwOMYmqlN7q41tS8s9KqY9eSI4XrcrD8d7YeiYpcBryGe7p85P1u6Qg9WTafKPy6o2YpwZBpMRvKsAJNQ4Rpb7rmMqymAYEEs60pup6O5dl6t6sPvDF85ipccs7y92x4R2h4WnKb/EcOpd3GhzYDo2SMvTOLmUussLncXUxY02BE6S0MIpfQHvJ1QpFqH/NuiTmp0CxXidqo5V0w8nHMN0l9U0DAqyHWZWo1/uk8qNVcPORn0Se1hk/4KfKkugobHrKWIoLXXTkB5ZdCXFYrY7b+lSVuYpKvsMo9/MOw7yTn3c6m3diI58BbH9vpHblP85jtixmvSt3tOaXBXHMD9ZC8aWskK8fnhb98ocl83mHujee6hi1J4kYfPHnU3MEAvP0U1SG2p1gfvGOvZatF49N7Rb3avHvf/7r3ZJ1pGpE1Mt8eek7w+DWJrBqk74jLm3S91mCOPFKxGm5MXcR/rDryG72LN7zfCPvIvwZRnCt7tmXrqcyaz2CG16AR2sql48jS8Ezz+0YZIdJjFezuu9piNnka/vN0DoQ3IWP056SOue4J9ZeaEoP4sL7rNFyOGtYt281nQYhk8GZJPWqJ3UYpJ01f1Q76QJdwablA6UA/PhnlETdOLnYf/GJJbyPX8cmrJ8zvs6++c4h0FRQsB7ZiaqJp+hIBcf8VvE3czKVOTwtQ5O34QPqY8c7r8enUHapSYfrnJq7Vufep/4u3uUkLZuQZKVPsj4nc+Ryqu9NwiJkwmraun958S146s6nmvqxY6t0D+LS4wffp6ErXdgTeVgSM/JJ+AqLt0aP7zX/iz/YNvwz7MdgNUCWLz9+VPUH/6+1JcfBqkbFV1RQseSb/wDodouGARwAAA==',
  'leaflet.js': '...',
  'leaflet.css': 'H4sIAH622WoC/70ba2/ktvFzAuQ/MD6ksYPVerX2+rFG2qZ5oFfcFUGbBv1WUBJ3xVgrqnp41z74v3f4kkiKelwS9Az7VuRoOJz3DLmXX6GS/LehJUlQVT9npEJfXX726WefLjOCdxmpgwLnZGE81zSzng+4fCRlQGOWe4arFCfs6L4fAHCNaU7KhbMS+iOqnvae0RjnT7gyJ14YOwQRO5lj9ID3JMjws41ZDKAPn336ScEqWlOWbxGOKpY1NXmA0Yzs6i1a8Y81K9SnVwNBS69Awp5IucvYcYtSmiQkd6F/DYsE4uBIokdaB03Fx0lGYiArZ7kgEiEUHNjL4KT4NzBpIU5KvO8mgfDLr9CPJXkieV2ht9+HIdqV7AB726cZ/NY03yO+pQrRHEVZQ4SKWLvdbuWawFmxkQjHj/uSNXmyRXWJ86rAJaCH9eRy/8Q7XFJQvTwhZcVpCUoCC2GxEAIs6jEidQ1cP9I6RXVKqwWKmhp9mwKBBNEKHVlZOfRUErdFnyBKKodcE/a0bbnCipoe6ItUyxJXdceXFDYCC+MaFZpD6REJfarQWVUDmXEKyM7QMSU5yhhOOLtyclQsmyTN0a0jTep0i8Kb1ao4cTpSwmVgjmiyBWN3rDwErKR7Cjq96mvubN1LaFXAvrYgYRY/dhzw2ABY6BYEVJGaf0QHfAoE2SghcUZKLNSgSmlRgFcBnfkbWGqGP0fnv4gPS1buL9DV8iSY41+CHmCJAy64jKOSPQJvAdEPPyC6Q8+sAbk8kW5h4M5q9QVXGw/TO6TtEDdg2Kv0LXwLnAEGNm4b6HN6KFhZY6G2YlaLwjP9Or6gYrdYD7a2GIeWUpkLLZRoHBYmplzkAOLfzhn1Im5qxh8LnCTC+LSmThLcksH9BiHgdOu6qLaXl1Gzr5YxdwW0OXCluiwu9eMlraqGVJcJAYzZn2jyNZhPuF4J1fjkQE9BlIEjAIeaAMYia6pA+DpSSh/lo6qjiDVxKuKP9NqHSg3hWMYWkEZwEn+fZUwZmXydsRT32ENL0VzR8iBZ9APOMu5+EXgGbjDHlMZgmYxU+Zdgr03BZWO8JjliI9WxYWSpuWTPZ5p/UTU6HIxbb4iLoA1ZAJOx0hN7PEjwNJpyH+HzTbhA4R38Wa/vF2i1vL5wULaKuqMZKNIWPFYKkUZYwROtaEQzWj+PJQ0Bjx/gMz84bxiIXj0ZkBk4VmbMEA8iaQCooKIvwvIiVkIE5ENt5jA8/RJQiJenLbpbrbqgAGH3EYv4LtTMsMgXmoGHhyXF/9woq5Qd/wMzy3hPuSHe3d1dhfe2j/Y65OFs57WXoer8B31ALcXXQDGyQTtfaYOuFaifIA9Or682ATcuoBkCTMAbF7BmLKtpock0QTcOaMGKxgd42985xFIJKBNpEzrsk1p0kvAzyoB+OmScCwWx8hcneZGPYhGAlzkigSBOuXk1ZXb+JiE73GT1m5/fv+OG1eUjNM/AToM2LfGn8IImoZsiiWMZ0nA8H3NqGg1iVwQlySB1eSI+tS8YzcGmA5kDbpGwzoz8iPlw8oBg3bffo/sgXLWuVqQoPOxJD+ti0AHRFn1hxuSI1bVyl/5NmyL0kmkYjLmIQOmvdEouMDFfGj7k1U+V/OiB4QWVgDAqq9cBAUAlhQGGQ3IwyCBxyb1QnfppG0MiIHwb9r4EJglpcyAYEa4MFbV3Ovau5oDvdcGEsZclc3yvjuxUvaukY7ys1F9EWMwdM4QS+AD5HlecXl3PpwM+jWx3IivcAsci8qx6BYdSQwUBcXBdIW6guDSL1Hmg4t8M0Ndxult/NbaT0Bs9JX9UyJ0srJDYXjUO0O5poDQDpzpCwxHiZhCnON+TbYeml5S2L6K5+1Ec7u2R83pTobiJaBxE5IWS8ny1gB8YXoQXQ0K1xubj6Um8x7EZeF7HGeG2X3iq3YcY5JDbbxmY6+3En5wMCQqSQNLL86zMUIezpqxY2TNf4ed5kvwk0UgwSNJlBHDZtC9xZMHpnfOJB2Nc79ozLv7pccubl6yCDIDaBaUeRJM0t6Au4i6/WQwED43CF015+bHn0d/iwmISwkvw+GsquRMTOMrs7R2YzCl8rI8AzRD7PXOtCPTcq9ISSQD6A1KJK+/Vadq5Q7Wy7Y9t2Q60DLp+LVSzdTrc5FUZzGBq4idtSgoGVVOgDqUD4KZjnkAu0Hi39bsmhkK0gLHBGaqPBD9Wvky247HZdn2TJAlHw5pa5M9st6uIk4oP1MKq6n2zWt3efXM3WnXKYhHqguIE+6t55Hlzdbd7cIgRBfR6s1no39Vyc2H7uT0BAvg+nwsGGl6kz0NtPJHvwROkAgeagds8+yvJnkhNY4z+ThpytkDflBRnC9SOL1AFjhrqyJIK2sTrUPBCjA3XkiHG0Gp5uymJaKgI1rVVzHIzQDOUbBEuB05Q+Izk1UkZF08HQA5oA7+CNSLAAVNuJFdUCV7ihDagEdee1BRrYXVs1t2KN7vdzsDSpqiwGJQOFCQUx7HRmlvf2MWafrb2rgf7beKanOoAZ3QPMTAmOviI0YTETLaDO3tXREYZbrvM9q48rl4aIm8Y7ffKuRrb7mqjzeoL/mtrX1CSgmDRPVAfH4ba3TYh25Q700VveMfiphrk/TX/8ePb0bKCPaU0Swzj4fWHKAVceRvTItuf1Idthn34pQKMLaEgvKs4WuSrKMXiXXSkFQ+BiRUDVXX/MMU0PRJFUT84iM4g8tiAUuQr58hEP1uKfOUpt3yIZ0trPSGt9cz15gtvPS28tVEW6qJQ503+Zoj06zRfDE1BIGldL6/Pgc7wDnzKl++amCYYfQtFJsvIlwv0nuU4ZgtIfHJWQf1FWofA2xW5FYSGueLShWYBciq7CCH9+XrtFsnqAG+CIwpqpu++vnCjnuOItWw2/gg86edETiK7ZeJjdSnBl0W+vzD8+ZXjz/Wz1WCQx6u/DwHB+tTSYIAr5sPqbfR4nRZ3nwK1retre1vX11N8HESe0aoeCTPkBAkj78PPIK6NIz7H+LF4OV0Dh7H+NuWcxT6Yp25cGLxnJD4ooWife3V1NaTBI+tUUL1lmc5x9NWIAGiXMw/m6MmscRVRupU1bRbyGICVveadsm8/j2R7b1JXMhyRbID5Zrp41UsXw+XqDngnE8ZR8gsM2ZCi3zqj6YKHlaepFF5uVXAIBUJ64tON7eO/kyEW8doJ/esf75xjcBWBRWkVyPKFFydNBSrC8uyZn6/z4WDfkKri1W1KmhIUksaQPROC3i3f8qpMLyNql1HnYNZy2j10LhjXdUmjRtwV4O3KKsZgUoplI3VOj7smog9DHthXi6Duz2p5d2FyeqhhbS7mcyBiEwHPN2yzW2ntdmzNKS+uZyyrUp7B/Hry9X5a6wfr0tzeWg2/QsNpdxc0MAS7TB1aOyc5zv0AoIVXaJkuISIMZk6HrhGE0spa41ne3Nzc9y3P23qX0uk34DezO/99DDotHnFejk6YFbMy89vbW9cJaGk6ChJaVyjWyhMo53ZMaU0CkXBxBEcoon/7+e9cyxE6ojOkUBI1Gj46tmxzVp+b6faFk2+PMMstS8yoEKznCKW/uhzp0vGLfj7uo2hWMjvgP0ZzocV45eCmp/4zv9+K31VaK/Vdu6lfnNGC33SRAV6pUuf75fGMe9G0PbPxH3b6+wyOHa491Z3sIvM9w1sBN4tC9yS1JSkLMpfQ55FO8h6uJxYwvIPMFtD6mpuD+OQrSMPl1UfmGAfYsO9k3UtOYRN0C2SszFAHy5NDP+BJJPwegvd+4rVTauudjd2vVZ0Zy0QV+ZY/DjQy3z3bOSfcLe3WjYRb50qCenZ0wOCMTLVEo9Z7uK4atb6DQ/CVrMY1Ob/eJGTfP0QbhrHPFwdgrAMzD8yk9i8GuWX6exFQfBWCVQxzxQ6v/dXwQJfZJS5jFQnAK9YqhRtwAO2FhbKXOhvXx7xuQrc6nRpSP8ueRggZ9aWw1p9wyg54gX4mZYLzXvtY8+N2w39G253DN6F/BXP8qduM97pcTpO+ueM/fmWRhZsqHTsjtA4muptSWULdk3evs+WKzVs0nqN4Lw7HfrWo2hTdtD9ArS/dnRUl29Nk+92/3/Iy5CdtJsv3lJ8zsl29fI8hCJ/O34fh16vl7eo2XN3c3i3Q+3BtP6/DrwN7wAS4OHsw7vr931adlIDZDltMg/VzgDkSXcyVnjZPo6q9v7+304GEPsmq1ckIYFxUj8NlXR851AM28p/kdTp91/wvUF7ocxp+dbFOCSIZOfBQKe75p7iCEqs23nJv5o24KLPP8jDjcMYi3d8tvLILx/V6PZbrj32BZPg+Jbcf74R3cCgE93ukV9NRQfH0Y+41zLtJJy9R8pgRERC0/R0mNauyxmEAkZAMT8s4JOfHtGKQYVIHblodcCLESPBQ9gj+7sxqAtFSfg2n10Gx99xroUmFHWBhDzoYBp/kaScS44agnRuqPNDbNPYisS4iOiVBEPZPaBxjnNyHc2HSTmHXvhz2xndGM2PZ9uakD93rkA723xh+4WN1WuzSl7uPiMdYo3eX1L7A6EpHHHTN4FOPTOOmqc06Z4HSvNhvrKC+BgeW2l0Y/vOBJBRDbKeqruu+KScHeeNYfFmuJAf2xF90O6IVYru2q7mUXVPvTabWdwvEksAAJ780FWyCnMAlckI/GZ99lXv5H30mhcrWOQAA',
  'fonts/inter-latin-ext-wght-normal.woff2': '...',
  'fonts/inter-latin-wght-normal.woff2': '...'
}
for n,b in V.items():
    os.makedirs(os.path.dirname('/content/app/static/vendor/'+n),exist_ok=True); open('/content/app/static/vendor/'+n,'wb').write(gzip.decompress(base64.b64decode(b)))
print({n:os.path.getsize('/content/app/static/vendor/'+n) for n in V})

In [ ]:
# Start the app (server + background worker). The models load on the GPU inside the server process. All times in the app are India Standard Time (IST).
import os, sys, subprocess, time, urllib.request, secrets
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()
PERSIST = True     # True = the database (users, tickets, approvals and all saved Copilot chats) is backed up to MyDrive/franchise_ops_data/db every few minutes and restored automatically after a runtime reset
USE_DRIVE = True   # mounts Google Drive (allow access when Colab asks) so the app loads the RAG chunks your RAG Builder notebook saved in MyDrive/FranchiseOps_AI/synthetic_kb. Set False to use only the built-in knowledge pack
os.environ["APP_KIND"] = "Franchise"
if PERSIST or USE_DRIVE:
    try: from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("Drive not mounted:", e)
if PERSIST and os.path.isdir("/content/drive/MyDrive"): os.environ["DRIVE_DB_DIR"] = "/content/drive/MyDrive/franchise_ops_data/db"
os.environ.setdefault("INGEST_KEY", secrets.token_urlsafe(12))   # X-API-Key for POST /ingest/<table>
try:
    from google.colab import userdata
    for k in ("HF_TOKEN","JWT_SECRET_KEY","NGROK_AUTHTOKEN","NGROK_TOKEN","LLM_MODEL","EMAIL_ID","EMAIL_PASSWORD","ALERT_EMAIL_TO","SLACK_WEBHOOK_URL","TEAMS_WEBHOOK_URL","TWILIO_SID","TWILIO_TOKEN","TWILIO_FROM","ALERT_SMS_TO"):
        try: os.environ.setdefault(k, userdata.get(k))
        except Exception: pass
except Exception: pass
if os.path.exists("/content/srv.pid"):
    try: os.kill(int(open("/content/srv.pid").read()), 9)
    except Exception: pass
srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"], cwd="/content/app", stdout=open("/content/server.log","w"), stderr=subprocess.STDOUT)
open("/content/srv.pid","w").write(str(srv.pid))
for _ in range(120):
    try: urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2); break
    except Exception: time.sleep(1)
else: print(open("/content/server.log").read()[-3000:]); raise SystemExit("server did not start")
print("Server is up (IST clock:", time.strftime("%d %b %Y %I:%M %p"), ") INGEST_KEY =", os.environ["INGEST_KEY"])


In [ ]:
# Open the app: a clickable link. Colab's own proxy first (Google-hosted, no browser warning); ngrok is the fallback if you saved NGROK_AUTHTOKEN in Colab Secrets.
import os
from IPython.display import HTML, display
link = None
try:
    from google.colab import output
    link = output.eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})")
except Exception: pass
tok = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_TOKEN")
if tok:
    try:
        get_ipython().system("pip -q install pyngrok")
        from pyngrok import ngrok; ngrok.set_auth_token(tok); ng = ngrok.connect(8000).public_url
        display(HTML(f'<p style="font:16px sans-serif">Fallback (ngrok): <a href="{ng}" target="_blank">{ng}</a></p>'))
        link = link or ng
    except Exception as e: print("ngrok failed:", e)
if link: display(HTML(f'<p style="font:18px sans-serif"><b>Open the app:</b> <a href="{link}" target="_blank" style="font-size:20px">{link}</a></p>'))
else: print("Not on Colab: open http://localhost:8000")


In [ ]:
# Live GPU / model status (same data as Admin > GPU & models in the app)
import json, urllib.request
print(json.dumps(json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health")), indent=1))
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv